# BHA official-source feasibility study

## Purpose

Notebook 26 — the Great Britain race-population completeness audit — is paused after a successful 34/34 BHA-to-Database-v4 pilot reconciliation.

During that work, the current BHA results service exposed substantially more structured official racing data than had previously been assumed. Before using it for a larger completeness audit, this notebook asks a narrower source-feasibility question:

> **Can the current BHA structured service provide sufficiently complete, historically deep and internally coherent official racing data to serve Inside Rails?**

This is a source-capability investigation, not a Database v5 design exercise.

## Current question — what result detail does the service actually provide?

Historical depth is useful only if the underlying race and runner data are useful.

The first substantive step is therefore to take one known modern completed British race and inspect the current BHA resources at their apparent grains:

1. the individual race resource;
2. the corresponding full race-results resource.

We will inventory the returned structures rather than assume meanings from field names.

In particular, we want to establish whether the results resource represents the complete race result and what evidence it exposes for concepts such as runners, horse identity, finishing outcome, jockey, trainer, starting price, draw, weight, margins, non-runners and result amendments.

Only after the useful result capability has been established will we test whether materially equivalent evidence is available at the **2015 beginning of Inside Rails Source Version 1**.

## Established context

The BHA public results frontend currently uses a structured service rooted at:

`https://api09.horseracing.software/bha/v1`

Observed routes relevant to this investigation include:

- `/fixtures/{fixtureYear}/{fixtureId}/races`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/results`

Previous bounded exploration also established that BHA identifiers must be treated as external provenance rather than assumed global identities:

- `fixtureId` can be reused across years;
- `raceId` can be reused across years;
- divided races can share a `raceId`;
- the observed race-resource reference therefore includes `yearOfRace + raceId + divisionSequence`.

These observations do **not** create Inside Rails fixture or race identity rules.

## Source and security boundary

The service is demonstrably used by the current public BHA frontend, but it has not been established as a formally documented public API. Its behaviour must therefore be treated as observed current behaviour rather than a permanent contractual interface.

Where the public frontend's current Authorization value is required:

- obtain it from the current frontend application in memory only;
- never print it;
- never persist it;
- never cache credential-bearing application JavaScript;
- retain only a SHA-256 fingerprint of the application asset for provenance.

Every BHA data response used by the investigation must be bounded, cached and preserved with request provenance.

## Explicit non-goals

This notebook will not:

- locate the exact historical backend transition around 1999;
- design or build Database v5;
- create new Inside Rails fixture or race identities;
- bulk-download historical results before source suitability is established;
- infer field meaning solely from names;
- treat undocumented service behaviour as guaranteed;
- persist the BHA Authorization value.

The immediate decision is simpler:

> **First establish what a complete modern BHA race result contains. Then test whether that useful capability reaches 2015.**

In [1]:
# Modern BHA result-capability probe
#
# PURPOSE
# -------
# Fetch one already-known completed British race from the current BHA structured
# service at both observed result-related grains:
#
#   1. race detail;
#   2. full race results.
#
# We are inspecting the returned evidence before assigning meanings to fields.
#
# EXTERNAL-WRITE BOUNDARY
# -----------------------
# The two BHA JSON responses are cached under data/cache/ so a notebook rerun
# does not need to repeat the external requests.
#
# The public BHA app.js is deliberately NOT cached because it currently contains
# the Authorization value used by the frontend. Only its SHA-256 fingerprint is
# retained in the API-cache provenance.
#
# The Authorization value is held in memory only and is never printed or saved.

from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# Use the explicit repository root required by the notebook working rules.
PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "modern_capability_probe"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


# This race was already observed successfully in Notebook 26.
# These are BHA external-reference fields only; they are not an Inside Rails ID.
SAMPLE_RACE = {
    "yearOfRace": "2026",
    "raceId": 2959,
    "divisionSequence": 0,
    "raceDate": "2026-05-27",
    "raceTime": "14:05:00",
    "courseName": "Hamilton Park",
}

BHA_BASE = "https://api09.horseracing.software/bha/v1"
race_path = (
    f"{SAMPLE_RACE['yearOfRace']}/"
    f"{SAMPLE_RACE['raceId']}/"
    f"{SAMPLE_RACE['divisionSequence']}"
)

ENDPOINTS = {
    "race": f"{BHA_BASE}/races/{race_path}",
    "results": f"{BHA_BASE}/races/{race_path}/results",
}

APP_JS_URL = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js?ver=1.19"
)


def load_cached_payload(cache_path, expected_url):
    """Return a previously successful cached JSON payload, if present."""
    if not cache_path.exists():
        return None

    envelope = json.loads(cache_path.read_text(encoding="utf-8"))

    # Fail closed if a file has somehow been reused for a different request.
    assert envelope["request_url"] == expected_url
    assert envelope["response_status"] == 200
    assert "parsed_json" in envelope

    return envelope["parsed_json"]


# Determine whether either API response actually needs a new request.
missing_cache = [
    name
    for name, url in ENDPOINTS.items()
    if load_cached_payload(CACHE_DIR / f"{name}.json", url) is None
]


authorization_value = None
app_js_sha256 = None


if missing_cache:
    # Fetch the current public frontend configuration into memory only.
    app_request = Request(
        APP_JS_URL,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/javascript,*/*;q=0.8",
        },
    )

    with urlopen(app_request, timeout=30) as response:
        app_js_bytes = response.read()

    app_js_sha256 = sha256(app_js_bytes).hexdigest()
    app_js_text = app_js_bytes.decode("utf-8")

    # Match only the active Angular Authorization assignment.
    # Commented-out lines are deliberately ignored because Notebook 26 showed
    # that an obsolete Bearer value can remain in JavaScript comments.
    auth_pattern = re.compile(
        r"""
        \$httpProvider
        \.defaults
        \.headers
        \.common
        \[['"]Authorization['"]\]
        \s*=\s*
        ['"]
        (Bearer\s+[^'"]+)
        ['"]
        \s*;
        """,
        re.VERBOSE,
    )

    active_matches = []

    for line_number, line in enumerate(app_js_text.splitlines(), start=1):
        if line.lstrip().startswith("//"):
            continue

        match = auth_pattern.search(line)

        if match:
            active_matches.append(
                {
                    "line_number": line_number,
                    "value": match.group(1),
                }
            )

    # If BHA changes the frontend, stop rather than guessing which value to use.
    assert len(active_matches) == 1, (
        "Expected exactly one active BHA Authorization assignment; "
        f"found {len(active_matches)}."
    )

    authorization_value = active_matches[0]["value"]
    assert authorization_value.startswith("Bearer ")


def fetch_bha_json(name, url):
    """Reuse a cached response or fetch and cache one bounded BHA JSON request."""
    cache_path = CACHE_DIR / f"{name}.json"

    cached = load_cached_payload(cache_path, url)

    if cached is not None:
        return cached, "cache"

    request = Request(
        url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/racing/results/",
            "User-Agent": "Mozilla/5.0",
        },
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()

    try:
        with urlopen(request, timeout=30) as response:
            status = response.status
            content_type = response.headers.get("Content-Type")
            response_text = response.read().decode("utf-8")

    except (HTTPError, URLError) as error:
        # Preserve failure provenance without ever storing the credential.
        failure_envelope = {
            "provider": "British Horseracing Authority",
            "request_url": url,
            "request_parameters": {},
            "retrieved_at_utc": retrieved_at,
            "response_status": getattr(error, "code", None),
            "error_details": repr(error),
            "app_js_sha256": app_js_sha256,
        }

        temp_path = cache_path.with_suffix(".tmp")
        temp_path.write_text(
            json.dumps(failure_envelope, indent=2),
            encoding="utf-8",
        )
        temp_path.replace(cache_path)
        raise

    parsed_json = json.loads(response_text)

    envelope = {
        "provider": "British Horseracing Authority",
        "candidate_identity": SAMPLE_RACE,
        "request_url": url,
        "request_parameters": {},
        "retrieved_at_utc": retrieved_at,
        "response_status": status,
        "content_type": content_type,
        "app_js_url": APP_JS_URL,
        "app_js_sha256": app_js_sha256,
        "raw_response": response_text,
        "parsed_json": parsed_json,
        "error_details": None,
    }

    # Atomic replacement avoids leaving a half-written cache after interruption.
    temp_path = cache_path.with_suffix(".tmp")
    temp_path.write_text(json.dumps(envelope, indent=2), encoding="utf-8")
    temp_path.replace(cache_path)

    return parsed_json, "network"


def inspect_json(label, payload):
    """Print a bounded structural inventory without assigning field semantics."""
    print(f"\n{label}")
    print("=" * len(label))
    print("Top-level type:", type(payload).__name__)

    if isinstance(payload, dict):
        print("Top-level keys:", sorted(payload.keys()))

        for key, value in payload.items():
            if isinstance(value, list):
                print(f"{key}: list[{len(value)}]")

                if value and isinstance(value[0], dict):
                    print(f"  first-item keys: {sorted(value[0].keys())}")

            elif isinstance(value, dict):
                print(f"{key}: dict keys={sorted(value.keys())}")

            else:
                preview = repr(value)
                print(f"{key}: {type(value).__name__} = {preview[:180]}")

    elif isinstance(payload, list):
        print("Items:", len(payload))

        if payload and isinstance(payload[0], dict):
            print("First-item keys:", sorted(payload[0].keys()))

    # A bounded raw preview lets us see nesting and representative values
    # without flooding the notebook if the service returns a large structure.
    print("\nBounded JSON preview:")
    print(json.dumps(payload, indent=2, ensure_ascii=False)[:12_000])


payloads = {}

for name, url in ENDPOINTS.items():
    payloads[name], source = fetch_bha_json(name, url)
    print(f"{name}: loaded from {source}")

print("\nAuthorization value displayed: NO")
print("API cache directory:", CACHE_DIR)

inspect_json("RACE RESOURCE", payloads["race"])
inspect_json("RESULTS RESOURCE", payloads["results"])

race: loaded from network
results: loaded from network

Authorization value displayed: NO
API cache directory: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/modern_capability_probe

RACE RESOURCE
Top-level type: dict
Top-level keys: ['data', 'success', 'total']
success: bool = True
total: int = 1
data: list[1]
  first-item keys: ['abandonedReasonCode', 'ageLimit', 'animalType', 'atTheRaces', 'blackTypeRace', 'createdByHandicapper', 'currentStageCode', 'distanceChange', 'distanceChangeText', 'distanceFullText', 'distanceText', 'distanceUnits', 'distanceValue', 'divisionSequence', 'fixtureId', 'goingText', 'images', 'maxRunners', 'offTime', 'pastWinners', 'photoFinishHiResImage', 'photoFinishLowResImage', 'prizeAmount', 'prizeCurrency', 'raceCriteriaMinimumWeight', 'raceCriteriaRaceType', 'raceCriteriaWeightsRaised', 'raceDate', 'raceId', 'raceName', 'raceNumber', 'raceTime', 'racecardAvailable', 'racingUK', 'ratingBand', 'rawDistanceText', 'res

## Modern result capability — first observation

The first bounded modern probe establishes that the BHA service exposes substantially more than a race-level result flag.

For the sampled Hamilton Park race on 27 May 2026:

- the individual race resource returned one race record;
- the race record reported `runners = 6`;
- the `/results` resource returned six result records;
- all six returned records represented runners in this completed race.

### Race-level evidence observed

The race resource exposed, among other fields:

- BHA race reference fields;
- race date and advertised race time;
- race name;
- distance representations;
- going;
- prize amount;
- race type;
- runner count;
- winning time;
- result/racecard availability;
- abandonment reason code;
- photo-finish image references;
- a stewards-report locator.

This establishes useful race-level capability for the sample. It does not yet establish the semantics or historical completeness of every field.

### Runner/result-level evidence observed

The results resource exposed one record per runner in this six-runner sample, including:

- BHA horse identifier and horse name;
- finishing-position fields;
- runner/status fields;
- cloth number;
- drawn stall;
- trainer identifier and name;
- jockey identifier and name;
- owner identifier and name;
- age and sex;
- starting-price-like `bettingRatio`;
- beaten-distance text;
- individual finish-time fields;
- code-specific rating fields;
- headgear and colours/silks information.

The structure also contains fields relating to:

- non-finishers;
- DNF reasons;
- non-runners;
- withdrawal declarations.

Those fields were null in this all-finisher sample, so their practical behaviour remains untested.

### Important unresolved capability

No obvious carried-weight field was present in the `/results` records returned for this sample.

That is an observation about this representation, not yet a conclusion that BHA cannot provide carried weight elsewhere.

Likewise, this sample does not establish how the service represents:

- a horse that starts but does not finish;
- a declared non-runner;
- disqualification or amended-result cases.

### Decision

The service has passed the first capability threshold: it supplies meaningful official race and runner/result data at usable grains.

Before testing historical depth at 2015, perform one further bounded modern capability check aimed specifically at exceptional result states and the currently unobserved carried-weight concept.

Do not expand into bulk acquisition.

In [2]:
# Modern exceptional-result capability probe
#
# WHAT
# ----
# Inspect one known modern BHA National Hunt race that contains non-finishing
# outcomes and may also distinguish declarations from actual starters.
#
# For the same BHA race reference we request:
#
#   1. the race resource;
#   2. the results resource;
#   3. the entries resource.
#
# WHY
# ---
# The first modern sample contained six ordinary finishers. It therefore could
# not tell us how the structured service represents:
#
#   - pulled-up / fallen / unseated or other non-finishing outcomes;
#   - declared horses that did not start;
#   - carried weight, which was absent from the first /results representation.
#
# This is still a deliberately bounded capability test: one race only.
#
# READS
# -----
# Current public BHA app.js in memory only if a live request is required.
# Existing ignored research-cache files are reused when present.
#
# WRITES
# ------
# Raw BHA API responses and provenance envelopes only under:
#
#   data/cache/bha_official_source_feasibility/modern_exception_probe/
#
# The Authorization value is NEVER printed or persisted.
#
# EXPECTED RESULT
# ---------------
# A structural inventory showing how this one exceptional race is represented.
# Field names are observations only; we do not yet assign governed semantics.

EXCEPTION_RACE = {
    "yearOfRace": "2026",
    "raceId": 23016,
    "divisionSequence": 0,
    "raceDate": "2026-05-27",
    "raceTime": "14:53:00",
    "courseName": "Newton Abbot",
}

exception_cache_dir = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "modern_exception_probe"
)
exception_cache_dir.mkdir(parents=True, exist_ok=True)

exception_race_path = (
    f"{EXCEPTION_RACE['yearOfRace']}/"
    f"{EXCEPTION_RACE['raceId']}/"
    f"{EXCEPTION_RACE['divisionSequence']}"
)

exception_endpoints = {
    "race": f"{BHA_BASE}/races/{exception_race_path}",
    "results": f"{BHA_BASE}/races/{exception_race_path}/results",
    "entries": f"{BHA_BASE}/races/{exception_race_path}/entries",
}


def load_exception_cache(cache_path, expected_url):
    """Reuse only a successful cache that belongs to this exact request."""
    if not cache_path.exists():
        return None

    envelope = json.loads(cache_path.read_text(encoding="utf-8"))

    assert envelope["request_url"] == expected_url
    assert envelope["response_status"] == 200
    assert "parsed_json" in envelope

    return envelope["parsed_json"]


# Work out whether this cell needs live BHA access at all.
uncached_exception_requests = [
    name
    for name, url in exception_endpoints.items()
    if load_exception_cache(
        exception_cache_dir / f"{name}.json",
        url,
    )
    is None
]


# The earlier cell may already hold the current frontend Authorization value in
# memory. On a later fresh-kernel rerun, however, its two responses may come
# entirely from cache, so the credential may intentionally not have been loaded.
#
# If this cell needs a live request and no credential is currently in memory,
# recover the single ACTIVE frontend assignment again without displaying it.
if uncached_exception_requests and not authorization_value:
    app_request = Request(
        APP_JS_URL,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/javascript,*/*;q=0.8",
        },
    )

    with urlopen(app_request, timeout=30) as response:
        current_app_js_bytes = response.read()

    current_app_js_sha256 = sha256(current_app_js_bytes).hexdigest()
    current_app_js_text = current_app_js_bytes.decode("utf-8")

    authorization_assignment_pattern = re.compile(
        r"""
        \$httpProvider
        \.defaults
        \.headers
        \.common
        \[['"]Authorization['"]\]
        \s*=\s*
        ['"]
        (Bearer\s+[^'"]+)
        ['"]
        \s*;
        """,
        re.VERBOSE,
    )

    current_active_matches = []

    for line_number, line in enumerate(
        current_app_js_text.splitlines(),
        start=1,
    ):
        # Ignore the obsolete commented-out assignment observed previously.
        if line.lstrip().startswith("//"):
            continue

        match = authorization_assignment_pattern.search(line)

        if match:
            current_active_matches.append(
                {
                    "line_number": line_number,
                    "value": match.group(1),
                }
            )

    # Fail closed if BHA has changed its frontend configuration.
    assert len(current_active_matches) == 1, (
        "Expected exactly one active BHA Authorization assignment; "
        f"found {len(current_active_matches)}."
    )

    authorization_value = current_active_matches[0]["value"]
    app_js_sha256 = current_app_js_sha256

    assert authorization_value.startswith("Bearer ")


def fetch_exception_payload(name, url):
    """Reuse or acquire one exact BHA response without persisting credentials."""
    cache_path = exception_cache_dir / f"{name}.json"

    cached = load_exception_cache(cache_path, url)

    if cached is not None:
        return cached, "cache"

    request = Request(
        url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/racing/results/",
            "User-Agent": "Mozilla/5.0",
        },
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()

    try:
        with urlopen(request, timeout=30) as response:
            status = response.status
            content_type = response.headers.get("Content-Type")
            raw_response = response.read().decode("utf-8")

    except (HTTPError, URLError) as error:
        # Preserve the failed request itself for auditability, but no credential.
        failure_envelope = {
            "provider": "British Horseracing Authority",
            "candidate_identity": EXCEPTION_RACE,
            "request_url": url,
            "request_parameters": {},
            "retrieved_at_utc": retrieved_at,
            "response_status": getattr(error, "code", None),
            "error_details": repr(error),
            "app_js_sha256": app_js_sha256,
        }

        temp_path = cache_path.with_suffix(".tmp")
        temp_path.write_text(
            json.dumps(failure_envelope, indent=2),
            encoding="utf-8",
        )
        temp_path.replace(cache_path)
        raise

    parsed_json = json.loads(raw_response)

    envelope = {
        "provider": "British Horseracing Authority",
        "candidate_identity": EXCEPTION_RACE,
        "request_url": url,
        "request_parameters": {},
        "retrieved_at_utc": retrieved_at,
        "response_status": status,
        "content_type": content_type,
        "app_js_url": APP_JS_URL,
        "app_js_sha256": app_js_sha256,
        "raw_response": raw_response,
        "parsed_json": parsed_json,
        "error_details": None,
    }

    # Write atomically so an interrupted notebook run cannot leave a partial
    # response masquerading as a valid cache.
    temp_path = cache_path.with_suffix(".tmp")
    temp_path.write_text(
        json.dumps(envelope, indent=2),
        encoding="utf-8",
    )
    temp_path.replace(cache_path)

    return parsed_json, "network"


exception_payloads = {}

for name, url in exception_endpoints.items():
    exception_payloads[name], source = fetch_exception_payload(name, url)
    print(f"{name}: loaded from {source}")


# ---------------------------------------------------------------------------
# 1. Race-level observation
# ---------------------------------------------------------------------------
race_rows = exception_payloads["race"].get("data", [])

assert len(race_rows) == 1, (
    f"Expected one race record, found {len(race_rows)}."
)

race_record = race_rows[0]

print("\nRACE RESOURCE")
print("=============")
print("Keys:", sorted(race_record.keys()))

for key in [
    "raceId",
    "yearOfRace",
    "divisionSequence",
    "raceDate",
    "raceTime",
    "raceName",
    "runners",
    "maxRunners",
    "resultsAvailable",
    "abandonedReasonCode",
    "winTime",
]:
    print(f"{key}: {race_record.get(key)!r}")


# ---------------------------------------------------------------------------
# 2. Result-state observation
# ---------------------------------------------------------------------------
result_rows = exception_payloads["results"].get("data", [])

print("\nRESULTS RESOURCE")
print("================")
print("Returned rows:", len(result_rows))

if result_rows:
    print("Union of result-row keys:")
    print(
        sorted(
            {
                key
                for row in result_rows
                if isinstance(row, dict)
                for key in row.keys()
            }
        )
    )

    print("\nOne line per returned result record:")

    for row in result_rows:
        print(
            {
                "racehorseName": row.get("racehorseName"),
                "status": row.get("status"),
                "runner": row.get("runner"),
                "resultFinishPos": row.get("resultFinishPos"),
                "finalPosition": row.get("finalPosition"),
                "nonFinishingCode": row.get("nonFinishingCode"),
                "DNFReason": row.get("DNFReason"),
                "nonRunnerDeclaredDate": row.get("nonRunnerDeclaredDate"),
                "nonRunnerDeclaredTime": row.get("nonRunnerDeclaredTime"),
                "nonRunnerDeclaredReason": row.get("nonRunnerDeclaredReason"),
                "withdrawnTimestamp": row.get("withdrawnTimestamp"),
                "bettingRatio": row.get("bettingRatio"),
                "resultBtnDistance": row.get("resultBtnDistance"),
                "finishTime": row.get("finishTime"),
            }
        )


# ---------------------------------------------------------------------------
# 3. Entry-resource observation, especially carried-weight-like fields
# ---------------------------------------------------------------------------
entry_payload = exception_payloads["entries"]
entry_rows = entry_payload.get("data", [])

print("\nENTRIES RESOURCE")
print("================")
print("Top-level keys:", sorted(entry_payload.keys()))
print("Returned rows:", len(entry_rows))

if entry_rows:
    entry_key_union = sorted(
        {
            key
            for row in entry_rows
            if isinstance(row, dict)
            for key in row.keys()
        }
    )

    print("Union of entry-row keys:")
    print(entry_key_union)

    # Search only by field name at this stage. A name containing "weight" or
    # "allowance" is a candidate clue, not proof of carried-weight semantics.
    weight_candidate_keys = [
        key
        for key in entry_key_union
        if "weight" in key.lower() or "allow" in key.lower()
    ]

    print("\nWeight/allowance candidate field names:")
    print(weight_candidate_keys)

    if weight_candidate_keys:
        print("\nObserved values for those candidate fields:")

        for row in entry_rows:
            print(
                {
                    "horse": (
                        row.get("racehorseName")
                        or row.get("animalName")
                        or row.get("horseName")
                    ),
                    **{
                        key: row.get(key)
                        for key in weight_candidate_keys
                    },
                }
            )


print("\nAuthorization value displayed: NO")
print("API cache directory:", exception_cache_dir)

race: loaded from network
results: loaded from network
entries: loaded from network

RACE RESOURCE
Keys: ['abandonedReasonCode', 'ageLimit', 'animalType', 'atTheRaces', 'blackTypeRace', 'createdByHandicapper', 'currentStageCode', 'distanceChange', 'distanceChangeText', 'distanceFullText', 'distanceText', 'distanceUnits', 'distanceValue', 'divisionSequence', 'fixtureId', 'goingText', 'images', 'maxRunners', 'offTime', 'pastWinners', 'photoFinishHiResImage', 'photoFinishLowResImage', 'prizeAmount', 'prizeCurrency', 'raceCriteriaMinimumWeight', 'raceCriteriaRaceType', 'raceCriteriaWeightsRaised', 'raceDate', 'raceId', 'raceName', 'raceNumber', 'raceTime', 'racecardAvailable', 'racingUK', 'ratingBand', 'rawDistanceText', 'resultsAvailable', 'riderType', 'runners', 'sexLimit', 'stewardsReport', 'transparentWindowStatus', 'tvCoverage', 'winTime', 'yearOfRace']
raceId: 23016
yearOfRace: '2026'
divisionSequence: 0
raceDate: '2026-05-27'
raceTime: '14:53:00'
raceName: "THE STOCK EXE BUILDING SU

## Modern exceptional-result capability — observation

The second bounded modern probe tested a completed National Hunt race containing ordinary finishers and three different non-finishing outcomes.

For the Newton Abbot 14:53 race on 27 May 2026:

- the race resource reported `runners = 10`;
- the `/results` resource returned 10 records;
- seven records had finishing positions;
- three records had no finishing position and carried explicit non-finishing evidence;
- the `/entries` resource also returned 10 records.

### Non-finishing outcomes

The results resource represented the three non-finishers as follows in this sample:

| Horse | `nonFinishingCode` | `DNFReason` | finishing position |
|---|---:|---|---|
| Amhranai | 1 | `Pulled Up` | null |
| Frenati | 2 | `Fell` | null |
| Princess of Ballea | 3 | `Unseated Rider` | null |

All three remained:

- `status = "Runner"`;
- `runner = 1`.

This demonstrates that the current BHA results resource can distinguish at least these non-finishing outcomes from ordinary finishing positions.

The observed numeric codes should not yet be treated as a complete or permanent codebook. Their meanings are established here only for the values observed in this race.

### Important finish-time warning

For each of the three non-finishers, `finishTime` was populated with:

`4m 8.51s`

That is also the winner's recorded winning time.

Therefore `finishTime` cannot safely be interpreted as the individual horse's actual finish time merely because the field is populated.

For finishers in this race the values progressed consistently with finishing order, but the non-finisher behaviour means field semantics must remain evidence-led.

### Weight-related evidence

The `/entries` resource exposed the following fields:

- `weightText`;
- `weightValue`;
- `weightsJockeyClaiming`.

Every returned entry in this sample had populated weight representations, for example:

- `11st 7lbs` / `11-7`;
- `11st 0lbs` / `11-0`;
- `10st 8lbs` / `10-8`.

`weightsJockeyClaiming` was `0` for every horse in this sample.

This establishes that the current structured service exposes useful runner weight-related information through the entries resource.

It does **not** yet establish that `weightText` or `weightValue` is semantically identical to final carried weight in every circumstance, particularly where jockey claims, late changes or other adjustments apply.

### Non-runner capability remains unobserved

Although the result and entry schemas contain explicit fields for non-runner declarations, every returned record in this race was a runner.

No declared non-runner has therefore yet been observed.

Also, `maxRunners = 14` is a maximum-runner/capacity field. It is **not evidence that 14 horses were declared**, so the difference between `maxRunners = 14` and `runners = 10` must not be interpreted as four non-runners.

### Modern capability decision

The current BHA structured service has now demonstrated:

- useful race-level detail;
- one row per actual runner in the tested results;
- BHA horse, jockey, trainer and owner identifiers;
- finishing positions;
- explicit non-finishing outcomes;
- starting-price-like information;
- beaten-distance information;
- draw where relevant;
- race and runner timing fields, with an identified semantic warning;
- runner weight-related fields through `/entries`;
- explicit schema provision for non-runner information.

Exact non-runner behaviour and exact carried-weight semantics remain unresolved, but neither currently blocks the central feasibility question.

## Decision

The modern capability threshold is satisfied.



## Phase 2 — Map the current BHA structured data surface

The feasibility question is now broader than historical race-population completeness.

The immediate objective is:

> **What useful official BHA information can Inside Rails acquire, understand and potentially govern?**

Historical depth will be tested later, and only for data products that prove analytically useful.

### Method

Before downloading more racing records, first inventory the structured resources referenced by the current BHA public frontend.

For each observed resource, establish:

1. its apparent grain;
2. the fields it exposes;
3. whether it contains information already present in Database v4;
4. whether it offers useful validation/provenance;
5. whether it contains genuinely new analytical information;
6. which semantics or edge cases would need investigation before governance.

Observed frontend routes already include resource families such as:

- fixtures;
- fixture races;
- fixture officials;
- fixture going;
- race details;
- nominations;
- entries;
- transferred/related entry information;
- balloted entries;
- results.

This list is an observation from current frontend code, not yet a claim that every route is useful, populated, historically available or suitable for Inside Rails.

### Boundary

Do not design Database v5 during source discovery.

The sequence is:

**discover → inspect → understand → classify usefulness → test historical availability → design governed integration only where justified.**

The next step is therefore to inventory the current frontend's structured BHA routes systematically before making additional race-data requests.

In [3]:
# Current BHA structured-route inventory
#
# WHAT
# ----
# Inspect the JavaScript used by the current public BHA Results page and extract
# active URL constructions belonging to frontend factories whose urlBase is
# explicitly rooted at:
#
#     /bha/v1/
#
# WHY
# ---
# We want to discover the current structured BHA data surface before deciding
# which resources are worth probing or eventually governing in Inside Rails.
#
# This cell discovers routes only. It does NOT call any BHA data endpoint.
#
# READS
# -----
# 1. Current public BHA Results page.
# 2. The current app.js referenced by that page.
#
# WRITES
# ------
# data/cache/bha_official_source_feasibility/route_inventory/
#
# The Results-page HTML may be cached because it contains no API credential.
#
# app.js is NEVER persisted because the live file currently contains an
# Authorization value. For app.js we retain only:
#
#   - URL;
#   - retrieval timestamp;
#   - HTTP status / Content-Type;
#   - SHA-256 fingerprint;
#   - derived route inventory.
#
# EXPECTED RESULT
# ---------------
# A compact list of current /bha/v1/ route templates with the frontend function
# that references each route. These are observed frontend routes, not yet claims
# about usefulness, semantics, completeness or historical availability.

from datetime import datetime, timezone
from hashlib import sha256
import html
import json
from pathlib import Path
import re
from urllib.parse import urljoin
from urllib.request import Request, urlopen


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

ROUTE_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "route_inventory"
)
ROUTE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PAGE_URL = "https://www.britishhorseracing.com/racing/results/"

RESULTS_HTML_CACHE = ROUTE_CACHE_DIR / "results_page.html"
RESULTS_PROVENANCE_CACHE = ROUTE_CACHE_DIR / "results_page_provenance.json"
ROUTE_INVENTORY_CACHE = ROUTE_CACHE_DIR / "bha_v1_route_inventory.json"


def fetch_public_text(url):
    """Fetch one bounded public frontend resource without supplying credentials."""
    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "text/html,application/javascript,*/*;q=0.8",
        },
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()

    with urlopen(request, timeout=30) as response:
        status = response.status
        content_type = response.headers.get("Content-Type")
        body = response.read()

    return {
        "url": url,
        "retrieved_at_utc": retrieved_at,
        "status": status,
        "content_type": content_type,
        "body": body,
    }


# ---------------------------------------------------------------------------
# 1. Obtain the current Results page.
# ---------------------------------------------------------------------------
#
# This page itself is safe to cache and lets us discover the actual app.js URL
# currently referenced by BHA instead of assuming a version query string.

if RESULTS_HTML_CACHE.exists() and RESULTS_PROVENANCE_CACHE.exists():
    results_html = RESULTS_HTML_CACHE.read_text(encoding="utf-8")

    results_page_provenance = json.loads(
        RESULTS_PROVENANCE_CACHE.read_text(encoding="utf-8")
    )

    results_page_source = "cache"

else:
    page_response = fetch_public_text(RESULTS_PAGE_URL)

    assert page_response["status"] == 200, (
        f"Unexpected Results-page status: {page_response['status']}"
    )

    results_html = page_response["body"].decode("utf-8")

    results_page_provenance = {
        "request_url": RESULTS_PAGE_URL,
        "retrieved_at_utc": page_response["retrieved_at_utc"],
        "response_status": page_response["status"],
        "content_type": page_response["content_type"],
        "sha256": sha256(page_response["body"]).hexdigest(),
    }

    # Atomic cache writes avoid treating an interrupted write as valid evidence.
    html_temp = RESULTS_HTML_CACHE.with_suffix(".tmp")
    html_temp.write_text(results_html, encoding="utf-8")
    html_temp.replace(RESULTS_HTML_CACHE)

    provenance_temp = RESULTS_PROVENANCE_CACHE.with_suffix(".tmp")
    provenance_temp.write_text(
        json.dumps(results_page_provenance, indent=2),
        encoding="utf-8",
    )
    provenance_temp.replace(RESULTS_PROVENANCE_CACHE)

    results_page_source = "network"


# ---------------------------------------------------------------------------
# 2. Discover the current app.js URL from the page itself.
# ---------------------------------------------------------------------------

script_sources = re.findall(
    r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
    results_html,
    flags=re.IGNORECASE,
)

script_urls = [
    urljoin(
        RESULTS_PAGE_URL,
        html.unescape(source),
    )
    for source in script_sources
]

app_js_candidates = [
    url
    for url in script_urls
    if re.search(r"/angular/app\.js(?:\?|$)", url)
]

assert len(app_js_candidates) == 1, (
    "Expected exactly one current angular/app.js reference on the BHA Results "
    f"page; found {len(app_js_candidates)}."
)

current_app_js_url = app_js_candidates[0]


# ---------------------------------------------------------------------------
# 3. Fetch app.js into memory only.
# ---------------------------------------------------------------------------
#
# SECURITY BOUNDARY:
# app.js may contain the frontend Authorization value.
#
# The raw JavaScript therefore MUST NOT be written to disk or displayed.

app_response = fetch_public_text(current_app_js_url)

assert app_response["status"] == 200, (
    f"Unexpected app.js status: {app_response['status']}"
)

app_js_bytes = app_response["body"]
app_js_text = app_js_bytes.decode("utf-8")
app_js_fingerprint = sha256(app_js_bytes).hexdigest()


# ---------------------------------------------------------------------------
# 4. Find frontend factory blocks whose urlBase is explicitly /bha/v1/.
# ---------------------------------------------------------------------------
#
# We preserve the frontend factory/function context because a route like
# "/entries" is more informative when we can see that the frontend calls it
# through something named getRaceEntries.
#
# No meaning is inferred from those function names yet.

factory_start_pattern = re.compile(
    r"""bhaCommon\.factory\(\s*['"]([^'"]+)['"]"""
)

factory_method_pattern = re.compile(
    r"""
    \b
    [A-Za-z_$][\w$]*
    \.
    ([A-Za-z_$][\w$]*)
    \s*=\s*function
    """,
    re.VERBOSE,
)

bha_base_pattern = re.compile(
    r"""
    \bvar\s+urlBase\s*=
    .*?
    ['"]/bha/v1/['"]
    """,
    re.VERBOSE,
)

route_line_pattern = re.compile(
    r"""
    \burl\s*:\s*
    urlBase
    (?P<tail>.*?)
    ,?\s*
    (?://.*)?
    $
    """,
    re.VERBOSE,
)

# Tokenise JavaScript string concatenation such as:
#
#   'races'+'/'+yearOfRace+'/'+raceId+'/'+divisionSequence+'/results'
#
# into literal path pieces and named placeholders.
route_token_pattern = re.compile(
    r"""
    (?:
        ['"](?P<literal>[^'"]*)['"]
    )
    |
    (?:
        \b(?P<identifier>[A-Za-z_$][\w$]*)\b
    )
    """,
    re.VERBOSE,
)


current_factory = None
current_method = None

# Track factories for which we have directly observed:
#
#     var urlBase = apiaddress + '/bha/v1/';
#
# rather than assuming every variable named urlBase points at the BHA service.
bha_v1_factories = set()

route_candidates = []


for line_number, raw_line in enumerate(app_js_text.splitlines(), start=1):
    stripped = raw_line.strip()

    # Ignore entire JavaScript // comment lines so historical/dead routes do not
    # enter the active route inventory.
    if stripped.startswith("//"):
        continue

    factory_match = factory_start_pattern.search(raw_line)

    if factory_match:
        current_factory = factory_match.group(1)
        current_method = None

    if current_factory and bha_base_pattern.search(raw_line):
        bha_v1_factories.add(current_factory)

    method_match = factory_method_pattern.search(raw_line)

    if method_match:
        current_method = method_match.group(1)

    route_match = route_line_pattern.search(raw_line)

    if not route_match:
        continue

    # Only retain a route after its current factory has explicitly established
    # /bha/v1/ as its urlBase.
    if current_factory not in bha_v1_factories:
        continue

    route_tail = route_match.group("tail")

    path_parts = []

    for token_match in route_token_pattern.finditer(route_tail):
        literal = token_match.group("literal")
        identifier = token_match.group("identifier")

        if literal is not None:
            path_parts.append(literal)

        elif identifier is not None:
            # JavaScript variables become placeholders in our observed template.
            path_parts.append("{" + identifier + "}")

    relative_path = "".join(path_parts)

    # Normalise only presentation slashes; this does not alter BHA identifiers.
    relative_path = re.sub(r"/+", "/", relative_path).strip("/")

    route_template = (
        "/bha/v1/"
        + relative_path
    )

    route_candidates.append(
        {
            "factory": current_factory,
            "frontend_function": current_method,
            "source_line": line_number,
            "route_template": route_template,
        }
    )


# ---------------------------------------------------------------------------
# 5. De-duplicate identical observations while preserving source provenance.
# ---------------------------------------------------------------------------

seen = set()
route_inventory = []

for row in route_candidates:
    identity = (
        row["factory"],
        row["frontend_function"],
        row["route_template"],
    )

    if identity in seen:
        continue

    seen.add(identity)
    route_inventory.append(row)

route_inventory.sort(
    key=lambda row: (
        row["route_template"],
        row["frontend_function"] or "",
    )
)


# ---------------------------------------------------------------------------
# 6. Persist ONLY credential-safe provenance and the derived route inventory.
# ---------------------------------------------------------------------------

inventory_envelope = {
    "provider": "British Horseracing Authority",
    "frontend_page_url": RESULTS_PAGE_URL,
    "frontend_page_source": results_page_source,
    "app_js_url": current_app_js_url,
    "app_js_retrieved_at_utc": app_response["retrieved_at_utc"],
    "app_js_response_status": app_response["status"],
    "app_js_content_type": app_response["content_type"],
    "app_js_sha256": app_js_fingerprint,
    "raw_app_js_persisted": False,
    "bha_v1_factories_observed": sorted(bha_v1_factories),
    "routes": route_inventory,
}

inventory_temp = ROUTE_INVENTORY_CACHE.with_suffix(".tmp")
inventory_temp.write_text(
    json.dumps(inventory_envelope, indent=2),
    encoding="utf-8",
)
inventory_temp.replace(ROUTE_INVENTORY_CACHE)


# ---------------------------------------------------------------------------
# 7. Print only the safe derived inventory.
# ---------------------------------------------------------------------------

print("BHA frontend route inventory")
print("============================")
print("Results page:", results_page_source)
print("app.js URL:", current_app_js_url)
print("app.js SHA-256:", app_js_fingerprint)
print("Raw app.js persisted: NO")
print("Observed /bha/v1/ factories:", sorted(bha_v1_factories))
print("Observed route references:", len(route_inventory))

for row in route_inventory:
    print(
        f"\n{row['route_template']}\n"
        f"  factory: {row['factory']}\n"
        f"  frontend function: {row['frontend_function']}\n"
        f"  app.js line: {row['source_line']}"
    )

print("\nDerived inventory cache:", ROUTE_INVENTORY_CACHE)

HTTPError: HTTP Error 403: Forbidden

In [4]:
# BHA structured-route inventory
#
# WHAT
# ----
# Download the current BHA frontend app.js and inventory the /bha/v1/ routes
# referenced by its fixtureFactory.
#
# WHY
# ---
# The BHA Results HTML page returns HTTP 403 to this notebook's urllib request.
# We do not actually need that HTML page: the current app.js resource is already
# known and is sufficient for mapping the structured BHA route surface.
#
# READS
# -----
# Current public BHA app.js.
#
# WRITES
# ------
# A credential-safe derived route inventory under:
#
#   data/cache/bha_official_source_feasibility/route_inventory/
#
# Raw app.js is NOT persisted because it contains the frontend Authorization
# value. Only its SHA-256 fingerprint and derived route observations are saved.
#
# EXPECTED RESULT
# ---------------
# A list of observed /bha/v1/ route templates. No API data routes are called.

from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import re
import subprocess


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "route_inventory"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

INVENTORY_FILE = CACHE_DIR / "bha_v1_route_inventory.json"

APP_JS_URL = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js?ver=1.19"
)


# Use curl because it has already successfully retrieved this exact resource on
# this machine, while BHA rejected urllib access to the surrounding HTML page.
curl_result = subprocess.run(
    [
        "curl",
        "-fsSL",
        "--max-time",
        "30",
        APP_JS_URL,
    ],
    check=True,
    capture_output=True,
)

app_js_bytes = curl_result.stdout

assert app_js_bytes, "BHA app.js response was empty."

app_js_text = app_js_bytes.decode("utf-8")

retrieved_at = datetime.now(timezone.utc).isoformat()
app_js_sha256 = sha256(app_js_bytes).hexdigest()


# Locate the fixtureFactory block. This is where the current frontend defines
# the fixture/race resources already used successfully in our earlier probes.
factory_start = app_js_text.find(
    "bhaCommon.factory('fixtureFactory'"
)

assert factory_start != -1, (
    "Could not find fixtureFactory in current BHA app.js."
)

# Stop at the next Angular factory declaration rather than searching unrelated
# frontend code and accidentally mixing resource families.
next_factory = app_js_text.find(
    "bhaCommon.factory(",
    factory_start + 1,
)

if next_factory == -1:
    fixture_factory_text = app_js_text[factory_start:]
else:
    fixture_factory_text = app_js_text[factory_start:next_factory]


# Confirm that this factory is currently based on /bha/v1/.
assert "/bha/v1/" in fixture_factory_text, (
    "fixtureFactory no longer visibly references /bha/v1/."
)


# Track the frontend method in which each url: expression occurs.
method_pattern = re.compile(
    r"""
    fixtureFactory\.
    (?P<method>[A-Za-z_$][\w$]*)
    \s*=\s*function
    """,
    re.VERBOSE,
)

url_pattern = re.compile(
    r"""
    \burl\s*:\s*
    urlBase
    (?P<expression>.*?)
    ,?\s*
    (?://.*)?
    $
    """,
    re.VERBOSE,
)

token_pattern = re.compile(
    r"""
    ['"](?P<literal>[^'"]*)['"]
    |
    \b(?P<identifier>[A-Za-z_$][\w$]*)\b
    """,
    re.VERBOSE,
)


current_method = None
route_rows = []

for line_number, line in enumerate(
    fixture_factory_text.splitlines(),
    start=1,
):
    stripped = line.strip()

    # Ignore entire // comments so obsolete routes do not enter the inventory.
    if stripped.startswith("//"):
        continue

    method_match = method_pattern.search(line)

    if method_match:
        current_method = method_match.group("method")

    url_match = url_pattern.search(line)

    if not url_match:
        continue

    expression = url_match.group("expression")

    path_parts = []

    for token in token_pattern.finditer(expression):
        literal = token.group("literal")
        identifier = token.group("identifier")

        if literal is not None:
            path_parts.append(literal)

        elif identifier is not None:
            # Variables become descriptive placeholders rather than values.
            path_parts.append("{" + identifier + "}")

    relative_path = "".join(path_parts)
    relative_path = re.sub(r"/+", "/", relative_path).strip("/")

    route_rows.append(
        {
            "frontend_function": current_method,
            "route_template": "/bha/v1/" + relative_path,
            "factory_relative_line": line_number,
        }
    )


# De-duplicate without inventing any route meaning.
seen = set()
route_inventory = []

for row in route_rows:
    identity = (
        row["frontend_function"],
        row["route_template"],
    )

    if identity in seen:
        continue

    seen.add(identity)
    route_inventory.append(row)

route_inventory.sort(
    key=lambda row: (
        row["route_template"],
        row["frontend_function"] or "",
    )
)


# Save only safe derived evidence. Raw app.js and its credential are excluded.
inventory_envelope = {
    "provider": "British Horseracing Authority",
    "app_js_url": APP_JS_URL,
    "retrieved_at_utc": retrieved_at,
    "app_js_sha256": app_js_sha256,
    "raw_app_js_persisted": False,
    "factory": "fixtureFactory",
    "routes": route_inventory,
}

temp_file = INVENTORY_FILE.with_suffix(".tmp")

temp_file.write_text(
    json.dumps(inventory_envelope, indent=2),
    encoding="utf-8",
)

temp_file.replace(INVENTORY_FILE)


print("BHA structured-route inventory")
print("==============================")
print("app.js SHA-256:", app_js_sha256)
print("Raw app.js persisted: NO")
print("Observed routes:", len(route_inventory))

for row in route_inventory:
    print(
        f"\n{row['route_template']}\n"
        f"  frontend function: {row['frontend_function']}"
    )

print("\nInventory cache:", INVENTORY_FILE)

BHA structured-route inventory
app.js SHA-256: 6582a0bf5e1c47374694fc32ebb0420219be1f4de2f148f8e76969101eba3cb9
Raw app.js persisted: NO
Observed routes: 11

/bha/v1/fixtures
  frontend function: getOtherFixtures

/bha/v1/fixtures/{fixtureYear}/{fixtureId}
  frontend function: getFixtureInformation

/bha/v1/fixtures/{fixtureYear}/{fixtureId}/going
  frontend function: getFixtureGoing

/bha/v1/fixtures/{fixtureYear}/{fixtureId}/officials
  frontend function: getFixtureOfficials

/bha/v1/fixtures/{fixtureYear}/{fixtureId}/races
  frontend function: getFixtureRaces

/bha/v1/races/{yearOfRace}/{raceId}/{divisionSequence}
  frontend function: getRaceDetails

/bha/v1/races/{yearOfRace}/{raceId}/{divisionSequence}/balloted
  frontend function: getRaceBalloted

/bha/v1/races/{yearOfRace}/{raceId}/{divisionSequence}/entries
  frontend function: getRaceEntries

/bha/v1/races/{yearOfRace}/{raceId}/{divisionSequence}/nominations
  frontend function: getRaceNominations

/bha/v1/races/{yearOfRace}/{

## Current BHA structured data surface — route inventory

Inspection of the current BHA frontend identified 11 active `/bha/v1/` route patterns.

### Fixture-level resources

- `/fixtures`
- `/fixtures/{fixtureYear}/{fixtureId}`
- `/fixtures/{fixtureYear}/{fixtureId}/going`
- `/fixtures/{fixtureYear}/{fixtureId}/officials`
- `/fixtures/{fixtureYear}/{fixtureId}/races`

### Race-level and race-lifecycle resources

- `/races/{yearOfRace}/{raceId}/{divisionSequence}`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/nominations`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/entries`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/trans`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/balloted`
- `/races/{yearOfRace}/{raceId}/{divisionSequence}/results`

### Resources already materially inspected

Previous bounded work has already shown useful evidence from:

- fixture-list records;
- fixture race collections;
- individual race detail;
- race entries;
- race results.

Those observations already expose substantial fixture, race and runner/result information.

### Unexplored structured resources

The remaining frontend resources whose actual returned structures have not yet been inventoried are:

1. fixture detail;
2. fixture going;
3. fixture officials;
4. race nominations;
5. transferred/related race-entry information (`trans`);
6. balloted entries.

Their frontend function names provide useful discovery clues but are not sufficient evidence of their actual data semantics.

## Decision

Use one already-known modern fixture and race to inspect these six remaining resources.

The purpose is source-surface discovery only:

> **What additional structured information does each remaining BHA resource actually expose?**

For each response preserve:

- HTTP/result status;
- top-level structure;
- row count;
- complete field-name inventory;
- a bounded representative payload.

Do not yet decide whether any field belongs in Database v5.

In [6]:
# Remaining BHA structured-resource probe
#
# WHAT
# ----
# Inspect the six /bha/v1/ resource types discovered in the frontend route
# inventory but not yet materially examined:
#
# Fixture level:
#   1. fixture detail
#   2. fixture going
#   3. fixture officials
#
# Race lifecycle:
#   4. nominations
#   5. transferred/related entries ("trans")
#   6. balloted entries
#
# WHY
# ---
# We are mapping what useful official BHA information is currently available
# before deciding what deserves deeper semantic work or future database
# governance.
#
# READS
# -----
# - Local BHA Authorization value from .env.local
# - Six bounded BHA API routes for one already-known modern fixture/race
#
# WRITES
# ------
# Raw response evidence and safe request provenance under:
#
#   data/cache/bha_official_source_feasibility/
#       remaining_resource_probe/
#
# The Authorization credential is NEVER written to these cache files or printed.
#
# EXPECTED RESULT
# ---------------
# For each route:
#
# - HTTP status;
# - JSON/non-JSON status;
# - top-level type and keys;
# - returned row count where applicable;
# - union of observed row fields;
# - bounded representative payload.
#
# Empty or unsuccessful resources are evidence too and must be preserved rather
# than treated as failures of the notebook.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

SECRETS_FILE = PROJECT_ROOT / ".env.local"

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "remaining_resource_probe"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# 1. Load the locally stored BHA credential.
# ---------------------------------------------------------------------------
#
# Do not use python-dotenv here: this tiny read keeps the notebook dependency-free
# and makes the secret-handling behaviour explicit.

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)

bha_authorization = None

for line in SECRETS_FILE.read_text(encoding="utf-8").splitlines():
    if line.startswith("BHA_AUTHORIZATION="):
        bha_authorization = line.split("=", 1)[1]
        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith("Bearer "), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 2. Use one already-known fixture and race.
# ---------------------------------------------------------------------------
#
# These are BHA external references only.
# They do NOT create Inside Rails fixture or race identities.

SAMPLE_FIXTURE = {
    "fixtureYear": "2026",
    "fixtureId": 10399,
    "courseName": "Newton Abbot",
    "fixtureDate": "2026-05-27",
}

SAMPLE_RACE = {
    "yearOfRace": "2026",
    "raceId": 23016,
    "divisionSequence": 0,
    "raceTime": "14:53:00",
}

BHA_BASE = "https://api09.horseracing.software/bha/v1"

fixture_path = (
    f"{SAMPLE_FIXTURE['fixtureYear']}/"
    f"{SAMPLE_FIXTURE['fixtureId']}"
)

race_path = (
    f"{SAMPLE_RACE['yearOfRace']}/"
    f"{SAMPLE_RACE['raceId']}/"
    f"{SAMPLE_RACE['divisionSequence']}"
)

RESOURCE_URLS = {
    "fixture_detail": f"{BHA_BASE}/fixtures/{fixture_path}",
    "fixture_going": f"{BHA_BASE}/fixtures/{fixture_path}/going",
    "fixture_officials": f"{BHA_BASE}/fixtures/{fixture_path}/officials",
    "race_nominations": f"{BHA_BASE}/races/{race_path}/nominations",
    "race_trans": f"{BHA_BASE}/races/{race_path}/trans",
    "race_balloted": f"{BHA_BASE}/races/{race_path}/balloted",
}


# ---------------------------------------------------------------------------
# 3. Read an existing cache if we have already made this exact request.
# ---------------------------------------------------------------------------

def load_cached_response(cache_path, expected_url):
    if not cache_path.exists():
        return None

    envelope = json.loads(
        cache_path.read_text(encoding="utf-8")
    )

    assert envelope["request_url"] == expected_url

    return envelope


# ---------------------------------------------------------------------------
# 4. Make one credential-safe BHA request.
# ---------------------------------------------------------------------------
#
# HTTP errors such as 404 are preserved as observations rather than raised out
# of the cell. That distinction matters when mapping an undocumented service.

def fetch_resource(name, url):
    cache_path = CACHE_DIR / f"{name}.json"

    cached = load_cached_response(
        cache_path,
        url,
    )

    if cached is not None:
        return cached, "cache"

    request = Request(
        url,
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/racing/results/",
            "User-Agent": "Mozilla/5.0",
        },
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(request, timeout=30) as response:
            status = response.status
            content_type = response.headers.get("Content-Type")
            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code
        content_type = error.headers.get("Content-Type")

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        transport_error = repr(error)

    except URLError as error:
        transport_error = repr(error)


    parsed_json = None
    json_error = None

    if response_text:
        try:
            parsed_json = json.loads(response_text)

        except json.JSONDecodeError as error:
            json_error = repr(error)


    envelope = {
        "provider": "British Horseracing Authority",
        "resource_name": name,
        "sample_fixture": SAMPLE_FIXTURE,
        "sample_race": SAMPLE_RACE,
        "request_url": url,
        "retrieved_at_utc": retrieved_at,
        "response_status": status,
        "content_type": content_type,
        "raw_response": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    # Atomic replacement protects the evidence cache from interrupted writes.
    temp_path = cache_path.with_suffix(".tmp")

    temp_path.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_path.replace(cache_path)

    return envelope, "network"


# ---------------------------------------------------------------------------
# 5. Structural inspection helper.
# ---------------------------------------------------------------------------
#
# This deliberately describes structure rather than assigning meanings.

def inspect_resource(name, envelope, source):
    print("\n" + "=" * 78)
    print(name)
    print("=" * 78)

    print("Loaded from:", source)
    print("HTTP status:", envelope["response_status"])
    print("Content-Type:", envelope["content_type"])

    payload = envelope["parsed_json"]

    if payload is None:
        print("Parsed JSON: NO")

        raw_preview = envelope["raw_response"][:3000]

        if raw_preview:
            print("\nBounded raw-response preview:")
            print(raw_preview)

        if envelope["transport_error"]:
            print("\nTransport observation:")
            print(envelope["transport_error"])

        return

    print("Parsed JSON: YES")
    print("Top-level type:", type(payload).__name__)


    if isinstance(payload, dict):
        print("Top-level keys:", sorted(payload.keys()))

        # Most observed BHA endpoints use a top-level `data` member, but we do
        # not assume that every resource follows the same structure.
        data = payload.get("data")

        if isinstance(data, list):
            print("data rows:", len(data))

            if data:
                dictionary_rows = [
                    row
                    for row in data
                    if isinstance(row, dict)
                ]

                if dictionary_rows:
                    field_union = sorted(
                        {
                            key
                            for row in dictionary_rows
                            for key in row.keys()
                        }
                    )

                    print("Union of row fields:")
                    print(field_union)

                    print("\nFirst returned row:")
                    print(
                        json.dumps(
                            dictionary_rows[0],
                            indent=2,
                            ensure_ascii=False,
                        )[:8000]
                    )

                else:
                    print("\nFirst returned data item:")
                    print(
                        repr(data[0])[:8000]
                    )

            else:
                print("Resource returned an empty data list.")

        elif isinstance(data, dict):
            print("data type: dict")
            print("data keys:", sorted(data.keys()))

            print("\nBounded data preview:")
            print(
                json.dumps(
                    data,
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )

        else:
            print("data member type:", type(data).__name__)


        # Preserve useful scalar metadata such as success/total without assuming
        # identical semantics across endpoints.
        scalar_metadata = {
            key: value
            for key, value in payload.items()
            if not isinstance(value, (dict, list))
        }

        if scalar_metadata:
            print("\nTop-level scalar metadata:")
            print(scalar_metadata)


    elif isinstance(payload, list):
        print("Top-level rows:", len(payload))

        if payload and isinstance(payload[0], dict):
            print(
                "Union of row fields:",
                sorted(
                    {
                        key
                        for row in payload
                        if isinstance(row, dict)
                        for key in row.keys()
                    }
                ),
            )

            print("\nFirst returned row:")
            print(
                json.dumps(
                    payload[0],
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )

    else:
        print("\nBounded payload:")
        print(repr(payload)[:8000])


# ---------------------------------------------------------------------------
# 6. Acquire and inspect exactly the six remaining resources.
# ---------------------------------------------------------------------------

resource_observations = {}

for resource_name, resource_url in RESOURCE_URLS.items():
    envelope, source = fetch_resource(
        resource_name,
        resource_url,
    )

    resource_observations[resource_name] = envelope

    inspect_resource(
        resource_name,
        envelope,
        source,
    )


print("\n" + "=" * 78)
print("PROBE COMPLETE")
print("=" * 78)
print("Resources tested:", len(resource_observations))
print("Authorization value displayed: NO")
print("Authorization written to evidence cache: NO")
print("Evidence cache:", CACHE_DIR)


fixture_detail
Loaded from: network
HTTP status: 200
Content-Type: application/json
Parsed JSON: YES
Top-level type: dict
Top-level keys: ['data', 'success', 'total']
data rows: 1
Union of row fields:
['BSTime', 'abandonedReasonCode', 'cotcInspectionPreCautionary', 'cotcInspectionStatus', 'cotcInspectionText', 'cotcInspectionUpdatedAt', 'courseId', 'courseName', 'fixtureDate', 'fixtureId', 'fixtureSession', 'fixtureType', 'fixtureYear', 'goingText', 'goingUpdatedAt', 'inspectionsText', 'inspectionsUpdatedAt', 'lastUpdated', 'meetingId', 'nextFixture', 'otherText', 'otherUpdatedAt', 'previousFixture', 'racePlanningCode', 'racingTrackType', 'railText', 'railUpdatedAt', 'resultsAvailable', 'stallsText', 'stallsUpdatedAt', 'stewardsReport', 'ticketsLink', 'transparentAvailable', 'wateringText', 'wateringUpdatedAt', 'weatherText', 'weatherUpdatedAt']

First returned row:
{
  "fixtureId": 10399,
  "fixtureYear": "2026",
  "fixtureDate": "2026-05-27",
  "meetingId": 13958,
  "courseId": 37,


## Remaining BHA resource surface — first findings

The six previously unexplored frontend resources were tested for one known completed Newton Abbot fixture/race.

All six routes returned HTTP 200 and valid JSON.

### Fixture detail

The fixture-detail resource returned one structured fixture record containing substantially more information than the fixture-list representation.

Observed fields included:

- BHA fixture, meeting and course references;
- fixture date/type/session;
- racing-track type;
- race-planning code;
- result availability;
- fixture-level stewards-report locator;
- going;
- weather;
- stalls;
- rails;
- watering;
- inspections;
- other fixture information;
- update timestamps for several of those fields.

It also returned `previousFixture` and `nextFixture` objects.

Their exact semantics must not be inferred merely from those names.

### Fixture going / conditions

The `/going` resource is substantially richer than a final going string.

For this sample it returned structured fixture conditions including:

- primary ground code and text;
- in-places ground code and text;
- GoingStick value and timestamp;
- rail information;
- weather and weather commentary;
- inspection-related fields;
- watering status;
- abandonment-related fields.

It also exposed separate nested resources named:

- `conditionsHistory`;
- `weatherHistory`;
- `wateringHistory`;
- `tracks`;
- `races`.

The observed `conditionsHistory` record included a creation timestamp and separate track definitions for Chase and Hurdle racing.

The race observations inside the conditions resource included race identifiers, distances, distance changes and runner counts.

This resource is therefore a high-priority candidate for deeper investigation.

### Fixture officials

The fixture-officials resource returned nine category records.

Each record contained:

- an official category;
- one or more official names.

This demonstrates a structured fixture-officials capability, although its analytical value to Inside Rails has not yet been assessed.

### Race nominations

For this completed sample race, `/nominations` returned 10 records.

The observed structure closely resembled the `/entries` representation and contained runner, horse, jockey, trainer, owner, weight, rating and declaration-related information.

This sample does **not** establish that nominations and entries are semantically identical.

The completed state of the race may affect what is returned.

### Transferred and balloted resources

Both:

- `/trans`;
- `/balloted`;

returned successful but empty datasets for this sample.

Their routes are therefore live, but their populated structure and semantics remain unobserved.

## Source-discovery decision

The highest-value newly discovered resource is currently fixture going/conditions history.

It may provide official time-dependent evidence for:

- going development;
- GoingStick changes;
- weather observations;
- watering;
- rail movements;
- track/sub-course distinctions;
- race-distance changes;
- inspections and abandonment context.

Before probing more races or designing database structures, inspect the complete shape and temporal content of this resource more carefully.

In [7]:
# Deep inspection of the BHA fixture-going resource
#
# WHAT
# ----
# Examine the complete structure of the already-cached Newton Abbot `/going`
# response, with particular attention to:
#
#   - current conditions;
#   - conditionsHistory;
#   - weatherHistory;
#   - wateringHistory;
#   - tracks;
#   - races;
#   - timestamps and apparent temporal grain.
#
# WHY
# ---
# The first surface probe showed that `/going` may contain genuinely useful
# time-dependent official evidence rather than merely a final going string.
#
# Before considering database integration or wider acquisition, establish
# exactly what this one response contains and whether the "history" collections
# really provide multiple historical observations.
#
# READS
# -----
# Existing ignored research cache only:
#
#   data/cache/bha_official_source_feasibility/
#       remaining_resource_probe/fixture_going.json
#
# WRITES
# ------
# None.
#
# EXPECTED RESULT
# ---------------
# A structural and temporal inventory of every major `/going` component.
#
# This cell describes observed structure only. It does not yet assign governed
# meanings to codes or decide what belongs in a future database.

from collections import Counter
import json
from pathlib import Path


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

GOING_CACHE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "remaining_resource_probe"
    / "fixture_going.json"
)

assert GOING_CACHE.is_file(), (
    f"Cached fixture-going response not found: {GOING_CACHE}"
)


# ---------------------------------------------------------------------------
# 1. Load the exact cached response from the previous bounded probe.
# ---------------------------------------------------------------------------

going_envelope = json.loads(
    GOING_CACHE.read_text(encoding="utf-8")
)

assert going_envelope["response_status"] == 200
assert going_envelope["parsed_json"] is not None

going_payload = going_envelope["parsed_json"]

assert going_payload.get("success") is True

going_data = going_payload.get("data")

assert isinstance(going_data, dict), (
    "Expected fixture-going data to be a dictionary."
)


print("BHA FIXTURE-GOING DEEP INSPECTION")
print("=================================")
print("Fixture:", going_data.get("fixtureDate"), going_data.get("fixtureId"))
print("Course ID:", going_data.get("courseId"))
print("Fixture type:", going_data.get("fixtureType"))
print("Top-level data keys:")
print(sorted(going_data.keys()))


# ---------------------------------------------------------------------------
# 2. Current conditions.
# ---------------------------------------------------------------------------

conditions = going_data.get("conditions")

print("\nCURRENT CONDITIONS")
print("==================")

if isinstance(conditions, dict):
    for key in sorted(conditions):
        value = conditions[key]

        if isinstance(value, (dict, list)):
            print(f"{key}: {type(value).__name__}")
        else:
            print(f"{key}: {value!r}")

else:
    print("No dictionary-shaped current conditions found.")


# ---------------------------------------------------------------------------
# 3. Generic history inspector.
# ---------------------------------------------------------------------------
#
# We want to know:
#
#   - whether a history really contains multiple observations;
#   - which fields exist across observations;
#   - which timestamp-like fields occur;
#   - whether observations themselves contain nested structures.

def inspect_history(name, rows):
    print(f"\n{name.upper()}")
    print("=" * len(name))

    if not isinstance(rows, list):
        print("Type:", type(rows).__name__)
        print("Expected a list; no row-level inspection performed.")
        return

    print("Rows:", len(rows))

    if not rows:
        print("Empty history.")
        return

    dictionary_rows = [
        row
        for row in rows
        if isinstance(row, dict)
    ]

    print("Dictionary rows:", len(dictionary_rows))

    field_union = sorted(
        {
            key
            for row in dictionary_rows
            for key in row.keys()
        }
    )

    print("Union of top-level fields:")
    print(field_union)

    timestamp_fields = [
        key
        for key in field_union
        if any(
            term in key.lower()
            for term in (
                "time",
                "date",
                "created",
                "updated",
                "timestamp",
            )
        )
    ]

    print("\nTimestamp/date-like top-level fields:")
    print(timestamp_fields)

    if timestamp_fields:
        print("\nObserved top-level timestamp/date values:")

        for index, row in enumerate(dictionary_rows, start=1):
            print(
                f"Row {index}:",
                {
                    key: row.get(key)
                    for key in timestamp_fields
                },
            )

    print("\nNested structures by row:")

    for index, row in enumerate(dictionary_rows, start=1):
        nested = {
            key: (
                f"list[{len(value)}]"
                if isinstance(value, list)
                else f"dict[{len(value)}]"
            )
            for key, value in row.items()
            if isinstance(value, (list, dict))
        }

        print(f"Row {index}:", nested or "none")

    print("\nFirst-row bounded preview:")
    print(
        json.dumps(
            dictionary_rows[0],
            indent=2,
            ensure_ascii=False,
        )[:10_000]
    )


# ---------------------------------------------------------------------------
# 4. Inspect each explicit history collection.
# ---------------------------------------------------------------------------

conditions_history = going_data.get("conditionsHistory")
weather_history = going_data.get("weatherHistory")
watering_history = going_data.get("wateringHistory")

inspect_history(
    "conditionsHistory",
    conditions_history,
)

inspect_history(
    "weatherHistory",
    weather_history,
)

inspect_history(
    "wateringHistory",
    watering_history,
)


# ---------------------------------------------------------------------------
# 5. Inspect track definitions.
# ---------------------------------------------------------------------------

tracks = going_data.get("tracks")

print("\nTRACKS")
print("======")

if isinstance(tracks, list):
    print("Rows:", len(tracks))

    if tracks:
        track_fields = sorted(
            {
                key
                for row in tracks
                if isinstance(row, dict)
                for key in row.keys()
            }
        )

        print("Union of track fields:")
        print(track_fields)

        print("\nTrack records:")

        for row in tracks:
            if isinstance(row, dict):
                print(
                    {
                        key: row.get(key)
                        for key in track_fields
                    }
                )

else:
    print("Type:", type(tracks).__name__)


# ---------------------------------------------------------------------------
# 6. Inspect race-level observations within the fixture-going resource.
# ---------------------------------------------------------------------------

going_races = going_data.get("races")

print("\nRACES WITHIN GOING RESOURCE")
print("===========================")

if isinstance(going_races, list):
    print("Rows:", len(going_races))

    if going_races:
        race_fields = sorted(
            {
                key
                for row in going_races
                if isinstance(row, dict)
                for key in row.keys()
            }
        )

        print("Union of race fields:")
        print(race_fields)

        print("\nRace observations:")

        for row in going_races:
            if isinstance(row, dict):
                print(
                    {
                        "raceId": row.get("raceId"),
                        "yearOfRace": row.get("yearOfRace"),
                        "divisionSequence": row.get("divisionSequence"),
                        "raceTime": row.get("raceTime"),
                        "raceDistanceText": row.get("raceDistanceText"),
                        "distanceChange": row.get("distanceChange"),
                        "ground": row.get("ground"),
                        "groundPlaces": row.get("groundPlaces"),
                        "numberOfRunners": row.get("numberOfRunners"),
                        "groundLastUpdatedAt": row.get(
                            "groundLastUpdatedAt"
                        ),
                        "abandonment": row.get("abandonment"),
                    }
                )

else:
    print("Type:", type(going_races).__name__)


# ---------------------------------------------------------------------------
# 7. Search the entire response for timestamp-like field names.
# ---------------------------------------------------------------------------
#
# This gives us a complete first-pass list of temporal fields even when they are
# buried inside nested history/track/race structures.

timestamp_occurrences = []


def walk_for_timestamps(value, path="data"):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"

            if any(
                term in key.lower()
                for term in (
                    "time",
                    "date",
                    "created",
                    "updated",
                    "timestamp",
                )
            ):
                timestamp_occurrences.append(
                    {
                        "path": child_path,
                        "value": child,
                    }
                )

            walk_for_timestamps(
                child,
                child_path,
            )

    elif isinstance(value, list):
        for index, child in enumerate(value):
            walk_for_timestamps(
                child,
                f"{path}[{index}]",
            )


walk_for_timestamps(going_data)


print("\nALL TIMESTAMP/DATE-LIKE PATHS")
print("=============================")
print("Occurrences:", len(timestamp_occurrences))

path_counts = Counter(
    # Replace list indexes so repeated history records collapse to one field path.
    __import__("re").sub(
        r"\[\d+\]",
        "[]",
        row["path"],
    )
    for row in timestamp_occurrences
)

for path, count in sorted(path_counts.items()):
    print(f"{path}: {count} occurrence(s)")


print("\nCACHE PROVENANCE")
print("================")
print("Request URL:", going_envelope["request_url"])
print("Retrieved at:", going_envelope["retrieved_at_utc"])
print("Network request made by this cell: NO")
print("Writes made by this cell: NONE")

BHA FIXTURE-GOING DEEP INSPECTION
Fixture: 2026-05-27 10399
Course ID: 37
Fixture type: JUMP
Top-level data keys:
['conditions', 'conditionsHistory', 'courseId', 'fixtureDate', 'fixtureId', 'fixtureType', 'fixtureYear', 'lastUpdate', 'races', 'tracks', 'wateringHistory', 'weatherHistory']

CURRENT CONDITIONS
abandonedWhen: None
abandonedWhenText: None
abandonment: None
abandonmentComment: None
abandonmentText: None
abandonmentUpdatedAt: None
conditionInPlaces: 3
conditionInPlacesComment: None
conditionInPlacesText: 'Good to Firm'
goingStick: '5.7'
goingStickAvailable: 1
goingStickComment: None
goingStickUpdatedAt: '2026-05-27 08:00:00'
ground: 4
groundComment: 'Home straight will be watered this morning'
groundText: 'Good'
inspectedAt: None
inspectionComment: None
inspectionStatus: '(Status: Pending)'
lowSunProvision: None
other: 'Family Fun Day\nAutism In Racing'
preCautionary: None
rails: 'Chase bends: 2yds - 6yds from innermost\nHurdle bends: 2yds - 6yds from innermost'
stalls: None

## Fixture-going history — substantive finding

Deep inspection confirms that the BHA `/going` resource contains genuine temporal history rather than only a current going description.

For the sampled Newton Abbot fixture on 27 May 2026, the response contained:

- 13 `conditionsHistory` records;
- 6 `wateringHistory` records;
- a structured `weatherHistory` object containing timestamped comment and forecast records;
- current fixture conditions;
- race-level condition/distance information;
- track-level structures;
- race timetable structures.

### Conditions history

Each of the 13 conditions-history records contained its own nested:

- `conditions`;
- `tracks`;
- `races`;
- `timetable`.

The nested `conditions` structures expose timestamp fields including:

- `creationTimestamp`;
- `groundLastUpdatedAt`;
- `goingStickUpdatedAt`.

This demonstrates that the service preserves multiple historical fixture-condition states.

The exact reason a new conditions-history record is created remains to be established. A record must therefore not yet be described as specifically a "going update".

### Watering history

Six watering-history records were observed, with creation timestamps between 19 and 22 May 2026.

This establishes explicit historical watering-state capability.

### Weather history

`weatherHistory` is not a simple list.

The response contains nested weather-history structures including at least:

- 10 timestamped comment observations;
- 2 timestamped forecast observations.

Their exact structure and semantics still require direct inspection.

### Track-level history

Although the current top-level `tracks` collection was empty for this completed fixture, individual conditions-history records contained two track records:

- Chase;
- Hurdle.

Those records carry stable-looking BHA `trackId` values in this sample.

This may be relevant to future sub-course/track research, but no Inside Rails track identity should be inferred from these IDs yet.

### Race-programme history

The race collections embedded in conditions-history snapshots are themselves historical observations.

An apparent change was observed in the final race represented within different snapshots:

- one state referenced BHA `raceId = 38746`;
- another referenced BHA `raceId = 68081`.

The reason for that change is unresolved.

It may represent a programme amendment or another BHA lifecycle behaviour and should be investigated rather than normalised away.

### Timetable structure

Conditions-history snapshots also expose per-race timetable fields such as:

- `horsesArriveInParadeRing`;
- `jockeysLeaveWeighingRoom`;
- `signalToMountIsGiven`;
- `lastHorseLeavesParadeRing`;
- `horsesArriveAtStart`.

Observed values such as `10`, `7`, `5`, `4` and `3` are not yet assigned a unit or semantic interpretation.

### Decision

The BHA fixture-going resource is now a high-value candidate source for Inside Rails.

Before considering acquisition or schema design, determine:

1. the chronological sequence of condition snapshots;
2. what actually changed between snapshots;
3. the detailed structure of `weatherHistory`;
4. the historical watering sequence;
5. the apparent race-programme change within the same fixture.

In [8]:
# BHA fixture-condition timeline reconstruction
#
# WHAT
# ----
# Reconstruct the chronological sequence of the 13 cached conditionsHistory
# snapshots for the sampled Newton Abbot fixture and identify what changed from
# one snapshot to the next.
#
# Also unpack the weatherHistory object directly, because the previous generic
# inspector established that it is a dictionary rather than a simple list.
#
# WHY
# ---
# We now know the `/going` resource preserves multiple historical states.
# The next question is whether those states form a meaningful timeline of
# changing going, weather, watering, rails, race programme or other conditions.
#
# READS
# -----
# Existing cached fixture-going response only:
#
#   data/cache/bha_official_source_feasibility/
#       remaining_resource_probe/fixture_going.json
#
# WRITES
# ------
# None.
#
# EXPECTED RESULT
# ---------------
# 1. One chronological summary row per conditionsHistory snapshot.
# 2. A compact list of fields that changed between consecutive snapshots.
# 3. Direct inspection of weatherHistory comments and forecasts.
# 4. A check for race-list changes across condition snapshots.
#
# No new BHA request is made.

from copy import deepcopy
from datetime import datetime
import json
from pathlib import Path


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

GOING_CACHE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "remaining_resource_probe"
    / "fixture_going.json"
)

assert GOING_CACHE.is_file()

going_envelope = json.loads(
    GOING_CACHE.read_text(encoding="utf-8")
)

going_data = going_envelope["parsed_json"]["data"]

conditions_history = going_data.get("conditionsHistory", [])
weather_history = going_data.get("weatherHistory")
watering_history = going_data.get("wateringHistory", [])

assert isinstance(conditions_history, list)
assert conditions_history


# ---------------------------------------------------------------------------
# 1. Put the historical condition snapshots into chronological order.
# ---------------------------------------------------------------------------
#
# The nested conditions.creationTimestamp is the clearest observed timestamp
# describing creation of each historical state.

def snapshot_timestamp(snapshot):
    conditions = snapshot.get("conditions") or {}
    value = conditions.get("creationTimestamp")

    if value is None:
        return datetime.min

    return datetime.fromisoformat(value)


ordered_snapshots = sorted(
    conditions_history,
    key=snapshot_timestamp,
)


# ---------------------------------------------------------------------------
# 2. Build a compact analytical representation of each snapshot.
# ---------------------------------------------------------------------------
#
# Preserve only fields useful for comparing state changes. The raw cached JSON
# remains available if we later need the full record.

def race_reference(row):
    return (
        row.get("yearOfRace"),
        row.get("raceId"),
        row.get("divisionSequence"),
    )


def compact_snapshot(snapshot):
    conditions = snapshot.get("conditions") or {}
    tracks = snapshot.get("tracks") or []
    races = snapshot.get("races") or []
    timetable = snapshot.get("timetable") or []

    return {
        "creationTimestamp": conditions.get("creationTimestamp"),
        "ground": conditions.get("ground"),
        "groundText": conditions.get("groundText"),
        "groundComment": conditions.get("groundComment"),
        "conditionInPlaces": conditions.get("conditionInPlaces"),
        "conditionInPlacesText": conditions.get("conditionInPlacesText"),
        "goingStick": conditions.get("goingStick"),
        "goingStickUpdatedAt": conditions.get("goingStickUpdatedAt"),
        "rails": conditions.get("rails"),
        "weather": conditions.get("weather"),
        "weatherComment": conditions.get("weatherComment"),
        "watering": conditions.get("watering"),
        "wateringStatus": conditions.get("wateringStatus"),
        "inspectionComment": conditions.get("inspectionComment"),
        "inspectedAt": conditions.get("inspectedAt"),
        "abandonment": conditions.get("abandonment"),
        "abandonedWhen": conditions.get("abandonedWhen"),
        "track_records": deepcopy(tracks),
        "race_refs": [race_reference(row) for row in races],
        "race_records": deepcopy(races),
        "timetable": deepcopy(timetable),
    }


timeline = [
    compact_snapshot(snapshot)
    for snapshot in ordered_snapshots
]


print("CONDITIONS-HISTORY TIMELINE")
print("===========================")
print("Snapshots:", len(timeline))


for index, row in enumerate(timeline, start=1):
    print(f"\nSnapshot {index}")
    print("-" * 40)

    for key in [
        "creationTimestamp",
        "groundText",
        "groundComment",
        "conditionInPlacesText",
        "goingStick",
        "goingStickUpdatedAt",
        "weather",
        "wateringStatus",
        "rails",
    ]:
        print(f"{key}: {row.get(key)!r}")

    print("Race references:")
    print(row["race_refs"])


# ---------------------------------------------------------------------------
# 3. Diff consecutive snapshots.
# ---------------------------------------------------------------------------
#
# This tells us whether the 13 history records represent genuinely changing
# state or merely repeated copies created for unrelated reasons.

comparison_fields = [
    "ground",
    "groundText",
    "groundComment",
    "conditionInPlaces",
    "conditionInPlacesText",
    "goingStick",
    "goingStickUpdatedAt",
    "rails",
    "weather",
    "weatherComment",
    "watering",
    "wateringStatus",
    "inspectionComment",
    "inspectedAt",
    "abandonment",
    "abandonedWhen",
    "track_records",
    "race_refs",
    "race_records",
    "timetable",
]


print("\n\nCHANGES BETWEEN CONSECUTIVE SNAPSHOTS")
print("=====================================")


for index in range(1, len(timeline)):
    previous = timeline[index - 1]
    current = timeline[index]

    changes = []

    for field in comparison_fields:
        if previous.get(field) != current.get(field):
            changes.append(field)

    print(
        f"\n{previous['creationTimestamp']}"
        f"  ->  {current['creationTimestamp']}"
    )

    if not changes:
        print("Changed fields: NONE")
        continue

    print("Changed fields:")
    print(changes)

    # For scalar/high-value fields, show before and after directly.
    for field in changes:
        if field not in {
            "track_records",
            "race_records",
            "timetable",
        }:
            print(
                f"  {field}: "
                f"{previous.get(field)!r} "
                f"-> {current.get(field)!r}"
            )


# ---------------------------------------------------------------------------
# 4. Isolate race-programme changes.
# ---------------------------------------------------------------------------

print("\n\nRACE-PROGRAMME CHANGES")
print("======================")

race_change_found = False

for index in range(1, len(timeline)):
    previous = timeline[index - 1]
    current = timeline[index]

    if previous["race_refs"] == current["race_refs"]:
        continue

    race_change_found = True

    print(
        f"\nAt snapshot transition:\n"
        f"  {previous['creationTimestamp']}\n"
        f"  -> {current['creationTimestamp']}"
    )

    previous_set = set(previous["race_refs"])
    current_set = set(current["race_refs"])

    print("Removed race references:")
    print(sorted(previous_set - current_set))

    print("Added race references:")
    print(sorted(current_set - previous_set))


if not race_change_found:
    print("No race-reference changes observed.")


# ---------------------------------------------------------------------------
# 5. Inspect weatherHistory directly.
# ---------------------------------------------------------------------------

print("\n\nWEATHER HISTORY")
print("===============")
print("Top-level type:", type(weather_history).__name__)

if isinstance(weather_history, dict):
    print("Keys:", sorted(weather_history.keys()))

    for key, value in weather_history.items():
        print(f"\n{key}")
        print("-" * len(key))
        print("Type:", type(value).__name__)

        if isinstance(value, list):
            print("Rows:", len(value))

            for index, row in enumerate(value, start=1):
                print(f"\nRow {index}")

                if isinstance(row, dict):
                    print(
                        json.dumps(
                            row,
                            indent=2,
                            ensure_ascii=False,
                        )[:5000]
                    )
                else:
                    print(repr(row)[:5000])

        elif isinstance(value, dict):
            print(
                json.dumps(
                    value,
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )

        else:
            print(repr(value))

else:
    print(
        json.dumps(
            weather_history,
            indent=2,
            ensure_ascii=False,
        )[:8000]
    )


# ---------------------------------------------------------------------------
# 6. Watering chronology, sorted explicitly.
# ---------------------------------------------------------------------------

print("\n\nWATERING HISTORY")
print("================")

ordered_watering = sorted(
    watering_history,
    key=lambda row: (
        row.get("creationTimestamp") or ""
    ),
)

for row in ordered_watering:
    print(
        {
            "creationTimestamp": row.get("creationTimestamp"),
            "status": row.get("status"),
            "value": row.get("value"),
        }
    )


print("\n\nPROVENANCE")
print("==========")
print("Source request:", going_envelope["request_url"])
print("Cached retrieval:", going_envelope["retrieved_at_utc"])
print("Network request made by this cell: NO")
print("Writes made by this cell: NONE")

CONDITIONS-HISTORY TIMELINE
Snapshots: 13

Snapshot 1
----------------------------------------
creationTimestamp: '2026-05-19 14:13:44'
groundText: {'code': 5, 'description': 'Good to Soft'}
groundComment: ''
conditionInPlacesText: {'code': 4, 'description': 'Good'}
goingStick: None
goingStickUpdatedAt: None
weather: 'Partly Cloudy'
wateringStatus: None
rails: 'Chase bends: tbc\nHurdle bends: tbc'
Race references:
[(2026, 23016, 0), (2026, 18080, 0), (2026, 5439, 0), (2026, 56297, 0), (2026, 5441, 0), (2026, 38746, 0)]

Snapshot 2
----------------------------------------
creationTimestamp: '2026-05-20 09:08:46'
groundText: {'code': 5, 'description': 'Good to Soft'}
groundComment: ''
conditionInPlacesText: {'code': 4, 'description': 'Good'}
goingStick: None
goingStickUpdatedAt: None
weather: 'Partly Cloudy'
wateringStatus: None
rails: 'Chase bends: tbc\nHurdle bends: tbc'
Race references:
[(2026, 23016, 0), (2026, 18080, 0), (2026, 5439, 0), (2026, 56297, 0), (2026, 5441, 0), (2026, 680

## Fixture-condition timeline — confirmed capability

The Newton Abbot sample confirms that the BHA `/going` resource preserves a genuine historical sequence of fixture state.

### Going development

Observed conditions evolved materially before raceday.

The first historical observation, on 19 May 2026, recorded:

- `Good to Soft`;
- `Good` in places.

By 22 May the primary going had changed to:

- `Good`;
- `Good to Soft` in places.

By 25 May it was:

- `Good`;
- `Good to Firm` in places.

The accompanying comments also changed over time and contain contextual statements about drying ground and intended watering.

This demonstrates that a single final-going value would discard useful official historical information.

### GoingStick history

GoingStick was not populated in the earliest snapshots.

Observed later measurements included:

- `5.9`, timestamped 25 May 2026 at 07:00;
- `5.7`, timestamped 27 May 2026 at 08:00.

An intermediate 27 May conditions snapshot had a null GoingStick even though an earlier measurement existed.

Therefore a null observation must not automatically be interpreted as evidence that no previous GoingStick measurement existed.

Historical observations must be preserved independently.

### Watering history

The explicit watering history records a progression of official intentions and activity:

- possible future watering;
- expected watering;
- stated intention to water;
- confirmed watering;
- `Watering (In Progress)`.

This is materially richer than a single fixture-level watering field.

### Weather history

The weather resource preserves repeated timestamped revisions.

For this fixture:

- two high-level forecast-state observations were returned;
- ten detailed weather-comment versions were preserved.

The detailed narrative changed repeatedly as the fixture approached, including revisions to:

- rainfall already received;
- expected daily temperatures;
- expected showers/thunderstorms;
- raceday conditions.

This establishes a source of official pre-race weather expectations as distinct from independent measured meteorological data.

### Race-programme history

Between the conditions snapshots created on:

- 19 May 2026 at 14:13:44;
- 20 May 2026 at 09:08:46;

the final race reference changed from:

- `2026 / 38746 / 0`;

to:

- `2026 / 68081 / 0`.

The reason remains unresolved.

The observation nevertheless demonstrates that the historical fixture resource can preserve race-programme changes over time.

### History-row semantics warning

Two consecutive snapshots on 25 May 2026 contained no material difference across the fields compared in this investigation.

Therefore:

> **a new `conditionsHistory` row does not necessarily imply a substantive racing-condition change.**

`creationTimestamp` should be treated as an observation timestamp, not automatically as the timestamp of a specific type of racing event.

### Analytical value

The BHA source has now demonstrated potentially valuable official evidence for:

- going evolution;
- GoingStick history;
- watering intention and activity;
- changing weather expectations;
- rail configuration;
- race-distance changes;
- race-programme changes;
- fixture inspection/abandonment context;
- track-specific structures.

These are materially different from merely obtaining a better final race result.

## Decision

The next phase should compare the useful BHA information discovered so far against the fields already governed in Database v4.

The objective is to separate:

1. information Database v4 already represents adequately;
2. BHA information useful mainly for validation/provenance;
3. genuinely new analytical information worth investigating for future governed storage;
4. low-value information that does not justify further work.

Do not design new database tables yet.

## Additional BHA source surface — Stewards' Reports

The BHA fixture and race resources expose links to official Stewards' Reports.

These reports may contain information that is analytically relevant but not represented in ordinary race/result records, including:

- race enquiries and incidents;
- interference;
- explanations for poor or unusual performance;
- runner notes;
- veterinary or equipment-related observations;
- possible amendments or disciplinary context;
- fixture-level incidents.

The existence of a report link does not establish how this information is technically exposed.

Before comparing the complete BHA information surface with Database v4, inspect one known modern Stewards' Report and determine:

1. whether the report is HTML, PDF, JSON or another structured resource;
2. whether individual races and runners can be identified reliably;
3. whether report categories are structurally distinguishable;
4. whether useful information is machine-readable or only free text;
5. whether additional underlying structured endpoints exist behind the report interface.

Use the already-known Newton Abbot fixture as the bounded sample.

Do not yet attempt bulk acquisition or database design.

In [11]:
# BHA Stewards' Report — first technical inspection
#
# WHAT
# ----
# Follow the official Stewards' Report URL already exposed by the cached BHA
# fixture-detail response for the Newton Abbot fixture.
#
# Determine:
#
#   - HTTP status;
#   - final URL after redirects;
#   - content type;
#   - whether the response is HTML, JSON, PDF or something else;
#   - page title / obvious structural clues if HTML;
#   - whether the page references additional API/JSON resources.
#
# WHY
# ---
# Stewards' Reports may expose analytically useful information not present in
# ordinary race/result records. Before examining their contents, establish how
# the report is technically delivered.
#
# READS
# -----
# Existing cached BHA fixture-detail evidence:
#
#   data/cache/bha_official_source_feasibility/
#       remaining_resource_probe/fixture_detail.json
#
# WRITES
# ------
# One ignored research-cache directory:
#
#   data/cache/bha_official_source_feasibility/
#       stewards_report_probe/
#
# The exact HTTP response body is cached locally.
#
# EXPECTED RESULT
# ---------------
# A bounded technical description of one official Stewards' Report response.
#
# Do not yet scrape individual report contents or infer report semantics.

import hashlib
import json
import re
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

FIXTURE_DETAIL_CACHE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "remaining_resource_probe"
    / "fixture_detail.json"
)

REPORT_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "stewards_report_probe"
)

REPORT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert FIXTURE_DETAIL_CACHE.is_file(), (
    f"Fixture-detail cache not found: {FIXTURE_DETAIL_CACHE}"
)


# ---------------------------------------------------------------------------
# 1. Recover the report locator from the already-cached official fixture data.
# ---------------------------------------------------------------------------

fixture_envelope = json.loads(
    FIXTURE_DETAIL_CACHE.read_text(encoding="utf-8")
)

fixture_rows = fixture_envelope["parsed_json"]["data"]

assert isinstance(fixture_rows, list)
assert len(fixture_rows) == 1

fixture_record = fixture_rows[0]

report_url = fixture_record.get("stewardsReport")

assert report_url, (
    "No Stewards' Report URL was present in the fixture-detail record."
)

print("BHA STEWARDS' REPORT — TECHNICAL PROBE")
print("======================================")
print("Fixture:", fixture_record.get("fixtureDate"), fixture_record.get("courseName"))
print("BHA fixture reference:", fixture_record.get("fixtureYear"), fixture_record.get("fixtureId"))
print("Report locator:", report_url)


# ---------------------------------------------------------------------------
# 2. Fetch exactly this one report.
# ---------------------------------------------------------------------------
#
# This is a public report locator supplied by the BHA fixture resource.
# No Authorization credential is sent to this separate report host.

request = Request(
    report_url,
    headers={
        "Accept": "*/*",
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.britishhorseracing.com/racing/results/",
    },
)

status = None
final_url = None
content_type = None
body = b""
error_text = None

try:
    with urlopen(request, timeout=30) as response:
        status = response.status
        final_url = response.geturl()
        content_type = response.headers.get("Content-Type")
        body = response.read()

except HTTPError as error:
    status = error.code
    final_url = error.geturl()
    content_type = error.headers.get("Content-Type")
    body = error.read()
    error_text = repr(error)

except URLError as error:
    error_text = repr(error)


# ---------------------------------------------------------------------------
# 3. Cache the exact body plus metadata.
# ---------------------------------------------------------------------------

body_hash = hashlib.sha256(body).hexdigest()

body_path = REPORT_CACHE_DIR / "newton_abbot_2026_10399_response.bin"
metadata_path = REPORT_CACHE_DIR / "newton_abbot_2026_10399_metadata.json"

body_path.write_bytes(body)

metadata = {
    "fixtureYear": fixture_record.get("fixtureYear"),
    "fixtureId": fixture_record.get("fixtureId"),
    "fixtureDate": fixture_record.get("fixtureDate"),
    "courseName": fixture_record.get("courseName"),
    "request_url": report_url,
    "final_url": final_url,
    "http_status": status,
    "content_type": content_type,
    "response_bytes": len(body),
    "sha256": body_hash,
    "error": error_text,
}

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# 4. Classify the response conservatively from headers and magic bytes.
# ---------------------------------------------------------------------------

content_type_lower = (content_type or "").lower()

if body.startswith(b"%PDF"):
    response_kind = "PDF"

elif "json" in content_type_lower:
    response_kind = "JSON"

elif (
    "html" in content_type_lower
    or b"<html" in body[:2000].lower()
    or b"<!doctype html" in body[:2000].lower()
):
    response_kind = "HTML"

else:
    response_kind = "OTHER / UNKNOWN"


print("\nHTTP / CONTENT")
print("==============")
print("HTTP status:", status)
print("Final URL:", final_url)
print("Content-Type:", content_type)
print("Response bytes:", len(body))
print("SHA-256:", body_hash)
print("Response kind:", response_kind)

if error_text:
    print("HTTP/transport observation:", error_text)


# ---------------------------------------------------------------------------
# 5. If HTML, inspect only structural clues.
# ---------------------------------------------------------------------------

if response_kind == "HTML":
    text = body.decode(
        "utf-8",
        errors="replace",
    )

    title_match = re.search(
        r"<title[^>]*>(.*?)</title>",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    title = None

    if title_match:
        title = re.sub(
            r"\s+",
            " ",
            title_match.group(1),
        ).strip()

    print("\nHTML STRUCTURE")
    print("==============")
    print("Title:", title)

    script_sources = sorted(
        set(
            re.findall(
                r"""<script[^>]+src=["']([^"']+)["']""",
                text,
                flags=re.IGNORECASE,
            )
        )
    )

    print("\nExternal script sources:", len(script_sources))

    for source in script_sources:
        print(" ", source)

    # Search for strings that may indicate structured resources or AJAX calls.
    interesting_strings = sorted(
        set(
            re.findall(
                r"""["']([^"']*(?:api|ajax|json|report|runner|steward)[^"']*)["']""",
                text,
                flags=re.IGNORECASE,
            )
        )
    )

    print("\nPotential structured-resource clues:")

    if interesting_strings:
        for value in interesting_strings[:100]:
            print(" ", value)
    else:
        print("  NONE FOUND")

    # Print only a bounded text preview after stripping scripts/styles/tags.
    visible = re.sub(
        r"<script\b[^>]*>.*?</script>",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    visible = re.sub(
        r"<style\b[^>]*>.*?</style>",
        " ",
        visible,
        flags=re.IGNORECASE | re.DOTALL,
    )

    visible = re.sub(
        r"<[^>]+>",
        " ",
        visible,
    )

    visible = re.sub(
        r"\s+",
        " ",
        visible,
    ).strip()

    print("\nBounded visible-text preview:")
    print(visible[:5000])


# ---------------------------------------------------------------------------
# 6. If JSON, describe its structure without interpreting it.
# ---------------------------------------------------------------------------

elif response_kind == "JSON":
    try:
        parsed = json.loads(
            body.decode(
                "utf-8",
                errors="strict",
            )
        )

        print("\nJSON STRUCTURE")
        print("==============")
        print("Top-level type:", type(parsed).__name__)

        if isinstance(parsed, dict):
            print("Top-level keys:", sorted(parsed.keys()))

        elif isinstance(parsed, list):
            print("Rows:", len(parsed))

            if parsed and isinstance(parsed[0], dict):
                print(
                    "Union of row fields:",
                    sorted(
                        {
                            key
                            for row in parsed
                            if isinstance(row, dict)
                            for key in row
                        }
                    ),
                )

        print("\nBounded JSON preview:")
        print(
            json.dumps(
                parsed,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

    except Exception as error:
        print("JSON parse observation:", repr(error))


# ---------------------------------------------------------------------------
# 7. If PDF, stop here.
# ---------------------------------------------------------------------------
#
# A PDF requires visual/page inspection rather than treating its bytes as text.

elif response_kind == "PDF":
    print("\nPDF response confirmed.")
    print("No PDF content interpreted by this cell.")


print("\nCACHE")
print("=====")
print("Body:", body_path)
print("Metadata:", metadata_path)
print("BHA Authorization sent: NO")

BHA STEWARDS' REPORT — TECHNICAL PROBE
Fixture: 2026-05-27 Newton Abbot
BHA fixture reference: 2026 10399
Report locator: https://crate.horseracing.software/stewardsreport/?fixtureId=10399&year=2026

HTTP / CONTENT
HTTP status: 200
Final URL: https://crate.horseracing.software/stewardsreport/?fixtureId=10399&year=2026
Content-Type: application/pdf
Response bytes: 18696
SHA-256: cda531285dc4b60dece0c2451a16e23e985c7768dddcfc1a5b5e392470ac4304
Response kind: PDF

PDF response confirmed.
No PDF content interpreted by this cell.

CACHE
=====
Body: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/stewards_report_probe/newton_abbot_2026_10399_response.bin
Metadata: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/stewards_report_probe/newton_abbot_2026_10399_metadata.json
BHA Authorization sent: NO


## Stewards' Report delivery — first observation

The sampled Newton Abbot fixture exposed an official Stewards' Report locator:

`https://crate.horseracing.software/stewardsreport/?fixtureId=10399&year=2026`

A direct request to that locator returned:

- HTTP 200;
- `Content-Type: application/pdf`;
- a PDF response of 18,696 bytes.

No BHA API Authorization credential was required for this request.

### Observation

The Stewards' Report is delivered as a directly accessible PDF document rather than as HTML or a JSON resource at the observed locator.

This means the report should be investigated as a document source rather than assumed to share the structure of the BHA race/result API.

### Next question

Determine whether the PDF contains machine-extractable text with reliable identifiers for:

- fixture;
- individual races;
- horses/runners;
- enquiry/report categories;
- incidents and explanations.

If the document is structurally readable, it may provide another useful official evidence source even if its contents are primarily narrative rather than field-based.

Do not yet attempt bulk acquisition or database integration.

In [13]:
# BHA Stewards' Report — text and structural inspection using pdftotext
#
# WHAT
# ----
# Extract machine-readable text from the already-cached Newton Abbot
# Stewards' Report PDF using the system `pdftotext` utility.
#
# WHY
# ---
# Neither pypdf nor PyPDF2 is installed in the current environment.
# There is no reason to add a Python dependency if the existing Ubuntu
# PDF-text tooling can inspect this one research document.
#
# READS
# -----
# Existing cached Stewards' Report only.
#
# WRITES
# ------
# None.
#
# EXPECTED RESULT
# ---------------
# - confirm whether pdftotext is available;
# - extract report text;
# - show clock times and possible structural headings;
# - display the report text with its approximate layout preserved.
#
# No network request is made.

import re
import shutil
import subprocess
from pathlib import Path


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

REPORT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "stewards_report_probe"
    / "newton_abbot_2026_10399_response.bin"
)

assert REPORT_FILE.is_file(), (
    f"Cached Stewards' Report not found: {REPORT_FILE}"
)


# ---------------------------------------------------------------------------
# 1. Confirm the existing system PDF-text tool is available.
# ---------------------------------------------------------------------------

PDFTOTEXT = shutil.which("pdftotext")

assert PDFTOTEXT is not None, (
    "`pdftotext` is not installed on this machine."
)

print("BHA STEWARDS' REPORT — TEXT INSPECTION")
print("======================================")
print("Extractor:", PDFTOTEXT)


# ---------------------------------------------------------------------------
# 2. Extract text to stdout without creating another file.
# ---------------------------------------------------------------------------
#
# `-layout` attempts to retain useful columns and line structure from the PDF.

result = subprocess.run(
    [
        PDFTOTEXT,
        "-layout",
        str(REPORT_FILE),
        "-",
    ],
    capture_output=True,
    text=True,
    check=True,
)

full_text = result.stdout.replace("\r\n", "\n").replace("\r", "\n")

print("Extracted characters:", len(full_text))
print("Machine-readable text present:", bool(full_text.strip()))


# ---------------------------------------------------------------------------
# 3. Inspect basic structural clues.
# ---------------------------------------------------------------------------

times = sorted(
    set(
        re.findall(
            r"\b(?:[01]?\d|2[0-3]):[0-5]\d\b",
            full_text,
        )
    )
)

print("\nCLOCK-TIME STRINGS")
print("==================")
print(times)


lines = [
    line.strip()
    for line in full_text.splitlines()
    if line.strip()
]

heading_candidates = []

for line in lines:
    if len(line) > 120:
        continue

    upper = line.upper()

    if (
        line == upper
        or any(
            term in upper
            for term in (
                "STEWARDS",
                "ENQUIR",
                "INCIDENT",
                "RUNNER",
                "RACE",
                "REPORT",
                "NOTES",
                "HORSE",
            )
        )
    ):
        heading_candidates.append(line)


print("\nPOSSIBLE HEADINGS / LABELS")
print("==========================")

for line in dict.fromkeys(heading_candidates):
    print(line)


# ---------------------------------------------------------------------------
# 4. Display the extracted report.
# ---------------------------------------------------------------------------

print("\nEXTRACTED REPORT TEXT")
print("=====================")

MAX_PRINT_CHARS = 30_000

print(full_text[:MAX_PRINT_CHARS])

if len(full_text) > MAX_PRINT_CHARS:
    print(
        f"\n[Output truncated after {MAX_PRINT_CHARS:,} characters]"
    )


print("\nPROVENANCE")
print("==========")
print("Cached PDF:", REPORT_FILE)
print("Network request made by this cell: NO")
print("Writes made by this cell: NONE")

BHA STEWARDS' REPORT — TEXT INSPECTION
Extractor: /usr/bin/pdftotext
Extracted characters: 4250
Machine-readable text present: True

CLOCK-TIME STRINGS
['00:56']

POSSIBLE HEADINGS / LABELS
NEWTON ABBOT STEWARDS' REPORT
Stewards:        Thomas Evetts (Chief Steward), Mark Elgar (Stewards' Panel Chair), Sophie Candy
(Steward), Beth Dowswell (Assistant Steward), Milly Bersey (Raceday Assistant).
Non-Runners
Race   Horse and Trainer                        Reason
3:23pm FERANDO (IRE), trained by Barry          Horse not Qualified
Chanin.                                  (Not changed since declaration but horse on course
Race   Horse and Trainer                             Reason
Race     Horse                Declared       Replacement        Reason
MAGIC (GB)             Davies                            Racecourse Medical Officer)
Race 1 - 2:53pm .
THE STOCK EXE BUILDING SUPPLIES MARES' 'NATIONAL HUNT' NOVICES' HURDLE RACE (CLASS 4)
(GBB RACE)
Following the race, the Veterinary Officer re

## Stewards' Report content — capability confirmed

Text extraction from the sampled Newton Abbot Stewards' Report succeeded without OCR.

The PDF therefore contains machine-readable text and exhibits a repeatable human-readable structure.

### Fixture-level information

The report identifies:

- racecourse;
- fixture date;
- named stewards and their roles;
- going;
- rail configuration.

### Pre-race / participation information

Separate sections were observed for:

- `Non-Runners`;
- `Withdrawals`;
- `Jockey Changes`.

The non-runner table included:

- race time;
- horse;
- trainer;
- reason.

Observed reasons included examples such as:

- going;
- bruised foot;
- dehydration;
- lameness;
- horse not qualified.

The withdrawal section provided additional contextual detail beyond a simple withdrawn status.

For example, one horse was withdrawn by the Starter because it arrived at the start wearing different headgear from that declared.

### Race-specific reports

The document contained a clearly labelled section for each of the six races.

Individual race sections contained identifiable runner-specific observations including:

- permission to wear equipment or go early to post;
- veterinary observations;
- lameness;
- lost shoes;
- jockey-reported poor performance;
- equipment/declaration irregularities;
- post-race veterinary examination;
- occasions where there was nothing to report.

These observations are linked in the document to identifiable race sections and named horses.

### Disciplinary information

A separate `Fines` section was present containing:

- person;
- race;
- horse;
- fine amount.

The sampled report recorded a £140 fine associated with the headgear incident.

### Structural assessment

The PDF is primarily a document source rather than a field-based API response, but its text is sufficiently structured to support reliable extraction experiments.

Potential analytical information includes:

- detailed non-runner reasons;
- withdrawals and their circumstances;
- jockey substitutions;
- veterinary findings;
- explanations for unusual performance;
- equipment incidents;
- race permissions;
- steward interventions;
- fines and disciplinary outcomes.

This information is materially richer than ordinary finishing-position data.

### Publication-timestamp warning

The PDF footer displayed:

`Published: Thursday, 13th August 2026, 00:56`

for a fixture held on 27 May 2026.

This timestamp corresponds closely to the time at which the report was retrieved/generated during this study.

It must therefore **not** be interpreted as the historical publication timestamp of the original stewards' report without further evidence.

### Decision

Stewards' Reports are a viable additional official BHA information source.

Their contents should be included in the eventual two-way comparison between BHA and Database v4.

Before attempting general extraction, determine whether the report-generation service exposes the same information through an underlying structured resource or whether PDF text extraction is the only practical interface.

Do not yet bulk acquire reports or design database structures.

In [14]:
# BHA Stewards' Report — backend/source-surface discovery
#
# WHAT
# ----
# Investigate whether the direct PDF report service exposes clues about an
# underlying structured source.
#
# The cell:
#
#   1. inspects metadata and printable strings inside the cached PDF;
#   2. requests the report host root;
#   3. requests /stewardsreport/ without fixture parameters;
#   4. searches returned content for API, JSON, route or application clues.
#
# WHY
# ---
# The report PDF is generated dynamically and contains useful structured-looking
# information. Before accepting PDF text extraction as the only interface,
# determine whether the report host exposes evidence of a structured backend.
#
# READS
# -----
# Existing cached Newton Abbot Stewards' Report PDF.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       stewards_report_backend_probe/
#
# EXPECTED RESULT
# ---------------
# Evidence for one of three outcomes:
#
#   - obvious structured/API surface exists;
#   - only a PDF-generating application is observable;
#   - evidence remains inconclusive.
#
# No BHA Authorization credential is used.
# No speculative bulk route enumeration is performed.

import json
import re
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

REPORT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "stewards_report_probe"
    / "newton_abbot_2026_10399_response.bin"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "stewards_report_backend_probe"
)

CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert REPORT_FILE.is_file()


print("BHA STEWARDS' REPORT — BACKEND DISCOVERY")
print("=========================================")


# ---------------------------------------------------------------------------
# 1. Inspect PDF metadata with the existing Ubuntu `pdfinfo` utility.
# ---------------------------------------------------------------------------

PDFINFO = shutil.which("pdfinfo")

print("\nPDF METADATA")
print("============")

if PDFINFO:
    result = subprocess.run(
        [PDFINFO, str(REPORT_FILE)],
        capture_output=True,
        text=True,
        check=False,
    )

    print(result.stdout.strip() or "[no metadata returned]")

    if result.stderr.strip():
        print("pdfinfo stderr:", result.stderr.strip())

else:
    print("pdfinfo not installed")


# ---------------------------------------------------------------------------
# 2. Search printable PDF strings for technical clues.
# ---------------------------------------------------------------------------
#
# This is not being used to interpret report content. We are looking only for
# implementation evidence such as generator names, URLs or route-like strings.

STRINGS = shutil.which("strings")

print("\nPDF TECHNICAL STRINGS")
print("=====================")

if STRINGS:
    result = subprocess.run(
        [STRINGS, str(REPORT_FILE)],
        capture_output=True,
        text=True,
        check=False,
    )

    technical_lines = []

    for line in result.stdout.splitlines():
        lower = line.lower()

        if any(
            token in lower
            for token in (
                "http",
                "api",
                "json",
                "swagger",
                "crate",
                "report",
                "producer",
                "creator",
                "fixture",
            )
        ):
            technical_lines.append(line.strip())

    if technical_lines:
        for line in dict.fromkeys(technical_lines):
            print(line)
    else:
        print("No obvious technical URL/API strings found.")

else:
    print("strings utility not installed")


# ---------------------------------------------------------------------------
# 3. Make two bounded requests to the report host.
# ---------------------------------------------------------------------------

PROBE_URLS = {
    "host_root": "https://crate.horseracing.software/",
    "stewardsreport_without_parameters":
        "https://crate.horseracing.software/stewardsreport/",
}


def fetch_and_cache(name, url):
    cache_path = CACHE_DIR / f"{name}.json"

    if cache_path.exists():
        envelope = json.loads(
            cache_path.read_text(encoding="utf-8")
        )
        return envelope, "cache"

    request = Request(
        url,
        headers={
            "Accept": "*/*",
            "User-Agent": "Mozilla/5.0",
            "Referer": "https://www.britishhorseracing.com/racing/results/",
        },
    )

    status = None
    final_url = None
    headers = {}
    body = b""
    error_text = None

    try:
        with urlopen(request, timeout=30) as response:
            status = response.status
            final_url = response.geturl()
            headers = dict(response.headers.items())
            body = response.read()

    except HTTPError as error:
        status = error.code
        final_url = error.geturl()
        headers = dict(error.headers.items())
        body = error.read()
        error_text = repr(error)

    except URLError as error:
        error_text = repr(error)

    envelope = {
        "request_url": url,
        "final_url": final_url,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "http_status": status,
        "headers": headers,
        "response_bytes": len(body),
        "body_text": body.decode("utf-8", errors="replace"),
        "error": error_text,
    }

    temp_path = cache_path.with_suffix(".tmp")

    temp_path.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_path.replace(cache_path)

    return envelope, "network"


for name, url in PROBE_URLS.items():
    envelope, source = fetch_and_cache(name, url)

    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)

    print("Loaded from:", source)
    print("HTTP status:", envelope["http_status"])
    print("Final URL:", envelope["final_url"])
    print("Response bytes:", envelope["response_bytes"])

    headers = envelope["headers"]

    for header_name in (
        "Content-Type",
        "Server",
        "X-Powered-By",
        "Location",
        "Content-Disposition",
    ):
        matching_value = next(
            (
                value
                for key, value in headers.items()
                if key.lower() == header_name.lower()
            ),
            None,
        )

        if matching_value is not None:
            print(f"{header_name}:", matching_value)

    text = envelope["body_text"]

    # Search only for technical/source-surface clues.
    clue_patterns = [
        r"https?://[^\s\"'<>]+",
        r"[/A-Za-z0-9_.-]*(?:api|swagger|json|report|fixture)[/A-Za-z0-9_?=&%.-]*",
    ]

    clues = []

    for pattern in clue_patterns:
        clues.extend(
            re.findall(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
        )

    clues = [
        clue
        for clue in dict.fromkeys(clues)
        if clue.strip()
    ]

    print("\nPotential technical clues:")

    if clues:
        for clue in clues[:100]:
            print(" ", clue)
    else:
        print("  NONE FOUND")

    print("\nBounded response preview:")

    # Binary PDF bodies are not useful as text previews.
    content_type = next(
        (
            value
            for key, value in headers.items()
            if key.lower() == "content-type"
        ),
        "",
    ).lower()

    if "pdf" in content_type:
        print("[PDF response — text preview suppressed]")
    else:
        compact = re.sub(
            r"\s+",
            " ",
            text,
        ).strip()

        print(compact[:5000] or "[empty response]")


print("\nEVIDENCE CACHE")
print("==============")
print(CACHE_DIR)
print("BHA Authorization sent: NO")
print("Requests made at most: 2")

BHA STEWARDS' REPORT — BACKEND DISCOVERY

PDF METADATA
Title:           NEWTON ABBOT - Stewards' Report: Wednesday 27 May 2026
Producer:        dompdf 3.1.4 + CPDF
CreationDate:    Thu Aug 13 00:56:08 2026 BST
ModDate:         Thu Aug 13 00:56:08 2026 BST
Custom Metadata: no
Metadata Stream: no
Tagged:          no
UserProperties:  no
Suspects:        no
Form:            none
JavaScript:      no
Pages:           3
Encrypted:       no
Page size:       595.28 x 841.89 pts (A4)
Page rot:        0
File size:       18696 bytes
Optimized:       no
PDF version:     1.7

PDF TECHNICAL STRINGS
/Producer (

host_root
Loaded from: network
HTTP status: 200
Final URL: https://crate.horseracing.software/
Response bytes: 88557
Content-Type: text/html; charset=UTF-8
Server: nginx/1.18.0

Potential technical clues:
  https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700&display=swap
  http://www.britishhorseracing.com
  //fonts.googleapis.com/css2?family=Nunito
  88nOItW8wH4JofYM/4dw8i/SjYMZH

## Stewards' Report backend investigation — conclusion

A bounded investigation was performed to determine whether the dynamically generated Stewards' Report PDF exposes evidence of an accessible structured backend.

### PDF generation

The sampled PDF metadata identified:

- producer: `dompdf 3.1.4 + CPDF`;
- creation time: 13 August 2026 at 00:56:08;
- modification time: the same timestamp.

This confirms that the PDF is generated dynamically when requested.

It also explains the report footer observed previously:

`Published: Thursday, 13th August 2026, 00:56`

That value is not evidence of the historical publication time of the May fixture report.

### Report host

The root of `crate.horseracing.software` returned an HTML application page.

A bounded inspection found no obvious:

- JSON endpoint;
- API route;
- Swagger/OpenAPI interface;
- structured report resource.

### Missing report parameters

Requesting `/stewardsreport/` without the required fixture parameters did not reveal a discovery interface.

The request redirected towards the BHA racing-results site.

### Decision

No accessible structured Stewards' Report backend has been demonstrated.

The PDF itself should therefore be treated as the observed source interface.

Because its text is machine-readable and contains recognisable fixture, race, runner and incident structure, this does not prevent it from being useful for research.

Do not spend further study time searching speculatively for hidden Stewards' Report API routes.

The Stewards' Report capability should now be included in the eventual two-way comparison between:

- information available in Database v4;
- information available from the wider BHA source surface.

## Additional BHA source surface — Racing Statistics

The BHA also publishes a separate Racing Statistics resource containing official aggregate racing information.

This is analytically distinct from the fixture/race/runner resources already investigated.

### Observed publication families

The Racing Statistics section includes:

- annual racing data packs;
- monthly racing data packs;
- Horse Population Reports.

Annual data packs are available across multiple years, including coverage back to 2015.

### Annual and monthly racing statistics

The published data packs contain official aggregate measures including examples such as:

- fixtures programmed and fixtures run;
- abandoned and additional fixtures;
- race counts by racing code/type;
- entries;
- declarations;
- eliminations;
- non-runners;
- average field sizes;
- divided races;
- handicap and weight-for-age race proportions;
- race values and prize-money measures;
- race punctuality and delay measures;
- race clashes;
- other industry-level racing KPIs.

Some of these measures may be reconstructible from individual race records.

Others may represent information or BHA-defined concepts not directly available from ordinary result data.

### Horse Population Reports

The BHA also publishes separate Horse Population Reports.

These provide official population-level information derived from BHA/Weatherbys administrative sources rather than merely from horses appearing in race results.

Observed subject areas include:

- horses in training;
- age distributions;
- racing-code populations;
- rating distributions;
- historical population comparisons.

These reports may therefore measure populations that cannot be reproduced simply by counting horses that raced.

### Analytical role

This source should not be treated as a replacement for race-level data.

Its main value to Inside Rails is likely to be:

1. **aggregate validation**
   - test whether Inside Rails can reproduce official BHA totals when equivalent definitions are used;

2. **definition evidence**
   - identify exactly how BHA defines concepts such as fixtures run, abandonments, additional fixtures, eliminations and other published measures;

3. **coverage diagnostics**
   - identify discrepancies between official aggregate totals and populations reconstructed from Database v4 or BHA race-level resources;

4. **additional analytical information**
   - retain useful official measures that cannot reasonably be reconstructed from individual race/result records;

5. **research context**
   - provide official historical industry trends against which Inside Rails findings can be interpreted.

### Important comparison principle

A disagreement between an Inside Rails reconstruction and a published BHA total does not automatically mean either source is wrong.

Possible explanations include:

- different population definitions;
- different timing/snapshot rules;
- missing records;
- different treatment of abandoned or additional fixtures;
- different racing-code classifications;
- information available administratively to BHA but absent from result data.

Any comparison must therefore establish the BHA statistic's definition before treating it as a validation target.

## Source-surface decision

The wider BHA evidence surface now includes at least:

- fixture-list resources;
- fixture detail;
- fixture conditions and condition history;
- fixture officials;
- race detail;
- nominations;
- entries;
- transferred/related-entry resources;
- balloted-entry resources;
- results;
- Stewards' Reports;
- annual Racing Statistics data packs;
- monthly Racing Statistics data packs;
- Horse Population Reports.

The next comparison with Database v4 should therefore be two-way and concept-based:

> **What useful information is available from each source, what overlaps, what exists only in one source, and which source provides the strongest evidence for each concept?**

Do not assume either Database v4 or the BHA source surface is globally superior.

## Phase 3 — Site-wide BHA public-source inventory

The earlier frontend investigation concentrated on fixture and race resources.

A broader search of the public BHA website demonstrates that this is not the complete BHA information surface.

### Additional public data-bearing surfaces identified

#### Official ratings

The BHA maintains a searchable official-ratings database.

The public interface supports downloadable Excel exports for:

- the full published ratings list;
- weekly rating changes;
- latest performance figures.

This is potentially important because performance figures and rating histories are analytically distinct from ordinary race-result fields.

#### Horse database

Public horse profiles expose information including:

- breeding/pedigree;
- owner;
- trainer;
- breeder;
- career performance statistics;
- published rating history;
- future entries;
- Great Britain training history;
- recent performance history;
- associated Stewards' Reports and disciplinary/appeal material.

#### Jockey information

Public jockey resources expose:

- licensed-jockey search;
- championship statistics;
- season/career information;
- Stewards' Report and appeal links;
- British winners and runs since 1 January 1995.

#### Trainer information

Public trainer resources expose:

- trainer search;
- performance information;
- championship statistics;
- trainer location/contact information;
- yard size;
- trainer-map information;
- quarterly non-runner statistics based on declarations.

#### Owner information

The public Owners Championship exposes measures including:

- wins;
- runs;
- prize money;
- leading earner.

#### Racecourse resources

Separate BHA racecourse resources include:

- Going Stick Average Readings;
- Going Stick Archive by Racecourse;
- historical changes to Jump race-distance measurements;
- starts and remeasurement data;
- track-design and racecourse technical documentation.

The Going Stick archive is a particularly high-priority discovery because the fixture `/going` resource has already demonstrated valuable historical conditions and GoingStick observations.

#### Fixture-list downloads

The BHA publishes official full-year Fixture Lists in downloadable formats including Excel and PDF.

These provide another official fixture source distinct from the live fixture API.

#### Racing Statistics

In addition to annual/monthly data packs and Horse Population Reports, the BHA publishes dedicated race off-times data including:

- racecourse-level punctuality;
- overall racing punctuality;
- stated primary reasons for delays.

#### Claiming races

The BHA provides a searchable public record relating to claimed horses and claimants.

#### Disciplinary and handicapping material

The website contains searchable:

- disciplinary decisions;
- appeal hearings;
- handicapping appeal decisions.

These may occasionally provide explanatory evidence for individual horses, participants or rating decisions.

### BHA resource index

The BHA's own site search currently indexes hundreds of downloadable resources.

This provides a useful discovery mechanism for finding datasets and historical technical publications that are not obvious from the main racing navigation.

## Revised source-discovery decision

Do not begin the Database v4 comparison yet.

First complete a bounded site-wide BHA source inventory.

The next technical investigation should determine whether the public dynamic interfaces for:

- horses;
- jockeys;
- trainers;
- owners;
- ratings;
- racecourses;

use additional structured frontend resources beyond the fixture/race routes already discovered.

Also inspect the published Going Stick archive as a priority.

The objective remains:

> **Know what useful official BHA information is publicly available before deciding what role it should play in Inside Rails.**

Do not yet bulk acquire data or design Database v5.

In [16]:
# Going Stick Archive — first bounded technical/content probe
#
# WHAT
# ----
# Inspect the public TurfTrax Going Stick archive resource linked from the BHA
# racecourse material for Newton Abbot.
#
# The known archive locator for the sampled course is:
#
#   https://maps.turftrax.co.uk/iframe/api_goingstickarchive.asp?courseid=37
#
# The cell determines:
#
#   - HTTP status;
#   - content type;
#   - response format;
#   - returned fields/records;
#   - apparent historical date coverage;
#   - whether individual raceday GoingStick readings are exposed.
#
# WHY
# ---
# The BHA fixture `/going` resource has already shown that GoingStick values can
# be preserved historically within a meeting.
#
# The separate BHA-linked TurfTrax archive may expose a much longer historical
# series by racecourse.
#
# Before considering wider acquisition, we need to know whether this archive is:
#
#   - a genuine structured historical dataset;
#   - a presentation-only summary;
#   - or something else.
#
# READS
# -----
# One public TurfTrax archive endpoint discovered through BHA racecourse
# resources.
#
# WRITES
# ------
# One ignored research-cache file under:
#
#   data/cache/bha_official_source_feasibility/
#       going_stick_archive_probe/
#
# The exact response is cached so this discovery request does not need to be
# repeated.
#
# EXPECTED RESULT
# ---------------
# A bounded structural description of the Newton Abbot archive showing:
#
#   - whether it is JSON or another format;
#   - the fields/records it exposes;
#   - first/last observed records where possible;
#   - apparent date coverage.
#
# This cell does NOT:
#
#   - request any other racecourse;
#   - bulk acquire archive data;
#   - infer that TurfTrax `courseid=37` is formally identical to BHA `courseId=37`;
#   - assign governed meanings to any returned fields.
#
# No BHA Authorization credential is sent.

from datetime import datetime, timezone
import json
import re
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit project and research-cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on the notebook's current working directory. The study should
# behave the same way regardless of where Jupyter was launched.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "going_stick_archive_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------------
# 2. Define exactly one bounded archive request.
# ---------------------------------------------------------------------------
#
# Newton Abbot is deliberately reused because it is already the controlled
# modern sample for the fixture-going investigation.
#
# The numeric `courseid=37` is observed in the archive locator and happens to
# match the BHA courseId seen for Newton Abbot in our sample. That apparent
# correspondence is NOT treated here as proven identifier equivalence.

ARCHIVE_URL = (
    "https://maps.turftrax.co.uk/iframe/"
    "api_goingstickarchive.asp?courseid=37"
)

CACHE_FILE = (
    CACHE_DIR
    / "newton_abbot_courseid_37.json"
)


# ---------------------------------------------------------------------------
# 3. Reuse cached evidence where available.
# ---------------------------------------------------------------------------
#
# External-source discovery should not repeatedly hit the same endpoint when an
# exact cached response already exists.

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(encoding="utf-8")
    )

    # Protect against accidentally reusing a cache generated from a different
    # request.
    assert envelope["request_url"] == ARCHIVE_URL

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 4. Make exactly one public archive request.
    # -----------------------------------------------------------------------
    #
    # No BHA API credential is involved. The request merely follows the public
    # TurfTrax archive resource discovered through BHA racecourse material.

    request = Request(
        ARCHIVE_URL,
        headers={
            "Accept": "*/*",
            "User-Agent": "Mozilla/5.0",
            "Referer": (
                "https://maps.turftrax.co.uk/"
                "iframe/goingstickarchiveindex.asp"
            ),
        },
    )

    status = None
    final_url = None
    content_type = None
    response_bytes = b""
    error_text = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:
            status = response.status
            final_url = response.geturl()
            content_type = response.headers.get(
                "Content-Type"
            )
            response_bytes = response.read()

    except HTTPError as error:
        # An HTTP error is still source evidence. Preserve the returned body
        # rather than allowing the notebook to discard it.
        status = error.code
        final_url = error.geturl()
        content_type = error.headers.get(
            "Content-Type"
        )
        response_bytes = error.read()
        error_text = repr(error)

    except URLError as error:
        # Transport failures are also cached so a failed request is not silently
        # mistaken for an empty archive.
        error_text = repr(error)


    # -----------------------------------------------------------------------
    # 5. Decode conservatively and test for JSON.
    # -----------------------------------------------------------------------
    #
    # The endpoint name contains `api`, but that is not evidence that the
    # response is JSON. Detect the actual format from the returned content.

    response_text = response_bytes.decode(
        "utf-8",
        errors="replace",
    )

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(error)


    # -----------------------------------------------------------------------
    # 6. Persist request provenance and exact response evidence.
    # -----------------------------------------------------------------------
    #
    # Store the raw decoded response alongside any parsed representation.
    # This preserves our ability to revisit parsing decisions later.

    envelope = {
        "provider": "TurfTrax",
        "discovered_via": (
            "BHA Racecourse - Going Stick Archive"
        ),
        "course_label": "Newton Abbot",
        "courseid_parameter": 37,
        "request_url": ARCHIVE_URL,
        "final_url": final_url,
        "retrieved_at_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
        "http_status": status,
        "content_type": content_type,
        "response_bytes": len(response_bytes),
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "error": error_text,
    }

    # Use an atomic replacement so an interrupted write cannot leave a partial
    # cache that later appears valid.
    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 7. Report transport-level observations first.
# ---------------------------------------------------------------------------
#
# Keep acquisition evidence separate from interpretation of the returned data.

print("GOING STICK ARCHIVE — NEWTON ABBOT")
print("===================================")
print("Loaded from:", source)
print("HTTP status:", envelope["http_status"])
print("Final URL:", envelope["final_url"])
print("Content-Type:", envelope["content_type"])
print("Response bytes:", envelope["response_bytes"])


# ---------------------------------------------------------------------------
# 8. Inspect JSON structurally if the response genuinely parsed as JSON.
# ---------------------------------------------------------------------------
#
# Do not assume a specific schema. Describe the returned structure first.

payload = envelope["parsed_json"]

if payload is not None:
    print("\nRESPONSE FORMAT")
    print("===============")
    print("JSON")
    print(
        "Top-level type:",
        type(payload).__name__,
    )

    if isinstance(payload, dict):
        print("Top-level keys:")
        print(
            sorted(payload.keys())
        )

        # Examine each top-level member independently because different archive
        # versions may expose metadata and records in different collections.
        for key, value in payload.items():

            if isinstance(value, list):
                print(
                    f"\n{key}: list[{len(value)}]"
                )

                dictionary_rows = [
                    row
                    for row in value
                    if isinstance(row, dict)
                ]

                if dictionary_rows:
                    # Use the union rather than only the first row so nullable or
                    # conditionally present fields are not missed.
                    fields = sorted(
                        {
                            field
                            for row in dictionary_rows
                            for field in row.keys()
                        }
                    )

                    print("  Union of fields:")
                    print(
                        " ",
                        fields,
                    )

                    print("  First row:")
                    print(
                        json.dumps(
                            dictionary_rows[0],
                            indent=2,
                            ensure_ascii=False,
                        )[:5000]
                    )

                    print("  Last row:")
                    print(
                        json.dumps(
                            dictionary_rows[-1],
                            indent=2,
                            ensure_ascii=False,
                        )[:5000]
                    )

            elif isinstance(value, dict):
                print(
                    f"\n{key}: dict"
                )
                print(
                    "  Keys:",
                    sorted(value.keys()),
                )

            else:
                print(
                    f"\n{key}: {value!r}"
                )


    elif isinstance(payload, list):
        print(
            "Rows:",
            len(payload),
        )

        dictionary_rows = [
            row
            for row in payload
            if isinstance(row, dict)
        ]

        if dictionary_rows:
            fields = sorted(
                {
                    field
                    for row in dictionary_rows
                    for field in row.keys()
                }
            )

            print(
                "Union of row fields:"
            )
            print(fields)

            print("\nFirst row:")
            print(
                json.dumps(
                    dictionary_rows[0],
                    indent=2,
                    ensure_ascii=False,
                )[:5000]
            )

            print("\nLast row:")
            print(
                json.dumps(
                    dictionary_rows[-1],
                    indent=2,
                    ensure_ascii=False,
                )[:5000]
            )


# ---------------------------------------------------------------------------
# 9. If the response is not JSON, inspect it as presentation/text evidence.
# ---------------------------------------------------------------------------
#
# A non-JSON response may still contain a useful table or historical series.
# Strip presentation markup only for a bounded readable preview; preserve the
# original response unchanged in the cache.

else:
    text = envelope["response_text"]

    print("\nRESPONSE FORMAT")
    print("===============")
    print("Non-JSON")

    visible = re.sub(
        r"<script\b[^>]*>.*?</script>",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    visible = re.sub(
        r"<style\b[^>]*>.*?</style>",
        " ",
        visible,
        flags=re.IGNORECASE | re.DOTALL,
    )

    visible = re.sub(
        r"<[^>]+>",
        "\n",
        visible,
    )

    visible = re.sub(
        r"\n\s*\n+",
        "\n",
        visible,
    ).strip()

    print("\nBounded readable preview:")
    print(
        visible[:12_000]
    )


    # -----------------------------------------------------------------------
    # 10. Look only for date-like values as a first historical-depth clue.
    # -----------------------------------------------------------------------
    #
    # This does NOT yet establish which dates represent readings, meetings, or
    # another concept. It merely tells us whether historical dates are present.

    date_candidates = sorted(
        set(
            re.findall(
                r"\b(?:"
                r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}"
                r"|"
                r"\d{4}-\d{2}-\d{2}"
                r")\b",
                text,
            )
        )
    )

    print("\nDATE-LIKE VALUES FOUND")
    print("======================")
    print(
        "Distinct values:",
        len(date_candidates),
    )

    if date_candidates:
        print(
            "First examples:",
            date_candidates[:10],
        )
        print(
            "Last examples:",
            date_candidates[-10:],
        )


# ---------------------------------------------------------------------------
# 11. Record provenance and bounded scope explicitly.
# ---------------------------------------------------------------------------

print("\nPROVENANCE")
print("==========")
print("Archive URL:", ARCHIVE_URL)
print("Cache:", CACHE_FILE)
print("BHA Authorization sent: NO")
print("Courses requested: 1")
print("Bulk acquisition performed: NO")

GOING STICK ARCHIVE — NEWTON ABBOT
Loaded from: cache
HTTP status: 200
Final URL: https://maps.turftrax.co.uk/iframe/api_goingstickarchive.asp?courseid=37
Content-Type: text/html
Response bytes: 15107

RESPONSE FORMAT
Non-JSON

Bounded readable preview:
TurfTrax Course Services - Course Going Stick Reading Archive
Newton Abbot GoingStick Archive
Website : 
-->
2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
meeting date
report date
official going
goingstick index
Sunday
19 July
Sunday
19 July at 13:18
Good to Firm, Good (in places)
6.7 on 19-07-2026 at 07:45
Sunday
19 July
Sunday
19 July at 07:54
Good, Good to Firm (in places)
6.7 on 19-07-2026 at 07:45
Monday
13 July
Monday
13 July at 13:48
Good to Firm, Good (in places)
Warm day forecast with strong breeze
6.7 on 13-07-2026 at 07:45
Monday
13 July
Monday
13 July at 07:48
Good, Good to Firm (in places)
Warm day forecast with strong breeze
6.7 on 13-07-2026 at 07:45
Friday
03 July
Friday
03 July at 08:00
Good, Good to Firm (in places

## Going Stick Archive — capability confirmed

The BHA-linked TurfTrax Going Stick Archive is a substantive historical data source.

A bounded probe of Newton Abbot returned a human-readable historical archive containing raceday GoingStick reports.

### Observed archive structure

The archive exposes fields corresponding to:

- meeting date;
- report date/time;
- official going;
- GoingStick index.

The report content can also include free-text context concerning:

- weather;
- rainfall;
- watering;
- expected drying or ground change;
- maintenance/activity affecting the racing surface.

### Multiple observations within a meeting

The archive can preserve more than one report for the same fixture date.

Observed examples include:

- two reports for 19 July 2026;
- multiple reports for 4 April 2026.

Therefore the archive is not merely a one-row-per-meeting final-value dataset.

It preserves at least some intraday evolution of official going information.

### Relationship with the fixture-going resource

For the already-studied Newton Abbot meeting on 27 May 2026, the archive contains:

- official going: `Good, Good to Firm (in places)`;
- GoingStick: `5.7`;
- GoingStick observation time: `08:00`;
- contextual note that the home straight would be watered that morning.

These observations correspond closely to information already observed in the BHA fixture `/going` resource for the same meeting.

This provides evidence that the two public source surfaces represent related official going information.

Their exact provenance relationship should not yet be assumed.

### Historical navigation

The archive interface visibly offers year selections from:

- 2018;
- through to 2026.

This suggests materially deeper historical GoingStick availability than the single modern fixture sample investigated so far.

However, the presence of a year selector is not sufficient evidence that every year contains records.

### Analytical value

The archive may provide an important historical source for research into:

- GoingStick levels;
- changes in official going during raceday;
- watering;
- weather/ground commentary;
- differences between subjective going descriptions and instrument readings;
- racecourse-specific historical ground behaviour.

It may also provide useful evidence for the previously proposed Inside Rails study comparing weather and going development.

## Decision

Before requesting older archive years, inspect the cached Newton Abbot HTML to determine exactly how:

- racecourse selection;
- year selection;
- archive navigation;

are represented technically.

Do not guess query parameters and do not bulk acquire other years or racecourses yet.

In [18]:
# Going Stick Archive — inspect cached navigation mechanics
#
# WHAT
# ----
# Inspect the already-cached Newton Abbot Going Stick Archive HTML to determine
# how the archive represents:
#
#   - racecourse selection;
#   - year selection;
#   - forms;
#   - links;
#   - query parameters;
#   - script-generated navigation.
#
# WHY
# ---
# The archive visibly exposes historical years from 2018 through 2026.
#
# Before making any further external requests, establish the actual technical
# navigation mechanism from cached evidence rather than guessing parameter names
# or URL structures.
#
# READS
# -----
# Existing ignored research cache only:
#
#   data/cache/bha_official_source_feasibility/
#       going_stick_archive_probe/
#       newton_abbot_courseid_37.json
#
# WRITES
# ------
# None.
#
# EXPECTED RESULT
# ---------------
# A structural inventory showing:
#
#   - all HTML forms and their methods/actions;
#   - all select controls and available options;
#   - all input names/values;
#   - links containing archive/course/year clues;
#   - JavaScript fragments that reference navigation parameters;
#   - any explicit evidence for how older years are requested.
#
# This cell makes NO network request and does NOT infer parameter semantics from
# names alone.

import html
import json
import re
from pathlib import Path
from urllib.parse import parse_qs, urlparse


# ---------------------------------------------------------------------------
# 1. Load the exact cached archive response from the previous probe.
# ---------------------------------------------------------------------------
#
# Keep the repository root explicit so the notebook does not depend on where
# Jupyter happens to have been launched.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_FILE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "going_stick_archive_probe"
    / "newton_abbot_courseid_37.json"
)

assert CACHE_FILE.is_file(), (
    f"Going Stick archive cache not found: {CACHE_FILE}"
)

envelope = json.loads(
    CACHE_FILE.read_text(encoding="utf-8")
)

assert envelope["http_status"] == 200

archive_html = envelope["response_text"]

assert archive_html.strip(), (
    "Cached archive response was empty."
)


print("GOING STICK ARCHIVE — NAVIGATION INSPECTION")
print("===========================================")
print("Cached request:", envelope["request_url"])
print("Response characters:", len(archive_html))


# ---------------------------------------------------------------------------
# 2. Inspect every HTML form.
# ---------------------------------------------------------------------------
#
# Forms are the strongest evidence for how the page expects navigation
# parameters to be submitted.

form_matches = list(
    re.finditer(
        r"<form\b([^>]*)>(.*?)</form>",
        archive_html,
        flags=re.IGNORECASE | re.DOTALL,
    )
)

print("\nFORMS")
print("=====")
print("Count:", len(form_matches))

for index, match in enumerate(form_matches, start=1):
    form_attrs = match.group(1)
    form_body = match.group(2)

    method_match = re.search(
        r"""\bmethod\s*=\s*["']?([^"'\s>]+)""",
        form_attrs,
        flags=re.IGNORECASE,
    )

    action_match = re.search(
        r"""\baction\s*=\s*["']([^"']*)["']""",
        form_attrs,
        flags=re.IGNORECASE,
    )

    method = (
        method_match.group(1)
        if method_match
        else None
    )

    action = (
        html.unescape(action_match.group(1))
        if action_match
        else None
    )

    print(f"\nForm {index}")
    print("-" * 40)
    print("Method:", method)
    print("Action:", action)

    # Preserve the names and values of hidden/text inputs because these often
    # carry course/year context that is not obvious from visible controls.
    inputs = re.findall(
        r"<input\b([^>]*)>",
        form_body,
        flags=re.IGNORECASE | re.DOTALL,
    )

    print("Inputs:")

    if not inputs:
        print("  NONE")

    for input_attrs in inputs:
        name_match = re.search(
            r"""\bname\s*=\s*["']([^"']+)["']""",
            input_attrs,
            flags=re.IGNORECASE,
        )

        value_match = re.search(
            r"""\bvalue\s*=\s*["']([^"']*)["']""",
            input_attrs,
            flags=re.IGNORECASE,
        )

        type_match = re.search(
            r"""\btype\s*=\s*["']([^"']+)["']""",
            input_attrs,
            flags=re.IGNORECASE,
        )

        print(
            {
                "type": (
                    type_match.group(1)
                    if type_match
                    else None
                ),
                "name": (
                    name_match.group(1)
                    if name_match
                    else None
                ),
                "value": (
                    html.unescape(value_match.group(1))
                    if value_match
                    else None
                ),
            }
        )


# ---------------------------------------------------------------------------
# 3. Inspect every SELECT control and its OPTION values.
# ---------------------------------------------------------------------------
#
# This should tell us whether the visible 2018–2026 choices are actual submitted
# values or merely presentation text.

select_matches = list(
    re.finditer(
        r"<select\b([^>]*)>(.*?)</select>",
        archive_html,
        flags=re.IGNORECASE | re.DOTALL,
    )
)

print("\nSELECT CONTROLS")
print("===============")
print("Count:", len(select_matches))

for index, match in enumerate(select_matches, start=1):
    select_attrs = match.group(1)
    select_body = match.group(2)

    name_match = re.search(
        r"""\bname\s*=\s*["']([^"']+)["']""",
        select_attrs,
        flags=re.IGNORECASE,
    )

    id_match = re.search(
        r"""\bid\s*=\s*["']([^"']+)["']""",
        select_attrs,
        flags=re.IGNORECASE,
    )

    onchange_match = re.search(
        r"""\bonchange\s*=\s*["']([^"']+)["']""",
        select_attrs,
        flags=re.IGNORECASE,
    )

    print(f"\nSelect {index}")
    print("-" * 40)
    print(
        "name:",
        name_match.group(1)
        if name_match
        else None,
    )
    print(
        "id:",
        id_match.group(1)
        if id_match
        else None,
    )
    print(
        "onchange:",
        html.unescape(onchange_match.group(1))
        if onchange_match
        else None,
    )

    option_matches = re.findall(
        r"<option\b([^>]*)>(.*?)</option>",
        select_body,
        flags=re.IGNORECASE | re.DOTALL,
    )

    print("Options:")

    for option_attrs, option_text in option_matches:
        value_match = re.search(
            r"""\bvalue\s*=\s*["']([^"']*)["']""",
            option_attrs,
            flags=re.IGNORECASE,
        )

        selected = bool(
            re.search(
                r"\bselected\b",
                option_attrs,
                flags=re.IGNORECASE,
            )
        )

        visible_text = re.sub(
            r"<[^>]+>",
            " ",
            option_text,
        )

        visible_text = re.sub(
            r"\s+",
            " ",
            html.unescape(visible_text),
        ).strip()

        print(
            {
                "value": (
                    html.unescape(value_match.group(1))
                    if value_match
                    else None
                ),
                "text": visible_text,
                "selected": selected,
            }
        )


# ---------------------------------------------------------------------------
# 4. Inventory archive-related links and their query parameters.
# ---------------------------------------------------------------------------
#
# Links provide direct evidence for parameter names and combinations already
# generated by the application.

hrefs = [
    html.unescape(value)
    for value in re.findall(
        r"""href\s*=\s*["']([^"']+)["']""",
        archive_html,
        flags=re.IGNORECASE,
    )
]

interesting_links = []

for href in hrefs:
    lower = href.lower()

    if any(
        token in lower
        for token in (
            "going",
            "stick",
            "archive",
            "course",
            "year",
            "asp",
        )
    ):
        interesting_links.append(href)


print("\nARCHIVE-RELATED LINKS")
print("=====================")
print("Distinct links:", len(set(interesting_links)))

for href in dict.fromkeys(interesting_links):
    print("\nLink:", href)

    parsed = urlparse(href)

    if parsed.query:
        print(
            "Query parameters:",
            parse_qs(parsed.query),
        )


# ---------------------------------------------------------------------------
# 5. Search scripts and inline event handlers for navigation logic.
# ---------------------------------------------------------------------------
#
# The archive may generate URLs in JavaScript rather than ordinary anchor tags.
# Show only bounded fragments containing likely navigation concepts.

script_blocks = re.findall(
    r"<script\b[^>]*>(.*?)</script>",
    archive_html,
    flags=re.IGNORECASE | re.DOTALL,
)

script_text = "\n".join(script_blocks)

technical_lines = []

for line in script_text.splitlines():
    compact = line.strip()

    if not compact:
        continue

    lower = compact.lower()

    if any(
        token in lower
        for token in (
            "courseid",
            "year",
            "archive",
            "goingstick",
            "location",
            "href",
            "submit",
        )
    ):
        technical_lines.append(compact)


print("\nNAVIGATION-RELATED JAVASCRIPT")
print("=============================")

if technical_lines:
    for line in dict.fromkeys(technical_lines):
        print(line[:3000])
else:
    print("No obvious navigation-related JavaScript found.")


# ---------------------------------------------------------------------------
# 6. Search the raw HTML for explicit parameter assignments.
# ---------------------------------------------------------------------------
#
# This is deliberately redundant with the DOM-like checks above. It protects
# against unusual markup where the navigation mechanism is embedded inside
# script strings or malformed HTML.

parameter_clues = sorted(
    set(
        re.findall(
            r"""(?:courseid|course|year|yr|season)\s*=\s*["']?([A-Za-z0-9_-]+)""",
            archive_html,
            flags=re.IGNORECASE,
        )
    )
)

print("\nRAW PARAMETER-VALUE CLUES")
print("=========================")
print(parameter_clues)


# ---------------------------------------------------------------------------
# 7. Show small source excerpts around each visible historical year.
# ---------------------------------------------------------------------------
#
# We know the page visibly presented years 2018–2026. Seeing their surrounding
# markup may reveal whether they are links, option values or JavaScript calls.

print("\nYEAR MARKUP EXCERPTS")
print("====================")

for year in range(2018, 2027):
    match = re.search(
        rf".{{0,250}}\b{year}\b.{{0,250}}",
        archive_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    print(f"\n{year}")
    print("-" * 20)

    if match:
        excerpt = re.sub(
            r"\s+",
            " ",
            match.group(0),
        ).strip()

        print(
            html.unescape(excerpt)
        )
    else:
        print("Not found")


# ---------------------------------------------------------------------------
# 8. State the evidence boundary explicitly.
# ---------------------------------------------------------------------------

print("\nPROVENANCE")
print("==========")
print("Cached response:", CACHE_FILE)
print("Network requests made by this cell: 0")
print("Writes made by this cell: NONE")
print("Older years requested: NO")
print("Other racecourses requested: NO")

GOING STICK ARCHIVE — NAVIGATION INSPECTION
Cached request: https://maps.turftrax.co.uk/iframe/api_goingstickarchive.asp?courseid=37
Response characters: 15107

FORMS
=====
Count: 0

SELECT CONTROLS
Count: 0

ARCHIVE-RELATED LINKS
Distinct links: 10

Link: ?courseid=37&year=2026
Query parameters: {'courseid': ['37'], 'year': ['2026']}

Link: ?courseid=37&year=2025
Query parameters: {'courseid': ['37'], 'year': ['2025']}

Link: ?courseid=37&year=2024
Query parameters: {'courseid': ['37'], 'year': ['2024']}

Link: ?courseid=37&year=2023
Query parameters: {'courseid': ['37'], 'year': ['2023']}

Link: ?courseid=37&year=2022
Query parameters: {'courseid': ['37'], 'year': ['2022']}

Link: ?courseid=37&year=2021
Query parameters: {'courseid': ['37'], 'year': ['2021']}

Link: ?courseid=37&year=2020
Query parameters: {'courseid': ['37'], 'year': ['2020']}

Link: ?courseid=37&year=2019
Query parameters: {'courseid': ['37'], 'year': ['2019']}

Link: ?courseid=37&year=2018
Query parameters: {'cour

## Going Stick Archive — year-navigation evidence

Inspection of the cached Newton Abbot archive HTML established the archive's historical navigation mechanism.

### Observed query structure

Historical archive pages are linked using:

`?courseid={courseid}&year={year}`

For the Newton Abbot sample, the page generated explicit links for:

- 2026;
- 2025;
- 2024;
- 2023;
- 2022;
- 2021;
- 2020;
- 2019;
- 2018;
- 2017.

This is direct evidence that the public archive interface supports year-specific requests for Newton Abbot across at least 2017–2026.

### Important evidence boundary

The HTML metadata also contains a copyright string listing years from 2003 onward.

That metadata is not evidence that GoingStick observations exist for those years.

Only years exposed through actual archive navigation should currently be treated as demonstrated historical archive years.

### Technical observation

No forms, select controls or JavaScript-generated navigation were required.

Course and year are passed directly as ordinary query parameters:

- `courseid`;
- `year`.

This gives us a stable observed mechanism for making a bounded historical test without guessing routes.

## Decision

Request exactly one older demonstrated year for Newton Abbot:

`2017`

The purpose is to establish whether the oldest year exposed by the current navigation actually contains historical GoingStick records and whether its structure is materially comparable with 2026.

Do not yet request every year or another racecourse.

In [19]:
# Going Stick Archive — oldest exposed year probe: Newton Abbot 2017
#
# WHAT
# ----
# Request exactly one older year from the public TurfTrax Going Stick Archive:
#
#   Newton Abbot
#   courseid = 37
#   year     = 2017
#
# Then inspect whether the returned page contains actual historical GoingStick
# observations and whether its presentation structure is materially comparable
# with the already-observed 2026 archive.
#
# WHY
# ---
# Cached navigation evidence showed explicit archive links for Newton Abbot from
# 2017 through 2026.
#
# The existence of a year link proves that the interface accepts the year, but
# not that the year necessarily contains useful historical observations.
#
# Before requesting an entire historical series, test the oldest currently
# exposed year only.
#
# READS
# -----
# One public TurfTrax archive page:
#
#   https://maps.turftrax.co.uk/iframe/
#       api_goingstickarchive.asp?courseid=37&year=2017
#
# WRITES
# ------
# One ignored research-cache file under:
#
#   data/cache/bha_official_source_feasibility/
#       going_stick_archive_probe/
#
# The exact returned HTML is preserved alongside request provenance.
#
# EXPECTED RESULT
# ---------------
# Evidence showing:
#
#   - HTTP/result status;
#   - whether 2017 returns an archive page;
#   - whether historical meeting/report records are present;
#   - observed GoingStick values/timestamps where present;
#   - a bounded readable preview;
#   - whether the page retains the same visible field labels seen in 2026.
#
# This cell does NOT:
#
#   - request 2018–2025;
#   - request another racecourse;
#   - infer complete archive coverage from one course;
#   - bulk acquire historical data.
#
# No BHA Authorization credential is sent.

from datetime import datetime, timezone
import html
import json
import re
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit project and cache locations.
# ---------------------------------------------------------------------------
#
# Keep path resolution independent of the notebook's current working directory.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "going_stick_archive_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------------
# 2. Define exactly one demonstrated historical archive request.
# ---------------------------------------------------------------------------
#
# The `year=2017` parameter is not guessed. It was discovered as an explicit
# archive link in the cached Newton Abbot 2026 HTML.

ARCHIVE_URL = (
    "https://maps.turftrax.co.uk/iframe/"
    "api_goingstickarchive.asp?courseid=37&year=2017"
)

CACHE_FILE = (
    CACHE_DIR
    / "newton_abbot_courseid_37_year_2017.json"
)


# ---------------------------------------------------------------------------
# 3. Reuse exact cached evidence if this request has already been made.
# ---------------------------------------------------------------------------

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(encoding="utf-8")
    )

    assert envelope["request_url"] == ARCHIVE_URL

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 4. Make the single bounded public request.
    # -----------------------------------------------------------------------
    #
    # Preserve HTTP errors and returned bodies as observations rather than
    # allowing an unsuccessful year request to disappear as a Python exception.

    request = Request(
        ARCHIVE_URL,
        headers={
            "Accept": "text/html,*/*",
            "User-Agent": "Mozilla/5.0",
            "Referer": (
                "https://maps.turftrax.co.uk/"
                "iframe/api_goingstickarchive.asp?courseid=37"
            ),
        },
    )

    status = None
    final_url = None
    content_type = None
    response_bytes = b""
    error_text = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:
            status = response.status
            final_url = response.geturl()
            content_type = response.headers.get(
                "Content-Type"
            )
            response_bytes = response.read()

    except HTTPError as error:
        status = error.code
        final_url = error.geturl()
        content_type = error.headers.get(
            "Content-Type"
        )
        response_bytes = error.read()
        error_text = repr(error)

    except URLError as error:
        error_text = repr(error)


    # -----------------------------------------------------------------------
    # 5. Decode and preserve the exact returned HTML.
    # -----------------------------------------------------------------------

    response_text = response_bytes.decode(
        "utf-8",
        errors="replace",
    )

    envelope = {
        "provider": "TurfTrax",
        "discovered_via": (
            "BHA Racecourse - Going Stick Archive"
        ),
        "course_label": "Newton Abbot",
        "courseid_parameter": 37,
        "year_parameter": 2017,
        "request_url": ARCHIVE_URL,
        "final_url": final_url,
        "retrieved_at_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
        "http_status": status,
        "content_type": content_type,
        "response_bytes": len(response_bytes),
        "response_text": response_text,
        "error": error_text,
    }

    # Atomic replacement prevents an interrupted write from becoming a later
    # false-positive cache hit.
    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 6. Report transport evidence separately from content interpretation.
# ---------------------------------------------------------------------------

print("GOING STICK ARCHIVE — NEWTON ABBOT 2017")
print("========================================")
print("Loaded from:", source)
print("HTTP status:", envelope["http_status"])
print("Final URL:", envelope["final_url"])
print("Content-Type:", envelope["content_type"])
print("Response bytes:", envelope["response_bytes"])

if envelope["error"]:
    print("HTTP/transport observation:", envelope["error"])


archive_html = envelope["response_text"]


# ---------------------------------------------------------------------------
# 7. Convert presentation HTML into a bounded readable representation.
# ---------------------------------------------------------------------------
#
# Preserve the original HTML in cache. This stripped version exists only to
# understand whether historical archive records are visibly present.

visible = re.sub(
    r"<script\b[^>]*>.*?</script>",
    " ",
    archive_html,
    flags=re.IGNORECASE | re.DOTALL,
)

visible = re.sub(
    r"<style\b[^>]*>.*?</style>",
    " ",
    visible,
    flags=re.IGNORECASE | re.DOTALL,
)

visible = re.sub(
    r"<[^>]+>",
    "\n",
    visible,
)

visible = html.unescape(
    visible
)

visible = re.sub(
    r"\n[ \t]*\n+",
    "\n",
    visible,
)

visible = "\n".join(
    line.strip()
    for line in visible.splitlines()
    if line.strip()
)


# ---------------------------------------------------------------------------
# 8. Test for the same visible archive schema observed in 2026.
# ---------------------------------------------------------------------------
#
# These are presentation labels, not governed field definitions. Their presence
# merely tells us whether the older page appears structurally comparable.

expected_labels = [
    "meeting date",
    "report date",
    "official going",
    "goingstick index",
]

print("\nVISIBLE ARCHIVE LABELS")
print("======================")

for label in expected_labels:
    present = (
        label.lower()
        in visible.lower()
    )

    print(
        f"{label}:",
        "YES" if present else "NO",
    )


# ---------------------------------------------------------------------------
# 9. Extract first-pass historical clues without assigning final semantics.
# ---------------------------------------------------------------------------
#
# We want to know whether this page contains actual observations, particularly
# GoingStick values with explicit dates/times.
#
# Do not attempt full table parsing yet.

goingstick_matches = re.findall(
    r"\b\d+(?:\.\d+)?\s+on\s+"
    r"\d{2}-\d{2}-\d{4}\s+at\s+"
    r"\d{1,2}:\d{2}\b",
    visible,
    flags=re.IGNORECASE,
)

date_matches = re.findall(
    r"\b\d{2}-\d{2}-2017\b",
    visible,
)

report_time_matches = re.findall(
    r"\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)"
    r"\s+\d{1,2}\s+[A-Za-z]+\s+at\s+\d{1,2}:\d{2}\b",
    visible,
    flags=re.IGNORECASE,
)


print("\nOBSERVED HISTORICAL CLUES")
print("=========================")
print(
    "GoingStick reading strings:",
    len(goingstick_matches),
)
print(
    "Distinct explicit 2017 GoingStick dates:",
    len(set(date_matches)),
)
print(
    "Report date/time strings:",
    len(report_time_matches),
)

if goingstick_matches:
    print("\nFirst GoingStick observations:")

    for value in goingstick_matches[:10]:
        print(" ", value)

    print("\nLast GoingStick observations:")

    for value in goingstick_matches[-10:]:
        print(" ", value)


# ---------------------------------------------------------------------------
# 10. Check whether the page looks populated rather than merely navigable.
# ---------------------------------------------------------------------------
#
# A valid 200 response with navigation only would not establish historical
# archive content. Use multiple independent clues rather than one substring.

archive_appears_populated = (
    len(goingstick_matches) > 0
    or (
        "official going" in visible.lower()
        and len(report_time_matches) > 0
    )
)

print("\nPOPULATION ASSESSMENT")
print("=====================")
print(
    "2017 archive appears populated:",
    "YES" if archive_appears_populated else "NO",
)


# ---------------------------------------------------------------------------
# 11. Show enough of the actual page to inspect its historical content.
# ---------------------------------------------------------------------------

print("\nBOUNDED READABLE PREVIEW")
print("========================")

MAX_PREVIEW_CHARS = 15_000

print(
    visible[:MAX_PREVIEW_CHARS]
)

if len(visible) > MAX_PREVIEW_CHARS:
    print(
        f"\n[Preview truncated after "
        f"{MAX_PREVIEW_CHARS:,} characters]"
    )


# ---------------------------------------------------------------------------
# 12. State the evidence boundary explicitly.
# ---------------------------------------------------------------------------

print("\nPROVENANCE")
print("==========")
print("Archive URL:", ARCHIVE_URL)
print("Cache:", CACHE_FILE)
print("BHA Authorization sent: NO")
print("Years requested by this cell: 1")
print("Requested year: 2017")
print("Other racecourses requested: NO")
print("Bulk historical acquisition performed: NO")

GOING STICK ARCHIVE — NEWTON ABBOT 2017
Loaded from: network
HTTP status: 200
Final URL: https://maps.turftrax.co.uk/iframe/api_goingstickarchive.asp?courseid=37&year=2017
Content-Type: text/html
Response bytes: 15888

VISIBLE ARCHIVE LABELS
meeting date: YES
report date: YES
official going: YES
goingstick index: YES

OBSERVED HISTORICAL CLUES
GoingStick reading strings: 0
Distinct explicit 2017 GoingStick dates: 0
Report date/time strings: 21

POPULATION ASSESSMENT
2017 archive appears populated: YES

BOUNDED READABLE PREVIEW
TurfTrax Course Services - Course Going Stick Reading Archive
Newton Abbot GoingStick Archive
Website :
-->
2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
meeting date
report date
official going
goingstick index
Friday
13 October
Friday
13 October at 07:22
Good Good to Soft in places
6.1 on Friday at 06:00
Monday
02 October
Monday
02 October at 06:00
Heavy
RACING GOES AHEAD
5.1 on Monday at 06:00
Friday
22 September
Friday
22 September at 05:24
Good to Soft So

## Going Stick Archive — 2017 historical capability confirmed

The oldest year currently exposed by the Newton Abbot archive navigation, 2017, contains substantive historical GoingStick records.

### Archive population

The 2017 page returned:

- HTTP 200;
- the same visible archive field headings as the 2026 page;
- 21 report date/time observations;
- official going descriptions;
- GoingStick readings;
- watering/contextual commentary.

Therefore the 2017 archive link is not merely an empty historical navigation placeholder.

### Historical observation structure

The archive includes multiple observations for some meeting dates.

Examples include repeated reports on:

- 16 June 2017;
- 5 June 2017;
- 15 April 2017.

This reinforces the earlier finding that the TurfTrax archive can preserve intraday changes rather than only one final value per fixture.

### Examples of historical information

Observed 2017 records include:

- official going descriptions;
- GoingStick values;
- watering status/context;
- report timestamps;
- raceday updates such as `After 2nd Race`;
- operational context such as `RACING GOES AHEAD`.

### Formatting difference

The first-pass parser reported zero GoingStick reading strings because it expected the modern format:

`5.7 on 27-05-2026 at 08:00`

The 2017 archive instead commonly uses forms such as:

`6.1 on Friday at 06:00`

This is a parsing-format difference, not absence of GoingStick data.

Any future archive parser must preserve and support both historical representations rather than assuming the modern date-explicit format.

### Historical coverage conclusion

For Newton Abbot, usable archive content has now been directly demonstrated in:

- 2017;
- 2026.

The archive interface exposes every intervening year from 2018 through 2025.

This establishes a demonstrated visible archive span of 2017–2026, but it does not yet prove that every intervening year contains observations.

Nor does it establish that 2017 is the earliest GoingStick data held by TurfTrax; it is only the earliest year exposed by the currently observed navigation.

## Decision

It is now reasonable to make one bounded completeness probe across the ten visible Newton Abbot archive years, 2017–2026.

For each year determine only:

- HTTP status;
- whether the archive is populated;
- apparent number of report observations;
- number of distinct meeting dates where recoverable;
- earliest/latest visible meeting;
- observed GoingStick-reading count.

Do not yet acquire other racecourses or build a general archive parser.

In [20]:
# Going Stick Archive — Newton Abbot visible-year completeness probe
#
# WHAT
# ----
# Test every year explicitly exposed by the Newton Abbot Going Stick Archive
# navigation:
#
#   2017 through 2026 inclusive.
#
# For each year, determine:
#
#   - HTTP status;
#   - whether the archive page appears populated;
#   - number of report observations;
#   - number of distinct meeting dates recoverable from the page;
#   - earliest/latest visible meeting date;
#   - number of GoingStick readings recognised.
#
# Existing cached evidence for 2017 and 2026 is reused.
# Only the eight missing years, 2018–2025, may require network requests.
#
# WHY
# ---
# We have directly demonstrated useful historical archive content in both 2017
# and 2026, while the public archive navigation exposes every intervening year.
#
# Before treating 2017–2026 as a usable continuous historical source, establish
# whether each visible year actually contains observations.
#
# This is a coverage check, not a bulk-data acquisition exercise.
#
# READS
# -----
# - Existing Newton Abbot Going Stick archive caches where available.
# - At most one public TurfTrax archive page for each missing visible year.
#
# WRITES
# ------
# One ignored cache file per newly requested year under:
#
#   data/cache/bha_official_source_feasibility/
#       going_stick_archive_probe/
#
# Each cache preserves the exact returned HTML and request provenance.
#
# EXPECTED RESULT
# ---------------
# One compact summary row for each year from 2017 through 2026 showing whether
# useful archive content is present and its approximate observation coverage.
#
# IMPORTANT ANALYTICAL LIMITS
# ---------------------------
# - This does NOT establish coverage for other racecourses.
# - This does NOT establish that 2017 is TurfTrax's true earliest data.
# - Report counts are based on visible archive report timestamps.
# - GoingStick parsing supports both observed modern and historical formats,
#   but this is still a discovery parser rather than a governed production
#   parser.
# - No database design or field governance is performed.
#
# No BHA Authorization credential is sent.

from datetime import datetime, timezone
import calendar
import html
import json
import re
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit project and research-cache locations.
# ---------------------------------------------------------------------------
#
# The notebook must not depend on its current working directory.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "going_stick_archive_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

COURSE_ID = 37
COURSE_LABEL = "Newton Abbot"

VISIBLE_YEARS = list(
    range(2017, 2027)
)


# ---------------------------------------------------------------------------
# 2. Define cache locations already created during earlier bounded probes.
# ---------------------------------------------------------------------------
#
# The first 2026 request used the archive's default-current-year URL and
# therefore has a slightly different cache filename from later explicit-year
# requests.
#
# Reuse it rather than making an unnecessary duplicate request.

EXISTING_CACHE_OVERRIDES = {
    2017: (
        CACHE_DIR
        / "newton_abbot_courseid_37_year_2017.json"
    ),
    2026: (
        CACHE_DIR
        / "newton_abbot_courseid_37.json"
    ),
}


# ---------------------------------------------------------------------------
# 3. Acquire one year's page, preferring exact cached evidence.
# ---------------------------------------------------------------------------
#
# HTTP failures are preserved as evidence rather than raised out of the cell.
# That distinction matters when investigating historical source coverage.

def acquire_year(year):
    archive_url = (
        "https://maps.turftrax.co.uk/iframe/"
        f"api_goingstickarchive.asp?courseid={COURSE_ID}&year={year}"
    )

    standard_cache = (
        CACHE_DIR
        / f"newton_abbot_courseid_37_year_{year}.json"
    )

    cache_file = EXISTING_CACHE_OVERRIDES.get(
        year,
        standard_cache,
    )

    if cache_file.exists():
        envelope = json.loads(
            cache_file.read_text(encoding="utf-8")
        )

        return envelope, "cache", cache_file


    # -----------------------------------------------------------------------
    # Only years not already cached are requested.
    # -----------------------------------------------------------------------

    request = Request(
        archive_url,
        headers={
            "Accept": "text/html,*/*",
            "User-Agent": "Mozilla/5.0",
            "Referer": (
                "https://maps.turftrax.co.uk/iframe/"
                f"api_goingstickarchive.asp?courseid={COURSE_ID}"
            ),
        },
    )

    status = None
    final_url = None
    content_type = None
    response_bytes = b""
    error_text = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:
            status = response.status
            final_url = response.geturl()
            content_type = response.headers.get(
                "Content-Type"
            )
            response_bytes = response.read()

    except HTTPError as error:
        status = error.code
        final_url = error.geturl()
        content_type = error.headers.get(
            "Content-Type"
        )
        response_bytes = error.read()
        error_text = repr(error)

    except URLError as error:
        error_text = repr(error)


    response_text = response_bytes.decode(
        "utf-8",
        errors="replace",
    )

    envelope = {
        "provider": "TurfTrax",
        "discovered_via": (
            "BHA Racecourse - Going Stick Archive"
        ),
        "course_label": COURSE_LABEL,
        "courseid_parameter": COURSE_ID,
        "year_parameter": year,
        "request_url": archive_url,
        "final_url": final_url,
        "retrieved_at_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
        "http_status": status,
        "content_type": content_type,
        "response_bytes": len(response_bytes),
        "response_text": response_text,
        "error": error_text,
    }

    # Atomic replacement protects against interrupted cache writes.
    temp_file = standard_cache.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        standard_cache
    )

    return envelope, "network", standard_cache


# ---------------------------------------------------------------------------
# 4. Convert archive HTML into stable readable text for discovery parsing.
# ---------------------------------------------------------------------------
#
# Raw HTML remains untouched in the cache. This normalisation exists only to
# recognise visible archive rows consistently across historical page formats.

def visible_archive_text(raw_html):
    text = re.sub(
        r"<script\b[^>]*>.*?</script>",
        " ",
        raw_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    text = re.sub(
        r"<style\b[^>]*>.*?</style>",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    text = re.sub(
        r"<[^>]+>",
        "\n",
        text,
    )

    text = html.unescape(
        text
    )

    text = "\n".join(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )

    return text


# ---------------------------------------------------------------------------
# 5. Recover meeting dates from the visible archive-row structure.
# ---------------------------------------------------------------------------
#
# Observed rows repeat the meeting day/date immediately before the report
# day/date/time, for example:
#
#   Friday
#   13 October
#   Friday
#   13 October at 07:22
#
# Matching this repeated structure is more defensible than counting arbitrary
# date-looking strings elsewhere in the page.

WEEKDAYS = (
    "Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday"
)

MONTHS = (
    "January|February|March|April|May|June|"
    "July|August|September|October|November|December"
)

MEETING_REPORT_PATTERN = re.compile(
    rf"\b({WEEKDAYS})\s+"
    rf"(\d{{1,2}}\s+(?:{MONTHS}))\s+"
    rf"({WEEKDAYS})\s+"
    rf"\2\s+at\s+(\d{{1,2}}:\d{{2}})\b",
    flags=re.IGNORECASE,
)


def extract_meeting_observations(text, year):
    collapsed = re.sub(
        r"\s+",
        " ",
        text,
    )

    observations = []

    for match in MEETING_REPORT_PATTERN.finditer(
        collapsed
    ):
        meeting_label = match.group(2)
        report_time = match.group(4)

        # Attach the known archive year so dates can be ordered correctly.
        parsed_date = datetime.strptime(
            f"{meeting_label} {year}",
            "%d %B %Y",
        ).date()

        observations.append(
            {
                "meeting_date": parsed_date,
                "report_time": report_time,
            }
        )

    return observations


# ---------------------------------------------------------------------------
# 6. Recognise both GoingStick formats already observed in this study.
# ---------------------------------------------------------------------------
#
# Modern example:
#
#   5.7 on 27-05-2026 at 08:00
#
# Historical example:
#
#   6.1 on Friday at 06:00
#
# These patterns are deliberately narrow. We would rather under-count an
# unfamiliar representation than silently invent a parse.

MODERN_GOINGSTICK_PATTERN = re.compile(
    r"\b\d+(?:\.\d+)?\s+on\s+"
    r"\d{2}-\d{2}-\d{4}\s+at\s+"
    r"\d{1,2}:\d{2}\b",
    flags=re.IGNORECASE,
)

HISTORICAL_GOINGSTICK_PATTERN = re.compile(
    rf"\b\d+(?:\.\d+)?\s+on\s+"
    rf"(?:{WEEKDAYS})\s+at\s+"
    r"\d{1,2}:\d{2}\b",
    flags=re.IGNORECASE,
)


# ---------------------------------------------------------------------------
# 7. Build one bounded coverage summary for every visible year.
# ---------------------------------------------------------------------------

coverage_rows = []

network_requests = 0

for year in VISIBLE_YEARS:
    envelope, source, cache_file = acquire_year(
        year
    )

    if source == "network":
        network_requests += 1

    raw_html = envelope.get(
        "response_text",
        "",
    )

    visible = visible_archive_text(
        raw_html
    )

    meeting_observations = extract_meeting_observations(
        visible,
        year,
    )

    distinct_meeting_dates = sorted(
        {
            row["meeting_date"]
            for row in meeting_observations
        }
    )

    modern_readings = MODERN_GOINGSTICK_PATTERN.findall(
        visible
    )

    historical_readings = HISTORICAL_GOINGSTICK_PATTERN.findall(
        visible
    )

    goingstick_count = (
        len(modern_readings)
        + len(historical_readings)
    )

    # Require actual visible report rows rather than merely a successful page
    # or archive navigation links.
    populated = (
        envelope.get("http_status") == 200
        and len(meeting_observations) > 0
    )

    coverage_rows.append(
        {
            "year": year,
            "source": source,
            "http_status": envelope.get(
                "http_status"
            ),
            "populated": populated,
            "report_observations": len(
                meeting_observations
            ),
            "distinct_meetings": len(
                distinct_meeting_dates
            ),
            "earliest_meeting": (
                distinct_meeting_dates[0]
                if distinct_meeting_dates
                else None
            ),
            "latest_meeting": (
                distinct_meeting_dates[-1]
                if distinct_meeting_dates
                else None
            ),
            "goingstick_readings": goingstick_count,
            "cache_file": cache_file,
        }
    )


# ---------------------------------------------------------------------------
# 8. Print a compact year-by-year coverage table.
# ---------------------------------------------------------------------------
#
# This is deliberately a discovery summary rather than persistence of a derived
# dataset. The raw source evidence remains in the cache files.

print("NEWTON ABBOT GOING STICK ARCHIVE — 2017–2026")
print("=============================================")

header = (
    f"{'Year':<6}"
    f"{'HTTP':<7}"
    f"{'Populated':<11}"
    f"{'Reports':<9}"
    f"{'Meetings':<10}"
    f"{'GoingStick':<12}"
    f"{'Earliest':<13}"
    f"{'Latest':<13}"
    f"{'Source':<8}"
)

print(header)
print("-" * len(header))

for row in coverage_rows:
    print(
        f"{row['year']:<6}"
        f"{str(row['http_status']):<7}"
        f"{('YES' if row['populated'] else 'NO'):<11}"
        f"{row['report_observations']:<9}"
        f"{row['distinct_meetings']:<10}"
        f"{row['goingstick_readings']:<12}"
        f"{str(row['earliest_meeting'] or ''):<13}"
        f"{str(row['latest_meeting'] or ''):<13}"
        f"{row['source']:<8}"
    )


# ---------------------------------------------------------------------------
# 9. State only conclusions directly supported by this bounded course probe.
# ---------------------------------------------------------------------------

populated_years = [
    row["year"]
    for row in coverage_rows
    if row["populated"]
]

empty_or_unresolved_years = [
    row["year"]
    for row in coverage_rows
    if not row["populated"]
]


print("\nCOVERAGE SUMMARY")
print("================")
print(
    "Visible years tested:",
    f"{VISIBLE_YEARS[0]}–{VISIBLE_YEARS[-1]}",
)
print(
    "Populated years:",
    populated_years,
)
print(
    "Empty/unresolved years:",
    empty_or_unresolved_years,
)

if populated_years:
    print(
        "Earliest demonstrated populated year:",
        min(populated_years),
    )
    print(
        "Latest demonstrated populated year:",
        max(populated_years),
    )

print(
    "Total recognised report observations:",
    sum(
        row["report_observations"]
        for row in coverage_rows
    ),
)

print(
    "Total recognised GoingStick readings:",
    sum(
        row["goingstick_readings"]
        for row in coverage_rows
    ),
)


# ---------------------------------------------------------------------------
# 10. Preserve the scope/provenance boundary.
# ---------------------------------------------------------------------------

print("\nPROVENANCE")
print("==========")
print("Course:", COURSE_LABEL)
print("Archive courseid parameter:", COURSE_ID)
print("Years tested:", VISIBLE_YEARS)
print("Network requests made by this cell:", network_requests)
print("Maximum possible new requests:", 8)
print("BHA Authorization sent: NO")
print("Other racecourses requested: NO")
print("Database writes: NONE")

NEWTON ABBOT GOING STICK ARCHIVE — 2017–2026
Year  HTTP   Populated  Reports  Meetings  GoingStick  Earliest     Latest       Source  
-----------------------------------------------------------------------------------------
2017  200    YES        21       18        21          2017-04-15   2017-10-13   cache   
2018  200    YES        24       20        23          2018-03-31   2018-10-12   network 
2019  200    YES        23       19        20          2019-04-20   2019-10-31   network 
2020  200    YES        22       22        10          2020-04-11   2020-10-29   network 
2021  200    YES        26       19        25          2021-04-03   2021-10-21   network 
2022  200    YES        18       18        18          2022-03-25   2022-10-15   network 
2023  200    YES        21       18        20          2023-04-08   2023-10-21   network 
2024  200    YES        17       14        16          2024-04-16   2024-10-30   network 
2025  200    YES        21       18        20          

## Going Stick Archive — Newton Abbot 2017–2026 coverage

A bounded coverage probe tested every year explicitly exposed by the Newton Abbot Going Stick Archive.

### Coverage result

All ten visible archive years were populated:

- 2017;
- 2018;
- 2019;
- 2020;
- 2021;
- 2022;
- 2023;
- 2024;
- 2025;
- 2026.

No empty or unresolved year was observed.

Across the ten pages the discovery parser recognised:

- 209 report observations;
- 188 GoingStick-reading strings.

The earliest demonstrated populated archive year is therefore 2017.

The latest demonstrated year is 2026.

### Meeting-level depth

Completed historical years generally contained observations covering numerous separate Newton Abbot meetings rather than isolated examples.

Observed distinct meeting counts were:

- 2017: 18;
- 2018: 20;
- 2019: 19;
- 2020: 22;
- 2021: 19;
- 2022: 18;
- 2023: 18;
- 2024: 14;
- 2025: 18.

The 2026 page contained 11 distinct meetings through the latest observation currently present in the archive.

### Multiple observations

Report counts are sometimes greater than distinct meeting counts.

This confirms across multiple seasons that the archive can preserve more than one official update for a meeting rather than functioning solely as a one-row-per-fixture dataset.

### GoingStick parsing warning

The discovery parser recognised fewer GoingStick readings than report observations in several years.

The largest difference occurred in 2020:

- 22 report observations;
- 10 currently recognised GoingStick strings.

This must not yet be interpreted as 12 genuinely absent measurements.

Earlier investigation already demonstrated historical formatting differences in how TurfTrax represents GoingStick readings.

The discrepancy could therefore represent:

- genuinely absent readings;
- another historical text format not recognised by the current discovery regex;
- a mixture of both.

Raw archive evidence has been preserved, so this can be resolved later if the dataset is selected for governed acquisition.

### Conclusion

For Newton Abbot, the BHA-linked TurfTrax resource is demonstrated to provide a substantial and continuous visible historical archive across 2017–2026.

It contains considerably more than final raceday GoingStick values, including repeated official updates and contextual going information.

This makes the source potentially valuable for historical research into:

- GoingStick;
- official going evolution;
- watering;
- weather/ground commentary;
- within-meeting changes;
- racecourse-specific ground behaviour.

### Evidence boundary

This result establishes coverage only for Newton Abbot.

It does not establish:

- equivalent historical depth for every British racecourse;
- that 2017 is TurfTrax's true earliest retained data;
- that every report necessarily contains a GoingStick measurement;
- that the current discovery parser captures every historical representation.

## Decision

Before considering general acquisition, test a small contrasting set of racecourses to establish whether this historical archive capability is widespread rather than peculiar to Newton Abbot.

The comparison should deliberately include different racing surfaces/codes where appropriate.

Do not yet request every racecourse or build a production archive parser.

## Phase 4 — Systematic BHA public-source inventory

The GoingStick investigation demonstrated an important methodological point:

> useful BHA information is distributed across multiple public website sections and external services, not only the fixture/race API.

The purpose of this phase is therefore to map the wider public BHA information surface before comparing it with Database v4.

This is **source discovery**, not detailed investigation of every dataset.

### Inventory questions

For each candidate source determine only:

1. **What information does it expose?**
2. **What is its apparent grain?**
   - horse;
   - jockey;
   - trainer;
   - owner;
   - fixture;
   - race;
   - runner;
   - racecourse;
   - aggregate/statistical;
   - disciplinary/documentary.
3. **How is it delivered?**
   - structured API;
   - searchable web interface;
   - HTML table;
   - Excel/CSV download;
   - PDF;
   - external linked service.
4. **Does it preserve history or only current state?**
5. **Does it appear potentially useful to Inside Rails?**
6. **Does it warrant deeper investigation later?**

### Candidate BHA source families already identified

The site-wide search has identified at least the following public source families for bounded inspection:

- official ratings database and downloadable rating files;
- horse profiles/database;
- jockey information and statistics;
- trainer information and statistics;
- owner information and championships;
- full-year fixture-list downloads;
- Racing Statistics annual data packs;
- Racing Statistics monthly data packs;
- Horse Population Reports;
- race off-times / punctuality statistics;
- claiming-race records;
- disciplinary decisions;
- handicapping appeals;
- racecourse technical resources;
- Going Stick archives;
- fixture/race structured resources already investigated;
- Stewards' Reports already investigated.

### Scope discipline

Do **not** now perform a deep study of each source.

A source only needs enough inspection to classify:

- its information content;
- its technical accessibility;
- its historical depth where immediately apparent;
- its likely value to Inside Rails.

Detailed semantic validation belongs later, and only for information we actually decide is worth using.

### Intended output

At the end of this phase produce a BHA source inventory in which each source is classified as:

- **high potential value**;
- **possible validation/provenance value**;
- **specialist/contextual value**;
- **low apparent value**;
- **requires further investigation**.

Only after this inventory is complete should the study perform the two-way comparison:

> **What useful information exists in BHA but not Database v4, what exists in Database v4 but not BHA, and where do both sources overlap?**

Do not design Database v5 during this phase.

In [22]:
# BHA Official Ratings — bounded source-surface inventory
#
# WHAT
# ----
# Inventory the public BHA Official Ratings source surface without repeatedly
# attempting to fetch the human-facing ratings page, which the BHA/Cloudflare
# layer has blocked from scripted notebook requests.
#
# This cell combines two bounded evidence sources:
#
#   1. safe export paths already observed from the public BHA ratings page;
#   2. the current BHA frontend app.js, inspected for ratings-related structured
#      resource calls used by the interactive website.
#
# WHY
# ---
# The purpose of this study phase is to discover what useful official BHA
# information exists, not to reverse-engineer or bulk-download every dataset.
#
# The earlier cell failed because `curl --fail` converted a scripted-page HTTP
# block into an exception. A blocked presentation page should not stop source
# discovery when the public export/resource paths are already observable.
#
# READS
# -----
# - Current BHA frontend JavaScript:
#
#     https://www.britishhorseracing.com/
#         wp-content/themes/bha/library/js/angular/app.js?ver=1.19
#
# - Three safe ratings export paths already observed from the public ratings
#   page during this study. Temporary signed query values are deliberately
#   excluded.
#
# WRITES
# ------
# One ignored derived inventory:
#
#   data/cache/bha_official_source_feasibility/
#       ratings_surface_inventory/
#       ratings_surface_inventory.json
#
# Raw app.js is NOT persisted.
#
# EXPECTED RESULT
# ---------------
# A compact ratings-source inventory containing:
#
#   - public spreadsheet export families;
#   - ratings-related frontend route/path strings;
#   - bounded frontend code snippets around ratings-related calls;
#   - app.js provenance/fingerprint.
#
# This cell does NOT:
#
#   - download the ratings spreadsheets;
#   - persist rating records;
#   - expose or preserve temporary signed URL values;
#   - investigate handicapping semantics;
#   - compare ratings with Database v4;
#   - design database structures.

from datetime import datetime, timezone
import hashlib
import json
import re
import subprocess
from pathlib import Path


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored cache locations.
# ---------------------------------------------------------------------------
#
# Keep notebook behaviour independent of the directory from which Jupyter was
# started.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "ratings_surface_inventory"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_FILE = (
    CACHE_DIR
    / "ratings_surface_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Record only the safe parts of the three public export resources.
# ---------------------------------------------------------------------------
#
# These paths were observed from the current public BHA ratings page during the
# site-wide source search.
#
# The live links also carried temporary `expires` and `sig` query values.
# Those operational values are unnecessary for source inventory and are
# intentionally excluded.
#
# The weekly-changes link was observed using the same ratings export route with
# a `diff` query parameter. We record the resource family rather than the
# temporary complete URL.

PUBLIC_EXPORT_RESOURCES = [
    {
        "label": "Full published ratings",
        "path": "/bha/v1/ratings/csv/ratings",
        "observed_query_role": "ordinary ratings export",
    },
    {
        "label": "Weekly rating changes",
        "path": "/bha/v1/ratings/csv/ratings",
        "observed_query_role": "ratings export with diff parameter",
    },
    {
        "label": "Latest performance figures",
        "path": "/bha/v1/ratings/csv/performance-figures",
        "observed_query_role": "performance-figures export",
    },
]


# ---------------------------------------------------------------------------
# 3. Fetch the current frontend JavaScript without treating HTTP errors as a
#    Python exception.
# ---------------------------------------------------------------------------
#
# The BHA presentation pages have intermittently blocked scripted access during
# this study, while app.js has previously been directly retrievable.
#
# Capture the HTTP status explicitly rather than using `curl --fail`, so any
# future access change becomes evidence rather than an opaque notebook crash.

APP_JS_URL = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js?ver=1.19"
)

STATUS_MARKER = "__BHA_HTTP_STATUS__"

result = subprocess.run(
    [
        "curl",
        "-L",
        "--silent",
        "--show-error",
        "--max-time",
        "30",
        "--user-agent",
        "Mozilla/5.0",
        "--write-out",
        f"\n{STATUS_MARKER}%{{http_code}}",
        APP_JS_URL,
    ],
    capture_output=True,
    text=True,
    check=False,
)

combined_output = result.stdout

marker_position = combined_output.rfind(
    f"\n{STATUS_MARKER}"
)

assert marker_position != -1, (
    "Could not recover HTTP status from app.js request."
)

app_js = combined_output[:marker_position]

status_text = combined_output[
    marker_position + len(f"\n{STATUS_MARKER}") :
].strip()

app_js_http_status = int(
    status_text
)

print("BHA OFFICIAL RATINGS — SOURCE-SURFACE INVENTORY")
print("===============================================")
print("app.js HTTP status:", app_js_http_status)
print("curl return code:", result.returncode)

if result.stderr.strip():
    print(
        "curl stderr:",
        result.stderr.strip(),
    )


# ---------------------------------------------------------------------------
# 4. Stop interpretation cleanly if app.js itself is unavailable.
# ---------------------------------------------------------------------------
#
# Do not mistake an access-control response for absence of ratings resources.

if app_js_http_status != 200 or not app_js.strip():
    print("\nFrontend JavaScript could not be inspected.")
    print(
        "This is an access observation, not evidence that ratings "
        "resources are absent."
    )

else:
    # -----------------------------------------------------------------------
    # 5. Locate every occurrence of `ratings` in the current frontend code.
    # -----------------------------------------------------------------------
    #
    # We deliberately search broadly first. Different Angular components may
    # construct route URLs in different ways, so looking only for complete
    # `/bha/v1/ratings/...` literals could miss concatenated paths.

    ratings_occurrences = [
        match.start()
        for match in re.finditer(
            r"ratings",
            app_js,
            flags=re.IGNORECASE,
        )
    ]

    print(
        "\nOccurrences of 'ratings' in app.js:",
        len(ratings_occurrences),
    )


    # -----------------------------------------------------------------------
    # 6. Recover quoted strings containing ratings-related resource clues.
    # -----------------------------------------------------------------------
    #
    # These may be:
    #
    #   - complete API routes;
    #   - relative route fragments;
    #   - template/page references;
    #   - function arguments.
    #
    # We preserve them as discovery evidence without assigning semantics solely
    # from their names.

    quoted_ratings_strings = sorted(
        set(
            match.group(2)
            for match in re.finditer(
                r"""(['"])([^'"]*ratings[^'"]*)\1""",
                app_js,
                flags=re.IGNORECASE,
            )
        )
    )

    print("\nRATINGS-RELATED QUOTED STRINGS")
    print("==============================")

    if quoted_ratings_strings:
        for value in quoted_ratings_strings:
            print(value[:3000])
    else:
        print("NONE RECOVERED")


    # -----------------------------------------------------------------------
    # 7. Recover explicit BHA-v1 path fragments containing ratings.
    # -----------------------------------------------------------------------
    #
    # This catches paths even where surrounding JavaScript quoting is awkward.

    bha_v1_ratings_paths = sorted(
        set(
            re.findall(
                r"""(?:/bha/v1/)?ratings[^\s'")},;]+""",
                app_js,
                flags=re.IGNORECASE,
            )
        )
    )

    print("\nRATINGS RESOURCE/PATH FRAGMENTS")
    print("===============================")

    if bha_v1_ratings_paths:
        for value in bha_v1_ratings_paths:
            print(value)
    else:
        print("NONE RECOVERED")


    # -----------------------------------------------------------------------
    # 8. Print bounded code context around each unique ratings occurrence.
    # -----------------------------------------------------------------------
    #
    # Function names and nearby URL construction are often more informative
    # than isolated route fragments.
    #
    # Merge nearby occurrences so the same Angular block is not printed dozens
    # of times.

    CONTEXT_RADIUS = 700

    context_ranges = []

    for position in ratings_occurrences:
        start = max(
            0,
            position - CONTEXT_RADIUS,
        )

        end = min(
            len(app_js),
            position + CONTEXT_RADIUS,
        )

        if (
            context_ranges
            and start <= context_ranges[-1][1]
        ):
            context_ranges[-1] = (
                context_ranges[-1][0],
                max(
                    context_ranges[-1][1],
                    end,
                ),
            )

        else:
            context_ranges.append(
                (start, end)
            )

    # Keep the notebook output bounded. If ratings appears throughout a very
    # large controller, the first few distinct code regions are sufficient for
    # source-surface discovery.
    MAX_CONTEXT_BLOCKS = 12

    bounded_contexts = []

    print("\nBOUNDED FRONTEND CONTEXT")
    print("========================")

    for index, (start, end) in enumerate(
        context_ranges[:MAX_CONTEXT_BLOCKS],
        start=1,
    ):
        snippet = app_js[start:end]

        # Collapse excessive whitespace while retaining enough syntax to show
        # route construction and function context.
        compact = re.sub(
            r"\s+",
            " ",
            snippet,
        ).strip()

        bounded_contexts.append(
            compact
        )

        print(f"\nContext {index}")
        print("-" * 40)
        print(
            compact[:5000]
        )


    # -----------------------------------------------------------------------
    # 9. Fingerprint the exact frontend version used for this inventory.
    # -----------------------------------------------------------------------
    #
    # The raw JavaScript is deliberately not persisted, but the SHA-256 gives
    # us reproducible provenance if the site's implementation later changes.

    app_js_sha256 = hashlib.sha256(
        app_js.encode("utf-8")
    ).hexdigest()


    # -----------------------------------------------------------------------
    # 10. Persist only the derived, secret-safe source inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
        "source_family": "BHA Official Ratings",
        "official_page": (
            "https://www.britishhorseracing.com/"
            "regulation/official-ratings/ratings-database/"
        ),
        "app_js_url": APP_JS_URL,
        "app_js_http_status": app_js_http_status,
        "app_js_sha256": app_js_sha256,
        "public_export_resources": PUBLIC_EXPORT_RESOURCES,
        "quoted_ratings_strings": quoted_ratings_strings,
        "ratings_path_fragments": bha_v1_ratings_paths,
        "bounded_frontend_contexts": bounded_contexts,
        "temporary_signed_query_values_persisted": False,
        "raw_app_js_persisted": False,
        "ratings_records_downloaded": False,
    }

    temp_file = INVENTORY_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_FILE
    )


    # -----------------------------------------------------------------------
    # 11. Present the already-established export families separately.
    # -----------------------------------------------------------------------

    print("\nPUBLIC EXPORT FAMILIES")
    print("======================")

    for resource in PUBLIC_EXPORT_RESOURCES:
        print("\nLabel:", resource["label"])
        print("Path:", resource["path"])
        print(
            "Observed role:",
            resource["observed_query_role"],
        )


    print("\nPROVENANCE")
    print("==========")
    print("app.js SHA-256:", app_js_sha256)
    print("Derived inventory:", INVENTORY_FILE)
    print("Raw app.js persisted: NO")
    print("Ratings records downloaded: NO")
    print("Signed query values displayed: NO")
    print("Signed query values persisted: NO")

BHA OFFICIAL RATINGS — SOURCE-SURFACE INVENTORY
app.js HTTP status: 200
curl return code: 0

Occurrences of 'ratings' in app.js: 0

RATINGS-RELATED QUOTED STRINGS
NONE RECOVERED

RATINGS RESOURCE/PATH FRAGMENTS
NONE RECOVERED

BOUNDED FRONTEND CONTEXT

PUBLIC EXPORT FAMILIES

Label: Full published ratings
Path: /bha/v1/ratings/csv/ratings
Observed role: ordinary ratings export

Label: Weekly rating changes
Path: /bha/v1/ratings/csv/ratings
Observed role: ratings export with diff parameter

Label: Latest performance figures
Path: /bha/v1/ratings/csv/performance-figures
Observed role: performance-figures export

PROVENANCE
app.js SHA-256: 2315abeb6d4a4d8d6f726daecd7ccb255ca986252ad31645d5fc8f290b6c0f66
Derived inventory: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/ratings_surface_inventory/ratings_surface_inventory.json
Raw app.js persisted: NO
Ratings records downloaded: NO
Signed query values displayed: NO
Signed query values persisted: NO


In [23]:
# BHA Official Ratings — discover the frontend asset that powers the ratings table
#
# WHAT
# ----
# Discover which JavaScript asset(s) power the public BHA Official Ratings table.
#
# The previous investigation proved that the general BHA `app.js` currently
# contains no ratings-related strings, despite the public Ratings Database page
# visibly using Angular expressions such as:
#
#   {{lastUpdatedRatings}}
#   rating.diffFlat
#   rating.diffAwt
#   rating.diffChase
#   rating.diffHurdle
#
# This cell therefore:
#
#   1. attempts to retrieve the Ratings Database HTML using a browser-like
#      request without treating HTTP blocks as Python failures;
#   2. extracts the JavaScript assets referenced by that page if successful;
#   3. if the ratings page is blocked, tries a small set of other BHA pages to
#      recover the site's shared frontend asset inventory;
#   4. downloads only BHA-owned JavaScript assets from that inventory;
#   5. searches those assets for exact ratings-template markers and
#      ratings/API path clues;
#   6. prints bounded code context around any matches.
#
# WHY
# ---
# We are currently mapping the complete BHA public information surface.
#
# The ratings table may expose structured resources beyond the three spreadsheet
# downloads already identified.
#
# We should therefore locate its actual frontend/controller implementation
# before deciding that the downloadable exports represent the whole ratings
# source surface.
#
# READS
# -----
# A bounded set of public BHA webpages and JavaScript assets.
#
# Candidate HTML pages:
#
#   - Official Ratings database;
#   - Horses;
#   - Results;
#   - BHA home page.
#
# Only BHA-owned JavaScript assets discovered from those pages are inspected.
#
# WRITES
# ------
# One ignored derived inventory:
#
#   data/cache/bha_official_source_feasibility/
#       ratings_frontend_discovery/
#       ratings_frontend_discovery.json
#
# Raw HTML and raw JavaScript are NOT persisted.
#
# EXPECTED RESULT
# ---------------
# Either:
#
#   A. one or more frontend assets containing ratings-specific controller,
#      factory, service or route logic;
#
# or:
#
#   B. explicit evidence that the pages available to this scripted client do
#      not expose the relevant asset, leaving the ratings frontend location
#      unresolved.
#
# IMPORTANT SCOPE LIMITS
# ----------------------
# - No ratings records are downloaded.
# - No speculative API route guessing is performed.
# - No temporary signed export tokens are printed or persisted.
# - No third-party JavaScript libraries are crawled.
# - No more than 30 BHA-owned JavaScript assets are inspected.
# - This is frontend discovery only, not handicapping research.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on the notebook's current working directory.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "ratings_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_FILE = (
    CACHE_DIR
    / "ratings_frontend_discovery.json"
)


# ---------------------------------------------------------------------------
# 2. Define the small set of BHA pages used for script discovery.
# ---------------------------------------------------------------------------
#
# Ratings is attempted first.
#
# The fallback pages are not substitutes for the ratings page. They exist only
# to recover shared BHA frontend assets if Cloudflare blocks the ratings page
# from this notebook client.

PAGE_URLS = [
    (
        "ratings_database",
        "https://www.britishhorseracing.com/"
        "regulation/official-ratings/ratings-database/",
    ),
    (
        "horses",
        "https://www.britishhorseracing.com/racing/horses/",
    ),
    (
        "results",
        "https://www.britishhorseracing.com/racing/results/",
    ),
    (
        "home",
        "https://www.britishhorseracing.com/",
    ),
]


# ---------------------------------------------------------------------------
# 3. Fetch a public page while preserving HTTP status as evidence.
# ---------------------------------------------------------------------------
#
# Do NOT use `curl --fail`.
#
# A 403/other block is an observation about access from this scripted client,
# not a reason for the notebook cell to crash.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            (
                "Accept: text/html,application/xhtml+xml,"
                "application/xml;q=0.9,*/*;q=0.8"
            ),
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[:status_position]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(status_text)
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Attempt the four bounded HTML pages and extract script references.
# ---------------------------------------------------------------------------
#
# We retain only derived script URLs and page fingerprints.
# Raw page HTML is not written to disk.

page_observations = []
script_sources = []

for page_name, page_url in PAGE_URLS:
    response = curl_public_text(
        page_url
    )

    body = response["body"]

    body_sha256 = (
        hashlib.sha256(
            body.encode("utf-8")
        ).hexdigest()
        if body
        else None
    )

    page_scripts = []

    if response["status"] == 200 and body:
        # Standard external script source.
        page_scripts.extend(
            re.findall(
                r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
                body,
                flags=re.IGNORECASE | re.DOTALL,
            )
        )

        # Some sites preload script assets separately.
        page_scripts.extend(
            re.findall(
                r"""<link\b[^>]*\bhref\s*=\s*["']([^"']+\.js[^"']*)["'][^>]*>""",
                body,
                flags=re.IGNORECASE | re.DOTALL,
            )
        )

    resolved_scripts = []

    for source in page_scripts:
        source = html.unescape(
            source
        )

        resolved = urljoin(
            response["final_url"] or page_url,
            source,
        )

        resolved_scripts.append(
            resolved
        )

        script_sources.append(
            resolved
        )

    page_observations.append(
        {
            "page_name": page_name,
            "request_url": page_url,
            "http_status": response["status"],
            "final_url": response["final_url"],
            "curl_return_code": response["curl_return_code"],
            "stderr": response["stderr"],
            "html_sha256": body_sha256,
            "script_count": len(
                set(resolved_scripts)
            ),
            "scripts": sorted(
                set(resolved_scripts)
            ),
        }
    )


# ---------------------------------------------------------------------------
# 5. Keep only first-party BHA JavaScript assets.
# ---------------------------------------------------------------------------
#
# We are looking for the BHA controller/service code, not jQuery, Google,
# analytics or other third-party libraries.
#
# Static assets served from the main BHA host are eligible. Query strings are
# retained for the request because they may represent asset versions.

BHA_HOSTS = {
    "www.britishhorseracing.com",
    "britishhorseracing.com",
}

first_party_scripts = []

for script_url in sorted(
    set(script_sources)
):
    parsed = urlsplit(
        script_url
    )

    if parsed.hostname not in BHA_HOSTS:
        continue

    if ".js" not in parsed.path.lower():
        continue

    first_party_scripts.append(
        script_url
    )


# ---------------------------------------------------------------------------
# 6. Bound the inspection before making asset requests.
# ---------------------------------------------------------------------------
#
# Thirty first-party scripts is already considerably more than expected and is
# sufficient for source discovery. If the site exposes more, stop rather than
# silently turning this into a broad crawl.

MAX_SCRIPT_ASSETS = 30

scripts_to_inspect = first_party_scripts[
    :MAX_SCRIPT_ASSETS
]

script_inventory_truncated = (
    len(first_party_scripts)
    > MAX_SCRIPT_ASSETS
)


# ---------------------------------------------------------------------------
# 7. Search each first-party script for exact ratings implementation clues.
# ---------------------------------------------------------------------------
#
# Exact page-template variables are particularly strong evidence because they
# tie a script directly to the public ratings table.
#
# Broader terms are also included to reveal route construction or resource
# names.

SEARCH_MARKERS = [
    "lastUpdatedRatings",
    "diffFlat",
    "diffAwt",
    "diffChase",
    "diffHurdle",
    "performance-figures",
    "ratings/csv",
    "/ratings",
    "ratings?",
    "ratingFactory",
    "ratingsFactory",
    "ratingService",
    "ratingsService",
    "ratingController",
    "ratingsController",
]

asset_observations = []
matching_assets = []

for script_url in scripts_to_inspect:
    response = curl_public_text(
        script_url
    )

    script_text = response["body"]

    matched_markers = [
        marker
        for marker in SEARCH_MARKERS
        if marker.lower() in script_text.lower()
    ]

    contexts = []

    if matched_markers:
        # ---------------------------------------------------------------
        # Recover bounded source context around each distinct marker.
        # ---------------------------------------------------------------
        #
        # This should expose function/controller names and concrete URL
        # construction without printing entire minified source bundles.

        seen_contexts = set()

        for marker in matched_markers:
            for match in re.finditer(
                re.escape(marker),
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 900,
                )

                end = min(
                    len(script_text),
                    match.end() + 1200,
                )

                context = script_text[
                    start:end
                ]

                compact = re.sub(
                    r"\s+",
                    " ",
                    context,
                ).strip()

                # Avoid printing essentially the same minified block repeatedly.
                context_key = hashlib.sha256(
                    compact.encode("utf-8")
                ).hexdigest()

                if context_key in seen_contexts:
                    continue

                seen_contexts.add(
                    context_key
                )

                contexts.append(
                    {
                        "marker": marker,
                        "context": compact[:5000],
                    }
                )

                # Two contexts per marker are plenty for discovery.
                if (
                    sum(
                        1
                        for item in contexts
                        if item["marker"] == marker
                    )
                    >= 2
                ):
                    break

    observation = {
        "script_url": script_url,
        "http_status": response["status"],
        "final_url": response["final_url"],
        "sha256": (
            hashlib.sha256(
                script_text.encode("utf-8")
            ).hexdigest()
            if script_text
            else None
        ),
        "matched_markers": matched_markers,
        "contexts": contexts,
    }

    asset_observations.append(
        observation
    )

    if matched_markers:
        matching_assets.append(
            observation
        )


# ---------------------------------------------------------------------------
# 8. Search matching assets for concrete route-like strings.
# ---------------------------------------------------------------------------
#
# Do this only after a script has demonstrated a ratings relationship.
#
# We preserve route/path clues but deliberately discard any query values that
# look like temporary signatures/tokens.

route_clues = []

for asset in matching_assets:
    for context_item in asset["contexts"]:
        context = context_item["context"]

        candidate_strings = re.findall(
            r"""['"]([^'"]*(?:ratings|performance-figures)[^'"]*)['"]""",
            context,
            flags=re.IGNORECASE,
        )

        for candidate in candidate_strings:
            # Never retain signed query material in this source inventory.
            safe_candidate = re.sub(
                r"([?&](?:sig|signature|token|expires)=[^&\s]+)",
                "",
                candidate,
                flags=re.IGNORECASE,
            )

            route_clues.append(
                safe_candidate
            )

route_clues = sorted(
    set(route_clues)
)


# ---------------------------------------------------------------------------
# 9. Persist only the derived frontend-discovery inventory.
# ---------------------------------------------------------------------------

inventory = {
    "observed_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "source_family": "BHA Official Ratings",
    "page_observations": page_observations,
    "first_party_scripts_discovered": first_party_scripts,
    "scripts_inspected": len(
        scripts_to_inspect
    ),
    "script_inventory_truncated": script_inventory_truncated,
    "asset_observations": asset_observations,
    "matching_assets": [
        {
            "script_url": item["script_url"],
            "http_status": item["http_status"],
            "sha256": item["sha256"],
            "matched_markers": item["matched_markers"],
            "contexts": item["contexts"],
        }
        for item in matching_assets
    ],
    "route_clues": route_clues,
    "raw_html_persisted": False,
    "raw_javascript_persisted": False,
    "signed_query_values_persisted": False,
    "ratings_records_downloaded": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 10. Present page-access and script-discovery evidence first.
# ---------------------------------------------------------------------------

print("BHA OFFICIAL RATINGS — FRONTEND DISCOVERY")
print("=========================================")

print("\nPAGE ACCESS")
print("===========")

for page in page_observations:
    print(
        f"{page['page_name']}: "
        f"HTTP {page['http_status']} | "
        f"scripts={page['script_count']}"
    )

    if page["final_url"]:
        print(
            "  final URL:",
            page["final_url"],
        )


print("\nFIRST-PARTY SCRIPT INVENTORY")
print("============================")
print(
    "First-party BHA scripts discovered:",
    len(first_party_scripts),
)
print(
    "Scripts inspected:",
    len(scripts_to_inspect),
)
print(
    "Inspection truncated:",
    "YES" if script_inventory_truncated else "NO",
)

for script_url in scripts_to_inspect:
    print(
        " ",
        script_url,
    )


# ---------------------------------------------------------------------------
# 11. Report only assets that actually contain ratings evidence.
# ---------------------------------------------------------------------------

print("\nRATINGS-MATCHING ASSETS")
print("=======================")
print(
    "Matching assets:",
    len(matching_assets),
)

if not matching_assets:
    print("NONE FOUND")

else:
    for index, asset in enumerate(
        matching_assets,
        start=1,
    ):
        print(f"\nAsset {index}")
        print("-" * 60)
        print(
            "URL:",
            asset["script_url"],
        )
        print(
            "HTTP:",
            asset["http_status"],
        )
        print(
            "SHA-256:",
            asset["sha256"],
        )
        print(
            "Matched markers:",
            asset["matched_markers"],
        )

        for context_index, context_item in enumerate(
            asset["contexts"],
            start=1,
        ):
            print(
                f"\n  Context {context_index} "
                f"[{context_item['marker']}]"
            )
            print(
                "  ",
                context_item["context"],
            )


# ---------------------------------------------------------------------------
# 12. Report concrete route/resource clues separately.
# ---------------------------------------------------------------------------

print("\nRATINGS ROUTE / RESOURCE CLUES")
print("==============================")

if route_clues:
    for clue in route_clues:
        print(clue)
else:
    print("NONE RECOVERED")


# ---------------------------------------------------------------------------
# 13. State the evidence boundary explicitly.
# ---------------------------------------------------------------------------

ratings_page_observation = next(
    (
        page
        for page in page_observations
        if page["page_name"] == "ratings_database"
    ),
    None,
)

print("\nDISCOVERY BOUNDARY")
print("==================")

if (
    ratings_page_observation
    and ratings_page_observation["http_status"] == 200
):
    print(
        "Ratings page itself was available to the notebook client: YES"
    )
else:
    print(
        "Ratings page itself was available to the notebook client: NO"
    )
    print(
        "Any scripts discovered only from fallback pages may represent "
        "shared assets and cannot prove page-specific completeness."
    )

print("\nPROVENANCE")
print("==========")
print(
    "Derived inventory:",
    INVENTORY_FILE,
)
print(
    "Raw HTML persisted: NO"
)
print(
    "Raw JavaScript persisted: NO"
)
print(
    "Signed query values displayed: NO"
)
print(
    "Signed query values persisted: NO"
)
print(
    "Ratings records downloaded: NO"
)

BHA OFFICIAL RATINGS — FRONTEND DISCOVERY

PAGE ACCESS
ratings_database: HTTP 200 | scripts=28
  final URL: https://www.britishhorseracing.com/regulation/official-ratings/ratings-database/
horses: HTTP 200 | scripts=28
  final URL: https://www.britishhorseracing.com/racing/horses/
results: HTTP 200 | scripts=29
  final URL: https://www.britishhorseracing.com/racing/results/
home: HTTP 200 | scripts=30
  final URL: https://www.britishhorseracing.com/

FIRST-PARTY SCRIPT INVENTORY
First-party BHA scripts discovered: 29
Scripts inspected: 29
Inspection truncated: NO
  https://www.britishhorseracing.com/cdn-cgi/scripts/5c5dd728/cloudflare-static/email-decode.min.js
  https://www.britishhorseracing.com/wp-content/plugins/magic-liquidizer-responsive-table/idjs/ml.responsive.table.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/flickity.pkgd.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/functions.min.js
  https://www.britishhorseracing

## BHA Official Ratings — structured frontend resource discovered

Further frontend discovery identified the JavaScript asset that powers the public BHA Official Ratings database:

`/wp-content/themes/bha/library/js/angular/pages/ratings.js?ver=1.1`

### Frontend controller

The asset defines a `RacehorseRatings` Angular controller.

It sets the ratings resource to:

`apiaddress + '/bha/v1/ratings'`

This establishes a structured public ratings resource distinct from the downloadable CSV exports.

### Observed query capabilities

The frontend sends the following parameters to the ratings resource:

- `page`;
- `per_page`;
- `sortby`;
- `q`.

The default page size used by the public interface is 250 records.

The `q` parameter is used by the frontend's name-search function.

The frontend also requests:

`per_page=1`

with:

`sortby=modifiedTimestamp:desc`

to determine the most recently updated ratings record.

### Observed response structure

The frontend expects the response to contain:

- `data`;
- `last_page`;
- `per_page`;
- `total`;
- `current_page`;
- `from`;
- `to`.

This demonstrates that `/bha/v1/ratings` is a paginated structured resource rather than merely a file-download endpoint.

### Separate ratings source surfaces now demonstrated

The Official Ratings family therefore contains at least:

1. interactive structured ratings data:
   - `/bha/v1/ratings`;

2. full published ratings export:
   - `/bha/v1/ratings/csv/ratings`;

3. weekly rating-change export:
   - ratings CSV resource with a difference/query distinction;

4. latest performance-figures export:
   - `/bha/v1/ratings/csv/performance-figures`.

### Historical implementation clue

The ratings frontend source itself contains version history dating its Angular implementation to 2015.

This is implementation provenance only.

It does **not** establish that the current ratings API exposes records back to 2015.

### Decision

The structured ratings API should be inspected with one bounded request to determine its actual record schema.

Use:

- one record only;
- most recently modified first;
- the locally stored BHA credential;
- the existing ignored research cache.

Do not download the full ratings population yet.

In [25]:
# BHA Official Ratings — one-record structured API probe
#
# WHAT
# ----
# Make one bounded request to the structured Official Ratings resource:
#
#   /bha/v1/ratings
#
# using the same query pattern observed in the public ratings frontend:
#
#   per_page = 1
#   sortby   = modifiedTimestamp:desc
#
# The purpose is to inspect the actual schema of one current ratings record.
#
# WHY
# ---
# Frontend discovery established that the public ratings table is backed by a
# paginated structured endpoint, not merely CSV downloads.
#
# Before moving on to another BHA source family, establish what information one
# ratings record actually contains.
#
# READS
# -----
# - BHA Authorization credential from repo-root `.env.local`;
# - one public BHA structured ratings endpoint.
#
# WRITES
# ------
# One ignored research-cache file:
#
#   data/cache/bha_official_source_feasibility/
#       ratings_record_probe/
#       latest_modified_rating.json
#
# The Authorization credential is NEVER written to the cache or printed.
#
# EXPECTED RESULT
# ---------------
# A bounded description showing:
#
#   - HTTP status;
#   - response top-level structure;
#   - pagination metadata;
#   - number of records returned;
#   - complete field inventory for the returned rating record;
#   - one representative record.
#
# IMPORTANT
# ---------
# This cell does NOT:
#
#   - download the full ratings population;
#   - infer historical coverage;
#   - assign governed meanings to fields;
#   - compare ratings with Database v4;
#   - design database structures.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and cache locations.
# ---------------------------------------------------------------------------
#
# Keep the notebook independent of the directory from which Jupyter was started.

PROJECT_ROOT = Path("/home/rob/Documents/inside-rails-horse-racing")

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "ratings_record_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_FILE = (
    CACHE_DIR
    / "latest_modified_rating.json"
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization value.
# ---------------------------------------------------------------------------
#
# Do not use python-dotenv here. The tiny explicit parser keeps secret handling
# visible and dependency-free.
#
# The credential value must never be printed.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Reproduce the exact bounded query pattern used by the public frontend.
# ---------------------------------------------------------------------------
#
# The frontend uses this request to recover the most recently updated ratings
# record.
#
# Requesting one record is sufficient for schema discovery.

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

PARAMS = {
    "per_page": 1,
    "sortby": "modifiedTimestamp:desc",
}

REQUEST_URL = (
    f"{BHA_BASE}/ratings?"
    f"{urlencode(PARAMS)}"
)


# ---------------------------------------------------------------------------
# 4. Reuse exact cached evidence if this request has already been made.
# ---------------------------------------------------------------------------
#
# Avoid repeated external requests during notebook iteration.

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        envelope["request_url"]
        == REQUEST_URL
    )

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 5. Make exactly one authenticated BHA ratings request.
    # -----------------------------------------------------------------------
    #
    # HTTP errors are preserved as source observations rather than allowed to
    # terminate the notebook before the response body is cached.

    request = Request(
        REQUEST_URL,
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/official-ratings/"
                "ratings-database/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = (
                response.headers.get(
                    "Content-Type"
                )
            )

            response_text = (
                response.read().decode(
                    "utf-8",
                    errors="replace",
                )
            )

    except HTTPError as error:
        status = error.code

        content_type = (
            error.headers.get(
                "Content-Type"
            )
        )

        response_text = (
            error.read().decode(
                "utf-8",
                errors="replace",
            )
        )

        transport_error = repr(
            error
        )

    except URLError as error:
        transport_error = repr(
            error
        )


    # -----------------------------------------------------------------------
    # 6. Parse JSON conservatively.
    # -----------------------------------------------------------------------
    #
    # An unsuccessful or non-JSON response must remain distinguishable from an
    # empty ratings result.

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # 7. Persist the exact response evidence without the Authorization value.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Official Ratings"
        ),
        "request_url": REQUEST_URL,
        "request_parameters": PARAMS,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    # Atomic replacement prevents a partial cache file from later looking like
    # valid source evidence.
    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 8. Report transport evidence before interpreting record structure.
# ---------------------------------------------------------------------------

print("BHA OFFICIAL RATINGS — ONE-RECORD PROBE")
print("=======================================")
print("Loaded from:", source)
print(
    "HTTP status:",
    envelope["response_status"],
)
print(
    "Content-Type:",
    envelope["content_type"],
)

if envelope["transport_error"]:
    print(
        "Transport observation:",
        envelope["transport_error"],
    )


# ---------------------------------------------------------------------------
# 9. Inspect the response structure without assuming it matches fixture APIs.
# ---------------------------------------------------------------------------
#
# The frontend expects pagination metadata, but the notebook should observe the
# actual response rather than imposing that schema.

payload = envelope["parsed_json"]

if payload is None:
    print("\nParsed JSON: NO")

    if envelope["response_text"]:
        print(
            "\nBounded raw-response preview:"
        )

        print(
            envelope["response_text"][:5000]
        )

else:
    print("\nParsed JSON: YES")
    print(
        "Top-level type:",
        type(payload).__name__,
    )

    if isinstance(
        payload,
        dict,
    ):
        print(
            "Top-level keys:",
            sorted(
                payload.keys()
            ),
        )


        # -------------------------------------------------------------------
        # 10. Separate pagination/scalar metadata from the actual record list.
        # -------------------------------------------------------------------

        scalar_metadata = {
            key: value
            for key, value in payload.items()
            if not isinstance(
                value,
                (dict, list),
            )
        }

        print(
            "\nTop-level scalar metadata:"
        )
        print(
            scalar_metadata
        )


        # -------------------------------------------------------------------
        # 11. Inspect returned rating records.
        # -------------------------------------------------------------------

        data = payload.get(
            "data"
        )

        print(
            "\ndata type:",
            type(data).__name__,
        )

        if isinstance(
            data,
            list,
        ):
            print(
                "data rows:",
                len(data),
            )

            dictionary_rows = [
                row
                for row in data
                if isinstance(
                    row,
                    dict,
                )
            ]

            if dictionary_rows:
                # Use the union of observed fields rather than trusting only the
                # first row as a complete schema definition.
                field_union = sorted(
                    {
                        key
                        for row in dictionary_rows
                        for key in row.keys()
                    }
                )

                print(
                    "\nUnion of rating-record fields:"
                )
                print(
                    field_union
                )

                print(
                    "\nRepresentative rating record:"
                )

                print(
                    json.dumps(
                        dictionary_rows[0],
                        indent=2,
                        ensure_ascii=False,
                    )[:12_000]
                )

            elif data:
                print(
                    "\nFirst returned data item:"
                )
                print(
                    repr(
                        data[0]
                    )[:5000]
                )

            else:
                print(
                    "Ratings resource returned an empty data list."
                )

        elif isinstance(
            data,
            dict,
        ):
            print(
                "data keys:",
                sorted(
                    data.keys()
                ),
            )

            print(
                "\nBounded data preview:"
            )

            print(
                json.dumps(
                    data,
                    indent=2,
                    ensure_ascii=False,
                )[:12_000]
            )


    elif isinstance(
        payload,
        list,
    ):
        print(
            "Rows:",
            len(payload),
        )

        if (
            payload
            and isinstance(
                payload[0],
                dict,
            )
        ):
            print(
                "Union of fields:",
                sorted(
                    {
                        key
                        for row in payload
                        if isinstance(
                            row,
                            dict,
                        )
                        for key in row.keys()
                    }
                ),
            )

            print(
                "\nRepresentative record:"
            )

            print(
                json.dumps(
                    payload[0],
                    indent=2,
                    ensure_ascii=False,
                )[:12_000]
            )


# ---------------------------------------------------------------------------
# 12. State scope and provenance explicitly.
# ---------------------------------------------------------------------------

print("\nPROVENANCE")
print("==========")
print(
    "Request URL:",
    REQUEST_URL,
)
print(
    "Cache:",
    CACHE_FILE,
)
print(
    "Authorization displayed: NO"
)
print(
    "Authorization written to cache: NO"
)
print(
    "Requested records: 1"
)
print(
    "Full ratings population downloaded: NO"
)

BHA OFFICIAL RATINGS — ONE-RECORD PROBE
Loaded from: network
HTTP status: 200
Content-Type: application/json

Parsed JSON: YES
Top-level type: dict
Top-level keys: ['current_page', 'data', 'first_page_url', 'from', 'last_page', 'last_page_url', 'links', 'next_page_url', 'path', 'per_page', 'prev_page_url', 'to', 'total']

Top-level scalar metadata:
{'current_page': 1, 'first_page_url': 'https://api09.horseracing.software/bha/v1/ratings?page=1', 'from': 1, 'last_page': 11840, 'last_page_url': 'https://api09.horseracing.software/bha/v1/ratings?page=11840', 'next_page_url': 'https://api09.horseracing.software/bha/v1/ratings?page=2', 'path': 'https://api09.horseracing.software/bha/v1/ratings', 'per_page': 1, 'prev_page_url': None, 'to': 1, 'total': 11840}

data type: list
data rows: 1

Union of rating-record fields:
['animalId', 'animalName', 'awtCollateral', 'chaseCollateral', 'dam', 'diffAwt', 'diffChase', 'diffFlat', 'diffHurdle', 'flatCollateral', 'hurdleCollateral', 'lastUpdate', 'rat

## BHA Official Ratings — structured record capability

A one-record probe of the public structured ratings resource succeeded:

`/bha/v1/ratings`

The endpoint returned a paginated JSON response.

### Population

The response reported:

- `total = 11,840`;
- standard pagination metadata;
- one requested record.

The total should currently be treated as the number of records returned by this ratings resource, not automatically as a governed count of unique active British racehorses.

### Observed rating-record fields

The returned record exposed:

#### Horse identity

- `animalId`;
- `animalName`;
- `yof`;
- `sex`.

#### Pedigree

- `sire`;
- `dam`.

#### Published ratings

Separate fields exist for:

- `ratingFlat`;
- `ratingAwt`;
- `ratingChase`;
- `ratingHurdle`.

#### Rating-related companion fields

Each code also has fields such as:

- `diffFlat`;
- `diffAwt`;
- `diffChase`;
- `diffHurdle`;

and:

- `flatCollateral`;
- `awtCollateral`;
- `chaseCollateral`;
- `hurdleCollateral`.

The semantics of the `diff*` and `*Collateral` fields have not yet been established and must not be inferred solely from their names.

#### Trainer

- `trainerId`;
- `trainer`.

#### Source update

- `lastUpdate`.

The most recently modified record returned by the frontend's own sort pattern had:

`lastUpdate = 2026-08-11 07:00:00`

### Source assessment

The structured ratings resource is potentially high-value because it combines:

- official BHA horse identifiers;
- current ratings across multiple racing codes;
- pedigree information;
- trainer identifiers;
- rating-change/collateral fields;
- explicit update information.

It overlaps with several other potential source families, including:

- horse identity;
- pedigree;
- trainers;
- runner ratings.

This overlap should later be compared rather than automatically duplicated.

### Remaining Official Ratings surface

The ratings family also exposes downloadable resources for:

- full published ratings;
- weekly rating changes;
- latest performance figures.

The structured `/ratings` resource has now been materially inspected.

The performance-figures dataset remains a distinct potentially valuable resource whose actual schema has not yet been observed.

## Decision

Before closing the Official Ratings source family, inspect the **latest performance figures** resource just far enough to determine its record grain and fields.

Do not download the complete dataset or begin ratings analysis.

In [26]:
# BHA Official Ratings — bounded Performance Figures export probe
#
# WHAT
# ----
# Inspect the public BHA "Latest Performance Figures" export just far enough to
# determine:
#
#   - whether the resource is directly accessible with the same BHA
#     Authorization used by the public frontend;
#   - its response/file type;
#   - its apparent column schema;
#   - its apparent record grain from a small number of sample rows.
#
# The resource being tested is:
#
#   /bha/v1/ratings/csv/performance-figures
#
# WHY
# ---
# Performance figures are a distinct BHA ratings product and may contain
# race/performance-level information that is materially different from the
# current one-row-per-horse published ratings resource.
#
# Before closing the Official Ratings source-family inventory we need to know
# what this dataset actually looks like.
#
# READS
# -----
# - BHA Authorization credential from repo-root `.env.local`;
# - one public BHA performance-figures export request.
#
# WRITES
# ------
# One ignored research-cache envelope containing ONLY the bounded response
# prefix:
#
#   data/cache/bha_official_source_feasibility/
#       performance_figures_probe/
#       performance_figures_prefix.json
#
# The complete export is deliberately NOT downloaded or cached.
#
# EXPECTED RESULT
# ---------------
# A concise report showing:
#
#   - HTTP status;
#   - Content-Type;
#   - Content-Disposition where supplied;
#   - whether the response exceeded our inspection prefix;
#   - CSV header/field names if recoverable;
#   - first five complete data rows if recoverable.
#
# BOUNDED-ACQUISITION RULE
# ------------------------
# Read at most 64 KiB from the response body.
#
# If the server offers a much larger file, the connection is closed after that
# prefix. This prevents a source-inventory exercise from silently becoming a
# bulk ratings download.
#
# IMPORTANT
# ---------
# A prefix may end part-way through a CSV row. Only complete parsed rows are
# used for the sample.
#
# No field semantics are inferred solely from column names.

from datetime import datetime, timezone
import csv
import io
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "performance_figures_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_FILE = (
    CACHE_DIR
    / "performance_figures_prefix.json"
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the BHA Authorization value without printing or persisting it.
# ---------------------------------------------------------------------------

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the exact performance-figures resource and inspection limit.
# ---------------------------------------------------------------------------
#
# No temporary signed export parameters are required for this probe unless the
# server proves otherwise.
#
# Reading one extra byte beyond the nominal limit lets us distinguish:
#
#   response ended within 64 KiB
#
# from:
#
#   response definitely continued beyond 64 KiB.

REQUEST_URL = (
    "https://api09.horseracing.software/"
    "bha/v1/ratings/csv/performance-figures"
)

PREFIX_LIMIT_BYTES = 64 * 1024
READ_LIMIT_BYTES = PREFIX_LIMIT_BYTES + 1


# ---------------------------------------------------------------------------
# 4. Reuse the exact cached bounded probe if it already exists.
# ---------------------------------------------------------------------------

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        envelope["request_url"]
        == REQUEST_URL
    )

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 5. Make one authenticated request and read only the bounded prefix.
    # -----------------------------------------------------------------------
    #
    # The response object is closed immediately after the prefix read.
    #
    # This means a large export is NOT consumed merely because the endpoint is
    # capable of returning it.

    request = Request(
        REQUEST_URL,
        headers={
            "Authorization": bha_authorization,
            "Accept": (
                "text/csv,"
                "application/csv,"
                "application/octet-stream,"
                "*/*"
            ),
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/official-ratings/"
                "ratings-database/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    final_url = None
    content_type = None
    content_disposition = None
    content_length = None
    prefix_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status
            final_url = response.geturl()

            content_type = response.headers.get(
                "Content-Type"
            )

            content_disposition = response.headers.get(
                "Content-Disposition"
            )

            content_length = response.headers.get(
                "Content-Length"
            )

            # ---------------------------------------------------------------
            # Deliberately consume no more than 64 KiB + one sentinel byte.
            # ---------------------------------------------------------------

            prefix_bytes = response.read(
                READ_LIMIT_BYTES
            )

    except HTTPError as error:
        status = error.code
        final_url = error.geturl()

        content_type = error.headers.get(
            "Content-Type"
        )

        content_disposition = error.headers.get(
            "Content-Disposition"
        )

        content_length = error.headers.get(
            "Content-Length"
        )

        prefix_bytes = error.read(
            READ_LIMIT_BYTES
        )

        transport_error = repr(
            error
        )

    except URLError as error:
        transport_error = repr(
            error
        )


    # -----------------------------------------------------------------------
    # 6. Separate our evidence prefix from the sentinel byte.
    # -----------------------------------------------------------------------

    response_exceeded_prefix = (
        len(prefix_bytes)
        > PREFIX_LIMIT_BYTES
    )

    evidence_bytes = prefix_bytes[
        :PREFIX_LIMIT_BYTES
    ]

    response_text = evidence_bytes.decode(
        "utf-8-sig",
        errors="replace",
    )


    # -----------------------------------------------------------------------
    # 7. Cache only the bounded evidence actually inspected.
    # -----------------------------------------------------------------------
    #
    # The cache explicitly records that this may be a truncated response so it
    # can never be mistaken for a complete performance-figures dataset.

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Official Ratings — Performance Figures"
        ),
        "request_url": REQUEST_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "final_url": final_url,
        "content_type": content_type,
        "content_disposition": content_disposition,
        "content_length_header": content_length,
        "inspection_limit_bytes": PREFIX_LIMIT_BYTES,
        "prefix_bytes_preserved": len(
            evidence_bytes
        ),
        "response_exceeded_prefix": (
            response_exceeded_prefix
        ),
        "response_prefix_text": response_text,
        "transport_error": transport_error,
        "authorization_persisted": False,
        "complete_export_downloaded": False,
    }

    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 8. Present transport/file evidence before parsing the response.
# ---------------------------------------------------------------------------

print(
    "BHA PERFORMANCE FIGURES — BOUNDED EXPORT PROBE"
)
print(
    "=============================================="
)

print(
    "Loaded from:",
    source,
)

print(
    "HTTP status:",
    envelope["response_status"],
)

print(
    "Final URL:",
    envelope["final_url"],
)

print(
    "Content-Type:",
    envelope["content_type"],
)

print(
    "Content-Disposition:",
    envelope["content_disposition"],
)

print(
    "Content-Length header:",
    envelope["content_length_header"],
)

print(
    "Prefix bytes preserved:",
    envelope["prefix_bytes_preserved"],
)

print(
    "Response exceeds 64 KiB:",
    (
        "YES"
        if envelope["response_exceeded_prefix"]
        else "NO / NOT DEMONSTRATED"
    ),
)

if envelope["transport_error"]:
    print(
        "Transport observation:",
        envelope["transport_error"],
    )


# ---------------------------------------------------------------------------
# 9. Attempt bounded CSV parsing.
# ---------------------------------------------------------------------------
#
# The response prefix may terminate inside its final row.
#
# To avoid presenting a truncated record as genuine data, drop the final text
# fragment whenever the response demonstrably continued beyond our prefix.

response_text = envelope[
    "response_prefix_text"
]

if envelope["response_exceeded_prefix"]:
    last_newline = response_text.rfind(
        "\n"
    )

    parseable_text = (
        response_text[: last_newline + 1]
        if last_newline != -1
        else ""
    )

else:
    parseable_text = response_text


# ---------------------------------------------------------------------------
# 10. Detect obvious non-CSV responses before interpreting fields.
# ---------------------------------------------------------------------------

stripped = parseable_text.lstrip()

looks_like_html = (
    stripped.lower().startswith("<!doctype html")
    or stripped.lower().startswith("<html")
)

looks_like_json = (
    stripped.startswith("{")
    or stripped.startswith("[")
)


if not parseable_text.strip():
    print(
        "\nNo parseable response body recovered."
    )


elif looks_like_html:
    print(
        "\nResponse appears to be HTML rather than CSV."
    )

    print(
        "\nBounded response preview:"
    )

    print(
        parseable_text[:5000]
    )


elif looks_like_json:
    print(
        "\nResponse appears to be JSON rather than CSV."
    )

    try:
        json_payload = json.loads(
            parseable_text
        )

        print(
            json.dumps(
                json_payload,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

    except json.JSONDecodeError:
        print(
            parseable_text[:5000]
        )


else:
    # -----------------------------------------------------------------------
    # 11. Parse the bounded CSV prefix using Python's CSV parser.
    # -----------------------------------------------------------------------

    reader = csv.DictReader(
        io.StringIO(
            parseable_text
        )
    )

    fieldnames = (
        reader.fieldnames
        or []
    )

    sample_rows = []

    for row in reader:
        # Ignore completely blank rows.
        if not any(
            value not in (None, "")
            for value in row.values()
        ):
            continue

        sample_rows.append(
            row
        )

        if len(sample_rows) >= 5:
            break


    print(
        "\nCSV PARSE"
    )
    print(
        "========="
    )

    print(
        "Field count:",
        len(fieldnames),
    )

    print(
        "Fields:"
    )

    for field in fieldnames:
        print(
            " -",
            repr(field),
        )

    print(
        "\nComplete sample rows recovered:",
        len(sample_rows),
    )

    for index, row in enumerate(
        sample_rows,
        start=1,
    ):
        print(
            f"\nSample row {index}"
        )
        print(
            "-" * 40
        )

        print(
            json.dumps(
                row,
                indent=2,
                ensure_ascii=False,
            )
        )


# ---------------------------------------------------------------------------
# 12. Preserve the analytical boundary.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Request URL:",
    REQUEST_URL,
)

print(
    "Cache:",
    CACHE_FILE,
)

print(
    "Maximum response body read:",
    READ_LIMIT_BYTES,
    "bytes",
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Complete performance-figures export downloaded: NO"
)

print(
    "Database writes: NONE"
)

BHA PERFORMANCE FIGURES — BOUNDED EXPORT PROBE
Loaded from: network
HTTP status: 403
Final URL: https://api09.horseracing.software/bha/v1/ratings/csv/performance-figures
Content-Type: application/json
Content-Disposition: None
Content-Length header: None
Prefix bytes preserved: 51
Response exceeds 64 KiB: NO / NOT DEMONSTRATED
Transport observation: <HTTPError 403: 'Forbidden'>

Response appears to be JSON rather than CSV.
{
  "error": "URL signature required: missing expires"
}

PROVENANCE
Request URL: https://api09.horseracing.software/bha/v1/ratings/csv/performance-figures
Cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/performance_figures_probe/performance_figures_prefix.json
Maximum response body read: 65537 bytes
Authorization displayed: NO
Authorization written to cache: NO
Complete performance-figures export downloaded: NO
Database writes: NONE


In [27]:
# BHA Official Ratings — signed Performance Figures export probe
#
# WHAT
# ----
# Follow the public BHA Ratings Database's own download mechanism for the
# "Latest Performance Figures" export.
#
# The previous direct request to:
#
#   /bha/v1/ratings/csv/performance-figures
#
# returned:
#
#   HTTP 403
#   {"error": "URL signature required: missing expires"}
#
# This demonstrates that the export requires a temporary signed URL.
#
# This cell therefore:
#
#   1. fetches the public Ratings Database page;
#   2. finds the actual performance-figures download link generated there;
#   3. verifies that temporary signing parameters are present;
#   4. keeps the complete signed URL only in memory;
#   5. performs one bounded request to that signed URL;
#   6. reads at most 64 KiB + one sentinel byte;
#   7. inventories the apparent CSV schema and a few sample records.
#
# WHY
# ---
# We are not trying to bypass the BHA's access mechanism.
#
# We are reproducing the download path exposed by its own public webpage so we
# can establish what information the Performance Figures product contains.
#
# READS
# -----
# Two public BHA resources at most:
#
#   1. Ratings Database HTML page;
#   2. the temporary signed Performance Figures URL found on that page.
#
# The BHA Authorization credential is NOT required for the signed export link
# and is deliberately not sent. A normal browser following an <a href> download
# link would not attach the Angular `$http` Authorization header either.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       performance_figures_probe/
#
# Specifically:
#
#   ratings_page_signed_link_discovery.json
#   signed_performance_figures_prefix.json
#
# The ratings-page cache contains a SANITISED version of the HTML in which
# temporary signature values are replaced with `[REDACTED]`.
#
# The complete signed URL is NEVER:
#
#   - printed;
#   - written to cache;
#   - included in an exception message;
#   - retained after the cell finishes.
#
# EXPECTED RESULT
# ---------------
# If the current public page exposes a usable signed link:
#
#   - HTTP/file metadata for the export;
#   - whether the export exceeds our 64 KiB inspection boundary;
#   - CSV column names;
#   - up to five complete sample rows.
#
# If no signed link is present, record that as unresolved rather than guessing.
#
# BOUNDED ACQUISITION
# -------------------
# The signed export response is read only up to:
#
#   64 KiB + 1 byte.
#
# The extra byte is solely a sentinel allowing us to demonstrate that the
# response continues beyond the retained prefix.
#
# This is source discovery, NOT acquisition of the complete dataset.

from datetime import datetime, timezone
import base64
import csv
import hashlib
import html
import io
import json
import re
import subprocess
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import (
    parse_qs,
    urljoin,
    urlsplit,
)
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "performance_figures_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "ratings_page_signed_link_discovery.json"
)

EXPORT_CACHE_FILE = (
    CACHE_DIR
    / "signed_performance_figures_prefix.json"
)


# ---------------------------------------------------------------------------
# 2. Define the public page and exact expected export path.
# ---------------------------------------------------------------------------
#
# Matching the path exactly prevents an unrelated ratings link containing the
# words "performance-figures" from being treated as the desired export.

RATINGS_PAGE_URL = (
    "https://www.britishhorseracing.com/"
    "regulation/official-ratings/ratings-database/"
)

EXPECTED_EXPORT_PATH = (
    "/bha/v1/ratings/csv/performance-figures"
)

PREFIX_LIMIT_BYTES = 64 * 1024
READ_LIMIT_BYTES = PREFIX_LIMIT_BYTES + 1


# ---------------------------------------------------------------------------
# 3. Helper: remove temporary URL credentials from text before persistence.
# ---------------------------------------------------------------------------
#
# The public HTML may contain short-lived values such as:
#
#   expires=...
#   sig=...
#
# Preserve the existence and parameter names while destroying their values.
#
# Include a few common token-name variants defensively so a future page change
# cannot accidentally make this cache persist an operational credential.

def redact_temporary_url_values(text):
    return re.sub(
        r"""(?i)((?:expires|sig|signature|token)=)[^&"'<> \t\r\n]+""",
        r"\1[REDACTED]",
        text,
    )


# ---------------------------------------------------------------------------
# 4. Fetch the Ratings Database HTML using the browser-like curl pattern that
#    successfully returned HTTP 200 in the preceding frontend-discovery cell.
# ---------------------------------------------------------------------------
#
# We deliberately do NOT use `--fail`: an HTTP block is evidence that should be
# recorded, not converted into an unexplained notebook exception.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"

page_result = subprocess.run(
    [
        "curl",
        "-L",
        "--silent",
        "--show-error",
        "--compressed",
        "--max-time",
        "30",
        "--user-agent",
        (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/140.0 Safari/537.36"
        ),
        "--header",
        (
            "Accept: text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
        "--header",
        "Accept-Language: en-GB,en;q=0.9",
        "--write-out",
        (
            f"\n{STATUS_MARKER}%{{http_code}}"
            f"\n{URL_MARKER}%{{url_effective}}"
        ),
        RATINGS_PAGE_URL,
    ],
    capture_output=True,
    text=True,
    check=False,
)

page_output = page_result.stdout

status_position = page_output.rfind(
    f"\n{STATUS_MARKER}"
)

final_url_position = page_output.rfind(
    f"\n{URL_MARKER}"
)

assert status_position != -1, (
    "Could not recover HTTP status from Ratings Database request."
)

assert final_url_position != -1, (
    "Could not recover final URL from Ratings Database request."
)

page_html = page_output[
    :status_position
]

page_status_text = page_output[
    status_position + len(f"\n{STATUS_MARKER}") :
    final_url_position
].strip()

page_final_url = page_output[
    final_url_position + len(f"\n{URL_MARKER}") :
].strip()

page_status = int(
    page_status_text
)


# ---------------------------------------------------------------------------
# 5. Extract only links whose URL path exactly matches the Performance Figures
#    export resource.
# ---------------------------------------------------------------------------
#
# HTML entities such as `&amp;` are decoded before query inspection.
#
# Complete candidate URLs exist only in memory.

href_values = re.findall(
    r"""<a\b[^>]*\bhref\s*=\s*["']([^"']+)["']""",
    page_html,
    flags=re.IGNORECASE | re.DOTALL,
)

candidate_urls = []

for href in href_values:
    decoded_href = html.unescape(
        href
    )

    absolute_url = urljoin(
        page_final_url or RATINGS_PAGE_URL,
        decoded_href,
    )

    parsed = urlsplit(
        absolute_url
    )

    if parsed.path == EXPECTED_EXPORT_PATH:
        candidate_urls.append(
            absolute_url
        )

# Preserve order while removing exact duplicates.
candidate_urls = list(
    dict.fromkeys(
        candidate_urls
    )
)


# ---------------------------------------------------------------------------
# 6. Convert candidate URLs into SECRET-SAFE discovery metadata.
# ---------------------------------------------------------------------------
#
# Store:
#
#   - host;
#   - path;
#   - query parameter NAMES.
#
# Do not store query parameter values.

safe_candidates = []

for candidate_url in candidate_urls:
    parsed = urlsplit(
        candidate_url
    )

    query = parse_qs(
        parsed.query,
        keep_blank_values=True,
    )

    safe_candidates.append(
        {
            "host": parsed.hostname,
            "path": parsed.path,
            "query_parameter_names": sorted(
                query.keys()
            ),
            "has_expires": (
                "expires" in query
            ),
            "has_sig": (
                "sig" in query
            ),
        }
    )


# ---------------------------------------------------------------------------
# 7. Cache the page-request evidence with temporary credentials redacted.
# ---------------------------------------------------------------------------
#
# Keep a SHA-256 of the original in-memory response for provenance, but persist
# only sanitised HTML.
#
# This satisfies the research rule that external requests leave cached evidence
# while avoiding persistence of the temporary signed download credential.

page_envelope = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Official Ratings — Performance Figures"
    ),
    "request_url": RATINGS_PAGE_URL,
    "retrieved_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "response_status": page_status,
    "final_url": page_final_url,
    "curl_return_code": page_result.returncode,
    "stderr": page_result.stderr.strip(),
    "original_html_sha256": hashlib.sha256(
        page_html.encode(
            "utf-8"
        )
    ).hexdigest(),
    "sanitised_response_html": (
        redact_temporary_url_values(
            page_html
        )
    ),
    "performance_figures_link_count": len(
        candidate_urls
    ),
    "safe_candidate_metadata": safe_candidates,
    "temporary_query_values_persisted": False,
}

page_temp = PAGE_CACHE_FILE.with_suffix(
    ".tmp"
)

page_temp.write_text(
    json.dumps(
        page_envelope,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

page_temp.replace(
    PAGE_CACHE_FILE
)


# ---------------------------------------------------------------------------
# 8. Report link-discovery evidence without printing any candidate URL.
# ---------------------------------------------------------------------------

print(
    "BHA PERFORMANCE FIGURES — SIGNED LINK DISCOVERY"
)
print(
    "==============================================="
)

print(
    "Ratings page HTTP status:",
    page_status,
)

print(
    "Performance Figures links found:",
    len(candidate_urls),
)

for index, metadata in enumerate(
    safe_candidates,
    start=1,
):
    print(
        f"\nCandidate {index}"
    )
    print(
        "  Host:",
        metadata["host"],
    )
    print(
        "  Path:",
        metadata["path"],
    )
    print(
        "  Query parameter names:",
        metadata["query_parameter_names"],
    )
    print(
        "  expires present:",
        "YES" if metadata["has_expires"] else "NO",
    )
    print(
        "  sig present:",
        "YES" if metadata["has_sig"] else "NO",
    )


# ---------------------------------------------------------------------------
# 9. Select a usable signed URL conservatively.
# ---------------------------------------------------------------------------
#
# A URL is considered usable only when:
#
#   - its path is the expected Performance Figures path;
#   - it contains BOTH `expires` and `sig`.
#
# If several identical/rendered links exist, use the first usable one.
#
# Do not invent missing signing parameters.

usable_signed_urls = []

for candidate_url in candidate_urls:
    parsed = urlsplit(
        candidate_url
    )

    query = parse_qs(
        parsed.query,
        keep_blank_values=True,
    )

    if (
        parsed.path == EXPECTED_EXPORT_PATH
        and "expires" in query
        and "sig" in query
    ):
        usable_signed_urls.append(
            candidate_url
        )


if not usable_signed_urls:
    print(
        "\nUSABLE SIGNED LINK: NO"
    )
    print(
        "No export request made. "
        "Signing mechanism remains unresolved from this page response."
    )

else:
    print(
        "\nUSABLE SIGNED LINK: YES"
    )

    # -----------------------------------------------------------------------
    # 10. Keep the complete signed URL in memory only.
    # -----------------------------------------------------------------------
    #
    # No Authorization header is sent.
    #
    # The signed URL itself is the public webpage's browser-download mechanism,
    # and an ordinary anchor navigation cannot attach Angular's custom
    # Authorization header.

    signed_url = usable_signed_urls[0]

    signed_request = Request(
        signed_url,
        headers={
            "Accept": (
                "text/csv,"
                "application/csv,"
                "application/octet-stream,"
                "*/*"
            ),
            "Referer": RATINGS_PAGE_URL,
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
        },
    )

    status = None
    final_url = None
    content_type = None
    content_disposition = None
    content_length = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            signed_request,
            timeout=30,
        ) as response:

            status = response.status

            # Do not persist or print response.geturl(), because it contains
            # the same signed query parameters.
            final_url = response.geturl()

            content_type = response.headers.get(
                "Content-Type"
            )

            content_disposition = response.headers.get(
                "Content-Disposition"
            )

            content_length = response.headers.get(
                "Content-Length"
            )

            response_bytes = response.read(
                READ_LIMIT_BYTES
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        content_disposition = error.headers.get(
            "Content-Disposition"
        )

        content_length = error.headers.get(
            "Content-Length"
        )

        response_bytes = error.read(
            READ_LIMIT_BYTES
        )

        # Do not use repr(error), because some HTTP exception representations
        # can retain/request the complete signed URL.
        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        # Store only the reason class/text, never the request URL.
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # 11. Apply the 64 KiB evidence boundary.
    # -----------------------------------------------------------------------

    response_exceeded_prefix = (
        len(response_bytes)
        > PREFIX_LIMIT_BYTES
    )

    evidence_bytes = response_bytes[
        :PREFIX_LIMIT_BYTES
    ]


    # -----------------------------------------------------------------------
    # 12. Cache the bounded export response WITHOUT the signed request URL.
    # -----------------------------------------------------------------------
    #
    # Base64 preserves the exact inspected bytes regardless of whether the
    # server returns CSV, JSON or another file type.

    export_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Official Ratings — Performance Figures"
        ),
        "request_host": (
            urlsplit(
                signed_url
            ).hostname
        ),
        "request_path": EXPECTED_EXPORT_PATH,
        "request_query_parameter_names": sorted(
            parse_qs(
                urlsplit(
                    signed_url
                ).query,
                keep_blank_values=True,
            ).keys()
        ),
        "signed_query_values_persisted": False,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "content_disposition": content_disposition,
        "content_length_header": content_length,
        "inspection_limit_bytes": PREFIX_LIMIT_BYTES,
        "prefix_bytes_preserved": len(
            evidence_bytes
        ),
        "response_exceeded_prefix": (
            response_exceeded_prefix
        ),
        "response_prefix_base64": (
            base64.b64encode(
                evidence_bytes
            ).decode(
                "ascii"
            )
        ),
        "transport_error": transport_error,
        "complete_export_downloaded": False,
        "authorization_sent": False,
    }

    export_temp = EXPORT_CACHE_FILE.with_suffix(
        ".tmp"
    )

    export_temp.write_text(
        json.dumps(
            export_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    export_temp.replace(
        EXPORT_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 13. Report transport/file evidence without exposing the signed URL.
    # -----------------------------------------------------------------------

    print(
        "\nSIGNED EXPORT RESPONSE"
    )
    print(
        "======================"
    )

    print(
        "HTTP status:",
        status,
    )

    print(
        "Request path:",
        EXPECTED_EXPORT_PATH,
    )

    print(
        "Signed query values displayed: NO"
    )

    print(
        "Content-Type:",
        content_type,
    )

    print(
        "Content-Disposition:",
        content_disposition,
    )

    print(
        "Content-Length header:",
        content_length,
    )

    print(
        "Prefix bytes preserved:",
        len(
            evidence_bytes
        ),
    )

    print(
        "Response exceeds 64 KiB:",
        (
            "YES"
            if response_exceeded_prefix
            else "NO / NOT DEMONSTRATED"
        ),
    )

    if transport_error:
        print(
            "Transport observation:",
            transport_error,
        )


    # -----------------------------------------------------------------------
    # 14. Decode the bounded evidence for format/schema inspection.
    # -----------------------------------------------------------------------

    response_text = evidence_bytes.decode(
        "utf-8-sig",
        errors="replace",
    )

    if response_exceeded_prefix:
        # The final retained bytes may end halfway through a CSV record.
        # Restrict parsing to complete newline-terminated content.
        final_newline = response_text.rfind(
            "\n"
        )

        parseable_text = (
            response_text[
                :final_newline + 1
            ]
            if final_newline != -1
            else ""
        )

    else:
        parseable_text = response_text


    stripped = parseable_text.lstrip()

    looks_like_json = (
        stripped.startswith("{")
        or stripped.startswith("[")
    )

    looks_like_html = (
        stripped.lower().startswith(
            "<!doctype html"
        )
        or stripped.lower().startswith(
            "<html"
        )
    )


    # -----------------------------------------------------------------------
    # 15. Interpret only enough to establish the export's apparent schema.
    # -----------------------------------------------------------------------

    if not parseable_text.strip():
        print(
            "\nNo parseable response body recovered."
        )

    elif looks_like_json:
        print(
            "\nResponse appears to be JSON rather than CSV."
        )

        try:
            payload = json.loads(
                parseable_text
            )

            print(
                json.dumps(
                    payload,
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )

        except json.JSONDecodeError:
            print(
                parseable_text[:5000]
            )

    elif looks_like_html:
        print(
            "\nResponse appears to be HTML rather than CSV."
        )

        print(
            parseable_text[:5000]
        )

    else:
        reader = csv.DictReader(
            io.StringIO(
                parseable_text
            )
        )

        fieldnames = (
            reader.fieldnames
            or []
        )

        sample_rows = []

        for row in reader:
            if not any(
                value not in (None, "")
                for value in row.values()
            ):
                continue

            sample_rows.append(
                row
            )

            if len(sample_rows) >= 5:
                break


        print(
            "\nCSV SCHEMA"
        )
        print(
            "=========="
        )

        print(
            "Field count:",
            len(fieldnames),
        )

        print(
            "Fields:"
        )

        for field in fieldnames:
            print(
                " -",
                repr(field),
            )

        print(
            "\nComplete sample rows recovered:",
            len(sample_rows),
        )

        for index, row in enumerate(
            sample_rows,
            start=1,
        ):
            print(
                f"\nSample row {index}"
            )
            print(
                "-" * 40
            )

            print(
                json.dumps(
                    row,
                    indent=2,
                    ensure_ascii=False,
                )
            )


    # -----------------------------------------------------------------------
    # 16. Remove references to the complete signed URL as soon as it is no
    #     longer needed.
    # -----------------------------------------------------------------------

    del signed_request
    del signed_url
    del usable_signed_urls


# ---------------------------------------------------------------------------
# 17. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Ratings-page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Signed-export cache:",
    EXPORT_CACHE_FILE,
)

print(
    "Temporary signature values printed: NO"
)

print(
    "Temporary signature values persisted: NO"
)

print(
    "BHA Authorization sent to export: NO"
)

print(
    "Maximum signed response body read:",
    READ_LIMIT_BYTES,
    "bytes",
)

print(
    "Complete Performance Figures export downloaded: NO"
)

print(
    "Database writes: NONE"
)

BHA PERFORMANCE FIGURES — SIGNED LINK DISCOVERY
Ratings page HTTP status: 200
Performance Figures links found: 1

Candidate 1
  Host: api09.horseracing.software
  Path: /bha/v1/ratings/csv/performance-figures
  Query parameter names: ['expires', 'sig']
  expires present: YES
  sig present: YES

USABLE SIGNED LINK: YES

SIGNED EXPORT RESPONSE
HTTP status: 200
Request path: /bha/v1/ratings/csv/performance-figures
Signed query values displayed: NO
Content-Type: text/csv; charset=UTF-8
Content-Disposition: attachment; filename="performance-figures.csv"
Content-Length header: None
Prefix bytes preserved: 65536
Response exceeds 64 KiB: YES

CSV SCHEMA
Field count: 10
Fields:
 - 'Racehorse'
 - 'YOF'
 - 'Sex'
 - 'Trainer'
 - 'Latest'
 - '2 runs ago'
 - '3 runs ago'
 - '4 runs ago'
 - '5 runs ago'
 - '6 runs ago'

Complete sample rows recovered: 5

Sample row 1
----------------------------------------
{
  "Racehorse": "A BEAR AFFAIR (IRE)",
  "YOF": "2024",
  "Sex": "COLT",
  "Trainer": "Richar

## BHA Official Ratings — source-family conclusion

The Official Ratings investigation has now demonstrated multiple distinct
public BHA data products rather than a single ratings table.

### 1. Structured current-ratings resource

The public ratings frontend is powered by:

`/bha/v1/ratings`

The resource is paginated and searchable.

A bounded request reported 11,840 records at the time of observation.

Observed record fields included:

#### Horse identity

- `animalId`;
- `animalName`;
- `yof`;
- `sex`.

#### Pedigree

- `sire`;
- `dam`.

#### Current published ratings

- `ratingFlat`;
- `ratingAwt`;
- `ratingChase`;
- `ratingHurdle`.

#### Rating-change / collateral fields

- `diffFlat`;
- `diffAwt`;
- `diffChase`;
- `diffHurdle`;
- `flatCollateral`;
- `awtCollateral`;
- `chaseCollateral`;
- `hurdleCollateral`.

Their exact semantics have not all been established and must not be inferred
from field names alone.

#### Trainer

- `trainerId`;
- `trainer`.

#### Update information

- `lastUpdate`.

The endpoint therefore provides considerably more than a horse name and one
headline handicap rating.

---

### 2. Full published-ratings export

The public interface exposes a signed downloadable ratings export through:

`/bha/v1/ratings/csv/ratings`

This has not been bulk-downloaded during source discovery.

---

### 3. Weekly rating-change export

The same ratings export family is used by the public site to provide weekly
rating changes through an additional query-level distinction.

The exact historical depth and semantics of that export have not yet been
tested.

---

### 4. Latest Performance Figures export

The public ratings page provides a temporary signed URL for:

`/bha/v1/ratings/csv/performance-figures`

A bounded 64 KiB prefix was successfully retrieved as CSV.

The observed schema contained ten columns:

- `Racehorse`;
- `YOF`;
- `Sex`;
- `Trainer`;
- `Latest`;
- `2 runs ago`;
- `3 runs ago`;
- `4 runs ago`;
- `5 runs ago`;
- `6 runs ago`.

The apparent export grain is therefore:

> one horse row containing up to six recent performance-figure observations.

It is **not**, from the observed export schema, a race-keyed performance-history
table.

No race identifier, race date or explicit race reference was present in the
observed columns.

Consequently the export alone does not provide a direct governed link between
each figure and the race in which that figure was achieved.

### Performance-figure semantics

BHA documentation states that a performance figure represents the level the
handicappers believe a horse achieved in an individual race and that
performance figures are expressed on the same scale as handicap ratings.

BHA further states that it retains performance figures for each run of every
horse's career.

Source:

https://www.britishhorseracing.com/regulation/performance-figures/

The BHA handicapping guide defines its code annotation as:

- `T` — Turf;
- `A` — AW;
- `S` — Steeplechase;
- `H` — Hurdle.

Source:

https://www.britishhorseracing.com/regulation/guide-to-handicapping/

The observed Performance Figures export contained values such as:

- `T:83`;
- `A:51`;
- `S:106`.

These are consistent with the BHA's published code convention, but an
export-specific definition of the prefix format has not yet been demonstrated.

Values such as:

- `T:x`;
- `A:x`;
- `S:0`;

were also observed.

The precise meanings of `x` and `0` in this export remain unresolved and must
not be guessed.

---

## Source assessment

**Potential value: high.**

The Official Ratings family potentially contributes:

- official BHA horse identifiers;
- official current handicap ratings by racing code;
- pedigree information;
- trainer identifiers;
- rating-change information;
- collateral-rating information;
- recent BHA performance figures;
- explicit update provenance.

There is substantial overlap with information likely available through horse,
trainer and race/result resources.

That overlap should be evaluated before deciding what should ultimately be
governed inside Inside Rails.

### Important follow-on clue

The BHA states that performance figures are retained for **each run of every
horse's career**, while the downloadable Performance Figures file exposes only
the latest six observations and no explicit race identifiers.

Therefore the subsequent horse-profile/database investigation should check
whether the public horse resource exposes the richer race-by-race performance
history and its associated identifiers.

## Decision

Official Ratings is sufficiently mapped for the site-wide source inventory.

Do not bulk-download the ratings population or continue investigating rating
semantics during this phase.

Move to the **BHA horse database/profile source family**.

In [29]:
# BHA Horse Database/Profile — frontend route discovery
#
# WHAT
# ----
# Inspect the dedicated JavaScript asset used by the BHA public racehorse pages:
#
#   /wp-content/themes/bha/library/js/angular/pages/racehorses.js?ver=1.3
#
# The aim is to identify the structured resources used by the horse search and
# horse-profile interface BEFORE requesting any horse records.
#
# Specifically this cell will:
#
#   - fetch exactly one public JavaScript asset;
#   - cache the response for reproducible research evidence;
#   - identify controller/factory/service names;
#   - identify `/bha/v1/...` route strings;
#   - identify routes assembled with `apiaddress`;
#   - print bounded source context around HTTP calls and horse-related routes.
#
# WHY
# ---
# The site-wide source inventory is trying to establish what official BHA
# information is publicly exposed.
#
# We should therefore let the public frontend tell us which horse resources
# exist rather than guessing endpoint names.
#
# Of particular interest is whether the horse profile surface appears to expose:
#
#   - horse identity;
#   - pedigree;
#   - ownership/training;
#   - rating history;
#   - career/race history;
#   - future entries;
#   - performance figures;
#   - disciplinary/stewards links.
#
# This cell does NOT assume those capabilities exist. It only inventories what
# the frontend code actually references.
#
# READS
# -----
# One public BHA frontend asset:
#
#   https://www.britishhorseracing.com/
#       wp-content/themes/bha/library/js/angular/pages/racehorses.js?ver=1.3
#
# WRITES
# ------
# One ignored research-cache envelope:
#
#   data/cache/bha_official_source_feasibility/
#       horse_frontend_discovery/
#       racehorses_js_response.json
#
# The response is public JavaScript and contains no BHA Authorization value from
# `.env.local`; that local credential is not read by this cell.
#
# EXPECTED RESULT
# ---------------
# A bounded inventory showing:
#
#   - HTTP status;
#   - asset SHA-256;
#   - Angular controller/factory/service names;
#   - explicit BHA API route strings;
#   - route-building expressions involving `apiaddress`;
#   - bounded code contexts around relevant HTTP requests.
#
# DECISION BOUNDARY
# -----------------
# Do not call any newly discovered horse endpoint in this cell.
#
# First establish the route surface. Then choose the smallest useful horse
# request in the following notebook cell.

from datetime import datetime, timezone
import hashlib
import json
import re
import subprocess
from pathlib import Path


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored cache locations.
# ---------------------------------------------------------------------------
#
# Notebook execution must not depend on Jupyter's current working directory.

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_FILE = (
    CACHE_DIR
    / "racehorses_js_response.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact frontend asset observed on the BHA racehorse pages.
# ---------------------------------------------------------------------------

RACEHORSES_JS_URL = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/pages/"
    "racehorses.js?ver=1.3"
)


# ---------------------------------------------------------------------------
# 3. Reuse cached evidence if this exact resource was already requested.
# ---------------------------------------------------------------------------
#
# The cache stores the complete public JavaScript response because this request
# itself is part of the research evidence.
#
# No local credential is read or sent.

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        envelope["request_url"]
        == RACEHORSES_JS_URL
    )

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 4. Fetch exactly one public JavaScript asset.
    # -----------------------------------------------------------------------
    #
    # Do not use `curl --fail`.
    #
    # If the server returns a non-200 response, preserve that as evidence rather
    # than crashing before the response is cached.

    STATUS_MARKER = "__BHA_HTTP_STATUS__"
    URL_MARKER = "__BHA_FINAL_URL__"

    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            "Accept: application/javascript,text/javascript,*/*;q=0.8",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            RACEHORSES_JS_URL,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    assert status_position != -1, (
        "Could not recover HTTP status from racehorses.js request."
    )

    assert final_url_position != -1, (
        "Could not recover final URL from racehorses.js request."
    )

    response_text = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    response_status = int(
        status_text
    )


    # -----------------------------------------------------------------------
    # 5. Check whether the public script unexpectedly contains a literal
    #    Bearer credential before persisting it.
    # -----------------------------------------------------------------------
    #
    # The dedicated page script is expected to reference shared configuration,
    # not contain a credential itself.
    #
    # If that assumption changes, preserve a redacted response rather than
    # writing an operational credential to disk.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            response_text,
            flags=re.IGNORECASE,
        )
    )

    cached_response_text = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        response_text,
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # 6. Cache the external-request evidence atomically.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "request_url": RACEHORSES_JS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": response_status,
        "final_url": final_url,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
        "content_sha256": hashlib.sha256(
            response_text.encode(
                "utf-8"
            )
        ).hexdigest(),
        "response_text": cached_response_text,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_from_env_sent": False,
    }

    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 7. Report transport/provenance before interpreting JavaScript.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE DATABASE/PROFILE — FRONTEND ROUTE DISCOVERY"
)
print(
    "====================================================="
)

print(
    "Loaded from:",
    source,
)

print(
    "HTTP status:",
    envelope["response_status"],
)

print(
    "Final URL:",
    envelope["final_url"],
)

print(
    "SHA-256:",
    envelope["content_sha256"],
)

print(
    "Literal Bearer credential found in asset:",
    (
        "YES — REDACTED IN CACHE"
        if envelope["literal_bearer_was_present"]
        else "NO"
    ),
)


# ---------------------------------------------------------------------------
# 8. Stop interpretation cleanly if the asset was unavailable.
# ---------------------------------------------------------------------------

script_text = envelope[
    "response_text"
]

if (
    envelope["response_status"] != 200
    or not script_text.strip()
):
    print(
        "\nFrontend asset unavailable."
    )

    print(
        "No horse-resource conclusions should be drawn from this response."
    )

else:
    # -----------------------------------------------------------------------
    # 9. Inventory Angular components defined by this dedicated page asset.
    # -----------------------------------------------------------------------
    #
    # Controller/factory/service names help distinguish:
    #
    #   - horse search;
    #   - horse profile;
    #   - related profile widgets;
    #
    # without assigning semantics solely from URL names.

    controller_names = sorted(
        set(
            re.findall(
                r"""\.controller\(\s*['"]([^'"]+)['"]""",
                script_text,
                flags=re.IGNORECASE,
            )
        )
    )

    factory_names = sorted(
        set(
            re.findall(
                r"""\.factory\(\s*['"]([^'"]+)['"]""",
                script_text,
                flags=re.IGNORECASE,
            )
        )
    )

    service_names = sorted(
        set(
            re.findall(
                r"""\.service\(\s*['"]([^'"]+)['"]""",
                script_text,
                flags=re.IGNORECASE,
            )
        )
    )

    print(
        "\nANGULAR COMPONENTS"
    )
    print(
        "=================="
    )

    print(
        "Controllers:",
        controller_names or "NONE",
    )

    print(
        "Factories:",
        factory_names or "NONE",
    )

    print(
        "Services:",
        service_names or "NONE",
    )


    # -----------------------------------------------------------------------
    # 10. Recover complete quoted strings containing `/bha/v1/`.
    # -----------------------------------------------------------------------
    #
    # These are the strongest simple route clues because they explicitly name
    # the BHA v1 resource namespace.

    explicit_bha_strings = sorted(
        set(
            match.group(2)
            for match in re.finditer(
                r"""(['"])([^'"]*/bha/v1/[^'"]*)\1""",
                script_text,
                flags=re.IGNORECASE,
            )
        )
    )

    print(
        "\nEXPLICIT /bha/v1/ STRINGS"
    )
    print(
        "========================="
    )

    if explicit_bha_strings:
        for value in explicit_bha_strings:
            print(
                value
            )
    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 11. Recover lines/expressions that construct URLs from `apiaddress`.
    # -----------------------------------------------------------------------
    #
    # The ratings script demonstrated that routes may be written as:
    #
    #   apiaddress + '/bha/v1/ratings'
    #
    # rather than appearing as complete literal URLs.
    #
    # Preserve bounded expressions around every occurrence of `apiaddress`.

    apiaddress_contexts = []

    for match in re.finditer(
        r"apiaddress",
        script_text,
        flags=re.IGNORECASE,
    ):
        start = max(
            0,
            match.start() - 250,
        )

        end = min(
            len(script_text),
            match.end() + 500,
        )

        context = script_text[
            start:end
        ]

        compact = re.sub(
            r"\s+",
            " ",
            context,
        ).strip()

        if compact not in apiaddress_contexts:
            apiaddress_contexts.append(
                compact
            )

    print(
        "\nAPIADDRESS ROUTE CONTEXTS"
    )
    print(
        "========================="
    )

    if apiaddress_contexts:
        for index, context in enumerate(
            apiaddress_contexts,
            start=1,
        ):
            print(
                f"\nContext {index}"
            )
            print(
                "-" * 50
            )
            print(
                context[:4000]
            )
    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 12. Extract quoted route/path fragments containing horse-related terms.
    # -----------------------------------------------------------------------
    #
    # Search more broadly than `/horses` because the horse profile may call
    # resources named for:
    #
    #   - racehorses;
    #   - entries;
    #   - form;
    #   - ratings;
    #   - trainers;
    #   - owners;
    #   - pedigree.
    #
    # These are clues only. A matching word does not prove resource semantics.

    HORSE_RELATED_TERMS = (
        "horse",
        "racehorse",
        "animal",
        "pedigree",
        "rating",
        "performance",
        "entry",
        "entries",
        "trainer",
        "owner",
        "form",
        "race",
    )

    quoted_strings = re.findall(
        r"""(['"])([^'"\r\n]{1,500})\1""",
        script_text,
    )

    horse_related_strings = sorted(
        {
            value
            for _, value in quoted_strings
            if any(
                term in value.lower()
                for term in HORSE_RELATED_TERMS
            )
        }
    )

    print(
        "\nHORSE-RELATED QUOTED STRINGS"
    )
    print(
        "============================"
    )

    if horse_related_strings:
        for value in horse_related_strings:
            print(
                value
            )
    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 13. Locate all `$http` calls and retain only those whose nearby code
    #     contains horse/profile/API evidence.
    # -----------------------------------------------------------------------
    #
    # This is often the clearest way to see:
    #
    #   - HTTP method;
    #   - resource variable;
    #   - request parameters;
    #   - response fields subsequently consumed by the profile.
    #
    # Keep contexts bounded so a large/minified asset does not flood output.

    http_contexts = []

    for match in re.finditer(
        r"\$http",
        script_text,
        flags=re.IGNORECASE,
    ):
        start = max(
            0,
            match.start() - 900,
        )

        end = min(
            len(script_text),
            match.end() + 1800,
        )

        context = script_text[
            start:end
        ]

        context_lower = context.lower()

        if not (
            "apiaddress" in context_lower
            or "/bha/v1/" in context_lower
            or "horse" in context_lower
            or "racehorse" in context_lower
        ):
            continue

        compact = re.sub(
            r"\s+",
            " ",
            context,
        ).strip()

        # Avoid near-duplicate overlapping blocks.
        if any(
            compact in existing
            or existing in compact
            for existing in http_contexts
        ):
            continue

        http_contexts.append(
            compact
        )


    MAX_HTTP_CONTEXTS = 15

    print(
        "\nBOUNDED HTTP CALL CONTEXTS"
    )
    print(
        "=========================="
    )

    if http_contexts:
        for index, context in enumerate(
            http_contexts[:MAX_HTTP_CONTEXTS],
            start=1,
        ):
            print(
                f"\nHTTP context {index}"
            )
            print(
                "-" * 60
            )
            print(
                context[:6000]
            )

        if len(http_contexts) > MAX_HTTP_CONTEXTS:
            print(
                "\nAdditional HTTP contexts omitted:",
                len(http_contexts) - MAX_HTTP_CONTEXTS,
            )

    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 14. Produce a simple deduplicated route-token inventory.
    # -----------------------------------------------------------------------
    #
    # This regex deliberately captures relative route fragments beginning with
    # `/bha/v1/` up to JavaScript punctuation.
    #
    # Dynamic concatenations may still appear incomplete; those should remain
    # unresolved until the surrounding context is interpreted.

    route_tokens = sorted(
        set(
            re.findall(
                r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                script_text,
                flags=re.IGNORECASE,
            )
        )
    )

    print(
        "\nDEDUPLICATED BHA V1 ROUTE TOKENS"
    )
    print(
        "================================"
    )

    if route_tokens:
        for route in route_tokens:
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 15. State the evidence/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Cache:",
    CACHE_FILE,
)

print(
    "External requests made by this cell:",
    0 if source == "cache" else 1,
)

print(
    "BHA Authorization read from .env.local: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Horse records requested: 0"
)

print(
    "Database writes: NONE"
)

BHA HORSE DATABASE/PROFILE — FRONTEND ROUTE DISCOVERY
Loaded from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/wp-content/themes/bha/library/js/angular/pages/racehorses.js?ver=1.3
SHA-256: ce46d44960bf72223bd10026374b0ae5fe6eb1b8e5781676a4c76b08cef449d5
Literal Bearer credential found in asset: NO

ANGULAR COMPONENTS
Controllers: ['RacehorseHome', 'RacehorseSearchController']
Factories: NONE
Services: NONE

EXPLICIT /bha/v1/ STRINGS
/bha/v1/racehorses

APIADDRESS ROUTE CONTEXTS

Context 1
--------------------------------------------------
cation.search().pagenum) ? $location.search().pagenum : 1, per_page: 20, }; options = angular.extend({}, defaults, options); $http({ method: 'GET', url: apiaddress+'/bha/v1/racehorses', params: options, }) .success(function (response) { $scope.horsesList = (response.data || []); $scope.errorLoading = false; // reset error // Add Pagination $scope.pagination = { lastpage: response.last_page, perpage: response.per_page, total:

In [30]:
# BHA Horse Database/Profile — one-horse search-result probe
#
# WHAT
# ----
# Make one bounded request to the structured BHA racehorse search resource:
#
#   /bha/v1/racehorses
#
# using a horse already observed in the Official Ratings resource:
#
#   ZYNAK (FR)
#   BHA ratings animalId = 3090473
#
# Search using:
#
#   q        = ZYNAK
#   rated    = 1
#   page     = 1
#   per_page = 5
#
# WHY
# ---
# Frontend discovery established that `racehorses.js` powers only the public
# horse-search/results surface and calls `/bha/v1/racehorses`.
#
# Before investigating an individual horse profile, determine what identifiers,
# URLs and metadata the search result itself exposes.
#
# Using a horse already observed through the ratings resource also lets us check
# whether the horse-search identifier agrees with the ratings `animalId`.
#
# READS
# -----
# - BHA Authorization from repo-root `.env.local`;
# - exactly one BHA `/bha/v1/racehorses` request.
#
# WRITES
# ------
# One ignored research-cache envelope:
#
#   data/cache/bha_official_source_feasibility/
#       horse_search_probe/
#       zynak_search.json
#
# The Authorization value is never printed or persisted.
#
# EXPECTED RESULT
# ---------------
# A bounded report showing:
#
#   - HTTP status;
#   - pagination metadata;
#   - number of matching horses;
#   - complete union of fields returned by the search;
#   - all returned rows, capped at five;
#   - whether any returned identifier equals ratings animalId 3090473.
#
# PARTICULARLY IMPORTANT
# ----------------------
# Look for anything that can lead us to the individual horse profile, such as:
#
#   - horse/animal identifiers;
#   - moniker/slug fields;
#   - profile URLs;
#   - names used by the public site's profile links.
#
# Do NOT guess a horse-profile endpoint in this cell.
#
# If the search result supplies enough identity information, the next step will
# be to follow the PUBLIC horse-profile interface and discover its own resource
# surface.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_search_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_FILE = (
    CACHE_DIR
    / "zynak_search.json"
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization credential.
# ---------------------------------------------------------------------------
#
# Keep secret handling explicit and dependency-free.
#
# The value must never be printed or written into the research cache.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the bounded search using the same parameter family observed in the
#    public `racehorses.js` frontend.
# ---------------------------------------------------------------------------
#
# Five rows is enough to handle possible similarly named matches without
# turning this into a population download.

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

PARAMS = {
    "q": "ZYNAK",
    "rated": 1,
    "page": 1,
    "per_page": 5,
}

REQUEST_URL = (
    f"{BHA_BASE}/racehorses?"
    f"{urlencode(PARAMS)}"
)

KNOWN_RATINGS_ANIMAL_ID = 3090473


# ---------------------------------------------------------------------------
# 4. Reuse the exact cached request if it has already been made.
# ---------------------------------------------------------------------------

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        envelope["request_url"]
        == REQUEST_URL
    )

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 5. Make exactly one authenticated racehorse-search request.
    # -----------------------------------------------------------------------
    #
    # Preserve HTTP failures as source evidence rather than crashing before the
    # response can be cached.

    request = Request(
        REQUEST_URL,
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/horses/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # 6. Parse JSON without assuming success.
    # -----------------------------------------------------------------------
    #
    # An access/error response must remain distinct from an empty horse search.

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # 7. Cache the complete bounded response WITHOUT the Authorization value.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "request_url": REQUEST_URL,
        "request_parameters": PARAMS,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 8. Report request/transport evidence before interpreting the response.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE DATABASE — ONE-HORSE SEARCH PROBE"
)
print(
    "==========================================="
)

print(
    "Loaded from:",
    source,
)

print(
    "HTTP status:",
    envelope["response_status"],
)

print(
    "Content-Type:",
    envelope["content_type"],
)

if envelope["transport_error"]:
    print(
        "Transport observation:",
        envelope["transport_error"],
    )


# ---------------------------------------------------------------------------
# 9. Inspect the actual response shape.
# ---------------------------------------------------------------------------

payload = envelope[
    "parsed_json"
]

if payload is None:
    print(
        "\nParsed JSON: NO"
    )

    if envelope["response_text"]:
        print(
            "\nBounded response preview:"
        )

        print(
            envelope["response_text"][:5000]
        )

else:
    print(
        "\nParsed JSON: YES"
    )

    print(
        "Top-level type:",
        type(payload).__name__,
    )

    if isinstance(
        payload,
        dict,
    ):
        print(
            "Top-level keys:",
            sorted(
                payload.keys()
            ),
        )


        # -------------------------------------------------------------------
        # 10. Report pagination/scalar metadata separately from horse records.
        # -------------------------------------------------------------------

        scalar_metadata = {
            key: value
            for key, value in payload.items()
            if not isinstance(
                value,
                (dict, list),
            )
        }

        print(
            "\nTop-level scalar metadata:"
        )

        print(
            scalar_metadata
        )


        # -------------------------------------------------------------------
        # 11. Inventory the returned horse records.
        # -------------------------------------------------------------------

        data = payload.get(
            "data"
        )

        print(
            "\ndata type:",
            type(data).__name__,
        )

        if isinstance(
            data,
            list,
        ):
            print(
                "Returned rows:",
                len(data),
            )

            dictionary_rows = [
                row
                for row in data
                if isinstance(
                    row,
                    dict,
                )
            ]

            if dictionary_rows:
                # -----------------------------------------------------------
                # Use the union across all returned rows rather than assuming
                # the first row demonstrates every optional field.
                # -----------------------------------------------------------

                field_union = sorted(
                    {
                        key
                        for row in dictionary_rows
                        for key in row.keys()
                    }
                )

                print(
                    "\nUnion of horse-search fields:"
                )

                for field in field_union:
                    print(
                        " -",
                        field,
                    )


                # -----------------------------------------------------------
                # Print all returned rows because the request was explicitly
                # capped at five records.
                # -----------------------------------------------------------

                for index, row in enumerate(
                    dictionary_rows,
                    start=1,
                ):
                    print(
                        f"\nHorse search row {index}"
                    )
                    print(
                        "-" * 50
                    )

                    print(
                        json.dumps(
                            row,
                            indent=2,
                            ensure_ascii=False,
                        )
                    )


                # -----------------------------------------------------------
                # 12. Test the known cross-source identifier explicitly.
                # -----------------------------------------------------------
                #
                # Do not assume the horse-search field is named `animalId`.
                # Search all scalar values for the exact known integer/string
                # and report which fields, if any, contain it.

                id_matches = []

                for row_number, row in enumerate(
                    dictionary_rows,
                    start=1,
                ):
                    for field, value in row.items():
                        if str(value) == str(
                            KNOWN_RATINGS_ANIMAL_ID
                        ):
                            id_matches.append(
                                {
                                    "row": row_number,
                                    "field": field,
                                    "value": value,
                                }
                            )

                print(
                    "\nCROSS-SOURCE ID CHECK"
                )
                print(
                    "====================="
                )

                print(
                    "Known ratings animalId:",
                    KNOWN_RATINGS_ANIMAL_ID,
                )

                if id_matches:
                    print(
                        "Exact matching value found: YES"
                    )

                    for match in id_matches:
                        print(
                            f"  row={match['row']} "
                            f"field={match['field']} "
                            f"value={match['value']}"
                        )

                else:
                    print(
                        "Exact matching value found: NO"
                    )

            elif data:
                print(
                    "\nReturned data is not dictionary-shaped."
                )

                print(
                    repr(
                        data
                    )[:5000]
                )

            else:
                print(
                    "\nSearch returned no horse rows."
                )

        else:
            print(
                "\nUnexpected `data` structure:"
            )

            print(
                json.dumps(
                    data,
                    indent=2,
                    ensure_ascii=False,
                )[:8000]
            )


# ---------------------------------------------------------------------------
# 13. State the acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Request URL:",
    REQUEST_URL,
)

print(
    "Cache:",
    CACHE_FILE,
)

print(
    "Known ratings animalId used for comparison:",
    KNOWN_RATINGS_ANIMAL_ID,
)

print(
    "Maximum horse rows requested:",
    5,
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Horse profile requested: NO"
)

print(
    "Database writes: NONE"
)

BHA HORSE DATABASE — ONE-HORSE SEARCH PROBE
Loaded from: network
HTTP status: 200
Content-Type: application/json

Parsed JSON: YES
Top-level type: dict
Top-level keys: ['current_page', 'data', 'first_page_url', 'from', 'last_page', 'last_page_url', 'links', 'next_page_url', 'path', 'per_page', 'prev_page_url', 'to', 'total']

Top-level scalar metadata:
{'current_page': 1, 'first_page_url': 'https://api09.horseracing.software/bha/v1/racehorses?page=1', 'from': 1, 'last_page': 1, 'last_page_url': 'https://api09.horseracing.software/bha/v1/racehorses?page=1', 'next_page_url': None, 'path': 'https://api09.horseracing.software/bha/v1/racehorses', 'per_page': 5, 'prev_page_url': None, 'to': 1, 'total': 1}

data type: list
Returned rows: 1

Union of horse-search fields:
 - awtRating
 - chaseRating
 - dateOfBirth
 - flatRating
 - hurdleRating
 - id
 - moniker
 - name
 - nameWithNoCountry
 - ownerName
 - trainerName

Horse search row 1
--------------------------------------------------
{
  "id"

In [31]:
# BHA Horse Database/Profile — discover the public horse-profile link pattern
#
# WHAT
# ----
# Inspect the HTML template used by the public BHA racehorse search-results page:
#
#   /racing/horses/racehorse-search-results/
#
# The previous API probe established that the structured horse search returns:
#
#   id
#   name
#   dateOfBirth
#   moniker
#   trainerName
#   ownerName
#   flatRating
#   awtRating
#   chaseRating
#   hurdleRating
#
# and demonstrated that:
#
#   racehorses.id == ratings.animalId
#
# for the observed horse ZYNAK (FR), BHA ID 3090473.
#
# The search API did NOT return a profile URL or slug.
#
# This cell therefore inspects the public search-results HTML to determine how
# the BHA frontend constructs a clickable horse-profile link.
#
# WHY
# ---
# We want to reach the individual horse profile using the site's own published
# navigation mechanism rather than guessing an endpoint or URL structure.
#
# Discovering the profile page should then let us inspect the richer horse
# information surface, potentially including:
#
#   - pedigree;
#   - career/race history;
#   - training history;
#   - ownership;
#   - rating history;
#   - future entries;
#   - performance figures.
#
# None of those capabilities are assumed in this cell.
#
# READS
# -----
# Exactly one public BHA HTML page:
#
#   https://www.britishhorseracing.com/
#       racing/horses/racehorse-search-results/
#
# No BHA Authorization credential is required or read.
#
# WRITES
# ------
# One ignored research-cache envelope:
#
#   data/cache/bha_official_source_feasibility/
#       horse_profile_link_discovery/
#       racehorse_search_results_page.json
#
# EXPECTED RESULT
# ---------------
# Bounded HTML/template contexts showing any expressions involving:
#
#   - horsesList;
#   - horse.id;
#   - ng-href / href;
#   - profile/detail URL construction.
#
# Ideally this establishes the exact public URL pattern used when a user clicks
# a search result.
#
# IMPORTANT
# ---------
# This cell does NOT:
#
#   - request an individual horse profile;
#   - guess a profile URL;
#   - call another structured API endpoint;
#   - read the BHA Authorization credential;
#   - write to the database.

from datetime import datetime, timezone
import hashlib
import json
import re
import subprocess
from pathlib import Path


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_profile_link_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_FILE = (
    CACHE_DIR
    / "racehorse_search_results_page.json"
)


# ---------------------------------------------------------------------------
# 2. Define the search-results page reached by `racehorses.js`.
# ---------------------------------------------------------------------------
#
# The previously inspected frontend explicitly redirects searches to:
#
#   racehorse-search-results/
#
# from the BHA `/racing/horses/` section.

SEARCH_RESULTS_URL = (
    "https://www.britishhorseracing.com/"
    "racing/horses/racehorse-search-results/"
)


# ---------------------------------------------------------------------------
# 3. Reuse exact cached evidence if this page has already been requested.
# ---------------------------------------------------------------------------

if CACHE_FILE.exists():
    envelope = json.loads(
        CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        envelope["request_url"]
        == SEARCH_RESULTS_URL
    )

    source = "cache"


else:
    # -----------------------------------------------------------------------
    # 4. Fetch exactly one public HTML page.
    # -----------------------------------------------------------------------
    #
    # Do not use `curl --fail`.
    #
    # If the BHA returns an access-control page or another non-200 response,
    # preserve it as evidence instead of turning it into a notebook exception.

    STATUS_MARKER = "__BHA_HTTP_STATUS__"
    URL_MARKER = "__BHA_FINAL_URL__"

    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            (
                "Accept: text/html,application/xhtml+xml,"
                "application/xml;q=0.9,*/*;q=0.8"
            ),
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            SEARCH_RESULTS_URL,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    assert status_position != -1, (
        "Could not recover HTTP status from horse search-results page."
    )

    assert final_url_position != -1, (
        "Could not recover final URL from horse search-results page."
    )

    page_html = output[
        :status_position
    ]

    response_status = int(
        output[
            status_position + len(f"\n{STATUS_MARKER}") :
            final_url_position
        ].strip()
    )

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()


    # -----------------------------------------------------------------------
    # 5. Persist the public page response as reproducible source evidence.
    # -----------------------------------------------------------------------
    #
    # This page request carries no BHA Authorization credential.
    #
    # Still check defensively for an unexpected literal Bearer token before
    # writing the response.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    cached_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "request_url": SEARCH_RESULTS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": response_status,
        "final_url": final_url,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
        "content_sha256": hashlib.sha256(
            page_html.encode(
                "utf-8"
            )
        ).hexdigest(),
        "response_html": cached_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        CACHE_FILE
    )

    source = "network"


# ---------------------------------------------------------------------------
# 6. Present transport/provenance evidence before interpreting the template.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE PROFILE — PUBLIC LINK DISCOVERY"
)
print(
    "========================================="
)

print(
    "Loaded from:",
    source,
)

print(
    "HTTP status:",
    envelope["response_status"],
)

print(
    "Final URL:",
    envelope["final_url"],
)

print(
    "SHA-256:",
    envelope["content_sha256"],
)

print(
    "Literal Bearer credential found:",
    (
        "YES — REDACTED"
        if envelope["literal_bearer_was_present"]
        else "NO"
    ),
)


# ---------------------------------------------------------------------------
# 7. Stop cleanly if the public page itself was unavailable.
# ---------------------------------------------------------------------------

page_html = envelope[
    "response_html"
]

if (
    envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nSearch-results page unavailable."
    )

    print(
        "Horse-profile link pattern remains unresolved."
    )

else:
    # -----------------------------------------------------------------------
    # 8. Search for template markers associated with the horse-results list.
    # -----------------------------------------------------------------------
    #
    # These markers should lead us to the actual HTML element representing a
    # clickable racehorse result.

    SEARCH_MARKERS = [
        "horsesList",
        "horse.id",
        "horse.name",
        "ng-href",
        "racehorse",
        "profile",
    ]

    marker_counts = {}

    for marker in SEARCH_MARKERS:
        marker_counts[marker] = len(
            re.findall(
                re.escape(
                    marker
                ),
                page_html,
                flags=re.IGNORECASE,
            )
        )

    print(
        "\nTEMPLATE MARKER COUNTS"
    )
    print(
        "======================"
    )

    for marker, count in marker_counts.items():
        print(
            f"{marker}: {count}"
        )


    # -----------------------------------------------------------------------
    # 9. Recover HTML tags containing Angular href/profile expressions.
    # -----------------------------------------------------------------------
    #
    # Preserve whole anchor tags where possible because the link pattern may
    # combine several Angular expressions:
    #
    #   horse.id
    #   horse.name
    #   filters/slugs
    #
    # Seeing the complete tag is more reliable than extracting an isolated href
    # fragment.

    anchor_tags = re.findall(
        r"""<a\b[^>]*>.*?</a>""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    relevant_anchor_tags = []

    for tag in anchor_tags:
        tag_lower = tag.lower()

        if (
            "horse." in tag_lower
            or "horseslist" in tag_lower
            or "racehorse" in tag_lower
        ):
            compact = re.sub(
                r"\s+",
                " ",
                tag,
            ).strip()

            if compact not in relevant_anchor_tags:
                relevant_anchor_tags.append(
                    compact
                )


    print(
        "\nHORSE-RELATED ANCHOR TAGS"
    )
    print(
        "========================="
    )

    if relevant_anchor_tags:
        for index, tag in enumerate(
            relevant_anchor_tags,
            start=1,
        ):
            print(
                f"\nAnchor {index}"
            )
            print(
                "-" * 60
            )
            print(
                tag[:5000]
            )

    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 10. Search explicitly for href/ng-href attribute values.
    # -----------------------------------------------------------------------
    #
    # Keep raw Angular expressions intact; do not attempt to construct a live
    # horse URL yet.

    href_expressions = sorted(
        set(
            match.group(2)
            for match in re.finditer(
                r"""((?:ng-)?href)\s*=\s*["']([^"']+)["']""",
                page_html,
                flags=re.IGNORECASE,
            )
            if (
                "horse" in match.group(2).lower()
                or "{{" in match.group(2)
            )
        )
    )

    print(
        "\nHORSE / ANGULAR HREF EXPRESSIONS"
    )
    print(
        "================================"
    )

    if href_expressions:
        for expression in href_expressions:
            print(
                expression
            )
    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 11. Print bounded HTML context around `horse.id` and `horsesList`.
    # -----------------------------------------------------------------------
    #
    # This catches link construction split across tags/attributes that a simple
    # anchor-expression search might miss.

    CONTEXT_MARKERS = [
        "horse.id",
        "horsesList",
    ]

    context_blocks = []

    for marker in CONTEXT_MARKERS:
        for match in re.finditer(
            re.escape(
                marker
            ),
            page_html,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1200,
            )

            end = min(
                len(page_html),
                match.end() + 1800,
            )

            context = page_html[
                start:end
            ]

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            # Avoid overlapping duplicate blocks.
            if any(
                compact in existing["context"]
                or existing["context"] in compact
                for existing in context_blocks
            ):
                continue

            context_blocks.append(
                {
                    "marker": marker,
                    "context": compact,
                }
            )


    print(
        "\nBOUNDED HORSE-TEMPLATE CONTEXT"
    )
    print(
        "=============================="
    )

    if context_blocks:
        for index, item in enumerate(
            context_blocks[:10],
            start=1,
        ):
            print(
                f"\nContext {index} [{item['marker']}]"
            )
            print(
                "-" * 60
            )
            print(
                item["context"][:7000]
            )

    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 12. State the acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Cache:",
    CACHE_FILE,
)

print(
    "External requests made by this cell:",
    0 if source == "cache" else 1,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Individual horse profile requested: NO"
)

print(
    "Structured horse-profile endpoint guessed: NO"
)

print(
    "Database writes: NONE"
)

BHA HORSE PROFILE — PUBLIC LINK DISCOVERY
Loaded from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/horses/racehorse-search-results/
SHA-256: 1231671f89c05ce6aa315c226c0e73734550afa6d4e447d49abce12a4e863958
Literal Bearer credential found: NO

TEMPLATE MARKER COUNTS
horsesList: 11
horse.id: 1
horse.name: 1
ng-href: 4
racehorse: 20
profile: 0

HORSE-RELATED ANCHOR TAGS

Anchor 1
------------------------------------------------------------
<a href="https://www.britishhorseracing.com/regulation/anti-doping-medication-control/bha-centre-for-racehorse-studies/">Centre for Racehorse Studies</a>

Anchor 2
------------------------------------------------------------
<a href="https://www.britishhorseracing.com/regulation/ownership/become-a-racehorse-owner/">Become an Owner</a>

Anchor 3
------------------------------------------------------------
<a href="https://www.britishhorseracing.com/regulation/official-ratings/longines-worlds-best-racehorse-rankings/">Lon

In [32]:
# BHA Horse Database/Profile — individual profile frontend discovery
#
# WHAT
# ----
# Follow the exact public horse-profile URL pattern demonstrated by the BHA
# racehorse search-results template.
#
# For the previously observed horse:
#
#   ZYNAK (FR)
#   BHA horse ID = 3090473
#
# the public site constructs:
#
#   https://www.britishhorseracing.com/
#       racing/horses/horse/#!/3090473
#
# This cell:
#
#   1. requests the public horse-profile HTML page;
#   2. inventories the JavaScript assets loaded by that page;
#   3. identifies assets not already known from the horse-search page where
#      possible;
#   4. fetches only first-party BHA JavaScript assets;
#   5. searches them for horse-profile controllers, HTTP calls and `/bha/v1/`
#      resources;
#   6. prints bounded context around the relevant code.
#
# WHY
# ---
# We now know the public navigation path to an individual horse profile.
#
# The horse-search JavaScript exposed only:
#
#   /bha/v1/racehorses
#
# but the individual profile may rely on additional resources for:
#
#   - horse identity and pedigree;
#   - owners and trainers;
#   - race/performance history;
#   - rating history;
#   - future entries;
#   - training history;
#   - other profile information.
#
# We must discover those routes from the public frontend rather than guessing
# endpoint names.
#
# READS
# -----
# - one public BHA horse-profile HTML page;
# - first-party BHA JavaScript assets referenced by that page.
#
# No BHA Authorization value is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       horse_profile_frontend_discovery/
#
# Specifically:
#
#   horse_profile_page.json
#   horse_profile_script_inventory.json
#
# Raw public JavaScript is not persisted individually.
#
# EXPECTED RESULT
# ---------------
# A bounded inventory showing:
#
#   - profile page HTTP status;
#   - first-party script assets;
#   - Angular controllers/factories/services associated with the profile;
#   - `/bha/v1/...` route strings;
#   - `apiaddress` route construction;
#   - HTTP-call contexts;
#   - likely profile-data resource families.
#
# DECISION BOUNDARY
# -----------------
# Do NOT call any newly discovered structured horse-profile endpoint yet.
#
# First map the profile resource surface. The next cell can then probe the
# smallest useful route using horse ID 3090473.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_profile_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "horse_profile_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "horse_profile_script_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Use the exact public URL pattern demonstrated by the BHA search template.
# ---------------------------------------------------------------------------
#
# The fragment `#!/3090473` is client-side state and is not transmitted to the
# web server in the HTTP request.
#
# We nevertheless retain the complete public browser URL as provenance because
# it establishes how a normal BHA-site user reaches this horse.

HORSE_ID = 3090473

PUBLIC_PROFILE_URL = (
    "https://www.britishhorseracing.com/"
    f"racing/horses/horse/#!/{HORSE_ID}"
)

PROFILE_PAGE_REQUEST_URL = (
    "https://www.britishhorseracing.com/"
    "racing/horses/horse/"
)


# ---------------------------------------------------------------------------
# 3. Helper for public text requests that preserves HTTP status as evidence.
# ---------------------------------------------------------------------------
#
# Do not use `curl --fail`.
#
# A blocked or unavailable asset should become a recorded observation rather
# than terminate the notebook before evidence can be inspected.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url, accept):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            f"Accept: {accept}",
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch the public horse-profile page once.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == PROFILE_PAGE_REQUEST_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        PROFILE_PAGE_REQUEST_URL,
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # 5. Defensively redact any unexpected literal Bearer token.
    # -----------------------------------------------------------------------
    #
    # None is expected in this page template, but do not persist one if the
    # site's implementation changes.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "public_browser_url": PUBLIC_PROFILE_URL,
        "horse_id": HORSE_ID,
        "request_url": PROFILE_PAGE_REQUEST_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 6. Report page access before interpreting its frontend.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE PROFILE — FRONTEND DISCOVERY"
)
print(
    "====================================="
)

print(
    "Public browser URL:",
    PUBLIC_PROFILE_URL,
)

print(
    "Loaded profile HTML from:",
    page_source,
)

print(
    "Profile page HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Profile HTML SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 7. Stop route discovery cleanly if the profile page was unavailable.
# ---------------------------------------------------------------------------

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nHorse-profile HTML unavailable."
    )

    print(
        "Profile resource surface remains unresolved."
    )

else:
    # -----------------------------------------------------------------------
    # 8. Extract all JavaScript assets referenced by the horse-profile page.
    # -----------------------------------------------------------------------

    script_sources = re.findall(
        r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    resolved_scripts = []

    for source in script_sources:
        decoded_source = html.unescape(
            source
        )

        resolved_scripts.append(
            urljoin(
                page_envelope["final_url"]
                or PROFILE_PAGE_REQUEST_URL,
                decoded_source,
            )
        )

    resolved_scripts = sorted(
        set(
            resolved_scripts
        )
    )


    # -----------------------------------------------------------------------
    # 9. Restrict inspection to first-party BHA JavaScript assets.
    # -----------------------------------------------------------------------
    #
    # Third-party libraries cannot define the BHA profile-resource API surface
    # we are trying to discover.

    BHA_HOSTS = {
        "www.britishhorseracing.com",
        "britishhorseracing.com",
    }

    first_party_scripts = []

    for script_url in resolved_scripts:
        parsed = urlsplit(
            script_url
        )

        if parsed.hostname not in BHA_HOSTS:
            continue

        if ".js" not in parsed.path.lower():
            continue

        first_party_scripts.append(
            script_url
        )


    # -----------------------------------------------------------------------
    # 10. Bound the frontend inspection.
    # -----------------------------------------------------------------------
    #
    # The earlier site-wide inventory showed roughly 30 first-party scripts.
    # Forty is therefore a generous ceiling while still preventing an
    # accidental broad crawl if the page structure changes.

    MAX_SCRIPT_ASSETS = 40

    scripts_to_inspect = first_party_scripts[
        :MAX_SCRIPT_ASSETS
    ]

    script_inventory_truncated = (
        len(first_party_scripts)
        > MAX_SCRIPT_ASSETS
    )


    # -----------------------------------------------------------------------
    # 11. Search each script for profile-specific evidence.
    # -----------------------------------------------------------------------
    #
    # A script becomes relevant if it contains one or more of:
    #
    #   - BHA v1 API paths;
    #   - `apiaddress`;
    #   - profile/horse controller language;
    #   - route parameters involving horse IDs.
    #
    # We inspect the complete public response in memory but persist only derived
    # metadata and bounded contexts.

    asset_observations = []

    all_route_tokens = set()

    for script_url in scripts_to_inspect:
        script_response = curl_public_text(
            script_url,
            (
                "application/javascript,"
                "text/javascript,*/*;q=0.8"
            ),
        )

        script_text = script_response[
            "body"
        ]

        script_sha256 = (
            hashlib.sha256(
                script_text.encode(
                    "utf-8"
                )
            ).hexdigest()
            if script_text
            else None
        )


        # -------------------------------------------------------------------
        # 12. Recover Angular component names from this asset.
        # -------------------------------------------------------------------

        controllers = sorted(
            set(
                re.findall(
                    r"""\.controller\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        factories = sorted(
            set(
                re.findall(
                    r"""\.factory\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        services = sorted(
            set(
                re.findall(
                    r"""\.service\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )


        # -------------------------------------------------------------------
        # 13. Recover explicit BHA-v1 route tokens.
        # -------------------------------------------------------------------

        route_tokens = sorted(
            set(
                re.findall(
                    r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        all_route_tokens.update(
            route_tokens
        )


        # -------------------------------------------------------------------
        # 14. Recover bounded contexts around `apiaddress` and `/bha/v1/`.
        # -------------------------------------------------------------------
        #
        # These contexts usually reveal dynamic concatenation such as:
        #
        #   apiaddress + '/bha/v1/...' + horseId
        #
        # which a simple route-token regex cannot fully reconstruct.

        marker_contexts = []

        for marker_pattern in [
            r"apiaddress",
            r"/bha/v1/",
        ]:
            for match in re.finditer(
                marker_pattern,
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 900,
                )

                end = min(
                    len(script_text),
                    match.end() + 1800,
                )

                context = script_text[
                    start:end
                ]

                compact = re.sub(
                    r"\s+",
                    " ",
                    context,
                ).strip()

                if compact not in marker_contexts:
                    marker_contexts.append(
                        compact
                    )


        # -------------------------------------------------------------------
        # 15. Recover bounded `$http` call contexts.
        # -------------------------------------------------------------------

        http_contexts = []

        for match in re.finditer(
            r"\$http",
            script_text,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1000,
            )

            end = min(
                len(script_text),
                match.end() + 2200,
            )

            context = script_text[
                start:end
            ]

            lower_context = context.lower()

            if not (
                "apiaddress" in lower_context
                or "/bha/v1/" in lower_context
                or "horse" in lower_context
                or "animal" in lower_context
            ):
                continue

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            if compact not in http_contexts:
                http_contexts.append(
                    compact
                )


        # -------------------------------------------------------------------
        # 16. Decide whether this asset is relevant enough to report.
        # -------------------------------------------------------------------

        relevance_terms = (
            "horse",
            "racehorse",
            "animal",
            "profile",
            "pedigree",
            "owner",
            "trainer",
            "rating",
            "performance",
            "entry",
            "entries",
        )

        component_text = " ".join(
            controllers
            + factories
            + services
        ).lower()

        asset_is_relevant = bool(
            route_tokens
            or marker_contexts
            or http_contexts
            or any(
                term in component_text
                for term in relevance_terms
            )
        )

        asset_observations.append(
            {
                "script_url": script_url,
                "http_status": (
                    script_response["status"]
                ),
                "sha256": script_sha256,
                "controllers": controllers,
                "factories": factories,
                "services": services,
                "route_tokens": route_tokens,
                "marker_contexts": (
                    marker_contexts[:12]
                ),
                "http_contexts": (
                    http_contexts[:12]
                ),
                "relevant": asset_is_relevant,
            }
        )


    # -----------------------------------------------------------------------
    # 17. Persist only the derived frontend inventory.
    # -----------------------------------------------------------------------
    #
    # We do not need a second copy of every raw JavaScript asset merely to
    # establish route capability.

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "horse_id": HORSE_ID,
        "public_profile_url": PUBLIC_PROFILE_URL,
        "profile_page_request_url": (
            PROFILE_PAGE_REQUEST_URL
        ),
        "first_party_scripts": (
            first_party_scripts
        ),
        "scripts_inspected": len(
            scripts_to_inspect
        ),
        "script_inventory_truncated": (
            script_inventory_truncated
        ),
        "all_bha_v1_route_tokens": sorted(
            all_route_tokens
        ),
        "asset_observations": (
            asset_observations
        ),
        "authorization_sent": False,
        "horse_api_records_requested": 0,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 18. Report the page's first-party frontend inventory.
    # -----------------------------------------------------------------------

    print(
        "\nFIRST-PARTY PROFILE SCRIPTS"
    )
    print(
        "==========================="
    )

    print(
        "First-party scripts discovered:",
        len(first_party_scripts),
    )

    print(
        "Scripts inspected:",
        len(scripts_to_inspect),
    )

    print(
        "Inspection truncated:",
        (
            "YES"
            if script_inventory_truncated
            else "NO"
        ),
    )

    for script_url in scripts_to_inspect:
        print(
            " ",
            script_url,
        )


    # -----------------------------------------------------------------------
    # 19. Show only assets containing potentially relevant profile logic.
    # -----------------------------------------------------------------------

    relevant_assets = [
        item
        for item in asset_observations
        if item["relevant"]
    ]

    print(
        "\nRELEVANT PROFILE FRONTEND ASSETS"
    )
    print(
        "================================"
    )

    print(
        "Relevant assets:",
        len(relevant_assets),
    )

    for index, item in enumerate(
        relevant_assets,
        start=1,
    ):
        print(
            f"\nAsset {index}"
        )
        print(
            "-" * 70
        )

        print(
            "URL:",
            item["script_url"],
        )

        print(
            "HTTP:",
            item["http_status"],
        )

        print(
            "SHA-256:",
            item["sha256"],
        )

        print(
            "Controllers:",
            item["controllers"] or "NONE",
        )

        print(
            "Factories:",
            item["factories"] or "NONE",
        )

        print(
            "Services:",
            item["services"] or "NONE",
        )

        print(
            "Route tokens:",
            item["route_tokens"] or "NONE",
        )

        for context_index, context in enumerate(
            item["marker_contexts"],
            start=1,
        ):
            print(
                f"\n  API context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )

        for context_index, context in enumerate(
            item["http_contexts"],
            start=1,
        ):
            print(
                f"\n  HTTP context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )


    # -----------------------------------------------------------------------
    # 20. Give the deduplicated BHA-v1 route inventory separately.
    # -----------------------------------------------------------------------

    print(
        "\nDEDUPLICATED PROFILE-PAGE BHA V1 ROUTES"
    )
    print(
        "======================================"
    )

    if all_route_tokens:
        for route in sorted(
            all_route_tokens
        ):
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 21. State the evidence and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Horse ID:",
    HORSE_ID,
)

print(
    "Public browser URL:",
    PUBLIC_PROFILE_URL,
)

print(
    "Profile HTML cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived frontend inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Structured profile API requests made: 0"
)

print(
    "Database writes: NONE"
)

BHA HORSE PROFILE — FRONTEND DISCOVERY
Public browser URL: https://www.britishhorseracing.com/racing/horses/horse/#!/3090473
Loaded profile HTML from: network
Profile page HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/horses/horse/
Profile HTML SHA-256: 2cada284a52aa8e53ff3c12c669ca889dac07bccce5266555ade9731ff8291a6

FIRST-PARTY PROFILE SCRIPTS
First-party scripts discovered: 23
Scripts inspected: 23
Inspection truncated: NO
  https://www.britishhorseracing.com/cdn-cgi/scripts/5c5dd728/cloudflare-static/email-decode.min.js
  https://www.britishhorseracing.com/wp-content/plugins/magic-liquidizer-responsive-table/idjs/ml.responsive.table.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/flickity.pkgd.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/functions.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/iframeResizer.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha

In [33]:
# BHA Horse Database/Profile — bounded three-resource profile probe
#
# WHAT
# ----
# Reproduce the three structured requests made by the public BHA individual
# horse-profile page for one already-identified horse:
#
#   ZYNAK (FR)
#   BHA horse ID = 3090473
#
# The public `racehorses-profile.js` frontend demonstrated these resources:
#
#   1. /bha/v1/racehorses/{horseId}
#   2. /bha/v1/racehorses/{horseId}/performances
#   3. /bha/v1/racehorses/{horseId}/training-history
#
# The frontend calls all three when loading an individual horse profile.
#
# WHY
# ---
# We are inventorying what useful official BHA information exists.
#
# This is the smallest useful probe of the individual horse-profile surface:
# effectively one normal public profile-page load.
#
# In particular we want to establish whether the BHA exposes structured:
#
#   - identity / pedigree information;
#   - current owner/trainer information;
#   - handicapper-rating history;
#   - race-by-race performance history;
#   - training history;
#   - race identifiers capable of linking performances to the race resources.
#
# We do NOT assign governed semantics merely because a field name looks
# familiar.
#
# READS
# -----
# - BHA Authorization from repo-root `.env.local`;
# - exactly three structured BHA resources at most.
#
# WRITES
# ------
# One ignored cache envelope for each resource under:
#
#   data/cache/bha_official_source_feasibility/
#       horse_profile_resource_probe/
#
# The files are:
#
#   zynak_details.json
#   zynak_performances_page_1.json
#   zynak_training_history.json
#
# Every response is cached, including HTTP errors, empty responses and
# ambiguous responses.
#
# EXPECTED RESULT
# ---------------
# For each resource print:
#
#   - HTTP status;
#   - top-level response shape;
#   - record count where applicable;
#   - observed field names;
#   - bounded representative data.
#
# For nested horse details, also inventory nested objects/lists sufficiently to
# reveal structures such as `handicapperRatings` without dumping the whole
# response blindly.
#
# ACQUISITION BOUNDARY
# --------------------
# - one horse only;
# - performances page 1 only;
# - no pagination beyond page 1;
# - no other horses;
# - maximum three network requests;
# - no Database v4 queries;
# - no database writes.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_profile_resource_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization value.
# ---------------------------------------------------------------------------
#
# Keep secret handling explicit.
#
# The credential is used only in HTTP headers and must never be printed or
# written into any cache envelope.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the exact horse and resources demonstrated by the public frontend.
# ---------------------------------------------------------------------------

HORSE_ID = 3090473

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

PROFILE_REFERER = (
    "https://www.britishhorseracing.com/"
    f"racing/horses/horse/#!/{HORSE_ID}"
)

RESOURCE_SPECS = [
    {
        "name": "details",
        "url": (
            f"{BHA_BASE}/racehorses/{HORSE_ID}"
        ),
        "cache_file": (
            CACHE_DIR
            / "zynak_details.json"
        ),
    },
    {
        "name": "performances",
        "url": (
            f"{BHA_BASE}/racehorses/{HORSE_ID}/performances?"
            f"{urlencode({'page': 1})}"
        ),
        "cache_file": (
            CACHE_DIR
            / "zynak_performances_page_1.json"
        ),
    },
    {
        "name": "training_history",
        "url": (
            f"{BHA_BASE}/racehorses/{HORSE_ID}/training-history"
        ),
        "cache_file": (
            CACHE_DIR
            / "zynak_training_history.json"
        ),
    },
]


# ---------------------------------------------------------------------------
# 4. Request/caching helper.
# ---------------------------------------------------------------------------
#
# Every external request must leave research evidence behind.
#
# HTTP failures are therefore returned and cached rather than raised out of the
# notebook before their bodies can be inspected.

def load_or_request_resource(spec):
    cache_file = spec[
        "cache_file"
    ]

    if cache_file.exists():
        envelope = json.loads(
            cache_file.read_text(
                encoding="utf-8"
            )
        )

        assert (
            envelope["request_url"]
            == spec["url"]
        )

        return envelope, "cache"


    request = Request(
        spec["url"],
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": PROFILE_REFERER,
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        # Deliberately avoid persisting repr(error) because request objects can
        # sometimes contain more request detail than we need.
        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Parse JSON conservatively.
    # -----------------------------------------------------------------------
    #
    # A non-JSON response remains distinguishable from a genuine empty JSON
    # response.

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # Persist the complete response evidence WITHOUT the Authorization header.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Horse Database/Profile"
        ),
        "resource_name": spec["name"],
        "horse_id": HORSE_ID,
        "request_url": spec["url"],
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    temp_file = cache_file.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        cache_file
    )

    return envelope, "network"


# ---------------------------------------------------------------------------
# 5. Load the three profile resources.
# ---------------------------------------------------------------------------
#
# This mirrors the resource family used by one normal public horse-profile
# load. No additional pagination or horse IDs are requested.

resource_results = {}

network_requests = 0

for spec in RESOURCE_SPECS:
    envelope, source = load_or_request_resource(
        spec
    )

    resource_results[
        spec["name"]
    ] = {
        "envelope": envelope,
        "source": source,
        "cache_file": spec["cache_file"],
    }

    if source == "network":
        network_requests += 1


# ---------------------------------------------------------------------------
# 6. Helpers for describing observed JSON structure.
# ---------------------------------------------------------------------------
#
# We want enough nested structure to discover useful source capabilities
# without flooding the notebook with the complete JSON response.

def value_type_name(value):
    if value is None:
        return "null"

    return type(value).__name__


def print_dict_field_inventory(
    record,
    *,
    indent="",
    max_list_sample=2,
):
    """
    Print one level of dictionary fields plus bounded nested information.

    This is intentionally descriptive rather than semantic: the function tells
    us what the source returned, not what a field is governed to mean.
    """

    for key in sorted(
        record.keys()
    ):
        value = record[
            key
        ]

        print(
            f"{indent}- {key}: "
            f"{value_type_name(value)}"
        )

        # ---------------------------------------------------------------
        # Nested dictionary:
        # show its immediate field names.
        # ---------------------------------------------------------------

        if isinstance(
            value,
            dict,
        ):
            print(
                f"{indent}    keys:",
                sorted(
                    value.keys()
                ),
            )


        # ---------------------------------------------------------------
        # Nested list:
        # show count and dictionary schema from a tiny sample.
        # ---------------------------------------------------------------

        elif isinstance(
            value,
            list,
        ):
            print(
                f"{indent}    rows/items:",
                len(value),
            )

            dictionary_items = [
                item
                for item in value[
                    :max_list_sample
                ]
                if isinstance(
                    item,
                    dict,
                )
            ]

            if dictionary_items:
                nested_union = sorted(
                    {
                        nested_key
                        for item in dictionary_items
                        for nested_key in item.keys()
                    }
                )

                print(
                    f"{indent}    sample item fields:",
                    nested_union,
                )


# ---------------------------------------------------------------------------
# 7. Generic paginated-response reporter.
# ---------------------------------------------------------------------------

def report_paginated_resource(
    label,
    payload,
    sample_limit=3,
):
    print(
        f"\n{label}"
    )
    print(
        "=" * len(label)
    )

    if not isinstance(
        payload,
        dict,
    ):
        print(
            "Unexpected top-level type:",
            type(payload).__name__,
        )

        print(
            json.dumps(
                payload,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

        return


    print(
        "Top-level keys:",
        sorted(
            payload.keys()
        ),
    )

    scalar_metadata = {
        key: value
        for key, value in payload.items()
        if not isinstance(
            value,
            (dict, list),
        )
    }

    print(
        "Scalar metadata:",
        scalar_metadata,
    )

    data = payload.get(
        "data"
    )

    print(
        "data type:",
        type(data).__name__,
    )

    if not isinstance(
        data,
        list,
    ):
        print(
            "Unexpected data value:"
        )

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

        return


    print(
        "Rows returned:",
        len(data),
    )

    dictionary_rows = [
        row
        for row in data
        if isinstance(
            row,
            dict,
        )
    ]

    if not dictionary_rows:
        if data:
            print(
                "First value:",
                repr(
                    data[0]
                )[:3000],
            )

        return


    field_union = sorted(
        {
            key
            for row in dictionary_rows
            for key in row.keys()
        }
    )

    print(
        "\nUnion of observed row fields:"
    )

    for field in field_union:
        print(
            " -",
            field,
        )


    print(
        f"\nRepresentative rows "
        f"(maximum {sample_limit}):"
    )

    for index, row in enumerate(
        dictionary_rows[
            :sample_limit
        ],
        start=1,
    ):
        print(
            f"\nRow {index}"
        )
        print(
            "-" * 50
        )

        print(
            json.dumps(
                row,
                indent=2,
                ensure_ascii=False,
            )[:12_000]
        )


# ---------------------------------------------------------------------------
# 8. Report transport evidence for all three resources first.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE PROFILE — THREE-RESOURCE PROBE"
)
print(
    "========================================"
)

for spec in RESOURCE_SPECS:
    result = resource_results[
        spec["name"]
    ]

    envelope = result[
        "envelope"
    ]

    print(
        f"\n{spec['name']}"
    )

    print(
        "  Loaded from:",
        result["source"],
    )

    print(
        "  HTTP status:",
        envelope["response_status"],
    )

    print(
        "  Content-Type:",
        envelope["content_type"],
    )

    if envelope["transport_error"]:
        print(
            "  Transport observation:",
            envelope["transport_error"],
        )


# ---------------------------------------------------------------------------
# 9. Inspect the main horse-details response.
# ---------------------------------------------------------------------------
#
# `racehorses-profile.js` expects response.data and then uses:
#
#   horseDetails.handicapperRatings
#
# We inspect the actual response before assuming its exact shape.

details_payload = resource_results[
    "details"
]["envelope"]["parsed_json"]

print(
    "\nHORSE DETAILS"
)
print(
    "============="
)

if details_payload is None:
    print(
        "No parsed JSON."
    )

    preview = resource_results[
        "details"
    ]["envelope"]["response_text"]

    print(
        preview[:5000]
    )

else:
    print(
        "Top-level type:",
        type(details_payload).__name__,
    )

    if isinstance(
        details_payload,
        dict,
    ):
        print(
            "Top-level keys:",
            sorted(
                details_payload.keys()
            ),
        )

        details_data = details_payload.get(
            "data"
        )

        # ---------------------------------------------------------------
        # The frontend explicitly allows either:
        #
        #   response.data[0]
        #
        # or:
        #
        #   response.data
        #
        # so support both observed possibilities.
        # ---------------------------------------------------------------

        if (
            isinstance(
                details_data,
                list,
            )
            and details_data
            and isinstance(
                details_data[0],
                dict,
            )
        ):
            horse_record = details_data[0]

            print(
                "Details data shape: list"
            )

            print(
                "Details rows:",
                len(details_data),
            )

        elif isinstance(
            details_data,
            dict,
        ):
            horse_record = details_data

            print(
                "Details data shape: dict"
            )

        else:
            horse_record = None

            print(
                "Unexpected details data shape:",
                type(details_data).__name__,
            )


        if horse_record is not None:
            print(
                "\nHorse-detail field inventory:"
            )

            print_dict_field_inventory(
                horse_record
            )


            # -----------------------------------------------------------
            # 10. Print the complete bounded representative detail record.
            # -----------------------------------------------------------
            #
            # One horse record is small enough to inspect directly, but cap the
            # rendering defensively in case a future response embeds a large
            # history object.

            print(
                "\nRepresentative horse-detail record:"
            )

            print(
                json.dumps(
                    horse_record,
                    indent=2,
                    ensure_ascii=False,
                )[:20_000]
            )


            # -----------------------------------------------------------
            # 11. Give handicapperRatings special structural treatment.
            # -----------------------------------------------------------
            #
            # This field was explicitly consumed by the public frontend and is
            # therefore an already-demonstrated source capability.

            handicapper_ratings = horse_record.get(
                "handicapperRatings"
            )

            print(
                "\nHANDICAPPER RATINGS STRUCTURE"
            )
            print(
                "============================="
            )

            print(
                "Type:",
                type(
                    handicapper_ratings
                ).__name__,
            )

            if isinstance(
                handicapper_ratings,
                list,
            ):
                print(
                    "Rows:",
                    len(
                        handicapper_ratings
                    ),
                )

                rating_rows = [
                    row
                    for row in handicapper_ratings
                    if isinstance(
                        row,
                        dict,
                    )
                ]

                if rating_rows:
                    rating_field_union = sorted(
                        {
                            key
                            for row in rating_rows
                            for key in row.keys()
                        }
                    )

                    print(
                        "Fields:",
                        rating_field_union,
                    )

                    print(
                        "\nFirst three rating-history rows:"
                    )

                    for row in rating_rows[:3]:
                        print(
                            json.dumps(
                                row,
                                indent=2,
                                ensure_ascii=False,
                            )
                        )

            elif isinstance(
                handicapper_ratings,
                dict,
            ):
                print(
                    "Keys:",
                    sorted(
                        handicapper_ratings.keys()
                    ),
                )

                print(
                    json.dumps(
                        handicapper_ratings,
                        indent=2,
                        ensure_ascii=False,
                    )[:10_000]
                )


# ---------------------------------------------------------------------------
# 12. Inspect racehorse performances.
# ---------------------------------------------------------------------------
#
# Only page 1 has been requested.
#
# We are especially interested in whether each performance contains BHA race
# identifiers, dates, courses and performance figures that can reconnect the
# horse-centric history to the race-centric source already investigated.

performances_payload = resource_results[
    "performances"
]["envelope"]["parsed_json"]

if performances_payload is None:
    print(
        "\nHORSE PERFORMANCES"
    )
    print(
        "=================="
    )

    print(
        "No parsed JSON."
    )

    print(
        resource_results[
            "performances"
        ]["envelope"]["response_text"][:5000]
    )

else:
    report_paginated_resource(
        "HORSE PERFORMANCES — PAGE 1",
        performances_payload,
        sample_limit=3,
    )


# ---------------------------------------------------------------------------
# 13. Inspect training history.
# ---------------------------------------------------------------------------
#
# The frontend expects a `.data` list but does not demonstrate pagination.
# Use the same reporter if the response is paginated; otherwise describe the
# actual shape directly.

training_payload = resource_results[
    "training_history"
]["envelope"]["parsed_json"]

print(
    "\nHORSE TRAINING HISTORY"
)
print(
    "======================"
)

if training_payload is None:
    print(
        "No parsed JSON."
    )

    print(
        resource_results[
            "training_history"
        ]["envelope"]["response_text"][:5000]
    )

elif isinstance(
    training_payload,
    dict,
):
    print(
        "Top-level keys:",
        sorted(
            training_payload.keys()
        ),
    )

    training_data = training_payload.get(
        "data"
    )

    print(
        "data type:",
        type(
            training_data
        ).__name__,
    )

    if isinstance(
        training_data,
        list,
    ):
        print(
            "Rows returned:",
            len(
                training_data
            ),
        )

        training_rows = [
            row
            for row in training_data
            if isinstance(
                row,
                dict,
            )
        ]

        if training_rows:
            training_field_union = sorted(
                {
                    key
                    for row in training_rows
                    for key in row.keys()
                }
            )

            print(
                "\nUnion of training-history fields:"
            )

            for field in training_field_union:
                print(
                    " -",
                    field,
                )


            print(
                "\nRepresentative training-history rows:"
            )

            for index, row in enumerate(
                training_rows[:5],
                start=1,
            ):
                print(
                    f"\nRow {index}"
                )
                print(
                    "-" * 50
                )

                print(
                    json.dumps(
                        row,
                        indent=2,
                        ensure_ascii=False,
                    )
                )

        elif training_data:
            print(
                repr(
                    training_data[:5]
                )
            )

    else:
        print(
            "\nBounded data preview:"
        )

        print(
            json.dumps(
                training_data,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

else:
    print(
        "Unexpected top-level type:",
        type(
            training_payload
        ).__name__,
    )

    print(
        json.dumps(
            training_payload,
            indent=2,
            ensure_ascii=False,
        )[:8000]
    )


# ---------------------------------------------------------------------------
# 14. State provenance and the acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Horse:",
    "ZYNAK (FR)"
)

print(
    "BHA horse ID:",
    HORSE_ID,
)

print(
    "Network requests made by this cell:",
    network_requests,
)

print(
    "Maximum possible network requests:",
    3,
)

for spec in RESOURCE_SPECS:
    print(
        f"{spec['name']} cache:",
        spec["cache_file"],
    )

print(
    "Performance pages requested: 1 only"
)

print(
    "Other horses requested: NO"
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA HORSE PROFILE — THREE-RESOURCE PROBE

details
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

performances
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

training_history
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

HORSE DETAILS
Top-level type: dict
Top-level keys: ['data', 'success']
Details data shape: dict

Horse-detail field inventory:
- dateOfBirth: str
- deathDate: null
- entryDetails: list
    rows/items: 0
- handicapperRatings: dict
    keys: ['awt', 'chase', 'flat', 'hurdle']
- handicapperRatingsHistory: dict
    keys: ['awt', 'chase', 'flat', 'hurdle']
- id: int
- lineage: dict
    keys: ['dam', 'sire']
- moniker: str
- name: str
- nameWithNoCountry: str
- ownerName: str
- ownerType: str
- performanceDetails: list
    rows/items: 1
    sample item fields: ['totalPlaces', 'totalPrizeMoney', 'totalRuns', 'totalWins']
- trainerId: int
- trainerName: str

Representative horse-detail recor

## BHA Horse Database/Profile — source-family conclusion

The individual BHA horse profile is backed by three demonstrated structured
resources:

1. horse details:

   `/bha/v1/racehorses/{horseId}`

2. career performances:

   `/bha/v1/racehorses/{horseId}/performances`

3. training history:

   `/bha/v1/racehorses/{horseId}/training-history`

A bounded probe was performed for:

- horse: `ZYNAK (FR)`;
- BHA horse ID: `3090473`.

The same identifier had already been observed as:

- `id = 3090473` in the racehorse-search resource;
- `animalId = 3090473` in the Official Ratings resource.

This provides direct cross-source evidence that these BHA resources share the
same horse identity for this example.

---

### Horse details

The individual horse-details resource exposed:

#### Identity

- `id`;
- `name`;
- `nameWithNoCountry`;
- `dateOfBirth`;
- `deathDate`;
- `moniker`.

Unlike the ratings resource, which exposed only year of foaling in the observed
record, the horse profile supplied a full date of birth.

#### Current trainer

- `trainerId`;
- `trainerName`.

#### Current owner

- `ownerName`;
- `ownerType`.

For the observed horse the owner type was:

`Syndicate`

No owner identifier was present in the observed horse-detail record.

#### Pedigree / lineage

The `lineage` object contained structured sire and dam identities.

Observed sire:

- `animalId = 2451002`;
- `animalName = Zarak (FR)`.

Observed dam:

- `animalId = 1631689`;
- `animalName = Zanatiya (FR)`.

This is stronger identity evidence than pedigree names alone because the BHA
provides horse identifiers for the parents.

#### Career summary

`performanceDetails` exposed summary fields including:

- `totalWins`;
- `totalRuns`;
- `totalPlaces`;
- `totalPrizeMoney`.

For the observed horse the profile reported:

- 3 runs;
- 0 wins;
- 1 place;
- total prize money `604.00`.

The semantics and currency of `totalPrizeMoney` have not yet been independently
established and should not be assumed solely from this record.

#### Current handicapper ratings

The profile contained a structured `handicapperRatings` object divided into:

- `flat`;
- `awt`;
- `hurdle`;
- `chase`.

For the observed horse:

- Flat rating = 62;
- rating date = `2026-08-11`.

The other code ratings were null.

#### Handicapper rating history

The profile also contained:

`handicapperRatingsHistory`

again divided into:

- `flat`;
- `awt`;
- `hurdle`;
- `chase`.

The observed Flat history record contained:

- `value`;
- `publishedAt`;
- `date`;
- `trainerId`;
- `trainerName`.

For Zynak the observed historical record was:

- value = 62;
- `publishedAt = 2026-06-23`;
- `date = 2026-06-30`;
- trainer ID = 1101978;
- trainer = Kevin Philippart de Foy.

The precise distinction between `publishedAt` and `date` remains unresolved and
must not be inferred from field names alone.

#### Entries

The details resource contained an `entryDetails` list.

For the observed horse this list was empty.

This demonstrates that the field exists but does **not** establish the schema
of a populated entry record.

---

## Career performance history

The `/performances` resource is paginated.

For Zynak it reported:

- 3 total records;
- 10 records per page;
- one page.

All three career performances were therefore observed in this sample.

### Race identity

Each performance contained:

- `raceId`;
- `yearOfRace`;
- `divisionSequence`.

These are the same components already observed in the BHA race-resource
identity:

> `yearOfRace + raceId + divisionSequence`

The horse-centric career history can therefore potentially be linked directly
to the race-centric BHA resource without relying on race name/date matching.

This is an important source capability.

### Race information

Observed fields included:

- `courseName`;
- `raceDateTime`;
- `raceDate`;
- `raceTime`;
- `raceClass`;
- `raceType`;
- `raceDistanceText`;
- `raceName`;
- `raceRunners`;
- `going`.

### Horse result information

Observed fields included:

- `resultPosition`;
- `bettingRatio`;
- `weightCarried`;
- `prizeMoney`;
- `status`;
- `nonRunnerDeclaredReason`;
- `nonRunnerDeclaredTimestamp`.

The units or governed semantics of fields such as `weightCarried` should not be
assigned solely from their numeric representation.

### Participant identity

Observed fields included:

- `animalId`;
- `jockeyId`;
- `jockeyName`;
- `trainerId`;
- `trainerName`.

This provides another horse-centric route into the BHA participant identity
network.

---

## Important rating-field warning

Every one of Zynak's three observed historical performance rows contained:

`ratingFlat = 62`

The races occurred on:

- 15 May 2026;
- 8 June 2026;
- 20 June 2026.

However, the horse profile's observed handicapper-rating history showed the
62 rating with:

`publishedAt = 23 June 2026`

All three races therefore occurred before that observed publication date.

Consequently:

> the `ratingFlat`, `ratingAWT`, `ratingChase` and `ratingHurdle` fields on a
> historical performance row must **not** currently be interpreted as the
> horse's rating applicable on that race date.

The observed evidence is consistent with those fields carrying a later/current
profile rating onto historical performance rows, but that interpretation has
not yet been formally established.

This is a material semantic trap for future use of the source.

---

## Performance figures

The career-performance endpoint did **not** expose an explicit BHA performance
figure in the observed field set.

This matters because BHA documentation states that performance figures are
retained for individual runs, while the separately investigated Performance
Figures CSV exposes only the latest six figures without explicit race IDs.

Therefore the investigation has **not yet demonstrated a structured public
resource that directly joins an individual BHA performance figure to its BHA
race identity**.

Do not assume that the horse `/performances` endpoint solves that problem.

---

## Training history

The `/training-history` resource returned structured trainer-history records.

Observed fields were:

- `dateStart`;
- `dateEnd`;
- `trainingType`;
- `trainerId`;
- `trainerName`.

For Zynak the observed record was:

- start = `2024-12-06`;
- end = null;
- training type = `Flat/Jump training`;
- trainer ID = 1101978;
- trainer = Kevin Philippart de Foy.

This demonstrates that the BHA horse source contains temporal training
information rather than merely the horse's current trainer.

---

## Source assessment

**Potential value: very high.**

The BHA horse source potentially contributes:

- stable BHA horse identity;
- full date of birth;
- death information;
- structured sire/dam identities;
- owner information;
- trainer identity;
- career summary;
- current ratings;
- rating history;
- race-linked career performance history;
- jockey/trainer identities within performances;
- non-runner information;
- training history;
- a route for future-entry information through `entryDetails`.

Several of these overlap with Database v4 and other BHA source families.

They should later be compared rather than automatically duplicated.

### Particularly notable BHA-only candidates for later comparison

The following deserve specific attention when the BHA-v4 comparison phase
begins:

- BHA parent horse IDs;
- full horse date of birth;
- owner type;
- handicapper rating history;
- temporal trainer history;
- BHA race identifiers attached to horse career history.

Whether these are actually absent from Database v4 has **not yet been tested**.

---

## Decision

The Horse Database/Profile source family is sufficiently mapped for the
site-wide inventory.

Do not now paginate through more horse histories or investigate additional
horses merely to increase sample size.

Record the historical-performance rating-field warning for later semantic
testing.

Move to the next public BHA source family:

**Jockey data.**

In [35]:
# BHA Jockey Data — search/frontend route discovery
#
# WHAT
# ----
# Inspect the public BHA jockey-search results page and the first-party
# JavaScript assets it loads.
#
# The purpose is to discover the structured resources used by the public
# jockey interface BEFORE requesting any jockey records.
#
# Specifically this cell will:
#
#   1. fetch the public jockey-search results HTML page;
#   2. extract first-party BHA JavaScript assets referenced by that page;
#   3. inspect those assets for jockey-related controllers/factories/services;
#   4. recover `/bha/v1/...` route strings and `apiaddress` URL construction;
#   5. print bounded HTTP-call contexts around relevant jockey code.
#
# WHY
# ---
# We are mapping the public BHA information surface, not guessing API routes.
#
# The public BHA jockey pages visibly expose information including:
#
#   - jockey identity;
#   - licence/permit type;
#   - age;
#   - geographical base;
#   - days since last win;
#   - championship information;
#   - career information;
#   - performance history.
#
# We need to establish which structured resources actually power those views.
#
# READS
# -----
# - one public BHA HTML page:
#
#     /racing/participants/jockeys/jockey-search-results/
#
# - first-party BHA JavaScript assets referenced by that page.
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       jockey_frontend_discovery/
#
# Specifically:
#
#   jockey_search_results_page.json
#   jockey_script_inventory.json
#
# Raw individual JavaScript assets are not persisted separately.
#
# EXPECTED RESULT
# ---------------
# A bounded inventory showing:
#
#   - profile/search page HTTP status;
#   - first-party JS assets;
#   - jockey-related Angular controllers/factories/services;
#   - concrete `/bha/v1/...` route clues;
#   - relevant `$http` request contexts;
#   - enough evidence to choose the smallest jockey API probe next.
#
# DECISION BOUNDARY
# -----------------
# Do NOT request any jockey records in this cell.
#
# First identify the public frontend resource surface.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on Jupyter's current working directory.

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "jockey_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "jockey_search_results_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "jockey_script_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact public jockey-search results page.
# ---------------------------------------------------------------------------

JOCKEY_SEARCH_RESULTS_URL = (
    "https://www.britishhorseracing.com/"
    "racing/participants/jockeys/jockey-search-results/"
)


# ---------------------------------------------------------------------------
# 3. Public text-request helper.
# ---------------------------------------------------------------------------
#
# Do not use `curl --fail`.
#
# An HTTP block or unavailable asset is research evidence and should not crash
# the notebook before it can be recorded.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url, accept):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            f"Accept: {accept}",
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the public jockey-search results page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == JOCKEY_SEARCH_RESULTS_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        JOCKEY_SEARCH_RESULTS_URL,
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # Defensively prevent persistence of any unexpected literal Bearer token.
    # -----------------------------------------------------------------------
    #
    # None is expected because this is ordinary public page HTML.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Jockey Data"
        ),
        "request_url": JOCKEY_SEARCH_RESULTS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 5. Report page-access evidence before inspecting the frontend.
# ---------------------------------------------------------------------------

print(
    "BHA JOCKEY DATA — FRONTEND ROUTE DISCOVERY"
)
print(
    "=========================================="
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Page SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 6. Stop cleanly if the public page itself was unavailable.
# ---------------------------------------------------------------------------

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nJockey search-results page unavailable."
    )

    print(
        "Jockey frontend resource surface remains unresolved."
    )

else:
    # -----------------------------------------------------------------------
    # 7. Extract JavaScript assets referenced by this jockey page.
    # -----------------------------------------------------------------------

    script_sources = re.findall(
        r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    resolved_scripts = []

    for source in script_sources:
        decoded_source = html.unescape(
            source
        )

        resolved_scripts.append(
            urljoin(
                page_envelope["final_url"]
                or JOCKEY_SEARCH_RESULTS_URL,
                decoded_source,
            )
        )

    resolved_scripts = sorted(
        set(
            resolved_scripts
        )
    )


    # -----------------------------------------------------------------------
    # 8. Keep only first-party BHA JavaScript assets.
    # -----------------------------------------------------------------------

    BHA_HOSTS = {
        "www.britishhorseracing.com",
        "britishhorseracing.com",
    }

    first_party_scripts = []

    for script_url in resolved_scripts:
        parsed = urlsplit(
            script_url
        )

        if parsed.hostname not in BHA_HOSTS:
            continue

        if ".js" not in parsed.path.lower():
            continue

        first_party_scripts.append(
            script_url
        )


    # -----------------------------------------------------------------------
    # 9. Bound the number of script requests.
    # -----------------------------------------------------------------------
    #
    # Earlier BHA pages exposed roughly 20–30 first-party assets.
    #
    # Forty is therefore a generous ceiling while preventing an accidental
    # broad crawl if the website structure changes.

    MAX_SCRIPT_ASSETS = 40

    scripts_to_inspect = first_party_scripts[
        :MAX_SCRIPT_ASSETS
    ]

    script_inventory_truncated = (
        len(first_party_scripts)
        > MAX_SCRIPT_ASSETS
    )


    # -----------------------------------------------------------------------
    # 10. Inspect each asset for jockey-specific frontend evidence.
    # -----------------------------------------------------------------------

    asset_observations = []
    all_route_tokens = set()

    JOCKEY_TERMS = (
        "jockey",
        "jockeys",
        "rider",
        "licence",
        "license",
        "championship",
        "dayssincelastwin",
        "lowestweight",
    )

    for script_url in scripts_to_inspect:
        script_response = curl_public_text(
            script_url,
            (
                "application/javascript,"
                "text/javascript,*/*;q=0.8"
            ),
        )

        script_text = script_response[
            "body"
        ]

        script_sha256 = (
            hashlib.sha256(
                script_text.encode(
                    "utf-8"
                )
            ).hexdigest()
            if script_text
            else None
        )


        # -------------------------------------------------------------------
        # 11. Recover Angular component names.
        # -------------------------------------------------------------------

        controllers = sorted(
            set(
                re.findall(
                    r"""\.controller\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        factories = sorted(
            set(
                re.findall(
                    r"""\.factory\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        services = sorted(
            set(
                re.findall(
                    r"""\.service\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )


        # -------------------------------------------------------------------
        # 12. Recover explicit `/bha/v1/...` route tokens.
        # -------------------------------------------------------------------

        route_tokens = sorted(
            set(
                re.findall(
                    r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        all_route_tokens.update(
            route_tokens
        )


        # -------------------------------------------------------------------
        # 13. Recover jockey-related quoted strings.
        # -------------------------------------------------------------------
        #
        # This helps reveal relative routes or controller configuration where
        # the API path is dynamically concatenated.

        quoted_strings = re.findall(
            r"""(['"])([^'"\r\n]{1,500})\1""",
            script_text,
        )

        jockey_strings = sorted(
            {
                value
                for _, value in quoted_strings
                if any(
                    term in value.lower()
                    for term in JOCKEY_TERMS
                )
            }
        )


        # -------------------------------------------------------------------
        # 14. Capture bounded contexts around jockey terms and `apiaddress`.
        # -------------------------------------------------------------------

        marker_contexts = []

        marker_patterns = [
            r"apiaddress",
            r"/bha/v1/",
            r"jockey",
            r"daysSinceLastWin",
            r"lowestRidingWeight",
        ]

        for marker_pattern in marker_patterns:
            for match in re.finditer(
                marker_pattern,
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 900,
                )

                end = min(
                    len(script_text),
                    match.end() + 1800,
                )

                context = script_text[
                    start:end
                ]

                compact = re.sub(
                    r"\s+",
                    " ",
                    context,
                ).strip()

                if compact not in marker_contexts:
                    marker_contexts.append(
                        compact
                    )


        # -------------------------------------------------------------------
        # 15. Capture bounded `$http` contexts relevant to jockey resources.
        # -------------------------------------------------------------------

        http_contexts = []

        for match in re.finditer(
            r"\$http",
            script_text,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1000,
            )

            end = min(
                len(script_text),
                match.end() + 2200,
            )

            context = script_text[
                start:end
            ]

            lower_context = context.lower()

            if not (
                "jockey" in lower_context
                or "apiaddress" in lower_context
                or "/bha/v1/" in lower_context
            ):
                continue

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            if compact not in http_contexts:
                http_contexts.append(
                    compact
                )


        # -------------------------------------------------------------------
        # 16. Decide whether this script is materially relevant.
        # -------------------------------------------------------------------

        component_text = " ".join(
            controllers
            + factories
            + services
        ).lower()

        asset_is_relevant = bool(
            jockey_strings
            or any(
                "jockey" in route.lower()
                for route in route_tokens
            )
            or any(
                term in component_text
                for term in JOCKEY_TERMS
            )
        )

        asset_observations.append(
            {
                "script_url": script_url,
                "http_status": (
                    script_response["status"]
                ),
                "sha256": script_sha256,
                "controllers": controllers,
                "factories": factories,
                "services": services,
                "route_tokens": route_tokens,
                "jockey_strings": jockey_strings,
                "marker_contexts": (
                    marker_contexts[:15]
                ),
                "http_contexts": (
                    http_contexts[:15]
                ),
                "relevant": asset_is_relevant,
            }
        )


    # -----------------------------------------------------------------------
    # 17. Persist the derived frontend inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Jockey Data"
        ),
        "page_url": JOCKEY_SEARCH_RESULTS_URL,
        "first_party_scripts": (
            first_party_scripts
        ),
        "scripts_inspected": len(
            scripts_to_inspect
        ),
        "script_inventory_truncated": (
            script_inventory_truncated
        ),
        "all_bha_v1_route_tokens": sorted(
            all_route_tokens
        ),
        "asset_observations": (
            asset_observations
        ),
        "authorization_sent": False,
        "jockey_records_requested": 0,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 18. Report the first-party script inventory.
    # -----------------------------------------------------------------------

    print(
        "\nFIRST-PARTY JOCKEY-PAGE SCRIPTS"
    )
    print(
        "==============================="
    )

    print(
        "First-party scripts discovered:",
        len(first_party_scripts),
    )

    print(
        "Scripts inspected:",
        len(scripts_to_inspect),
    )

    print(
        "Inspection truncated:",
        (
            "YES"
            if script_inventory_truncated
            else "NO"
        ),
    )

    for script_url in scripts_to_inspect:
        print(
            " ",
            script_url,
        )


    # -----------------------------------------------------------------------
    # 19. Report only materially jockey-related frontend assets.
    # -----------------------------------------------------------------------

    relevant_assets = [
        item
        for item in asset_observations
        if item["relevant"]
    ]

    print(
        "\nRELEVANT JOCKEY FRONTEND ASSETS"
    )
    print(
        "==============================="
    )

    print(
        "Relevant assets:",
        len(relevant_assets),
    )

    if not relevant_assets:
        print(
            "NONE RECOVERED"
        )

    for index, item in enumerate(
        relevant_assets,
        start=1,
    ):
        print(
            f"\nAsset {index}"
        )
        print(
            "-" * 70
        )

        print(
            "URL:",
            item["script_url"],
        )

        print(
            "HTTP:",
            item["http_status"],
        )

        print(
            "SHA-256:",
            item["sha256"],
        )

        print(
            "Controllers:",
            item["controllers"] or "NONE",
        )

        print(
            "Factories:",
            item["factories"] or "NONE",
        )

        print(
            "Services:",
            item["services"] or "NONE",
        )

        print(
            "Route tokens:",
            item["route_tokens"] or "NONE",
        )

        print(
            "Jockey-related strings:",
            item["jockey_strings"] or "NONE",
        )

        for context_index, context in enumerate(
            item["marker_contexts"],
            start=1,
        ):
            print(
                f"\n  Marker context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )

        for context_index, context in enumerate(
            item["http_contexts"],
            start=1,
        ):
            print(
                f"\n  HTTP context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )


    # -----------------------------------------------------------------------
    # 20. Present the deduplicated BHA-v1 route inventory separately.
    # -----------------------------------------------------------------------

    print(
        "\nDEDUPLICATED JOCKEY-PAGE BHA V1 ROUTES"
    )
    print(
        "======================================"
    )

    if all_route_tokens:
        for route in sorted(
            all_route_tokens
        ):
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 21. State the evidence/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived frontend inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Jockey records requested: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA JOCKEY DATA — FRONTEND ROUTE DISCOVERY
Loaded page from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/participants/jockeys/jockey-search-results/
Page SHA-256: 1536077e57efd9d90aaa8e67af493c535e3096d1344fd832fade4e359d7512f4

FIRST-PARTY JOCKEY-PAGE SCRIPTS
First-party scripts discovered: 23
Scripts inspected: 23
Inspection truncated: NO
  https://www.britishhorseracing.com/cdn-cgi/scripts/5c5dd728/cloudflare-static/email-decode.min.js
  https://www.britishhorseracing.com/wp-content/plugins/magic-liquidizer-responsive-table/idjs/ml.responsive.table.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/flickity.pkgd.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/functions.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/iframeResizer.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/js.cookie.js
  https://www.britishhorseracing.com/wp-content/theme

In [36]:
# BHA Jockey Data — bounded profile + championship probe
#
# WHAT
# ----
# Inspect two demonstrated BHA jockey resource families:
#
#   1. individual jockey profile:
#
#        /bha/v1/jockeys/{jockeyId}
#
#   2. jockey championship rankings:
#
#        /bha/v1/championships/jockeys
#
# Use a jockey identity already observed in the horse-performance source:
#
#   Laura Pearson
#   BHA jockey ID = 1132245
#
# For the championship endpoint request only five current Flat rows:
#
#   type     = flat
#   sort     = rank:asc
#   page     = 1
#   per_page = 5
#
# WHY
# ---
# Frontend discovery has already established that the public BHA jockey surface
# consists of at least:
#
#   - searchable jockey records;
#   - individual jockey profiles;
#   - Flat and Jump championship rankings.
#
# We do not need to crawl jockeys or championship tables. We only need enough
# structured evidence to determine what information these source families
# actually expose.
#
# Using Laura Pearson also tests whether the jockey identifier already attached
# to horse-performance records links directly into the jockey-profile resource.
#
# READS
# -----
# - BHA Authorization from repo-root `.env.local`;
# - exactly two structured BHA requests at most.
#
# WRITES
# ------
# Ignored research cache:
#
#   data/cache/bha_official_source_feasibility/
#       jockey_resource_probe/
#
# Files:
#
#   laura_pearson_profile.json
#   flat_championship_top5.json
#
# Every response is cached, including errors or empty responses.
#
# EXPECTED RESULT
# ---------------
# For the individual jockey profile:
#
#   - complete top-level field inventory;
#   - bounded nested structures;
#   - representative profile record.
#
# For the Flat championship:
#
#   - pagination metadata;
#   - record schema;
#   - five representative ranking rows.
#
# PARTICULARLY IMPORTANT
# ----------------------
# Look for information such as:
#
#   - BHA jockey identity;
#   - licence/permit information;
#   - date of birth / age;
#   - location;
#   - career totals;
#   - season totals;
#   - wins/runs;
#   - prize money;
#   - championship rank;
#   - days since last win;
#   - lowest riding weight;
#   - links or nested history.
#
# Do NOT assign semantics merely from field names.
#
# ACQUISITION BOUNDARY
# --------------------
# - one jockey profile only;
# - five Flat championship rows only;
# - no Jump championship request yet;
# - no jockey population download;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "jockey_resource_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization value.
# ---------------------------------------------------------------------------
#
# The credential must never be printed or persisted.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the exact jockey and two frontend-demonstrated resources.
# ---------------------------------------------------------------------------

JOCKEY_ID = 1132245
JOCKEY_NAME = "Laura Pearson"

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

PROFILE_URL = (
    f"{BHA_BASE}/jockeys/{JOCKEY_ID}"
)

CHAMPIONSHIP_PARAMS = {
    "type": "flat",
    "sort": "rank:asc",
    "page": 1,
    "per_page": 5,
}

CHAMPIONSHIP_URL = (
    f"{BHA_BASE}/championships/jockeys?"
    f"{urlencode(CHAMPIONSHIP_PARAMS)}"
)

RESOURCE_SPECS = [
    {
        "name": "jockey_profile",
        "url": PROFILE_URL,
        "cache_file": (
            CACHE_DIR
            / "laura_pearson_profile.json"
        ),
    },
    {
        "name": "flat_championship",
        "url": CHAMPIONSHIP_URL,
        "cache_file": (
            CACHE_DIR
            / "flat_championship_top5.json"
        ),
    },
]


# ---------------------------------------------------------------------------
# 4. Request/cache helper.
# ---------------------------------------------------------------------------
#
# Every external request leaves evidence in the ignored cache.
#
# HTTP failures are preserved rather than allowed to terminate the notebook
# before their response bodies can be inspected.

def load_or_request(spec):
    cache_file = spec[
        "cache_file"
    ]

    if cache_file.exists():
        envelope = json.loads(
            cache_file.read_text(
                encoding="utf-8"
            )
        )

        assert (
            envelope["request_url"]
            == spec["url"]
        )

        return envelope, "cache"


    request = Request(
        spec["url"],
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/participants/jockeys/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Parse JSON conservatively.
    # -----------------------------------------------------------------------

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # Persist response evidence WITHOUT the Authorization value.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Jockey Data"
        ),
        "resource_name": spec["name"],
        "request_url": spec["url"],
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    temp_file = cache_file.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        cache_file
    )

    return envelope, "network"


# ---------------------------------------------------------------------------
# 5. Make/reuse the two bounded resource requests.
# ---------------------------------------------------------------------------

results = {}
network_requests = 0

for spec in RESOURCE_SPECS:
    envelope, source = load_or_request(
        spec
    )

    results[
        spec["name"]
    ] = {
        "envelope": envelope,
        "source": source,
        "cache_file": spec["cache_file"],
    }

    if source == "network":
        network_requests += 1


# ---------------------------------------------------------------------------
# 6. Helper for one-level nested field inventory.
# ---------------------------------------------------------------------------
#
# We want to discover the source schema without blindly dumping arbitrarily
# large nested structures.

def describe_record(record):
    for key in sorted(
        record.keys()
    ):
        value = record[
            key
        ]

        if value is None:
            type_name = "null"
        else:
            type_name = type(
                value
            ).__name__

        print(
            f" - {key}: {type_name}"
        )

        if isinstance(
            value,
            dict,
        ):
            print(
                "     keys:",
                sorted(
                    value.keys()
                ),
            )

        elif isinstance(
            value,
            list,
        ):
            print(
                "     items:",
                len(value),
            )

            sample_dicts = [
                item
                for item in value[:3]
                if isinstance(
                    item,
                    dict,
                )
            ]

            if sample_dicts:
                nested_fields = sorted(
                    {
                        nested_key
                        for item in sample_dicts
                        for nested_key in item.keys()
                    }
                )

                print(
                    "     sample item fields:",
                    nested_fields,
                )


# ---------------------------------------------------------------------------
# 7. Report transport evidence before interpretation.
# ---------------------------------------------------------------------------

print(
    "BHA JOCKEY DATA — PROFILE + CHAMPIONSHIP PROBE"
)
print(
    "=============================================="
)

for spec in RESOURCE_SPECS:
    result = results[
        spec["name"]
    ]

    envelope = result[
        "envelope"
    ]

    print(
        f"\n{spec['name']}"
    )

    print(
        "  Loaded from:",
        result["source"],
    )

    print(
        "  HTTP status:",
        envelope["response_status"],
    )

    print(
        "  Content-Type:",
        envelope["content_type"],
    )

    if envelope["transport_error"]:
        print(
            "  Transport observation:",
            envelope["transport_error"],
        )


# ---------------------------------------------------------------------------
# 8. Inspect the individual jockey-profile response.
# ---------------------------------------------------------------------------
#
# The frontend expects:
#
#   response.data[0] || response.data
#
# so support either observed structure.

profile_payload = results[
    "jockey_profile"
]["envelope"]["parsed_json"]

print(
    "\nJOCKEY PROFILE"
)
print(
    "=============="
)

if profile_payload is None:
    print(
        "No parsed JSON."
    )

    print(
        results[
            "jockey_profile"
        ]["envelope"]["response_text"][:5000]
    )

elif isinstance(
    profile_payload,
    dict,
):
    print(
        "Top-level keys:",
        sorted(
            profile_payload.keys()
        ),
    )

    profile_data = profile_payload.get(
        "data"
    )

    if (
        isinstance(
            profile_data,
            list,
        )
        and profile_data
        and isinstance(
            profile_data[0],
            dict,
        )
    ):
        profile_record = profile_data[0]

        print(
            "Profile data shape: list"
        )

        print(
            "Profile rows:",
            len(
                profile_data
            ),
        )

    elif isinstance(
        profile_data,
        dict,
    ):
        profile_record = profile_data

        print(
            "Profile data shape: dict"
        )

    else:
        profile_record = None

        print(
            "Unexpected profile data type:",
            type(
                profile_data
            ).__name__,
        )


    if profile_record is not None:
        print(
            "\nProfile field inventory:"
        )

        describe_record(
            profile_record
        )


        # -------------------------------------------------------------------
        # Print one bounded representative profile record.
        # -------------------------------------------------------------------
        #
        # Cap output defensively in case the profile embeds sizeable history.

        print(
            "\nRepresentative profile record:"
        )

        print(
            json.dumps(
                profile_record,
                indent=2,
                ensure_ascii=False,
            )[:25_000]
        )


        # -------------------------------------------------------------------
        # 9. Explicitly test the cross-source jockey identity.
        # -------------------------------------------------------------------
        #
        # The jockey ID came from Zynak's BHA horse-performance records.

        id_matches = []

        for field, value in profile_record.items():
            if str(value) == str(
                JOCKEY_ID
            ):
                id_matches.append(
                    {
                        "field": field,
                        "value": value,
                    }
                )

        print(
            "\nCROSS-SOURCE ID CHECK"
        )
        print(
            "====================="
        )

        print(
            "Horse-performance jockeyId:",
            JOCKEY_ID,
        )

        if id_matches:
            print(
                "Exact matching profile value found: YES"
            )

            for match in id_matches:
                print(
                    f"  field={match['field']} "
                    f"value={match['value']}"
                )

        else:
            print(
                "Exact matching profile value found: NO"
            )

else:
    print(
        "Unexpected top-level profile type:",
        type(
            profile_payload
        ).__name__,
    )

    print(
        json.dumps(
            profile_payload,
            indent=2,
            ensure_ascii=False,
        )[:8000]
    )


# ---------------------------------------------------------------------------
# 10. Inspect the Flat championship response.
# ---------------------------------------------------------------------------

championship_payload = results[
    "flat_championship"
]["envelope"]["parsed_json"]

print(
    "\nFLAT JOCKEY CHAMPIONSHIP — TOP FIVE"
)
print(
    "==================================="
)

if championship_payload is None:
    print(
        "No parsed JSON."
    )

    print(
        results[
            "flat_championship"
        ]["envelope"]["response_text"][:5000]
    )

elif isinstance(
    championship_payload,
    dict,
):
    print(
        "Top-level keys:",
        sorted(
            championship_payload.keys()
        ),
    )

    scalar_metadata = {
        key: value
        for key, value in championship_payload.items()
        if not isinstance(
            value,
            (dict, list),
        )
    }

    print(
        "Scalar metadata:",
        scalar_metadata,
    )

    championship_data = championship_payload.get(
        "data"
    )

    print(
        "data type:",
        type(
            championship_data
        ).__name__,
    )

    if isinstance(
        championship_data,
        list,
    ):
        print(
            "Rows returned:",
            len(
                championship_data
            ),
        )

        championship_rows = [
            row
            for row in championship_data
            if isinstance(
                row,
                dict,
            )
        ]

        if championship_rows:
            field_union = sorted(
                {
                    key
                    for row in championship_rows
                    for key in row.keys()
                }
            )

            print(
                "\nUnion of championship fields:"
            )

            for field in field_union:
                print(
                    " -",
                    field,
                )


            print(
                "\nRepresentative championship rows:"
            )

            for index, row in enumerate(
                championship_rows[:5],
                start=1,
            ):
                print(
                    f"\nRank/sample row {index}"
                )
                print(
                    "-" * 50
                )

                print(
                    json.dumps(
                        row,
                        indent=2,
                        ensure_ascii=False,
                    )
                )

        elif championship_data:
            print(
                repr(
                    championship_data[:5]
                )
            )

    else:
        print(
            "\nUnexpected championship data shape:"
        )

        print(
            json.dumps(
                championship_data,
                indent=2,
                ensure_ascii=False,
            )[:8000]
        )

else:
    print(
        "Unexpected championship top-level type:",
        type(
            championship_payload
        ).__name__,
    )

    print(
        json.dumps(
            championship_payload,
            indent=2,
            ensure_ascii=False,
        )[:8000]
    )


# ---------------------------------------------------------------------------
# 11. State the evidence/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Jockey:",
    JOCKEY_NAME,
)

print(
    "BHA jockey ID:",
    JOCKEY_ID,
)

print(
    "Network requests made by this cell:",
    network_requests,
)

print(
    "Maximum possible network requests:",
    2,
)

print(
    "Jockey profile cache:",
    results[
        "jockey_profile"
    ]["cache_file"],
)

print(
    "Flat championship cache:",
    results[
        "flat_championship"
    ]["cache_file"],
)

print(
    "Flat championship rows requested:",
    5,
)

print(
    "Jump championship requested: NO"
)

print(
    "Other jockey profiles requested: NO"
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA JOCKEY DATA — PROFILE + CHAMPIONSHIP PROBE

jockey_profile
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

flat_championship
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

JOCKEY PROFILE
Top-level keys: ['data', 'success']
Profile data shape: dict

Profile field inventory:
 - age: int
 - career: dict
     keys: ['groupListedWins', 'prizeMoney', 'rides', 'wins', 'winsToRides']
 - county: str
 - dateOfBirth: str
 - daysSinceLastWin: int
 - id: int
 - jockeyImage: null
 - licenseType: str
 - lowestRidingWeight: int
 - name: str
 - recentPerformances: list
     items: 10
     sample item fields: ['animalId', 'animalName', 'awtRating', 'bettingRatio', 'chaseRating', 'courseId', 'courseName', 'divisionSequence', 'flatRating', 'going', 'hurdleRating', 'jockeyId', 'jockeyName', 'prizeMoney', 'raceClass', 'raceDate', 'raceDateTime', 'raceDistanceText', 'raceId', 'raceName', 'raceRunners', 'raceTime', 'raceType', 'resultPosition', 'sta

## BHA Jockey Data — source-family conclusion

The public BHA jockey surface is backed by at least two distinct structured
resource families:

1. jockey search/profile resources:

   - `/bha/v1/jockeys`;
   - `/bha/v1/jockeys/{jockeyId}`;

2. jockey championship resources:

   - `/bha/v1/championships/jockeys`.

A bounded profile probe used:

- jockey: Laura Pearson;
- BHA jockey ID: `1132245`.

This identifier had already been observed in BHA horse-performance records.

The individual jockey profile returned:

`id = 1132245`

providing direct cross-source evidence that the horse-performance and jockey
profile resources share the same BHA jockey identity for this example.

---

### Individual jockey profile

The observed profile exposed:

#### Identity and personal information

- `id`;
- `name`;
- `dateOfBirth`;
- `age`;
- `county`.

For the observed jockey:

- date of birth = `2001-03-01`;
- age = 25;
- county = Cambridgeshire.

The `age` field should be treated as a derived/current presentation field unless
its update semantics are later demonstrated.

#### Licensing / riding information

Observed fields included:

- `licenseType`;
- `lowestRidingWeight`.

For Laura Pearson:

- licence type = `Flat`;
- lowest riding weight = 114.

The unit represented by `lowestRidingWeight` has not yet been formally
established from this field alone and should not be assumed solely from the
numeric value.

#### Activity indicator

The profile exposed:

- `daysSinceLastWin`.

For the observed profile:

`daysSinceLastWin = 30`

This is clearly time-sensitive current-state information and would require
observation-date provenance if ever governed or analysed.

#### Career statistics

The nested `career` object contained:

- `wins`;
- `rides`;
- `winsToRides`;
- `groupListedWins`;
- `prizeMoney`.

Observed values for Laura Pearson were:

- wins = 124;
- rides = 1,383;
- wins-to-rides = 8.97;
- Group/Listed wins = 2;
- prize money = 1,457,166.68.

These are BHA-provided career aggregates.

Their exact population, jurisdictional scope and prize-money definition have
not yet been independently established and should not be reconstructed or
interpreted beyond the source evidence at this stage.

---

## Recent jockey performances

The jockey profile contained a `recentPerformances` list.

Ten performance records were returned in the observed profile.

Observed fields included:

### Race identity

- `raceId`;
- `yearOfRace`;
- `divisionSequence`.

These are the same BHA race-reference components already observed in the
race-centric and horse-centric resources.

### Course/race information

- `courseId`;
- `courseName`;
- `raceDateTime`;
- `raceDate`;
- `raceTime`;
- `raceClass`;
- `raceType`;
- `raceDistanceText`;
- `raceName`;
- `raceRunners`;
- `going`.

### Horse identity

- `animalId`;
- `animalName`.

### Jockey identity

- `jockeyId`;
- `jockeyName`.

### Trainer identity

- `trainerId`;
- `trainerName`.

### Result information

- `resultPosition`;
- `bettingRatio`;
- `weightCarried`;
- `prizeMoney`;
- `status`.

### Rating fields

- `flatRating`;
- `awtRating`;
- `chaseRating`;
- `hurdleRating`.

These rating fields should NOT yet be assumed to represent ratings applicable
on the date of each race.

A similar semantic issue has already been demonstrated in the horse-performance
resource, where historical performance rows carried a later/current rating.

The jockey-profile performance rows therefore require separate semantic
validation before any race-date rating interpretation.

---

## Flat jockey championship

The public frontend also uses:

`/bha/v1/championships/jockeys`

with parameters including:

- `type`;
- `sort`;
- `page`;
- `per_page`.

A bounded Flat championship request returned five records from a reported
population of 375.

That `total = 375` should currently be interpreted only as the population
returned by this championship resource under the observed query.

It should NOT automatically be interpreted as:

- all licensed Flat jockeys;
- all active British jockeys;
- all jockeys appearing in the database.

### Championship fields

Observed fields were:

- `jockeyId`;
- `jockeyName`;
- `prizemoney`;
- `championshipWins`;
- `championshipRides`;
- `strikeRate`;
- `rank`;
- `startdate`;
- `enddate`.

The explicit dates are particularly important.

For the observed 2026 Flat championship data:

- start = `2026-05-02`;
- end = `2026-10-17`.

Therefore:

> BHA championship statistics are explicitly competition-period statistics and
> must not be treated as generic calendar-year or career jockey statistics.

This is an important semantic distinction for future comparison or analysis.

---

## Source assessment

**Potential value: high.**

The jockey source potentially contributes:

- stable BHA jockey identity;
- date of birth;
- age/current-age presentation;
- county/location;
- licence type;
- lowest riding weight;
- days since last win;
- career wins and rides;
- career strike-rate measure;
- Group/Listed wins;
- career prize money;
- recent race-linked performance history;
- championship rank;
- championship wins/rides;
- championship prize money;
- explicit championship date boundaries.

Several of these cannot be obtained simply by reading one race result.

Some may be reconstructible from complete race history, but the BHA aggregate
definitions and population boundaries would still need to be understood before
claiming equivalence.

### Important distinction

The jockey source contains at least three different analytical concepts:

1. **identity/profile state**;
2. **career aggregates**;
3. **championship-period aggregates**.

These must remain distinct.

A jockey's career strike rate is not the same measure as a championship strike
rate, and championship rank is meaningful only within its stated championship
period.

## Decision

The BHA jockey source family is sufficiently mapped for the site-wide source
inventory.

Do not download the full jockey population or championship tables during this
phase.

Do not yet investigate historical championship seasons.

Move to the next public BHA source family:

**Trainer data.**

In [37]:
# BHA Trainer Data — search/frontend route discovery
#
# WHAT
# ----
# Inspect the public BHA trainer-search results page and the first-party
# JavaScript assets it loads.
#
# The purpose is to discover the structured resources used by the public
# trainer interface BEFORE requesting any trainer records.
#
# Specifically this cell will:
#
#   1. fetch the public trainer-search results HTML page;
#   2. extract first-party BHA JavaScript assets referenced by that page;
#   3. inspect those assets for trainer-related controllers/factories/services;
#   4. recover `/bha/v1/...` route strings and `apiaddress` URL construction;
#   5. print bounded HTTP-call contexts around relevant trainer code.
#
# WHY
# ---
# The site-wide inventory has now mapped:
#
#   - ratings;
#   - horse profiles;
#   - jockey profiles/championships.
#
# Trainer data is the next distinct participant source family.
#
# We want to determine what the BHA itself exposes beyond trainer names carried
# on race-result records, potentially including:
#
#   - trainer identity;
#   - location;
#   - contact/profile information;
#   - performance/career statistics;
#   - championship information;
#   - yard information;
#   - non-runner statistics.
#
# None of those capabilities are assumed here.
#
# As with the previous source families, we let the public frontend reveal its
# actual structured resources rather than guessing endpoint names.
#
# READS
# -----
# - one public BHA HTML page:
#
#     /racing/participants/trainers/trainer-search-results/
#
# - first-party BHA JavaScript assets referenced by that page.
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       trainer_frontend_discovery/
#
# Specifically:
#
#   trainer_search_results_page.json
#   trainer_script_inventory.json
#
# Raw individual JavaScript assets are not persisted separately.
#
# EXPECTED RESULT
# ---------------
# A bounded inventory showing:
#
#   - trainer search-page HTTP status;
#   - first-party JS assets;
#   - trainer-related Angular controllers/factories/services;
#   - concrete `/bha/v1/...` route clues;
#   - relevant `$http` request contexts;
#   - enough evidence to choose the smallest trainer-resource probe next.
#
# DECISION BOUNDARY
# -----------------
# Do NOT request any trainer records in this cell.
#
# First map the public trainer frontend surface.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on Jupyter's current working directory.

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "trainer_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "trainer_search_results_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "trainer_script_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the public trainer-search results page.
# ---------------------------------------------------------------------------
#
# This mirrors the search-results pattern already demonstrated for horses and
# jockeys, but the response itself will establish whether this page exists and
# what frontend assets it actually uses.

TRAINER_SEARCH_RESULTS_URL = (
    "https://www.britishhorseracing.com/"
    "racing/participants/trainers/trainer-search-results/"
)


# ---------------------------------------------------------------------------
# 3. Public text-request helper.
# ---------------------------------------------------------------------------
#
# Do not use `curl --fail`.
#
# A non-200 response is useful evidence about the public source surface and
# should be cached rather than converted into an unexplained notebook failure.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url, accept):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            f"Accept: {accept}",
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the public trainer-search results page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == TRAINER_SEARCH_RESULTS_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        TRAINER_SEARCH_RESULTS_URL,
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # 5. Defensively redact any unexpected literal Bearer token.
    # -----------------------------------------------------------------------
    #
    # None is expected in ordinary public HTML, but the cache should never
    # persist an operational credential if the website implementation changes.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # 6. Cache the page response as research evidence.
    # -----------------------------------------------------------------------

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Trainer Data"
        ),
        "request_url": TRAINER_SEARCH_RESULTS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 7. Report page-access evidence before interpreting frontend code.
# ---------------------------------------------------------------------------

print(
    "BHA TRAINER DATA — FRONTEND ROUTE DISCOVERY"
)
print(
    "==========================================="
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Page SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 8. Stop cleanly if this guessed public search-results page is unavailable.
# ---------------------------------------------------------------------------
#
# Importantly, a 404 here would NOT mean the BHA has no trainer data.
# It would mean only that this exact page path was not demonstrated and the
# next step should discover the route from the trainers landing page instead.

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nTrainer search-results page unavailable."
    )

    print(
        "Do not infer that trainer data is unavailable."
    )

    print(
        "The public trainer landing page should be inspected next."
    )

else:
    # -----------------------------------------------------------------------
    # 9. Extract JavaScript assets referenced by the trainer page.
    # -----------------------------------------------------------------------

    script_sources = re.findall(
        r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    resolved_scripts = []

    for source in script_sources:
        decoded_source = html.unescape(
            source
        )

        resolved_scripts.append(
            urljoin(
                page_envelope["final_url"]
                or TRAINER_SEARCH_RESULTS_URL,
                decoded_source,
            )
        )

    resolved_scripts = sorted(
        set(
            resolved_scripts
        )
    )


    # -----------------------------------------------------------------------
    # 10. Keep only first-party BHA JavaScript assets.
    # -----------------------------------------------------------------------
    #
    # Third-party libraries are not useful for discovering the BHA's trainer
    # resource interface.

    BHA_HOSTS = {
        "www.britishhorseracing.com",
        "britishhorseracing.com",
    }

    first_party_scripts = []

    for script_url in resolved_scripts:
        parsed = urlsplit(
            script_url
        )

        if parsed.hostname not in BHA_HOSTS:
            continue

        if ".js" not in parsed.path.lower():
            continue

        first_party_scripts.append(
            script_url
        )


    # -----------------------------------------------------------------------
    # 11. Bound the number of JavaScript requests.
    # -----------------------------------------------------------------------
    #
    # Previous participant pages exposed about two dozen first-party scripts.
    # Forty is therefore a safe discovery ceiling without creating a site crawl.

    MAX_SCRIPT_ASSETS = 40

    scripts_to_inspect = first_party_scripts[
        :MAX_SCRIPT_ASSETS
    ]

    script_inventory_truncated = (
        len(first_party_scripts)
        > MAX_SCRIPT_ASSETS
    )


    # -----------------------------------------------------------------------
    # 12. Inspect each asset for trainer-specific frontend evidence.
    # -----------------------------------------------------------------------

    asset_observations = []
    all_route_tokens = set()

    TRAINER_TERMS = (
        "trainer",
        "trainers",
        "yard",
        "stable",
        "nonrunner",
        "non-runner",
        "championship",
        "county",
        "postcode",
        "location",
    )

    for script_url in scripts_to_inspect:
        script_response = curl_public_text(
            script_url,
            (
                "application/javascript,"
                "text/javascript,*/*;q=0.8"
            ),
        )

        script_text = script_response[
            "body"
        ]

        script_sha256 = (
            hashlib.sha256(
                script_text.encode(
                    "utf-8"
                )
            ).hexdigest()
            if script_text
            else None
        )


        # -------------------------------------------------------------------
        # 13. Recover Angular component names.
        # -------------------------------------------------------------------

        controllers = sorted(
            set(
                re.findall(
                    r"""\.controller\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        factories = sorted(
            set(
                re.findall(
                    r"""\.factory\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        services = sorted(
            set(
                re.findall(
                    r"""\.service\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )


        # -------------------------------------------------------------------
        # 14. Recover explicit BHA-v1 route tokens.
        # -------------------------------------------------------------------

        route_tokens = sorted(
            set(
                re.findall(
                    r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        all_route_tokens.update(
            route_tokens
        )


        # -------------------------------------------------------------------
        # 15. Recover trainer-related quoted strings.
        # -----------------------------------------------------------------------
        #
        # These can reveal:
        #
        #   - relative frontend routes;
        #   - API fragments built dynamically;
        #   - map/non-runner/championship resource names.

        quoted_strings = re.findall(
            r"""(['"])([^'"\r\n]{1,500})\1""",
            script_text,
        )

        trainer_strings = sorted(
            {
                value
                for _, value in quoted_strings
                if any(
                    term in value.lower()
                    for term in TRAINER_TERMS
                )
            }
        )


        # -------------------------------------------------------------------
        # 16. Capture bounded contexts around likely trainer-resource markers.
        # -------------------------------------------------------------------

        marker_contexts = []

        marker_patterns = [
            r"apiaddress",
            r"/bha/v1/",
            r"trainer",
            r"championship",
            r"non.?runner",
            r"yard",
            r"stable",
        ]

        for marker_pattern in marker_patterns:
            for match in re.finditer(
                marker_pattern,
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 900,
                )

                end = min(
                    len(script_text),
                    match.end() + 1800,
                )

                context = script_text[
                    start:end
                ]

                compact = re.sub(
                    r"\s+",
                    " ",
                    context,
                ).strip()

                if compact not in marker_contexts:
                    marker_contexts.append(
                        compact
                    )


        # -------------------------------------------------------------------
        # 17. Capture bounded `$http` contexts relevant to trainer resources.
        # -------------------------------------------------------------------

        http_contexts = []

        for match in re.finditer(
            r"\$http",
            script_text,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1000,
            )

            end = min(
                len(script_text),
                match.end() + 2200,
            )

            context = script_text[
                start:end
            ]

            lower_context = context.lower()

            if not (
                "trainer" in lower_context
                or "apiaddress" in lower_context
                or "/bha/v1/" in lower_context
            ):
                continue

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            if compact not in http_contexts:
                http_contexts.append(
                    compact
                )


        # -------------------------------------------------------------------
        # 18. Decide whether this asset is materially trainer-related.
        # -------------------------------------------------------------------

        component_text = " ".join(
            controllers
            + factories
            + services
        ).lower()

        asset_is_relevant = bool(
            trainer_strings
            or any(
                "trainer" in route.lower()
                for route in route_tokens
            )
            or any(
                term in component_text
                for term in TRAINER_TERMS
            )
        )

        asset_observations.append(
            {
                "script_url": script_url,
                "http_status": (
                    script_response["status"]
                ),
                "sha256": script_sha256,
                "controllers": controllers,
                "factories": factories,
                "services": services,
                "route_tokens": route_tokens,
                "trainer_strings": trainer_strings,
                "marker_contexts": (
                    marker_contexts[:18]
                ),
                "http_contexts": (
                    http_contexts[:18]
                ),
                "relevant": asset_is_relevant,
            }
        )


    # -----------------------------------------------------------------------
    # 19. Persist only the derived frontend inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Trainer Data"
        ),
        "page_url": TRAINER_SEARCH_RESULTS_URL,
        "first_party_scripts": (
            first_party_scripts
        ),
        "scripts_inspected": len(
            scripts_to_inspect
        ),
        "script_inventory_truncated": (
            script_inventory_truncated
        ),
        "all_bha_v1_route_tokens": sorted(
            all_route_tokens
        ),
        "asset_observations": (
            asset_observations
        ),
        "authorization_sent": False,
        "trainer_records_requested": 0,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 20. Report the first-party script inventory.
    # -----------------------------------------------------------------------

    print(
        "\nFIRST-PARTY TRAINER-PAGE SCRIPTS"
    )
    print(
        "================================"
    )

    print(
        "First-party scripts discovered:",
        len(first_party_scripts),
    )

    print(
        "Scripts inspected:",
        len(scripts_to_inspect),
    )

    print(
        "Inspection truncated:",
        (
            "YES"
            if script_inventory_truncated
            else "NO"
        ),
    )

    for script_url in scripts_to_inspect:
        print(
            " ",
            script_url,
        )


    # -----------------------------------------------------------------------
    # 21. Report only materially trainer-related frontend assets.
    # -----------------------------------------------------------------------

    relevant_assets = [
        item
        for item in asset_observations
        if item["relevant"]
    ]

    print(
        "\nRELEVANT TRAINER FRONTEND ASSETS"
    )
    print(
        "================================"
    )

    print(
        "Relevant assets:",
        len(relevant_assets),
    )

    if not relevant_assets:
        print(
            "NONE RECOVERED"
        )

    for index, item in enumerate(
        relevant_assets,
        start=1,
    ):
        print(
            f"\nAsset {index}"
        )
        print(
            "-" * 70
        )

        print(
            "URL:",
            item["script_url"],
        )

        print(
            "HTTP:",
            item["http_status"],
        )

        print(
            "SHA-256:",
            item["sha256"],
        )

        print(
            "Controllers:",
            item["controllers"] or "NONE",
        )

        print(
            "Factories:",
            item["factories"] or "NONE",
        )

        print(
            "Services:",
            item["services"] or "NONE",
        )

        print(
            "Route tokens:",
            item["route_tokens"] or "NONE",
        )

        print(
            "Trainer-related strings:",
            item["trainer_strings"] or "NONE",
        )

        for context_index, context in enumerate(
            item["marker_contexts"],
            start=1,
        ):
            print(
                f"\n  Marker context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )

        for context_index, context in enumerate(
            item["http_contexts"],
            start=1,
        ):
            print(
                f"\n  HTTP context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )


    # -----------------------------------------------------------------------
    # 22. Present the deduplicated BHA-v1 route inventory separately.
    # -----------------------------------------------------------------------

    print(
        "\nDEDUPLICATED TRAINER-PAGE BHA V1 ROUTES"
    )
    print(
        "======================================="
    )

    if all_route_tokens:
        for route in sorted(
            all_route_tokens
        ):
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 23. State the evidence/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived frontend inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Trainer records requested: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA TRAINER DATA — FRONTEND ROUTE DISCOVERY
Loaded page from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/participants/trainers/trainer-search-results/
Page SHA-256: 4b6e90d5460f960414d7efc328885ea7066cd5eaf0aac794bbac84bd6db0634c

FIRST-PARTY TRAINER-PAGE SCRIPTS
First-party scripts discovered: 23
Scripts inspected: 23
Inspection truncated: NO
  https://www.britishhorseracing.com/cdn-cgi/scripts/5c5dd728/cloudflare-static/email-decode.min.js
  https://www.britishhorseracing.com/wp-content/plugins/magic-liquidizer-responsive-table/idjs/ml.responsive.table.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/flickity.pkgd.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/functions.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/iframeResizer.min.js
  https://www.britishhorseracing.com/wp-content/themes/bha/assets/js/js.cookie.js
  https://www.britishhorseracing.com/wp-content/t

In [38]:
# BHA Trainer Data — bounded three-resource profile probe
#
# WHAT
# ----
# Reproduce the three structured requests made by the public BHA trainer profile
# for one already-identified trainer:
#
#   Kevin Philippart de Foy
#   BHA trainer ID = 1101978
#
# Frontend discovery demonstrated that a normal trainer-profile load calls:
#
#   1. /bha/v1/trainers/{trainerId}
#   2. /bha/v1/trainers/{trainerId}/performances
#   3. /bha/v1/trainers/{trainerId}/nonrunners
#
# This cell requests:
#
#   - one trainer-detail record;
#   - performances page 1 only;
#   - non-runner statistics page 1 only.
#
# WHY
# ---
# We want to establish what the BHA trainer profile actually contributes beyond
# trainer names already carried on race-result records.
#
# Of particular interest are:
#
#   - trainer identity;
#   - yard/location/contact information;
#   - career/profile aggregates;
#   - race-linked performance history;
#   - non-runner statistics and their period/grain.
#
# The frontend explicitly labels the third resource as the trainer's
# "Non-runner Rate", but we must inspect the returned fields before deciding
# what that measure actually means.
#
# READS
# -----
# - BHA Authorization from repo-root `.env.local`;
# - at most three structured BHA requests.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       trainer_profile_resource_probe/
#
# Files:
#
#   kevin_philippart_de_foy_details.json
#   kevin_philippart_de_foy_performances_page_1.json
#   kevin_philippart_de_foy_nonrunners_page_1.json
#
# Every response is cached, including HTTP errors and empty responses.
#
# EXPECTED RESULT
# ---------------
# For each resource:
#
#   - HTTP status;
#   - top-level response shape;
#   - pagination metadata where present;
#   - field inventory;
#   - bounded representative records.
#
# ACQUISITION BOUNDARY
# --------------------
# - one trainer only;
# - performances page 1 only;
# - non-runners page 1 only;
# - no championship request yet;
# - no global trainer non-runner leaderboard request yet;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "trainer_profile_resource_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization value.
# ---------------------------------------------------------------------------
#
# The credential is used only in request headers.
#
# It must never be printed or written to the research cache.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the exact trainer and frontend-demonstrated profile resources.
# ---------------------------------------------------------------------------

TRAINER_ID = 1101978
TRAINER_NAME = "Kevin Philippart de Foy"

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

PROFILE_REFERER = (
    "https://www.britishhorseracing.com/"
    f"racing/participants/trainers/trainer/#!/{TRAINER_ID}"
)

RESOURCE_SPECS = [
    {
        "name": "details",
        "url": (
            f"{BHA_BASE}/trainers/{TRAINER_ID}"
        ),
        "cache_file": (
            CACHE_DIR
            / "kevin_philippart_de_foy_details.json"
        ),
    },
    {
        "name": "performances",
        "url": (
            f"{BHA_BASE}/trainers/{TRAINER_ID}/performances?"
            f"{urlencode({'page': 1})}"
        ),
        "cache_file": (
            CACHE_DIR
            / "kevin_philippart_de_foy_performances_page_1.json"
        ),
    },
    {
        "name": "nonrunners",
        "url": (
            f"{BHA_BASE}/trainers/{TRAINER_ID}/nonrunners?"
            f"{urlencode({'page': 1})}"
        ),
        "cache_file": (
            CACHE_DIR
            / "kevin_philippart_de_foy_nonrunners_page_1.json"
        ),
    },
]


# ---------------------------------------------------------------------------
# 4. Request/cache helper.
# ---------------------------------------------------------------------------
#
# Every external request is cached.
#
# HTTP errors are preserved as evidence rather than allowed to terminate the
# notebook before their response bodies can be inspected.

def load_or_request(spec):
    cache_file = spec[
        "cache_file"
    ]

    if cache_file.exists():
        envelope = json.loads(
            cache_file.read_text(
                encoding="utf-8"
            )
        )

        assert (
            envelope["request_url"]
            == spec["url"]
        )

        return envelope, "cache"


    request = Request(
        spec["url"],
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": PROFILE_REFERER,
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Parse JSON conservatively.
    # -----------------------------------------------------------------------
    #
    # An empty response, JSON error response and valid empty dataset are
    # analytically different things.

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # Persist complete response evidence WITHOUT the Authorization value.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Trainer Data"
        ),
        "resource_name": spec["name"],
        "trainer_id": TRAINER_ID,
        "request_url": spec["url"],
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    temp_file = cache_file.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        cache_file
    )

    return envelope, "network"


# ---------------------------------------------------------------------------
# 5. Load the three trainer-profile resources.
# ---------------------------------------------------------------------------

results = {}
network_requests = 0

for spec in RESOURCE_SPECS:
    envelope, source = load_or_request(
        spec
    )

    results[
        spec["name"]
    ] = {
        "envelope": envelope,
        "source": source,
        "cache_file": spec["cache_file"],
    }

    if source == "network":
        network_requests += 1


# ---------------------------------------------------------------------------
# 6. Helper: describe one dictionary record without assigning semantics.
# ---------------------------------------------------------------------------

def describe_record(record):
    for key in sorted(
        record.keys()
    ):
        value = record[
            key
        ]

        if value is None:
            type_name = "null"
        else:
            type_name = type(
                value
            ).__name__

        print(
            f" - {key}: {type_name}"
        )

        if isinstance(
            value,
            dict,
        ):
            print(
                "     keys:",
                sorted(
                    value.keys()
                ),
            )

        elif isinstance(
            value,
            list,
        ):
            print(
                "     items:",
                len(value),
            )

            sample_dicts = [
                item
                for item in value[:3]
                if isinstance(
                    item,
                    dict,
                )
            ]

            if sample_dicts:
                nested_fields = sorted(
                    {
                        nested_key
                        for item in sample_dicts
                        for nested_key in item.keys()
                    }
                )

                print(
                    "     sample item fields:",
                    nested_fields,
                )


# ---------------------------------------------------------------------------
# 7. Helper: inspect a paginated BHA response.
# ---------------------------------------------------------------------------

def report_paginated_resource(
    label,
    payload,
    *,
    sample_limit=3,
):
    print(
        f"\n{label}"
    )

    print(
        "=" * len(label)
    )

    if not isinstance(
        payload,
        dict,
    ):
        print(
            "Unexpected top-level type:",
            type(
                payload
            ).__name__,
        )

        print(
            json.dumps(
                payload,
                indent=2,
                ensure_ascii=False,
            )[:10_000]
        )

        return


    print(
        "Top-level keys:",
        sorted(
            payload.keys()
        ),
    )

    scalar_metadata = {
        key: value
        for key, value in payload.items()
        if not isinstance(
            value,
            (dict, list),
        )
    }

    print(
        "Scalar metadata:",
        scalar_metadata,
    )

    data = payload.get(
        "data"
    )

    print(
        "data type:",
        type(
            data
        ).__name__,
    )

    if not isinstance(
        data,
        list,
    ):
        print(
            "\nUnexpected data value:"
        )

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False,
            )[:10_000]
        )

        return


    print(
        "Rows returned:",
        len(
            data
        ),
    )

    dictionary_rows = [
        row
        for row in data
        if isinstance(
            row,
            dict,
        )
    ]

    if not dictionary_rows:
        if data:
            print(
                "\nFirst non-dictionary value:"
            )

            print(
                repr(
                    data[0]
                )[:5000]
            )

        return


    field_union = sorted(
        {
            key
            for row in dictionary_rows
            for key in row.keys()
        }
    )

    print(
        "\nUnion of observed row fields:"
    )

    for field in field_union:
        print(
            " -",
            field,
        )


    print(
        f"\nRepresentative rows "
        f"(maximum {sample_limit}):"
    )

    for index, row in enumerate(
        dictionary_rows[
            :sample_limit
        ],
        start=1,
    ):
        print(
            f"\nRow {index}"
        )

        print(
            "-" * 50
        )

        print(
            json.dumps(
                row,
                indent=2,
                ensure_ascii=False,
            )[:15_000]
        )


# ---------------------------------------------------------------------------
# 8. Report transport evidence before interpreting resource contents.
# ---------------------------------------------------------------------------

print(
    "BHA TRAINER DATA — THREE-RESOURCE PROFILE PROBE"
)
print(
    "==============================================="
)

for spec in RESOURCE_SPECS:
    result = results[
        spec["name"]
    ]

    envelope = result[
        "envelope"
    ]

    print(
        f"\n{spec['name']}"
    )

    print(
        "  Loaded from:",
        result["source"],
    )

    print(
        "  HTTP status:",
        envelope["response_status"],
    )

    print(
        "  Content-Type:",
        envelope["content_type"],
    )

    if envelope["transport_error"]:
        print(
            "  Transport observation:",
            envelope["transport_error"],
        )


# ---------------------------------------------------------------------------
# 9. Inspect the trainer-details response.
# ---------------------------------------------------------------------------
#
# The frontend allows either:
#
#   response.data[0]
#
# or:
#
#   response.data
#
# so handle both rather than assuming one shape.

details_payload = results[
    "details"
]["envelope"]["parsed_json"]

print(
    "\nTRAINER DETAILS"
)
print(
    "==============="
)

if details_payload is None:
    print(
        "No parsed JSON."
    )

    print(
        results[
            "details"
        ]["envelope"]["response_text"][:5000]
    )

elif isinstance(
    details_payload,
    dict,
):
    print(
        "Top-level keys:",
        sorted(
            details_payload.keys()
        ),
    )

    details_data = details_payload.get(
        "data"
    )

    if (
        isinstance(
            details_data,
            list,
        )
        and details_data
        and isinstance(
            details_data[0],
            dict,
        )
    ):
        trainer_record = details_data[0]

        print(
            "Details data shape: list"
        )

        print(
            "Details rows:",
            len(
                details_data
            ),
        )

    elif isinstance(
        details_data,
        dict,
    ):
        trainer_record = details_data

        print(
            "Details data shape: dict"
        )

    else:
        trainer_record = None

        print(
            "Unexpected details data type:",
            type(
                details_data
            ).__name__,
        )


    if trainer_record is not None:
        print(
            "\nTrainer-detail field inventory:"
        )

        describe_record(
            trainer_record
        )


        # -------------------------------------------------------------------
        # Show one bounded complete representative record.
        # -------------------------------------------------------------------

        print(
            "\nRepresentative trainer-detail record:"
        )

        print(
            json.dumps(
                trainer_record,
                indent=2,
                ensure_ascii=False,
            )[:30_000]
        )


        # -------------------------------------------------------------------
        # Explicitly test the cross-source trainer identity.
        # -------------------------------------------------------------------
        #
        # This trainer ID has already appeared in:
        #
        #   - BHA horse profile details;
        #   - horse training history;
        #   - horse performance rows;
        #   - jockey recent-performance rows.

        id_matches = []

        for field, value in trainer_record.items():
            if str(value) == str(
                TRAINER_ID
            ):
                id_matches.append(
                    {
                        "field": field,
                        "value": value,
                    }
                )

        print(
            "\nCROSS-SOURCE ID CHECK"
        )
        print(
            "====================="
        )

        print(
            "Known BHA trainerId:",
            TRAINER_ID,
        )

        if id_matches:
            print(
                "Exact matching profile value found: YES"
            )

            for match in id_matches:
                print(
                    f"  field={match['field']} "
                    f"value={match['value']}"
                )

        else:
            print(
                "Exact matching profile value found: NO"
            )

else:
    print(
        "Unexpected trainer-details top-level type:",
        type(
            details_payload
        ).__name__,
    )


# ---------------------------------------------------------------------------
# 10. Inspect page 1 of trainer performances.
# ---------------------------------------------------------------------------
#
# The frontend supports infinite scrolling, so this resource is explicitly
# paginated. Do not request subsequent pages during source discovery.

performances_payload = results[
    "performances"
]["envelope"]["parsed_json"]

if performances_payload is None:
    print(
        "\nTRAINER PERFORMANCES — PAGE 1"
    )
    print(
        "============================="
    )

    print(
        "No parsed JSON."
    )

    print(
        results[
            "performances"
        ]["envelope"]["response_text"][:5000]
    )

else:
    report_paginated_resource(
        "TRAINER PERFORMANCES — PAGE 1",
        performances_payload,
        sample_limit=3,
    )


# ---------------------------------------------------------------------------
# 11. Inspect page 1 of trainer-specific non-runner statistics.
# ---------------------------------------------------------------------------
#
# The frontend calls this resource from the trainer profile and labels the
# resulting view "Trainer Non-runner Rate".
#
# We must inspect the actual schema before deciding whether rows represent:
#
#   - quarters;
#   - rolling periods;
#   - individual declarations;
#   - code-specific summaries;
#   - something else.
#
# Do not interpret the metric from the frontend variable name alone.

nonrunners_payload = results[
    "nonrunners"
]["envelope"]["parsed_json"]

if nonrunners_payload is None:
    print(
        "\nTRAINER NON-RUNNERS — PAGE 1"
    )
    print(
        "============================"
    )

    print(
        "No parsed JSON."
    )

    print(
        results[
            "nonrunners"
        ]["envelope"]["response_text"][:5000]
    )

else:
    report_paginated_resource(
        "TRAINER NON-RUNNERS — PAGE 1",
        nonrunners_payload,
        sample_limit=5,
    )


# ---------------------------------------------------------------------------
# 12. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Trainer:",
    TRAINER_NAME,
)

print(
    "BHA trainer ID:",
    TRAINER_ID,
)

print(
    "Network requests made by this cell:",
    network_requests,
)

print(
    "Maximum possible network requests:",
    3,
)

for spec in RESOURCE_SPECS:
    print(
        f"{spec['name']} cache:",
        spec["cache_file"],
    )

print(
    "Performance pages requested: 1 only"
)

print(
    "Non-runner pages requested: 1 only"
)

print(
    "Championship endpoint requested: NO"
)

print(
    "Global trainer non-runner leaderboard requested: NO"
)

print(
    "Other trainers requested: NO"
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA TRAINER DATA — THREE-RESOURCE PROFILE PROBE

details
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

performances
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

nonrunners
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

TRAINER DETAILS
Top-level keys: ['count', 'data', 'success']
Details data shape: list
Details rows: 1

Trainer-detail field inventory:
 - championships: list
     items: 2
     sample item fields: ['championshipType', 'groupListedWins', 'leadingEarnerHorse', 'raceType', 'rank', 'strikeRate', 'totalPrizeMoney', 'totalPrizes', 'totalRides', 'totalRuns', 'totalWins']
 - county: str
 - email: null
 - fax: null
 - fullName: str
 - horsesInTraining: int
 - id: int
 - licenceType: str
 - name: str
 - numberOfDaysSinceLastWin: int
 - phone: null
 - totalCareerRunners: int
 - totalCareerWins: int
 - trainerImage: null
 - trainerType: str
 - trainingSinceYear: int
 - website: null

Representa

## BHA Trainer Data — source-family conclusion

The public BHA trainer surface is backed by several distinct structured
resources.

Frontend discovery demonstrated:

- trainer search:

  `/bha/v1/trainers`

- individual trainer details:

  `/bha/v1/trainers/{trainerId}`

- paginated trainer performances:

  `/bha/v1/trainers/{trainerId}/performances`

- trainer-specific non-runner statistics:

  `/bha/v1/trainers/{trainerId}/nonrunners`

- trainer championships:

  `/bha/v1/championships/trainers`

- a separate Flat/Jump trainer non-runner listing:

  `/bha/v1/trainers/nonrunners`

A bounded individual-profile probe used:

- trainer: Kevin Philippart de Foy;
- BHA trainer ID: `1101978`.

That identifier had already been observed in:

- horse profile details;
- horse training history;
- horse career performances;
- jockey recent performances.

The trainer profile returned:

`id = 1101978`

providing another direct cross-source identity link within the BHA resource
network.

---

### Trainer identity and licence information

The observed trainer-detail record exposed:

- `id`;
- `name`;
- `fullName`;
- `licenceType`;
- `trainerType`;
- `trainingSinceYear`;
- `county`.

For the observed trainer:

- name = Kevin Philippart de Foy;
- full name = Kevin Jean Daniel Philippart de Foy;
- licence type = `Combined`;
- trainer type = `Licensed`;
- training since year = 2020;
- county = Suffolk.

This is substantially richer participant information than a trainer name alone.

---

### Current trainer-state information

The profile also exposed:

- `numberOfDaysSinceLastWin`;
- `horsesInTraining`.

For the observed trainer:

- days since last win = 11;
- horses in training = 81.

These are time-sensitive current-state fields.

If ever governed or analysed, they would require observation-date provenance
rather than being treated as timeless trainer attributes.

---

### Contact/profile fields

The trainer schema included:

- `phone`;
- `fax`;
- `email`;
- `website`;
- `trainerImage`.

All were null in the observed profile.

Their presence demonstrates that the schema supports this information, but one
null example does not establish population coverage across trainers.

---

## Career aggregates

The trainer profile exposed:

- `totalCareerWins`;
- `totalCareerRunners`.

For Kevin Philippart de Foy:

- `totalCareerWins = 251`;
- `totalCareerRunners = 1821`.

The exact BHA population and scope represented by these career totals have not
yet been independently established.

Importantly, the trainer-performance endpoint separately reported:

`total = 1802`

for its paginated performance history.

Therefore:

> `totalCareerRunners` and the number of records exposed by
> `/performances` must NOT currently be treated as equivalent measures.

The observed difference is 19 records.

Possible causes include differing populations, definitions, eligibility rules,
historical coverage or update timing, but none has yet been demonstrated.

The discrepancy should remain unresolved rather than being "fixed" by
assumption.

---

## Championship information inside the trainer profile

The trainer-detail record contained a `championships` list.

Observed championship fields included:

- `raceType`;
- `championshipType`;
- `rank`;
- `totalWins`;
- `totalRides`;
- `strikeRate`;
- `totalRuns`;
- `totalPrizes`;
- `totalPrizeMoney`;
- `leadingEarnerHorse`;
- `groupListedWins`.

For the observed trainer there were separate:

- British Flat;
- British Jump

championship records.

The Flat record reported:

- rank = 36;
- wins = 23;
- runs = 172;
- strike rate = 13;
- prizes = 82;
- prize money = `546898.70`.

The Jump record reported:

- rank = 301;
- wins = 1;
- runs = 2;
- strike rate = 50;
- prizes = 0;
- prize money = `7206.15`.

`totalRides` was null in both observed records.

The exact semantics of `totalPrizes`, and the distinction between fields such
as `totalRides` and `totalRuns`, remain unresolved.

The separately demonstrated `/championships/trainers` resource has not been
bulk or historically investigated during this phase.

---

# Trainer performance history

The trainer-performance resource is paginated.

For the observed trainer it reported:

- 1,802 total records;
- 10 records per page;
- 181 pages.

Only page 1 was requested.

Observed performance fields included:

### BHA race identity

- `raceId`;
- `yearOfRace`;
- `divisionSequence`.

These provide the same race-reference components already observed in the
race-, horse- and jockey-centric resources.

### Course/race information

- `courseId`;
- `courseName`;
- `raceDateTime`;
- `raceDate`;
- `raceTime`;
- `raceClass`;
- `raceType`;
- `raceDistanceText`;
- `raceName`;
- `raceRunners`;
- `going`.

### Horse identity

- `animalId`;
- `animalName`.

### Jockey identity

- `jockeyId`;
- `jockeyName`.

### Trainer identity

- `trainerId`;
- `trainerName`.

### Result information

- `resultPosition`;
- `bettingRatio`;
- `weightCarried`;
- `prizeMoney`.

### Rating fields

- `flatRating`;
- `awtRating`;
- `chaseRating`;
- `hurdleRating`.

As with the horse-performance investigation, these fields must not yet be
assumed to represent the rating applicable on the historical race date.

---

# Major new finding — race-linked BHA performance figures

The trainer-performance resource exposed additional fields not observed in the
horse `/performances` resource:

- `performanceFigure`;
- `performanceFigureCreatedBy`;
- `HRO`.

One observed race-performance record contained:

- `performanceFigure = 100`;
- `performanceFigureCreatedBy = "Graeme D. Smith"`;
- `HRO = 0`.

That same row also contained:

- `yearOfRace = 2026`;
- `raceId = 5326`;
- `divisionSequence = 0`;
- `animalId = 3397104`;
- race date = `2026-08-08`;
- course = Newmarket;
- horse = Nuit d'Eclair (IRE).

This establishes an important source capability:

> at least some BHA individual performance figures are available in structured
> form together with BHA horse identity and BHA race identity.

This resolves an earlier source-capability question raised during the horse
profile investigation.

The horse-performance resource did not expose an explicit performance figure,
while the Performance Figures CSV exposed recent figures without explicit race
identifiers.

The trainer-performance resource demonstrates that a race-linked structured
representation does exist.

### Important limitations

The fields were not populated on every observed performance.

For example, two other sampled trainer-performance rows did not contain an
observed `performanceFigure` value.

Therefore:

> performance-figure population rules remain unresolved.

Do not interpret an absent figure as:

- no official figure was produced;
- figure = zero;
- horse did not achieve a measurable performance;
- record is outside handicapping scope.

None of those interpretations has yet been established.

The meaning of `HRO` is also unresolved.

Do not expand the abbreviation or infer its semantics from the numeric value.

The provenance field:

`performanceFigureCreatedBy`

is potentially especially valuable because it may preserve source-level
authorship of the assessment.

Its exact administrative meaning should be established before governance.

---

# Trainer non-runner statistics

The trainer-specific non-runner resource returned one structured record for the
observed trainer.

Fields were:

- `trainerId`;
- `trainerName`;
- `raceType`;
- `rate`;
- `declarations`;
- `nonrunners`;
- `dateFrom`;
- `dateTo`.

Observed values were:

- race type = `flat`;
- rate = `5.54`;
- declarations = 325;
- non-runners = 18;
- date from = `2025-07-01`;
- date to = `2026-06-30`.

This is a particularly important source because it explicitly supplies both the
numerator/denominator-style counts and the reporting-period boundaries.

The measure is therefore not merely an undated "trainer non-runner rate".

It is an administrative statistic tied to:

- a trainer;
- a racing code;
- a defined date interval;
- declarations;
- non-runners.

The observed period spans:

`1 July 2025` through `30 June 2026`.

Do not currently generalise this one observed interval into a permanent annual,
quarterly or rolling-window rule.

Further periods would need to be observed before that temporal rule was
established.

---

## Separate global trainer non-runner resource

The frontend also demonstrated:

`/bha/v1/trainers/nonrunners`

with separate:

- `type=flat`;
- `type=jump`

requests.

The frontend appears to obtain the resulting lists directly rather than
requesting an explicitly bounded page size.

Because the current study is avoiding unnecessary population downloads, that
global listing has not been requested during this phase.

Its existence is established; its complete schema/population remains
unobserved.

---

## Source assessment

**Potential value: very high.**

The BHA trainer source potentially contributes:

- stable BHA trainer identity;
- full trainer name;
- licence type;
- trainer type;
- training-since year;
- county/location;
- current horses-in-training count;
- current days-since-last-win measure;
- career aggregates;
- championship aggregates;
- race-linked trainer performance history;
- horse and jockey identities within trainer performances;
- race-linked BHA performance figures;
- performance-figure creator provenance;
- official trainer non-runner statistics based on declarations;
- explicit non-runner reporting periods.

Several of these are administrative or derived BHA measures that should not be
assumed to be reproducible exactly from ordinary race-result records.

---

## Particularly important findings for later Database v4 comparison

The following deserve explicit comparison against Database v4:

- BHA trainer IDs;
- trainer licence/type;
- training-since year;
- horses in training;
- career totals;
- championship aggregates;
- race-linked `performanceFigure`;
- `performanceFigureCreatedBy`;
- trainer declaration/non-runner statistics;
- non-runner reporting-period boundaries.

Whether any of these are genuinely BHA-only has **not yet been tested against
Database v4**.

---

## Decision

The Trainer Data source family is sufficiently mapped for the site-wide
inventory.

Do not paginate through the trainer's 1,802 performance records.

Do not fetch the unbounded global trainer non-runner lists during this phase.

Preserve as unresolved:

- the 1,821 career-runner versus 1,802 performance-record discrepancy;
- performance-figure population rules;
- `HRO` semantics;
- championship-field semantics where unclear;
- the general temporal rule governing non-runner reporting periods.

The most important new capability discovered here is:

> structured BHA performance figures can be linked directly to specific horses
> and specific BHA races.

Move to the next public BHA source family:

**Owner data / owner championships.**

In [39]:
# BHA Owner Data — search/frontend route discovery
#
# WHAT
# ----
# Inspect the public BHA owner-search results page and the first-party
# JavaScript assets it loads.
#
# The purpose is to discover the structured resources used by the public
# owner interface BEFORE requesting any owner records.
#
# Specifically this cell will:
#
#   1. fetch the public owner-search results HTML page;
#   2. extract first-party BHA JavaScript assets referenced by that page;
#   3. inspect those assets for owner-related controllers/factories/services;
#   4. recover `/bha/v1/...` route strings and `apiaddress` URL construction;
#   5. print bounded HTTP-call contexts around relevant owner code.
#
# WHY
# ---
# We have now mapped:
#
#   - official ratings;
#   - horse profiles;
#   - jockey profiles/championships;
#   - trainer profiles/performances/non-runner statistics.
#
# Owner data is the next distinct participant source family.
#
# We want to establish whether the BHA exposes structured information such as:
#
#   - stable owner identity;
#   - owner name/type;
#   - championship rank;
#   - wins/runs;
#   - prize money;
#   - leading earner;
#   - race-linked performance history;
#   - other owner-specific profile information.
#
# None of those capabilities are assumed in this cell.
#
# As with the previous participant investigations, the public frontend is used
# to discover the actual resource surface rather than guessing API routes.
#
# READS
# -----
# - one public BHA HTML page:
#
#     /racing/participants/owners/owner-search-results/
#
# - first-party BHA JavaScript assets referenced by that page.
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       owner_frontend_discovery/
#
# Specifically:
#
#   owner_search_results_page.json
#   owner_script_inventory.json
#
# Raw individual JavaScript assets are not persisted separately.
#
# EXPECTED RESULT
# ---------------
# A bounded inventory showing:
#
#   - owner search-page HTTP status;
#   - first-party JS assets;
#   - owner-related Angular controllers/factories/services;
#   - concrete `/bha/v1/...` route clues;
#   - relevant `$http` request contexts;
#   - enough evidence to choose the smallest owner-resource probe next.
#
# DECISION BOUNDARY
# -----------------
# Do NOT request any owner records in this cell.
#
# First map the public owner frontend surface.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on Jupyter's current working directory.

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "owner_frontend_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "owner_search_results_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "owner_script_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the public owner-search results page.
# ---------------------------------------------------------------------------
#
# This follows the same public participant-search pattern already demonstrated
# for jockeys and trainers.
#
# If the path is unavailable, that is evidence about the frontend structure,
# not evidence that owner data itself does not exist.

OWNER_SEARCH_RESULTS_URL = (
    "https://www.britishhorseracing.com/"
    "racing/participants/owners/owner-search-results/"
)


# ---------------------------------------------------------------------------
# 3. Public text-request helper.
# ---------------------------------------------------------------------------
#
# Do not use `curl --fail`.
#
# Non-200 responses and redirects are part of the source-surface evidence and
# should remain inspectable.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url, accept):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            f"Accept: {accept}",
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the public owner-search results page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == OWNER_SEARCH_RESULTS_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        OWNER_SEARCH_RESULTS_URL,
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # 5. Defensively redact any unexpected literal Bearer token.
    # -----------------------------------------------------------------------
    #
    # None is expected in ordinary public HTML.
    #
    # Still protect the research cache from accidentally persisting an
    # operational credential if the site's implementation changes.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # 6. Persist the page response as source evidence.
    # -----------------------------------------------------------------------

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Owner Data"
        ),
        "request_url": OWNER_SEARCH_RESULTS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 7. Report page-access evidence before interpreting frontend code.
# ---------------------------------------------------------------------------

print(
    "BHA OWNER DATA — FRONTEND ROUTE DISCOVERY"
)
print(
    "========================================="
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Page SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 8. Stop cleanly if this exact search-results page is unavailable.
# ---------------------------------------------------------------------------
#
# A missing page would mean only that our assumed public page path was wrong or
# retired. It would NOT establish that BHA owner data is absent.

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nOwner search-results page unavailable."
    )

    print(
        "Do not infer that owner data is unavailable."
    )

    print(
        "The public owners landing page should be inspected next."
    )

else:
    # -----------------------------------------------------------------------
    # 9. Extract JavaScript assets referenced by the owner page.
    # -----------------------------------------------------------------------

    script_sources = re.findall(
        r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    resolved_scripts = []

    for source in script_sources:
        decoded_source = html.unescape(
            source
        )

        resolved_scripts.append(
            urljoin(
                page_envelope["final_url"]
                or OWNER_SEARCH_RESULTS_URL,
                decoded_source,
            )
        )

    resolved_scripts = sorted(
        set(
            resolved_scripts
        )
    )


    # -----------------------------------------------------------------------
    # 10. Restrict inspection to first-party BHA JavaScript assets.
    # -----------------------------------------------------------------------

    BHA_HOSTS = {
        "www.britishhorseracing.com",
        "britishhorseracing.com",
    }

    first_party_scripts = []

    for script_url in resolved_scripts:
        parsed = urlsplit(
            script_url
        )

        if parsed.hostname not in BHA_HOSTS:
            continue

        if ".js" not in parsed.path.lower():
            continue

        first_party_scripts.append(
            script_url
        )


    # -----------------------------------------------------------------------
    # 11. Bound the frontend inspection.
    # -----------------------------------------------------------------------
    #
    # Previous BHA participant pages expose roughly two dozen first-party
    # scripts. Forty remains a conservative ceiling that prevents an accidental
    # broad crawl if the site structure changes.

    MAX_SCRIPT_ASSETS = 40

    scripts_to_inspect = first_party_scripts[
        :MAX_SCRIPT_ASSETS
    ]

    script_inventory_truncated = (
        len(first_party_scripts)
        > MAX_SCRIPT_ASSETS
    )


    # -----------------------------------------------------------------------
    # 12. Inspect each asset for owner-specific frontend evidence.
    # -----------------------------------------------------------------------

    asset_observations = []
    all_route_tokens = set()

    OWNER_TERMS = (
        "owner",
        "owners",
        "ownership",
        "championship",
        "leadingearner",
        "prizemoney",
        "syndicate",
        "partnership",
    )

    for script_url in scripts_to_inspect:
        script_response = curl_public_text(
            script_url,
            (
                "application/javascript,"
                "text/javascript,*/*;q=0.8"
            ),
        )

        script_text = script_response[
            "body"
        ]

        script_sha256 = (
            hashlib.sha256(
                script_text.encode(
                    "utf-8"
                )
            ).hexdigest()
            if script_text
            else None
        )


        # -------------------------------------------------------------------
        # 13. Recover Angular component names.
        # -------------------------------------------------------------------

        controllers = sorted(
            set(
                re.findall(
                    r"""\.controller\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        factories = sorted(
            set(
                re.findall(
                    r"""\.factory\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        services = sorted(
            set(
                re.findall(
                    r"""\.service\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )


        # -------------------------------------------------------------------
        # 14. Recover explicit BHA-v1 route tokens.
        # -------------------------------------------------------------------

        route_tokens = sorted(
            set(
                re.findall(
                    r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        all_route_tokens.update(
            route_tokens
        )


        # -------------------------------------------------------------------
        # 15. Recover owner-related quoted strings.
        # -----------------------------------------------------------------------
        #
        # Dynamic route construction may leave useful resource names in quoted
        # fragments even where a complete route is not recoverable by regex.

        quoted_strings = re.findall(
            r"""(['"])([^'"\r\n]{1,500})\1""",
            script_text,
        )

        owner_strings = sorted(
            {
                value
                for _, value in quoted_strings
                if any(
                    term in value.lower()
                    for term in OWNER_TERMS
                )
            }
        )


        # -------------------------------------------------------------------
        # 16. Capture bounded contexts around likely owner-resource markers.
        # -------------------------------------------------------------------

        marker_contexts = []

        marker_patterns = [
            r"apiaddress",
            r"/bha/v1/",
            r"owner",
            r"championship",
            r"leading.?earner",
            r"prize.?money",
        ]

        for marker_pattern in marker_patterns:
            for match in re.finditer(
                marker_pattern,
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 900,
                )

                end = min(
                    len(script_text),
                    match.end() + 1800,
                )

                context = script_text[
                    start:end
                ]

                compact = re.sub(
                    r"\s+",
                    " ",
                    context,
                ).strip()

                if compact not in marker_contexts:
                    marker_contexts.append(
                        compact
                    )


        # -------------------------------------------------------------------
        # 17. Capture bounded `$http` contexts relevant to owner resources.
        # -------------------------------------------------------------------

        http_contexts = []

        for match in re.finditer(
            r"\$http",
            script_text,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1000,
            )

            end = min(
                len(script_text),
                match.end() + 2200,
            )

            context = script_text[
                start:end
            ]

            lower_context = context.lower()

            if not (
                "owner" in lower_context
                or "apiaddress" in lower_context
                or "/bha/v1/" in lower_context
            ):
                continue

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            if compact not in http_contexts:
                http_contexts.append(
                    compact
                )


        # -------------------------------------------------------------------
        # 18. Decide whether this asset is materially owner-related.
        # -------------------------------------------------------------------

        component_text = " ".join(
            controllers
            + factories
            + services
        ).lower()

        asset_is_relevant = bool(
            owner_strings
            or any(
                "owner" in route.lower()
                for route in route_tokens
            )
            or any(
                term in component_text
                for term in OWNER_TERMS
            )
        )

        asset_observations.append(
            {
                "script_url": script_url,
                "http_status": (
                    script_response["status"]
                ),
                "sha256": script_sha256,
                "controllers": controllers,
                "factories": factories,
                "services": services,
                "route_tokens": route_tokens,
                "owner_strings": owner_strings,
                "marker_contexts": (
                    marker_contexts[:18]
                ),
                "http_contexts": (
                    http_contexts[:18]
                ),
                "relevant": asset_is_relevant,
            }
        )


    # -----------------------------------------------------------------------
    # 19. Persist only the derived frontend inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Owner Data"
        ),
        "page_url": OWNER_SEARCH_RESULTS_URL,
        "first_party_scripts": (
            first_party_scripts
        ),
        "scripts_inspected": len(
            scripts_to_inspect
        ),
        "script_inventory_truncated": (
            script_inventory_truncated
        ),
        "all_bha_v1_route_tokens": sorted(
            all_route_tokens
        ),
        "asset_observations": (
            asset_observations
        ),
        "authorization_sent": False,
        "owner_records_requested": 0,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 20. Report the first-party script inventory.
    # -----------------------------------------------------------------------

    print(
        "\nFIRST-PARTY OWNER-PAGE SCRIPTS"
    )
    print(
        "=============================="
    )

    print(
        "First-party scripts discovered:",
        len(first_party_scripts),
    )

    print(
        "Scripts inspected:",
        len(scripts_to_inspect),
    )

    print(
        "Inspection truncated:",
        (
            "YES"
            if script_inventory_truncated
            else "NO"
        ),
    )

    for script_url in scripts_to_inspect:
        print(
            " ",
            script_url,
        )


    # -----------------------------------------------------------------------
    # 21. Report only materially owner-related frontend assets.
    # -----------------------------------------------------------------------

    relevant_assets = [
        item
        for item in asset_observations
        if item["relevant"]
    ]

    print(
        "\nRELEVANT OWNER FRONTEND ASSETS"
    )
    print(
        "=============================="
    )

    print(
        "Relevant assets:",
        len(relevant_assets),
    )

    if not relevant_assets:
        print(
            "NONE RECOVERED"
        )

    for index, item in enumerate(
        relevant_assets,
        start=1,
    ):
        print(
            f"\nAsset {index}"
        )
        print(
            "-" * 70
        )

        print(
            "URL:",
            item["script_url"],
        )

        print(
            "HTTP:",
            item["http_status"],
        )

        print(
            "SHA-256:",
            item["sha256"],
        )

        print(
            "Controllers:",
            item["controllers"] or "NONE",
        )

        print(
            "Factories:",
            item["factories"] or "NONE",
        )

        print(
            "Services:",
            item["services"] or "NONE",
        )

        print(
            "Route tokens:",
            item["route_tokens"] or "NONE",
        )

        print(
            "Owner-related strings:",
            item["owner_strings"] or "NONE",
        )

        for context_index, context in enumerate(
            item["marker_contexts"],
            start=1,
        ):
            print(
                f"\n  Marker context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )

        for context_index, context in enumerate(
            item["http_contexts"],
            start=1,
        ):
            print(
                f"\n  HTTP context {context_index}"
            )
            print(
                "  " + "-" * 50
            )
            print(
                "  ",
                context[:7000],
            )


    # -----------------------------------------------------------------------
    # 22. Present the deduplicated BHA-v1 route inventory separately.
    # -----------------------------------------------------------------------

    print(
        "\nDEDUPLICATED OWNER-PAGE BHA V1 ROUTES"
    )
    print(
        "====================================="
    )

    if all_route_tokens:
        for route in sorted(
            all_route_tokens
        ):
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 23. State the evidence/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived frontend inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Owner records requested: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA OWNER DATA — FRONTEND ROUTE DISCOVERY
Loaded page from: network
HTTP status: 404
Final URL: https://www.britishhorseracing.com/racing/participants/owners/owner-search-results/
Page SHA-256: 1e1692274cb73d2b18982037ef104bd81cd0c21474859a07672d8cc271328325

Owner search-results page unavailable.
Do not infer that owner data is unavailable.
The public owners landing page should be inspected next.

PROVENANCE
Page cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/owner_frontend_discovery/owner_search_results_page.json
Derived frontend inventory: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/owner_frontend_discovery/owner_script_inventory.json
BHA Authorization read: NO
BHA Authorization sent: NO
Owner records requested: 0
Database v4 queried: NO
Database writes: NONE


In [40]:
# BHA Owner Data — owners landing-page frontend discovery
#
# WHAT
# ----
# Inspect the real public BHA owners landing page:
#
#   /racing/participants/owners/
#
# The previous probe showed that:
#
#   /racing/participants/owners/owner-search-results/
#
# returns HTTP 404.
#
# This cell therefore resets the discovery path and asks a simpler question:
#
#   What owner-related functionality does the actual BHA owners page expose?
#
# Specifically it will:
#
#   1. fetch/cache the owners landing-page HTML;
#   2. extract first-party JavaScript assets;
#   3. inspect those assets for owner-related Angular controllers and HTTP calls;
#   4. recover any `/bha/v1/...` owner/championship routes;
#   5. inspect the page HTML itself for owner-related links, templates and
#      Angular expressions.
#
# WHY
# ---
# Jockeys and trainers have obvious search/profile interfaces.
#
# Owners may be organised differently.
#
# We should therefore not continue guessing:
#
#   - owner search pages;
#   - owner profile pages;
#   - owner API endpoints.
#
# The real owners landing page should establish whether the useful BHA owner
# surface consists of:
#
#   - searchable owner profiles;
#   - championship tables only;
#   - ownership guidance/content;
#   - some other structured resource.
#
# READS
# -----
# - one public BHA owners HTML page;
# - first-party BHA JavaScript assets referenced by that page.
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       owner_landing_discovery/
#
# Files:
#
#   owners_landing_page.json
#   owners_frontend_inventory.json
#
# EXPECTED RESULT
# ---------------
# Evidence showing:
#
#   - whether the owners landing page has an Angular owner application;
#   - owner-related controllers;
#   - structured owner/championship routes;
#   - public links to any owner detail/search views;
#   - enough evidence to decide whether owner data warrants an API probe.
#
# DECISION BOUNDARY
# -----------------
# Do NOT request structured owner records yet.
#
# First establish the actual public owner source surface.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "owner_landing_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "owners_landing_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "owners_frontend_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the actual public owners landing page.
# ---------------------------------------------------------------------------

OWNERS_LANDING_URL = (
    "https://www.britishhorseracing.com/"
    "racing/participants/owners/"
)


# ---------------------------------------------------------------------------
# 3. Public text-request helper.
# ---------------------------------------------------------------------------
#
# Preserve non-200 responses as evidence rather than hiding them behind
# `curl --fail`.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url, accept):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            f"Accept: {accept}",
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the owners landing page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == OWNERS_LANDING_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        OWNERS_LANDING_URL,
        (
            "text/html,application/xhtml+xml,"
            "application/xml;q=0.9,*/*;q=0.8"
        ),
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # 5. Protect cache from any unexpected literal Bearer token.
    # -----------------------------------------------------------------------

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # 6. Persist the page response.
    # -----------------------------------------------------------------------

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Owner Data"
        ),
        "request_url": OWNERS_LANDING_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 7. Report page access before interpreting the frontend.
# ---------------------------------------------------------------------------

print(
    "BHA OWNER DATA — LANDING-PAGE DISCOVERY"
)
print(
    "======================================="
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Page SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 8. Stop cleanly if even the actual owners landing page is unavailable.
# ---------------------------------------------------------------------------

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nOwners landing page unavailable."
    )

    print(
        "Owner public-source surface remains unresolved."
    )

else:
    # -----------------------------------------------------------------------
    # 9. Inspect owner-related public links directly in the HTML.
    # -----------------------------------------------------------------------
    #
    # This matters because owners may be represented through ordinary WordPress
    # navigation rather than an Angular search/profile application.

    anchor_matches = re.findall(
        r"""<a\b[^>]*\bhref\s*=\s*["']([^"']+)["'][^>]*>(.*?)</a>""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    owner_links = []

    for href_value, anchor_body in anchor_matches:
        resolved_url = urljoin(
            page_envelope["final_url"]
            or OWNERS_LANDING_URL,
            html.unescape(
                href_value
            ),
        )

        anchor_text = re.sub(
            r"<[^>]+>",
            " ",
            anchor_body,
        )

        anchor_text = html.unescape(
            re.sub(
                r"\s+",
                " ",
                anchor_text,
            ).strip()
        )

        combined = (
            resolved_url
            + " "
            + anchor_text
        ).lower()

        if any(
            term in combined
            for term in (
                "owner",
                "ownership",
                "championship",
            )
        ):
            owner_links.append(
                {
                    "url": resolved_url,
                    "text": anchor_text,
                }
            )


    # Deduplicate without losing page order.
    seen_owner_links = set()
    unique_owner_links = []

    for item in owner_links:
        key = (
            item["url"],
            item["text"],
        )

        if key in seen_owner_links:
            continue

        seen_owner_links.add(
            key
        )

        unique_owner_links.append(
            item
        )


    print(
        "\nOWNER-RELATED PUBLIC LINKS"
    )
    print(
        "=========================="
    )

    if unique_owner_links:
        for index, item in enumerate(
            unique_owner_links[:50],
            start=1,
        ):
            print(
                f"\nLink {index}"
            )
            print(
                "  text:",
                item["text"] or "[no text]",
            )
            print(
                "  URL:",
                item["url"],
            )

    else:
        print(
            "NONE RECOVERED"
        )


    # -----------------------------------------------------------------------
    # 10. Inspect page-level Angular/controller/template clues.
    # -----------------------------------------------------------------------

    PAGE_MARKERS = [
        "ng-controller",
        "owner",
        "championship",
        "apiaddress",
        "/bha/v1/",
    ]

    print(
        "\nPAGE MARKER COUNTS"
    )
    print(
        "=================="
    )

    for marker in PAGE_MARKERS:
        count = len(
            re.findall(
                re.escape(
                    marker
                ),
                page_html,
                flags=re.IGNORECASE,
            )
        )

        print(
            f"{marker}: {count}"
        )


    # -----------------------------------------------------------------------
    # 11. Extract first-party JavaScript assets.
    # -----------------------------------------------------------------------

    script_sources = re.findall(
        r"""<script\b[^>]*\bsrc\s*=\s*["']([^"']+)["']""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    resolved_scripts = sorted(
        set(
            urljoin(
                page_envelope["final_url"]
                or OWNERS_LANDING_URL,
                html.unescape(
                    source
                ),
            )
            for source in script_sources
        )
    )

    BHA_HOSTS = {
        "www.britishhorseracing.com",
        "britishhorseracing.com",
    }

    first_party_scripts = []

    for script_url in resolved_scripts:
        parsed = urlsplit(
            script_url
        )

        if parsed.hostname not in BHA_HOSTS:
            continue

        if ".js" not in parsed.path.lower():
            continue

        first_party_scripts.append(
            script_url
        )


    # -----------------------------------------------------------------------
    # 12. Bound JavaScript inspection.
    # -----------------------------------------------------------------------

    MAX_SCRIPT_ASSETS = 40

    scripts_to_inspect = first_party_scripts[
        :MAX_SCRIPT_ASSETS
    ]

    script_inventory_truncated = (
        len(first_party_scripts)
        > MAX_SCRIPT_ASSETS
    )


    # -----------------------------------------------------------------------
    # 13. Inspect scripts specifically for owner/championship logic.
    # -----------------------------------------------------------------------

    asset_observations = []
    all_route_tokens = set()

    OWNER_TERMS = (
        "owner",
        "owners",
        "ownership",
        "championship",
        "leadingearner",
        "prizemoney",
    )

    for script_url in scripts_to_inspect:
        script_response = curl_public_text(
            script_url,
            (
                "application/javascript,"
                "text/javascript,*/*;q=0.8"
            ),
        )

        script_text = script_response[
            "body"
        ]

        script_sha256 = (
            hashlib.sha256(
                script_text.encode(
                    "utf-8"
                )
            ).hexdigest()
            if script_text
            else None
        )


        # -------------------------------------------------------------------
        # Recover Angular component names.
        # -------------------------------------------------------------------

        controllers = sorted(
            set(
                re.findall(
                    r"""\.controller\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        factories = sorted(
            set(
                re.findall(
                    r"""\.factory\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        services = sorted(
            set(
                re.findall(
                    r"""\.service\(\s*['"]([^'"]+)['"]""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )


        # -------------------------------------------------------------------
        # Recover explicit BHA-v1 route fragments.
        # -------------------------------------------------------------------

        route_tokens = sorted(
            set(
                re.findall(
                    r"""/bha/v1/[A-Za-z0-9_./{}:-]+""",
                    script_text,
                    flags=re.IGNORECASE,
                )
            )
        )

        all_route_tokens.update(
            route_tokens
        )


        # -------------------------------------------------------------------
        # Recover owner-related quoted strings.
        # -------------------------------------------------------------------

        quoted_strings = re.findall(
            r"""(['"])([^'"\r\n]{1,500})\1""",
            script_text,
        )

        owner_strings = sorted(
            {
                value
                for _, value in quoted_strings
                if any(
                    term in value.lower()
                    for term in OWNER_TERMS
                )
            }
        )


        # -------------------------------------------------------------------
        # Capture bounded contexts around owner/API markers.
        # -------------------------------------------------------------------

        marker_contexts = []

        for marker_pattern in [
            r"owner",
            r"championship",
            r"apiaddress",
            r"/bha/v1/",
        ]:
            for match in re.finditer(
                marker_pattern,
                script_text,
                flags=re.IGNORECASE,
            ):
                start = max(
                    0,
                    match.start() - 1000,
                )

                end = min(
                    len(script_text),
                    match.end() + 2200,
                )

                context = re.sub(
                    r"\s+",
                    " ",
                    script_text[
                        start:end
                    ],
                ).strip()

                if context not in marker_contexts:
                    marker_contexts.append(
                        context
                    )


        # -------------------------------------------------------------------
        # Capture bounded HTTP-request contexts.
        # -------------------------------------------------------------------

        http_contexts = []

        for match in re.finditer(
            r"\$http",
            script_text,
            flags=re.IGNORECASE,
        ):
            start = max(
                0,
                match.start() - 1000,
            )

            end = min(
                len(script_text),
                match.end() + 2500,
            )

            context = script_text[
                start:end
            ]

            lower_context = context.lower()

            if not (
                "owner" in lower_context
                or "championship" in lower_context
            ):
                continue

            compact = re.sub(
                r"\s+",
                " ",
                context,
            ).strip()

            if compact not in http_contexts:
                http_contexts.append(
                    compact
                )


        # -------------------------------------------------------------------
        # Keep only materially owner-related assets.
        # -------------------------------------------------------------------

        component_text = " ".join(
            controllers
            + factories
            + services
        ).lower()

        asset_is_relevant = bool(
            owner_strings
            or any(
                (
                    "owner" in route.lower()
                    or "championship" in route.lower()
                )
                for route in route_tokens
            )
            or any(
                term in component_text
                for term in OWNER_TERMS
            )
        )

        asset_observations.append(
            {
                "script_url": script_url,
                "http_status": (
                    script_response["status"]
                ),
                "sha256": script_sha256,
                "controllers": controllers,
                "factories": factories,
                "services": services,
                "route_tokens": route_tokens,
                "owner_strings": owner_strings,
                "marker_contexts": (
                    marker_contexts[:15]
                ),
                "http_contexts": (
                    http_contexts[:15]
                ),
                "relevant": asset_is_relevant,
            }
        )


    # -----------------------------------------------------------------------
    # 14. Persist the derived frontend inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Owner Data"
        ),
        "page_url": OWNERS_LANDING_URL,
        "owner_links": (
            unique_owner_links
        ),
        "first_party_scripts": (
            first_party_scripts
        ),
        "scripts_inspected": len(
            scripts_to_inspect
        ),
        "script_inventory_truncated": (
            script_inventory_truncated
        ),
        "all_bha_v1_route_tokens": sorted(
            all_route_tokens
        ),
        "asset_observations": (
            asset_observations
        ),
        "authorization_sent": False,
        "owner_records_requested": 0,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 15. Report script inventory.
    # -----------------------------------------------------------------------

    print(
        "\nFIRST-PARTY OWNERS-PAGE SCRIPTS"
    )
    print(
        "==============================="
    )

    print(
        "First-party scripts discovered:",
        len(first_party_scripts),
    )

    print(
        "Scripts inspected:",
        len(scripts_to_inspect),
    )

    print(
        "Inspection truncated:",
        (
            "YES"
            if script_inventory_truncated
            else "NO"
        ),
    )

    for script_url in scripts_to_inspect:
        print(
            " ",
            script_url,
        )


    # -----------------------------------------------------------------------
    # 16. Report only materially owner-related assets.
    # -----------------------------------------------------------------------

    relevant_assets = [
        item
        for item in asset_observations
        if item["relevant"]
    ]

    print(
        "\nRELEVANT OWNER FRONTEND ASSETS"
    )
    print(
        "=============================="
    )

    print(
        "Relevant assets:",
        len(relevant_assets),
    )

    if not relevant_assets:
        print(
            "NONE RECOVERED"
        )

    for index, item in enumerate(
        relevant_assets,
        start=1,
    ):
        print(
            f"\nAsset {index}"
        )

        print(
            "-" * 70
        )

        print(
            "URL:",
            item["script_url"],
        )

        print(
            "HTTP:",
            item["http_status"],
        )

        print(
            "SHA-256:",
            item["sha256"],
        )

        print(
            "Controllers:",
            item["controllers"] or "NONE",
        )

        print(
            "Factories:",
            item["factories"] or "NONE",
        )

        print(
            "Services:",
            item["services"] or "NONE",
        )

        print(
            "Route tokens:",
            item["route_tokens"] or "NONE",
        )

        print(
            "Owner-related strings:",
            item["owner_strings"] or "NONE",
        )

        for context_index, context in enumerate(
            item["marker_contexts"],
            start=1,
        ):
            print(
                f"\n  Marker context {context_index}"
            )

            print(
                "  " + "-" * 50
            )

            print(
                "  ",
                context[:7000],
            )

        for context_index, context in enumerate(
            item["http_contexts"],
            start=1,
        ):
            print(
                f"\n  HTTP context {context_index}"
            )

            print(
                "  " + "-" * 50
            )

            print(
                "  ",
                context[:7000],
            )


    # -----------------------------------------------------------------------
    # 17. Print the deduplicated BHA-v1 route inventory.
    # -----------------------------------------------------------------------

    print(
        "\nDEDUPLICATED OWNERS-PAGE BHA V1 ROUTES"
    )
    print(
        "======================================"
    )

    if all_route_tokens:
        for route in sorted(
            all_route_tokens
        ):
            print(
                route
            )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 18. State the evidence boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Landing-page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived frontend inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Structured owner records requested: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA OWNER DATA — LANDING-PAGE DISCOVERY
Loaded page from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/participants/owners/
Page SHA-256: 7fea281e5814bbff869f03cebca192163f806865965875b12ce1e657745fde31

OWNER-RELATED PUBLIC LINKS

Link 1
  text: [no text]
  URL: https://www.britishhorseracing.com/racing/participants/owners/

Link 2
  text: Updates
  URL: https://www.britishhorseracing.com/racing/participants/owners/

Link 3
  text: Owners championship
  URL: https://www.britishhorseracing.com/racing/participants/owners/

Link 4
  text: [no text]
  URL: https://www.britishhorseracing.com/racing/participants/owners/news-item/?news={{item.newsDuid}}

Link 5
  text: {{item.headLine}} / {{item.newsDateTime | moment:'DD MMM YY'}}
  URL: https://www.britishhorseracing.com/racing/participants/owners/news-item/?news={{item.newsDuid}}

Link 6
  text: More
  URL: https://www.britishhorseracing.com/racing/participants/owners/news-item/?news={{item.newsDuid}}

Link

In [41]:
# BHA Owner Data — bounded Flat + Jump championship probe
#
# WHAT
# ----
# Inspect the only structured owner-data resource demonstrated by the public
# BHA owners frontend:
#
#   /bha/v1/championships/owners
#
# Request two tiny samples:
#
#   - top five Flat owners;
#   - top five Jump owners.
#
# The public `owners.js` frontend demonstrated:
#
#   type     = flat | jump
#   sort     = rank:asc
#   page     = 1
#   per_page = configurable
#
# WHY
# ---
# Owner frontend discovery found:
#
#   - no owner search resource;
#   - no individual owner profile resource;
#   - no owner performance-history resource;
#   - one structured championship resource only.
#
# The purpose of this probe is therefore simply to establish the schema and
# semantics exposed by that championship dataset.
#
# We want to know whether it contributes:
#
#   - owner identity;
#   - championship rank;
#   - wins/runs;
#   - strike rate;
#   - prize money;
#   - leading earner;
#   - championship date boundaries;
#   - other administrative fields.
#
# READS
# -----
# - BHA Authorization from repo-root `.env.local`;
# - exactly two structured requests at most.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       owner_championship_probe/
#
# Files:
#
#   flat_top5.json
#   jump_top5.json
#
# Every response is cached, including HTTP errors and empty responses.
#
# EXPECTED RESULT
# ---------------
# For each championship:
#
#   - HTTP status;
#   - pagination metadata;
#   - reported total population;
#   - complete row-field union;
#   - five representative records.
#
# ACQUISITION BOUNDARY
# --------------------
# - five Flat rows only;
# - five Jump rows only;
# - no pagination beyond page 1;
# - no owner-profile guessing;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository, secret and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

SECRETS_FILE = (
    PROJECT_ROOT
    / ".env.local"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "owner_championship_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert SECRETS_FILE.is_file(), (
    f"Local secrets file not found: {SECRETS_FILE}"
)


# ---------------------------------------------------------------------------
# 2. Load the locally stored BHA Authorization value.
# ---------------------------------------------------------------------------
#
# The credential is used only in HTTP headers.
#
# It must never be printed or persisted.

bha_authorization = None

for line in SECRETS_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if line.startswith(
        "BHA_AUTHORIZATION="
    ):
        bha_authorization = line.split(
            "=",
            1,
        )[1].strip()

        break

assert bha_authorization, (
    "BHA_AUTHORIZATION was not found in .env.local."
)

assert bha_authorization.startswith(
    "Bearer "
), (
    "Unexpected BHA Authorization format."
)


# ---------------------------------------------------------------------------
# 3. Define the two exact frontend-demonstrated championship requests.
# ---------------------------------------------------------------------------

BHA_BASE = (
    "https://api09.horseracing.software/bha/v1"
)

CHAMPIONSHIP_BASE = (
    f"{BHA_BASE}/championships/owners"
)

RESOURCE_SPECS = []

for owner_type in (
    "flat",
    "jump",
):
    params = {
        "type": owner_type,
        "sort": "rank:asc",
        "page": 1,
        "per_page": 5,
    }

    RESOURCE_SPECS.append(
        {
            "name": (
                f"{owner_type}_championship"
            ),
            "owner_type": owner_type,
            "url": (
                f"{CHAMPIONSHIP_BASE}?"
                f"{urlencode(params)}"
            ),
            "cache_file": (
                CACHE_DIR
                / f"{owner_type}_top5.json"
            ),
        }
    )


# ---------------------------------------------------------------------------
# 4. Request/cache helper.
# ---------------------------------------------------------------------------
#
# Every external request leaves a cached research envelope.
#
# A failed or empty request is still evidence and must not disappear because
# the notebook raised an exception first.

def load_or_request(spec):
    cache_file = spec[
        "cache_file"
    ]

    if cache_file.exists():
        envelope = json.loads(
            cache_file.read_text(
                encoding="utf-8"
            )
        )

        assert (
            envelope["request_url"]
            == spec["url"]
        )

        return envelope, "cache"


    request = Request(
        spec["url"],
        headers={
            "Authorization": bha_authorization,
            "Accept": "application/json",
            "Origin": (
                "https://www.britishhorseracing.com"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/participants/owners/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_text = ""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_text = response.read().decode(
                "utf-8",
                errors="replace",
            )

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Parse JSON conservatively.
    # -----------------------------------------------------------------------

    parsed_json = None
    json_error = None

    if response_text.strip():
        try:
            parsed_json = json.loads(
                response_text
            )

        except json.JSONDecodeError as error:
            json_error = repr(
                error
            )


    # -----------------------------------------------------------------------
    # Persist response evidence WITHOUT Authorization.
    # -----------------------------------------------------------------------

    envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Owner Data / Owner Championships"
        ),
        "owner_type": spec["owner_type"],
        "request_url": spec["url"],
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "response_text": response_text,
        "parsed_json": parsed_json,
        "json_error": json_error,
        "transport_error": transport_error,
        "authorization_persisted": False,
    }

    temp_file = cache_file.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        cache_file
    )

    return envelope, "network"


# ---------------------------------------------------------------------------
# 5. Load the two bounded championship samples.
# ---------------------------------------------------------------------------

results = {}
network_requests = 0

for spec in RESOURCE_SPECS:
    envelope, source = load_or_request(
        spec
    )

    results[
        spec["name"]
    ] = {
        "envelope": envelope,
        "source": source,
        "cache_file": spec["cache_file"],
    }

    if source == "network":
        network_requests += 1


# ---------------------------------------------------------------------------
# 6. Helper for inspecting one paginated championship response.
# ---------------------------------------------------------------------------
#
# This reports observed structure only.
#
# It deliberately avoids interpreting fields such as `runs`, `prizemoney` or
# `leadingEarner` until we see exactly what the source returns.

def report_championship(
    label,
    payload,
):
    print(
        f"\n{label}"
    )

    print(
        "=" * len(label)
    )

    if not isinstance(
        payload,
        dict,
    ):
        print(
            "Unexpected top-level type:",
            type(
                payload
            ).__name__,
        )

        print(
            json.dumps(
                payload,
                indent=2,
                ensure_ascii=False,
            )[:10_000]
        )

        return


    print(
        "Top-level keys:",
        sorted(
            payload.keys()
        ),
    )


    # -----------------------------------------------------------------------
    # Separate pagination/scalar metadata from the actual row list.
    # -----------------------------------------------------------------------

    scalar_metadata = {
        key: value
        for key, value in payload.items()
        if not isinstance(
            value,
            (dict, list),
        )
    }

    print(
        "Scalar metadata:",
        scalar_metadata,
    )


    data = payload.get(
        "data"
    )

    print(
        "data type:",
        type(
            data
        ).__name__,
    )

    if not isinstance(
        data,
        list,
    ):
        print(
            "\nUnexpected data value:"
        )

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False,
            )[:10_000]
        )

        return


    print(
        "Rows returned:",
        len(
            data
        ),
    )


    # -----------------------------------------------------------------------
    # Build a union of all observed row fields.
    # -----------------------------------------------------------------------

    dictionary_rows = [
        row
        for row in data
        if isinstance(
            row,
            dict,
        )
    ]

    if not dictionary_rows:
        if data:
            print(
                "First value:",
                repr(
                    data[0]
                )[:3000],
            )

        return


    field_union = sorted(
        {
            key
            for row in dictionary_rows
            for key in row.keys()
        }
    )

    print(
        "\nUnion of observed owner-championship fields:"
    )

    for field in field_union:
        print(
            " -",
            field,
        )


    # -----------------------------------------------------------------------
    # Show all five requested records.
    # -----------------------------------------------------------------------
    #
    # Five rows are intentionally small enough to inspect without turning the
    # source-discovery notebook into a population download.

    print(
        "\nRepresentative rows:"
    )

    for index, row in enumerate(
        dictionary_rows[:5],
        start=1,
    ):
        print(
            f"\nRow {index}"
        )

        print(
            "-" * 50
        )

        print(
            json.dumps(
                row,
                indent=2,
                ensure_ascii=False,
            )
        )


# ---------------------------------------------------------------------------
# 7. Report transport evidence first.
# ---------------------------------------------------------------------------

print(
    "BHA OWNER DATA — CHAMPIONSHIP PROBE"
)
print(
    "==================================="
)

for spec in RESOURCE_SPECS:
    result = results[
        spec["name"]
    ]

    envelope = result[
        "envelope"
    ]

    print(
        f"\n{spec['owner_type']}"
    )

    print(
        "  Loaded from:",
        result["source"],
    )

    print(
        "  HTTP status:",
        envelope["response_status"],
    )

    print(
        "  Content-Type:",
        envelope["content_type"],
    )

    if envelope["transport_error"]:
        print(
            "  Transport observation:",
            envelope["transport_error"],
        )


# ---------------------------------------------------------------------------
# 8. Inspect the Flat championship sample.
# ---------------------------------------------------------------------------

flat_payload = results[
    "flat_championship"
]["envelope"]["parsed_json"]

if flat_payload is None:
    print(
        "\nFLAT OWNER CHAMPIONSHIP"
    )
    print(
        "======================="
    )

    print(
        "No parsed JSON."
    )

    print(
        results[
            "flat_championship"
        ]["envelope"]["response_text"][:5000]
    )

else:
    report_championship(
        "FLAT OWNER CHAMPIONSHIP — TOP FIVE",
        flat_payload,
    )


# ---------------------------------------------------------------------------
# 9. Inspect the Jump championship sample.
# ---------------------------------------------------------------------------

jump_payload = results[
    "jump_championship"
]["envelope"]["parsed_json"]

if jump_payload is None:
    print(
        "\nJUMP OWNER CHAMPIONSHIP"
    )
    print(
        "======================="
    )

    print(
        "No parsed JSON."
    )

    print(
        results[
            "jump_championship"
        ]["envelope"]["response_text"][:5000]
    )

else:
    report_championship(
        "JUMP OWNER CHAMPIONSHIP — TOP FIVE",
        jump_payload,
    )


# ---------------------------------------------------------------------------
# 10. Compare the observed schemas without assuming they must be identical.
# ---------------------------------------------------------------------------

print(
    "\nFLAT/JUMP SCHEMA COMPARISON"
)
print(
    "==========================="
)

schema_by_type = {}

for owner_type in (
    "flat",
    "jump",
):
    payload = results[
        f"{owner_type}_championship"
    ]["envelope"]["parsed_json"]

    if (
        isinstance(
            payload,
            dict,
        )
        and isinstance(
            payload.get("data"),
            list,
        )
    ):
        rows = [
            row
            for row in payload["data"]
            if isinstance(
                row,
                dict,
            )
        ]

        schema_by_type[
            owner_type
        ] = {
            key
            for row in rows
            for key in row.keys()
        }

    else:
        schema_by_type[
            owner_type
        ] = set()


flat_fields = schema_by_type[
    "flat"
]

jump_fields = schema_by_type[
    "jump"
]

print(
    "Fields common to both:",
    sorted(
        flat_fields
        & jump_fields
    ),
)

print(
    "Flat-only fields:",
    sorted(
        flat_fields
        - jump_fields
    ),
)

print(
    "Jump-only fields:",
    sorted(
        jump_fields
        - flat_fields
    ),
)


# ---------------------------------------------------------------------------
# 11. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Network requests made by this cell:",
    network_requests,
)

print(
    "Maximum possible network requests:",
    2,
)

for spec in RESOURCE_SPECS:
    print(
        f"{spec['owner_type']} cache:",
        spec["cache_file"],
    )

print(
    "Flat rows requested:",
    5,
)

print(
    "Jump rows requested:",
    5,
)

print(
    "Additional pages requested: NO"
)

print(
    "Owner profiles requested: NO"
)

print(
    "Authorization displayed: NO"
)

print(
    "Authorization written to cache: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA OWNER DATA — CHAMPIONSHIP PROBE

flat
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

jump
  Loaded from: network
  HTTP status: 200
  Content-Type: application/json

FLAT OWNER CHAMPIONSHIP — TOP FIVE
Top-level keys: ['current_page', 'data', 'first_page_url', 'from', 'last_page', 'last_page_url', 'links', 'next_page_url', 'path', 'per_page', 'prev_page_url', 'to', 'total']
Scalar metadata: {'current_page': 1, 'first_page_url': 'https://api09.horseracing.software/bha/v1/championships/owners?page=1', 'from': 1, 'last_page': 882, 'last_page_url': 'https://api09.horseracing.software/bha/v1/championships/owners?page=882', 'next_page_url': 'https://api09.horseracing.software/bha/v1/championships/owners?page=2', 'path': 'https://api09.horseracing.software/bha/v1/championships/owners', 'per_page': 5, 'prev_page_url': None, 'to': 5, 'total': 4409}
data type: list
Rows returned: 5

Union of observed owner-championship fields:
 - championshipType
 - leadingEarnerH

## BHA Owner Data — source-family conclusion

The public BHA owner surface differs materially from the horse, jockey and
trainer participant surfaces.

Initial discovery established that the analogous route:

`/racing/participants/owners/owner-search-results/`

returns HTTP 404.

Inspection of the actual public owners landing page then demonstrated that the
structured owner-data frontend uses only:

`/bha/v1/championships/owners`

with:

- `type = flat | jump`;
- sorting;
- pagination;
- configurable page size.

No public frontend evidence was found for:

- owner search;
- individual owner profiles;
- owner performance histories;
- owner-detail API resources.

Therefore the demonstrated BHA owner data surface is currently a
**championship aggregate source**, not a participant-profile source.

---

## Flat owner championship

A bounded request for the top five Flat owners returned a paginated resource
with:

- 5 rows requested;
- `total = 4409`;
- 882 pages at five records per page.

The reported total should be interpreted only as the population returned by
this championship query.

It must NOT automatically be interpreted as:

- 4,409 currently active owners;
- 4,409 licensed/registered ownership entities;
- 4,409 unique owners across all British racing;
- the complete historical owner population.

Those meanings have not been established.

---

## Jump owner championship

A bounded request for the top five Jump owners returned:

- 5 rows requested;
- `total = 1825`;
- 365 pages at five records per page.

Again, this represents the population returned by the observed Jump
championship query only.

It should not be reinterpreted as a general owner-population count.

---

## Observed championship schema

Flat and Jump returned the same observed schema:

- `ownerId`;
- `ownerName`;
- `championshipType`;
- `rank`;
- `totalOwnerChampionshipWins`;
- `totalOwnerChampionshipsRuns`;
- `totalOwnerPrizeMoneyWon`;
- `leadingEarnerHorse`.

No Flat-only or Jump-only fields were observed.

---

## Owner identity

The resource exposes a numeric:

`ownerId`

alongside `ownerName`.

Examples included:

- Godolphin — `506405`;
- Juddmonte — `525519`;
- Wathnan Racing — `1195013`.

This demonstrates that the championship source has structured BHA owner
identities rather than owner names alone.

Other BHA race resources investigated earlier also expose owner identifiers.

A later cross-source comparison should establish whether these are the same
owner-identity domain throughout the BHA resource network.

The current owner investigation has not yet demonstrated that with an exact
matched example.

---

## Championship measures

Observed owner championship fields provide:

### Rank

`rank`

### Wins

`totalOwnerChampionshipWins`

### Runs

`totalOwnerChampionshipsRuns`

### Prize money

`totalOwnerPrizeMoneyWon`

### Leading earner

`leadingEarnerHorse`

For example, the observed leading Flat owner record was:

- owner = Godolphin;
- rank = 1;
- wins = 51;
- runs = 187;
- prize money = `2984595.88`;
- leading earner = Ombudsman (IRE).

The observed leading Jump owner record was:

- owner = Mr Martin Gowing & Mr David Jewers;
- rank = 1;
- wins = 4;
- runs = 6;
- prize money = `64866.29`;
- leading earner = Queensbury Boy (IRE).

These are source-provided championship aggregates.

Their precise population rules should not be reconstructed solely from race
results without first understanding the BHA championship definition.

---

## Important temporal limitation

Unlike the previously investigated jockey championship response, the observed
owner championship rows did **not** expose:

- `startdate`;
- `enddate`;
- season identifier;
- championship year.

The query itself also contained no explicit date parameter.

Therefore:

> the temporal boundaries governing the observed owner championship totals are
> currently unresolved.

Do NOT assume that these figures represent:

- calendar-year totals;
- season-to-date totals;
- rolling-year totals;
- the same periods used by the jockey championship;
- the same Flat/Jump periods in every year.

The public page may provide human-readable championship context elsewhere, but
the structured rows observed here do not themselves carry that temporal
provenance.

This is a material limitation if these figures are ever persisted historically.

---

## Leading-earner identity limitation

`leadingEarnerHorse` was returned as a horse name only.

No accompanying BHA horse identifier was observed in the owner championship
schema.

Therefore linking the leading earner to a governed horse identity would require
additional resolution rather than treating the name string as an identifier.

---

## Source assessment

**Potential value: moderate to high for championship context; low as a general
owner-profile source.**

Useful demonstrated information includes:

- BHA owner ID;
- owner name;
- Flat/Jump championship classification;
- championship rank;
- championship wins;
- championship runs;
- championship prize money;
- leading-earner horse name.

The source does **not** currently provide a demonstrated public interface for:

- owner profiles;
- owner history;
- ownership periods;
- horse portfolios;
- owner contact/type information;
- race-by-race owner performance history.

Some owner information already appears elsewhere in BHA race and horse
resources, so later Database v4 comparison should consider the championship
resource separately from ordinary race-level ownership.

---

## Particularly important points for later Database v4 comparison

Compare:

- BHA `ownerId` availability;
- owner-name identity quality;
- race-level owner IDs;
- official owner championship rank;
- official championship wins/runs;
- official championship prize money;
- leading-earner information.

The championship aggregates may be reproducible approximately from race-level
data, but equivalence must not be assumed because:

- the championship population definition is not yet established;
- the temporal boundaries are not exposed in the observed structured response;
- official prize-money treatment may differ from a naive reconstruction.

---

## Decision

The Owner Data source family is sufficiently mapped for the site-wide
inventory.

Do not paginate through the 4,409 Flat or 1,825 Jump championship records.

Do not guess undiscovered owner-profile endpoints.

Preserve as unresolved:

- championship temporal boundaries;
- precise population/eligibility rules;
- prize-money definition;
- whether championship `ownerId` values exactly match owner IDs in race-level
  resources across the source;
- leading-earner identity beyond the returned horse-name string.

The demonstrated source should be classified as:

**official owner championship / aggregate data**, rather than a full owner
participant database.

Move to the next BHA source family:

**Full-year fixture-list downloads.**

In [42]:
# BHA Full-Year Fixture Lists — public download-surface discovery
#
# WHAT
# ----
# Inspect the public BHA "Full Year" fixtures page and recover the downloadable
# fixture-list resources it currently exposes.
#
# Specifically this cell will:
#
#   1. fetch/cache the public Full Year page;
#   2. extract all hyperlinks from the page;
#   3. identify links that appear to be:
#
#        - full fixture-list Excel files;
#        - full fixture-list PDF files;
#        - Premier Raceday lists;
#        - other clearly fixture-related downloads;
#
#   4. preserve each link's visible text, resolved URL and apparent file type;
#   5. print a bounded inventory without downloading the files themselves.
#
# WHY
# ---
# This source family is different from the structured BHA API resources already
# investigated.
#
# The research question here is:
#
#   What official annual fixture-list products does the BHA publish, in what
#   formats, and what might they contribute beyond the live fixture API?
#
# Before opening Excel/PDF files we first need to establish the actual public
# download surface and avoid guessing filenames or historical URLs.
#
# Of particular interest later will be whether these files contain fields such
# as:
#
#   - fixture date;
#   - racecourse;
#   - code;
#   - session;
#   - fixture classification;
#   - Premier status;
#   - planned fixture identity or other programme information.
#
# None of those contents are assumed in this discovery cell.
#
# READS
# -----
# - one public BHA HTML page:
#
#     https://www.britishhorseracing.com/racing/fixtures/full-year/
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       full_year_fixture_download_discovery/
#
# Files:
#
#   full_year_page.json
#   fixture_download_inventory.json
#
# No Excel or PDF fixture files are downloaded in this cell.
#
# EXPECTED RESULT
# ---------------
# A small inventory of the actual current fixture-related download links,
# including:
#
#   - visible link text;
#   - resolved URL;
#   - host;
#   - path suffix / apparent format;
#   - whether the link appears to represent Excel, PDF or another resource.
#
# DECISION BOUNDARY
# -----------------
# Do NOT download the fixture files yet.
#
# First establish exactly what products the public page exposes.

from datetime import datetime, timezone
import hashlib
import html
import json
import re
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------
#
# Do not depend on Jupyter's current working directory.

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "full_year_fixture_download_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "full_year_page.json"
)

INVENTORY_CACHE_FILE = (
    CACHE_DIR
    / "fixture_download_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the demonstrated public BHA Full Year fixture page.
# ---------------------------------------------------------------------------

FULL_YEAR_URL = (
    "https://www.britishhorseracing.com/"
    "racing/fixtures/full-year/"
)


# ---------------------------------------------------------------------------
# 3. Public text-request helper.
# ---------------------------------------------------------------------------
#
# Do not use `curl --fail`.
#
# Redirects and non-200 responses are research evidence and should remain
# inspectable rather than turning into an unexplained exception.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_text(url):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            (
                "Accept: text/html,application/xhtml+xml,"
                "application/xml;q=0.9,*/*;q=0.8"
            ),
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the public Full Year fixture page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == FULL_YEAR_URL
    )

    page_source = "cache"

else:
    page_response = curl_public_text(
        FULL_YEAR_URL
    )

    page_html = page_response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # Protect the research cache against any unexpected literal Bearer token.
    # -----------------------------------------------------------------------
    #
    # None is expected because this is a normal public HTML page.

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            page_html,
            flags=re.IGNORECASE,
        )
    )

    safe_page_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        page_html,
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # Persist the raw public page as ignored research evidence.
    # -----------------------------------------------------------------------

    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Full-Year Fixture Lists"
        ),
        "request_url": FULL_YEAR_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": page_response["status"],
        "final_url": page_response["final_url"],
        "curl_return_code": (
            page_response["curl_return_code"]
        ),
        "stderr": page_response["stderr"],
        "content_sha256": (
            hashlib.sha256(
                page_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if page_html
            else None
        ),
        "response_html": safe_page_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 5. Report access evidence before interpreting any download links.
# ---------------------------------------------------------------------------

print(
    "BHA FULL-YEAR FIXTURE LISTS — DOWNLOAD DISCOVERY"
)
print(
    "================================================"
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope["response_status"],
)

print(
    "Final URL:",
    page_envelope["final_url"],
)

print(
    "Page SHA-256:",
    page_envelope["content_sha256"],
)


page_html = page_envelope[
    "response_html"
]


# ---------------------------------------------------------------------------
# 6. Stop cleanly if the public page itself is unavailable.
# ---------------------------------------------------------------------------

if (
    page_envelope["response_status"] != 200
    or not page_html.strip()
):
    print(
        "\nFull Year fixture page unavailable."
    )

    print(
        "Download surface remains unresolved."
    )

else:
    # -----------------------------------------------------------------------
    # 7. Extract all hyperlinks and their visible text.
    # -----------------------------------------------------------------------
    #
    # Do not assume that download files live on the BHA hostname.
    #
    # WordPress media, CDN or document-storage hosts are all possible.

    anchor_matches = re.findall(
        r"""<a\b[^>]*\bhref\s*=\s*["']([^"']+)["'][^>]*>(.*?)</a>""",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    links = []

    for href_value, anchor_body in anchor_matches:
        decoded_href = html.unescape(
            href_value
        )

        resolved_url = urljoin(
            page_envelope["final_url"]
            or FULL_YEAR_URL,
            decoded_href,
        )

        anchor_text = re.sub(
            r"<[^>]+>",
            " ",
            anchor_body,
        )

        anchor_text = html.unescape(
            re.sub(
                r"\s+",
                " ",
                anchor_text,
            ).strip()
        )

        parsed = urlsplit(
            resolved_url
        )

        suffix = Path(
            parsed.path
        ).suffix.lower()

        links.append(
            {
                "text": anchor_text,
                "url": resolved_url,
                "host": parsed.hostname,
                "path": parsed.path,
                "suffix": suffix,
            }
        )


    # -----------------------------------------------------------------------
    # 8. Identify links plausibly belonging to the annual fixture-list family.
    # -----------------------------------------------------------------------
    #
    # Include:
    #
    #   - explicit fixture-list wording;
    #   - Excel/PDF links;
    #   - Premier Raceday resources;
    #
    # but require some fixture/racing context so unrelated page PDFs do not
    # become part of the candidate inventory.

    fixture_candidates = []

    for item in links:
        combined = (
            f"{item['text']} "
            f"{item['url']}"
        ).lower()

        has_fixture_context = any(
            token in combined
            for token in (
                "fixture",
                "premier raceday",
                "premier-raceday",
                "premier_raceday",
            )
        )

        has_download_format = (
            item["suffix"]
            in {
                ".xls",
                ".xlsx",
                ".pdf",
                ".csv",
            }
        )

        explicit_excel_word = (
            "excel" in combined
        )

        explicit_pdf_word = (
            "pdf" in combined
        )

        if has_fixture_context and (
            has_download_format
            or explicit_excel_word
            or explicit_pdf_word
            or "premier" in combined
        ):
            fixture_candidates.append(
                item
            )


    # -----------------------------------------------------------------------
    # 9. Deduplicate candidate resources while retaining page order.
    # -----------------------------------------------------------------------

    seen_urls = set()
    unique_candidates = []

    for item in fixture_candidates:
        if item["url"] in seen_urls:
            continue

        seen_urls.add(
            item["url"]
        )

        unique_candidates.append(
            item
        )


    # -----------------------------------------------------------------------
    # 10. Add a conservative apparent-resource classification.
    # -----------------------------------------------------------------------
    #
    # This classification describes the hyperlink representation only.
    #
    # It does NOT claim anything about the internal contents of the file.

    classified_candidates = []

    for item in unique_candidates:
        combined = (
            f"{item['text']} "
            f"{item['url']}"
        ).lower()

        if (
            item["suffix"]
            in {
                ".xls",
                ".xlsx",
            }
            or "excel" in combined
        ):
            apparent_format = "excel"

        elif (
            item["suffix"] == ".pdf"
            or "pdf" in combined
        ):
            apparent_format = "pdf"

        else:
            apparent_format = "other"

        if "premier" in combined:
            apparent_product = (
                "premier_racedays"
            )

        elif "fixture" in combined:
            apparent_product = (
                "fixture_list"
            )

        else:
            apparent_product = (
                "other_fixture_related"
            )

        classified_candidates.append(
            {
                **item,
                "apparent_format": (
                    apparent_format
                ),
                "apparent_product": (
                    apparent_product
                ),
            }
        )


    # -----------------------------------------------------------------------
    # 11. Persist the derived download inventory.
    # -----------------------------------------------------------------------

    inventory = {
        "observed_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "source_family": (
            "Full-Year Fixture Lists"
        ),
        "page_url": FULL_YEAR_URL,
        "candidate_count": len(
            classified_candidates
        ),
        "candidates": (
            classified_candidates
        ),
        "files_downloaded": 0,
        "authorization_sent": False,
    }

    temp_file = (
        INVENTORY_CACHE_FILE.with_suffix(
            ".tmp"
        )
    )

    temp_file.write_text(
        json.dumps(
            inventory,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        INVENTORY_CACHE_FILE
    )


    # -----------------------------------------------------------------------
    # 12. Report the candidate fixture-list products.
    # -----------------------------------------------------------------------

    print(
        "\nFIXTURE-RELATED DOWNLOAD CANDIDATES"
    )
    print(
        "==================================="
    )

    print(
        "Candidates recovered:",
        len(
            classified_candidates
        ),
    )

    if not classified_candidates:
        print(
            "NONE RECOVERED"
        )

    for index, item in enumerate(
        classified_candidates,
        start=1,
    ):
        print(
            f"\nCandidate {index}"
        )

        print(
            "-" * 70
        )

        print(
            "Visible text:",
            item["text"] or "[no text]",
        )

        print(
            "URL:",
            item["url"],
        )

        print(
            "Host:",
            item["host"],
        )

        print(
            "Path suffix:",
            item["suffix"] or "[none]",
        )

        print(
            "Apparent product:",
            item["apparent_product"],
        )

        print(
            "Apparent format:",
            item["apparent_format"],
        )


    # -----------------------------------------------------------------------
    # 13. Report any years visibly associated with fixture-download candidates.
    # -----------------------------------------------------------------------
    #
    # This is useful for later historical-availability work, but the current
    # phase does NOT attempt to manufacture historical URLs.

    visible_years = sorted(
        {
            int(year)
            for item in classified_candidates
            for year in re.findall(
                r"\b(20[0-9]{2})\b",
                (
                    f"{item['text']} "
                    f"{item['url']}"
                ),
            )
        }
    )

    print(
        "\nYEARS VISIBLE IN CANDIDATE LINKS"
    )
    print(
        "================================"
    )

    print(
        visible_years
        if visible_years
        else "NONE EXPLICITLY RECOVERED"
    )


# ---------------------------------------------------------------------------
# 14. State provenance and the acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived download inventory:",
    INVENTORY_CACHE_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Fixture Excel files downloaded: 0"
)

print(
    "Fixture PDF files downloaded: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA FULL-YEAR FIXTURE LISTS — DOWNLOAD DISCOVERY
Loaded page from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/racing/fixtures/full-year/
Page SHA-256: 0d18defb01486ac37a49b60752cbf8844de0bbc34d4d792bd20391fda3e2fb3a

FIXTURE-RELATED DOWNLOAD CANDIDATES
Candidates recovered: 3

Candidate 1
----------------------------------------------------------------------
Visible text: 2026 Fixture List – Excel
URL: https://media.britishhorseracing.com/bha/Fixture_List/2026_Fixture_List.xlsx
Host: media.britishhorseracing.com
Path suffix: .xlsx
Apparent product: fixture_list
Apparent format: excel

Candidate 2
----------------------------------------------------------------------
Visible text: 2026 Fixture List – PDF
URL: https://media.britishhorseracing.com/bha/Fixture_List/2026_Fixture_List.pdf
Host: media.britishhorseracing.com
Path suffix: .pdf
Apparent product: fixture_list
Apparent format: pdf

Candidate 3
------------------------------------------------------------

In [44]:
# BHA Full-Year Fixture Lists — bounded 2026 Excel structure probe
#
# WHAT
# ----
# Download and inspect the current official BHA:
#
#   2026 Fixture List — Excel
#
# without requiring `openpyxl`.
#
# XLSX files are ZIP containers containing XML documents. This cell therefore
# uses only Python's standard library to:
#
#   1. download/cache the workbook once;
#   2. preserve download provenance and SHA-256;
#   3. inspect the XLSX workbook structure;
#   4. recover worksheet names;
#   5. decode shared strings used by worksheet cells;
#   6. read a bounded sample of rows from each worksheet;
#   7. identify a plausible header row;
#   8. preserve a compact derived inventory.
#
# WHY
# ---
# `openpyxl` is not installed in the current project environment.
#
# Installing a new package merely to inspect this source is unnecessary because
# the XLSX format can be read sufficiently for this research question using
# Python's built-in ZIP and XML support.
#
# The analytical question remains:
#
#   What information does the official annual fixture workbook actually
#   contain, and at what grain?
#
# We want to determine whether it contributes useful fixture planning or
# classification information beyond the live BHA fixture resources.
#
# READS
# -----
# - one public BHA XLSX file;
# - no BHA Authorization;
# - no Database v4.
#
# WRITES
# ------
# Ignored research cache:
#
#   data/cache/bha_official_source_feasibility/
#       full_year_fixture_excel_probe/
#
# Files:
#
#   2026_Fixture_List.xlsx
#   2026_Fixture_List_metadata.json
#
# EXPECTED RESULT
# ---------------
# A structural inventory showing:
#
#   - workbook size and SHA-256;
#   - worksheet names;
#   - worksheet dimensions where encoded;
#   - plausible header rows;
#   - the first 15 non-empty rows from each sheet.
#
# ACQUISITION BOUNDARY
# --------------------
# - one current workbook only;
# - no historical fixture-file guessing;
# - no PDF download;
# - no full annual workbook dump;
# - no Database v4 comparison;
# - no database writes.

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import xml.etree.ElementTree as ET
import zipfile


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "full_year_fixture_excel_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

WORKBOOK_FILE = (
    CACHE_DIR
    / "2026_Fixture_List.xlsx"
)

METADATA_FILE = (
    CACHE_DIR
    / "2026_Fixture_List_metadata.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact workbook discovered from the public Full Year page.
# ---------------------------------------------------------------------------

WORKBOOK_URL = (
    "https://media.britishhorseracing.com/"
    "bha/Fixture_List/2026_Fixture_List.xlsx"
)


# ---------------------------------------------------------------------------
# 3. Download/cache the workbook.
# ---------------------------------------------------------------------------
#
# Preserve the original binary file exactly as served.
#
# A failed request is still research evidence, so write metadata before
# stopping.

download_source = "cache"

if not WORKBOOK_FILE.exists():
    download_source = "network"

    request = Request(
        WORKBOOK_URL,
        headers={
            "Accept": (
                "application/vnd.openxmlformats-officedocument."
                "spreadsheetml.sheet,*/*;q=0.8"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "racing/fixtures/full-year/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_bytes = response.read()

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_bytes = error.read()

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Preserve request evidence before asserting success.
    # -----------------------------------------------------------------------

    initial_metadata = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Full-Year Fixture Lists"
        ),
        "product": (
            "2026 Fixture List — Excel"
        ),
        "request_url": WORKBOOK_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "transport_error": transport_error,
        "bytes_received": len(
            response_bytes
        ),
        "sha256": (
            hashlib.sha256(
                response_bytes
            ).hexdigest()
            if response_bytes
            else None
        ),
        "authorization_sent": False,
    }

    METADATA_FILE.write_text(
        json.dumps(
            initial_metadata,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    assert status == 200, (
        "Fixture workbook request did not return HTTP 200. "
        f"See metadata: {METADATA_FILE}"
    )

    assert response_bytes, (
        "Fixture workbook response was empty."
    )


    # -----------------------------------------------------------------------
    # XLSX is a ZIP-based format and should begin with a ZIP signature.
    #
    # This protects us from silently caching an HTML error response under an
    # `.xlsx` filename.
    # -----------------------------------------------------------------------

    assert response_bytes.startswith(
        b"PK"
    ), (
        "Response does not look like an XLSX/ZIP file."
    )

    temp_file = WORKBOOK_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_bytes(
        response_bytes
    )

    temp_file.replace(
        WORKBOOK_FILE
    )


# ---------------------------------------------------------------------------
# 4. Recalculate provenance from the local cached binary.
# ---------------------------------------------------------------------------

workbook_bytes = WORKBOOK_FILE.read_bytes()

workbook_sha256 = hashlib.sha256(
    workbook_bytes
).hexdigest()

workbook_size = len(
    workbook_bytes
)

assert zipfile.is_zipfile(
    WORKBOOK_FILE
), (
    "Cached file is not a valid ZIP/XLSX container."
)


# ---------------------------------------------------------------------------
# 5. XML namespaces used by the Office Open XML workbook format.
# ---------------------------------------------------------------------------

MAIN_NS = (
    "http://schemas.openxmlformats.org/"
    "spreadsheetml/2006/main"
)

REL_NS = (
    "http://schemas.openxmlformats.org/"
    "officeDocument/2006/relationships"
)

PACKAGE_REL_NS = (
    "http://schemas.openxmlformats.org/"
    "package/2006/relationships"
)

NS = {
    "main": MAIN_NS,
    "rel": REL_NS,
}


# ---------------------------------------------------------------------------
# 6. Helpers for translating XLSX cell references.
# ---------------------------------------------------------------------------
#
# Examples:
#
#   A1  -> column 1
#   B7  -> column 2
#   AA4 -> column 27
#
# Explicit column positions are important because spreadsheets can contain
# blank cells between populated cells.

def column_letters_to_number(
    letters,
):
    number = 0

    for character in letters:
        number = (
            number * 26
            + ord(character.upper())
            - ord("A")
            + 1
        )

    return number


def cell_reference_to_column(
    reference,
):
    match = re.match(
        r"([A-Z]+)",
        reference or "",
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return column_letters_to_number(
        match.group(1)
    )


# ---------------------------------------------------------------------------
# 7. Read the workbook XML and worksheet relationship map.
# ---------------------------------------------------------------------------

with zipfile.ZipFile(
    WORKBOOK_FILE,
    "r",
) as archive:

    archive_names = set(
        archive.namelist()
    )

    assert (
        "xl/workbook.xml"
        in archive_names
    ), (
        "XLSX does not contain xl/workbook.xml."
    )

    assert (
        "xl/_rels/workbook.xml.rels"
        in archive_names
    ), (
        "XLSX does not contain workbook relationships."
    )

    workbook_root = ET.fromstring(
        archive.read(
            "xl/workbook.xml"
        )
    )

    relationships_root = ET.fromstring(
        archive.read(
            "xl/_rels/workbook.xml.rels"
        )
    )


    # -----------------------------------------------------------------------
    # Build relationship ID -> workbook component target.
    # -----------------------------------------------------------------------

    relationship_targets = {}

    for relationship in relationships_root:
        relationship_id = relationship.attrib.get(
            "Id"
        )

        target = relationship.attrib.get(
            "Target"
        )

        if relationship_id and target:
            relationship_targets[
                relationship_id
            ] = target


    # -----------------------------------------------------------------------
    # Recover workbook worksheet names and XML targets.
    # -----------------------------------------------------------------------

    worksheets = []

    sheets_element = workbook_root.find(
        "main:sheets",
        NS,
    )

    assert sheets_element is not None, (
        "Workbook contains no <sheets> element."
    )

    for sheet in sheets_element:
        sheet_name = sheet.attrib.get(
            "name"
        )

        relationship_id = sheet.attrib.get(
            f"{{{REL_NS}}}id"
        )

        target = relationship_targets.get(
            relationship_id
        )

        assert target, (
            f"No workbook target found for sheet {sheet_name!r}."
        )


        # -------------------------------------------------------------------
        # Relationship targets may be relative to the `xl/` directory.
        # -------------------------------------------------------------------

        if target.startswith(
            "/"
        ):
            worksheet_path = target.lstrip(
                "/"
            )

        else:
            worksheet_path = (
                "xl/"
                + target.lstrip(
                    "/"
                )
            )

        worksheets.append(
            {
                "name": sheet_name,
                "relationship_id": relationship_id,
                "path": worksheet_path,
            }
        )


    # -----------------------------------------------------------------------
    # 8. Decode the workbook shared-string table if one exists.
    # -----------------------------------------------------------------------
    #
    # Most Excel text cells store an integer pointer into sharedStrings.xml
    # rather than repeating the literal string inside each worksheet.

    shared_strings = []

    if (
        "xl/sharedStrings.xml"
        in archive_names
    ):
        shared_root = ET.fromstring(
            archive.read(
                "xl/sharedStrings.xml"
            )
        )

        for string_item in shared_root.findall(
            "main:si",
            NS,
        ):
            # Rich-text cells can contain several <t> nodes. Concatenate them
            # in document order to recover the displayed string.
            text_parts = []

            for text_node in string_item.iter(
                f"{{{MAIN_NS}}}t"
            ):
                if text_node.text is not None:
                    text_parts.append(
                        text_node.text
                    )

            shared_strings.append(
                "".join(
                    text_parts
                )
            )


    # -----------------------------------------------------------------------
    # 9. Cell decoder.
    # -----------------------------------------------------------------------
    #
    # Preserve numeric values as their XML text representation rather than
    # prematurely imposing types or date semantics.
    #
    # Excel dates are often stored as serial numbers whose interpretation can
    # depend on workbook style/date-system information. We do NOT guess that
    # semantic layer in this structural probe.

    def decode_cell(
        cell,
    ):
        cell_type = cell.attrib.get(
            "t"
        )

        value_element = cell.find(
            "main:v",
            NS,
        )


        # -------------------------------------------------------------------
        # Inline strings store text directly below <is>.
        # -------------------------------------------------------------------

        if cell_type == "inlineStr":
            inline_element = cell.find(
                "main:is",
                NS,
            )

            if inline_element is None:
                return None

            text_parts = []

            for text_node in inline_element.iter(
                f"{{{MAIN_NS}}}t"
            ):
                if text_node.text is not None:
                    text_parts.append(
                        text_node.text
                    )

            return "".join(
                text_parts
            )


        if value_element is None:
            return None

        raw_value = value_element.text


        # -------------------------------------------------------------------
        # Shared-string pointer.
        # -------------------------------------------------------------------

        if cell_type == "s":
            try:
                shared_index = int(
                    raw_value
                )

                return shared_strings[
                    shared_index
                ]

            except (
                TypeError,
                ValueError,
                IndexError,
            ):
                return (
                    f"[INVALID_SHARED_STRING:{raw_value}]"
                )


        # -------------------------------------------------------------------
        # Boolean cells.
        # -------------------------------------------------------------------

        if cell_type == "b":
            if raw_value == "1":
                return True

            if raw_value == "0":
                return False


        # -------------------------------------------------------------------
        # Ordinary numeric/formula result/string representations.
        #
        # Keep raw text in this phase.
        # -------------------------------------------------------------------

        return raw_value


    # -----------------------------------------------------------------------
    # 10. Helper to normalise printable text without changing data meaning.
    # -----------------------------------------------------------------------

    def normalise_display_value(
        value,
    ):
        if value is None:
            return None

        if isinstance(
            value,
            str,
        ):
            compact = " ".join(
                value.split()
            )

            if len(compact) > 180:
                return (
                    compact[:177]
                    + "..."
                )

            return compact

        return value


    # -----------------------------------------------------------------------
    # 11. Read each worksheet and retain only bounded structural evidence.
    # -----------------------------------------------------------------------

    sheet_inventory = []

    MAX_NONEMPTY_ROWS_TO_PRINT = 15
    MAX_HEADER_SCAN_ROWS = 30

    for worksheet_spec in worksheets:
        worksheet_name = worksheet_spec[
            "name"
        ]

        worksheet_path = worksheet_spec[
            "path"
        ]

        assert (
            worksheet_path
            in archive_names
        ), (
            f"Worksheet XML not found: {worksheet_path}"
        )

        worksheet_root = ET.fromstring(
            archive.read(
                worksheet_path
            )
        )


        # -------------------------------------------------------------------
        # Recover encoded worksheet dimension if present.
        #
        # Example:
        #
        #   A1:H1490
        #
        # This is workbook metadata rather than a count computed by us.
        # -------------------------------------------------------------------

        dimension_element = worksheet_root.find(
            "main:dimension",
            NS,
        )

        encoded_dimension = (
            dimension_element.attrib.get(
                "ref"
            )
            if dimension_element is not None
            else None
        )


        sheet_data = worksheet_root.find(
            "main:sheetData",
            NS,
        )

        assert sheet_data is not None, (
            f"Worksheet {worksheet_name!r} contains no sheetData."
        )


        # -------------------------------------------------------------------
        # Decode rows while retaining explicit Excel column numbers.
        # -------------------------------------------------------------------

        decoded_rows = []

        maximum_observed_column = 0

        for row_element in sheet_data.findall(
            "main:row",
            NS,
        ):
            row_number = int(
                row_element.attrib.get(
                    "r",
                    "0",
                )
            )

            values = {}

            for cell in row_element.findall(
                "main:c",
                NS,
            ):
                cell_reference = cell.attrib.get(
                    "r"
                )

                column_number = (
                    cell_reference_to_column(
                        cell_reference
                    )
                )

                if column_number is None:
                    continue

                maximum_observed_column = max(
                    maximum_observed_column,
                    column_number,
                )

                decoded_value = decode_cell(
                    cell
                )

                display_value = (
                    normalise_display_value(
                        decoded_value
                    )
                )

                if display_value not in (
                    None,
                    "",
                ):
                    values[
                        column_number
                    ] = display_value

            if values:
                decoded_rows.append(
                    {
                        "row_number": row_number,
                        "values": values,
                    }
                )


        # -------------------------------------------------------------------
        # 12. Find a plausible header among the first 30 worksheet rows.
        # -------------------------------------------------------------------
        #
        # This is deliberately heuristic.
        #
        # It helps locate the table without declaring the chosen row to be a
        # governed schema.

        header_candidates = []

        for row in decoded_rows:
            if (
                row["row_number"]
                > MAX_HEADER_SCAN_ROWS
            ):
                continue

            nonempty_values = list(
                row["values"].values()
            )

            text_values = [
                value
                for value in nonempty_values
                if isinstance(
                    value,
                    str,
                )
            ]

            if len(
                text_values
            ) < 3:
                continue

            score = (
                len(
                    set(
                        text_values
                    )
                ),
                len(
                    nonempty_values
                ),
            )

            header_candidates.append(
                {
                    "row_number": row[
                        "row_number"
                    ],
                    "values": row[
                        "values"
                    ],
                    "score": score,
                }
            )

        candidate_header = (
            max(
                header_candidates,
                key=lambda item: item[
                    "score"
                ],
            )
            if header_candidates
            else None
        )


        # -------------------------------------------------------------------
        # 13. Report worksheet evidence.
        # -------------------------------------------------------------------

        print(
            f"\nWORKSHEET: {worksheet_name}"
        )

        print(
            "=" * (
                len(
                    worksheet_name
                )
                + 11
            )
        )

        print(
            "XML path:",
            worksheet_path,
        )

        print(
            "Encoded dimension:",
            (
                encoded_dimension
                or "[not supplied]"
            ),
        )

        print(
            "Non-empty rows observed:",
            len(
                decoded_rows
            ),
        )

        print(
            "Maximum populated column observed:",
            maximum_observed_column,
        )


        if candidate_header:
            print(
                "Candidate header row:",
                candidate_header[
                    "row_number"
                ],
            )

            print(
                "Candidate header values:"
            )

            for (
                column_number,
                value,
            ) in sorted(
                candidate_header[
                    "values"
                ].items()
            ):
                print(
                    f"  C{column_number}: "
                    f"{value!r}"
                )

        else:
            print(
                "Candidate header row: "
                "NONE IDENTIFIED"
            )


        # -------------------------------------------------------------------
        # 14. Print only the first 15 non-empty rows.
        # -------------------------------------------------------------------

        print(
            "\nFirst non-empty rows:"
        )

        sample_rows = decoded_rows[
            :MAX_NONEMPTY_ROWS_TO_PRINT
        ]

        for row in sample_rows:
            print(
                f"\n  Row {row['row_number']}"
            )

            for (
                column_number,
                value,
            ) in sorted(
                row["values"].items()
            ):
                print(
                    f"    C{column_number}: "
                    f"{value!r}"
                )


        # -------------------------------------------------------------------
        # Preserve compact derived inventory for later interpretation.
        # -------------------------------------------------------------------

        sheet_inventory.append(
            {
                "sheet_name": (
                    worksheet_name
                ),
                "xml_path": (
                    worksheet_path
                ),
                "encoded_dimension": (
                    encoded_dimension
                ),
                "nonempty_rows_observed": len(
                    decoded_rows
                ),
                "maximum_populated_column_observed": (
                    maximum_observed_column
                ),
                "candidate_header_row": (
                    candidate_header[
                        "row_number"
                    ]
                    if candidate_header
                    else None
                ),
                "candidate_header_values": (
                    {
                        str(key): value
                        for key, value in candidate_header[
                            "values"
                        ].items()
                    }
                    if candidate_header
                    else None
                ),
                "sample_rows": [
                    {
                        "row_number": row[
                            "row_number"
                        ],
                        "values": {
                            str(key): value
                            for key, value in row[
                                "values"
                            ].items()
                        },
                    }
                    for row in sample_rows
                ],
            }
        )


# ---------------------------------------------------------------------------
# 15. Persist derived workbook metadata/inventory.
# ---------------------------------------------------------------------------

metadata = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Full-Year Fixture Lists"
    ),
    "product": (
        "2026 Fixture List — Excel"
    ),
    "request_url": WORKBOOK_URL,
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "local_file": str(
        WORKBOOK_FILE
    ),
    "bytes": workbook_size,
    "sha256": workbook_sha256,
    "worksheet_names": [
        worksheet[
            "name"
        ]
        for worksheet in worksheets
    ],
    "shared_string_count": len(
        shared_strings
    ),
    "sheet_inventory": (
        sheet_inventory
    ),
    "analysis_method": (
        "Python standard-library ZIP/XML inspection"
    ),
    "openpyxl_required": False,
    "authorization_sent": False,
    "pdf_downloaded": False,
    "database_v4_queried": False,
}

METADATA_FILE.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# 16. Report workbook-level provenance and acquisition boundary.
# ---------------------------------------------------------------------------

print(
    "\nBHA FULL-YEAR FIXTURE LIST — 2026 EXCEL PROBE"
)
print(
    "============================================="
)

print(
    "Loaded workbook from:",
    download_source,
)

print(
    "Workbook path:",
    WORKBOOK_FILE,
)

print(
    "Workbook bytes:",
    workbook_size,
)

print(
    "Workbook SHA-256:",
    workbook_sha256,
)

print(
    "Worksheet count:",
    len(
        worksheets
    ),
)

print(
    "Worksheet names:",
    [
        worksheet["name"]
        for worksheet in worksheets
    ],
)

print(
    "Shared strings:",
    len(
        shared_strings
    ),
)


print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Workbook cache:",
    WORKBOOK_FILE,
)

print(
    "Metadata/inventory:",
    METADATA_FILE,
)

print(
    "Network workbook requests:",
    (
        1
        if download_source == "network"
        else 0
    ),
)

print(
    "Workbook parser:",
    "Python standard library ZIP/XML"
)

print(
    "openpyxl required: NO"
)

print(
    "PDF files downloaded: 0"
)

print(
    "Historical fixture files requested: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)


WORKSHEET: List - Full Year
XML path: xl/worksheets/sheet1.xml
Encoded dimension: A1:I1459
Non-empty rows observed: 1459
Maximum populated column observed: 9
Candidate header row: 1
Candidate header values:
  C1: 'Date'
  C2: 'Weekday'
  C3: 'Course'
  C4: 'Time'
  C5: 'CourseGroup'
  C6: 'Region'
  C7: 'Code'
  C8: 'Surface'
  C9: 'Type'

First non-empty rows:

  Row 1
    C1: 'Date'
    C2: 'Weekday'
    C3: 'Course'
    C4: 'Time'
    C5: 'CourseGroup'
    C6: 'Region'
    C7: 'Code'
    C8: 'Surface'
    C9: 'Type'

  Row 2
    C1: '46023'
    C2: 'Thursday'
    C3: 'Windsor'
    C4: 'Afternoon'
    C5: 'Arena Racing Corporation Limited'
    C6: 'South'
    C7: 'Jump'
    C8: 'Turf'
    C9: 'National/BHA'

  Row 3
    C1: '46023'
    C2: 'Thursday'
    C3: 'Southwell'
    C4: 'Afternoon'
    C5: 'Arena Racing Corporation Limited'
    C6: 'Midlands'
    C7: 'Flat'
    C8: 'AWT'
    C9: 'National/BHA'

  Row 4
    C1: '46023'
    C2: 'Thursday'
    C3: 'Musselburgh'
    C4: 'Afterno

In [45]:
# BHA Full-Year Fixture Lists — category semantics + Premier subset probe
#
# WHAT
# ----
# Analyse the already-cached 2026 BHA fixture workbook locally.
#
# This cell will:
#
#   1. decode Excel date serials into calendar dates;
#   2. read the structured "List - Full Year" sheet;
#   3. inventory distinct values/counts for:
#
#        - Time;
#        - CourseGroup;
#        - Region;
#        - Code;
#        - Surface;
#        - Type;
#
#   4. verify the row count against the workbook's own "Fixture Numbers"
#      summary;
#   5. read the separate "Premier Fixture List";
#   6. test whether every Premier fixture has an exact matching
#      Date + Course record in the full-year list;
#   7. report any unmatched or duplicate Premier mappings.
#
# WHY
# ---
# The first workbook probe established that the annual fixture file contains
# significantly richer planning/classification information than date/course
# alone.
#
# Before concluding this source family we still need to establish:
#
#   - what categorical values actually occur;
#   - whether "Premier Fixture List" is genuinely a subset of the main list;
#   - whether the main 1,458-row fixture population agrees with the workbook's
#     own published total.
#
# This is deliberately NOT a Database v4 comparison yet.
#
# READS
# -----
# Cached BHA workbook only:
#
#   data/cache/bha_official_source_feasibility/
#       full_year_fixture_excel_probe/
#       2026_Fixture_List.xlsx
#
# No network requests.
# No BHA Authorization.
#
# WRITES
# ------
# One compact derived JSON file:
#
#   2026_fixture_category_inventory.json
#
# in the same ignored research-cache directory.
#
# EXPECTED RESULT
# ---------------
# Evidence for:
#
#   - actual category domains;
#   - 2026 fixture count;
#   - Premier fixture count;
#   - Premier/full-list relationship;
#   - date coverage.
#
# IMPORTANT
# ---------
# Do NOT infer the formal meaning of values such as:
#
#   National/BHA
#   Racecourse/Normal
#
# merely from their names.
#
# This cell inventories values and relationships only.

from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import re
import xml.etree.ElementTree as ET
import zipfile


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and cached-workbook locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "full_year_fixture_excel_probe"
)

WORKBOOK_FILE = (
    CACHE_DIR
    / "2026_Fixture_List.xlsx"
)

OUTPUT_FILE = (
    CACHE_DIR
    / "2026_fixture_category_inventory.json"
)

assert WORKBOOK_FILE.is_file(), (
    f"Cached workbook not found: {WORKBOOK_FILE}"
)


# ---------------------------------------------------------------------------
# 2. XLSX namespace constants and column-reference helpers.
# ---------------------------------------------------------------------------

MAIN_NS = (
    "http://schemas.openxmlformats.org/"
    "spreadsheetml/2006/main"
)

REL_NS = (
    "http://schemas.openxmlformats.org/"
    "officeDocument/2006/relationships"
)

NS = {
    "main": MAIN_NS,
    "rel": REL_NS,
}


def column_letters_to_number(letters):
    number = 0

    for character in letters:
        number = (
            number * 26
            + ord(character.upper())
            - ord("A")
            + 1
        )

    return number


def cell_reference_to_column(reference):
    match = re.match(
        r"([A-Z]+)",
        reference or "",
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return column_letters_to_number(
        match.group(1)
    )


# ---------------------------------------------------------------------------
# 3. Decode Excel's normal 1900-date-system serial.
# ---------------------------------------------------------------------------
#
# Excel's historical 1900 leap-year compatibility means serial dates are most
# conveniently converted using 1899-12-30 as the epoch.
#
# We use this only for the Date column whose workbook header explicitly says
# "Date". We are not applying date semantics to arbitrary numeric cells.

EXCEL_DATE_EPOCH = datetime(
    1899,
    12,
    30,
)


def decode_excel_date(raw_value):
    try:
        serial = float(
            raw_value
        )
    except (
        TypeError,
        ValueError,
    ):
        return None

    converted = (
        EXCEL_DATE_EPOCH
        + timedelta(
            days=serial
        )
    )

    return converted.date().isoformat()


# ---------------------------------------------------------------------------
# 4. Read workbook relationships, shared strings and target sheets.
# ---------------------------------------------------------------------------

with zipfile.ZipFile(
    WORKBOOK_FILE,
    "r",
) as archive:

    archive_names = set(
        archive.namelist()
    )

    workbook_root = ET.fromstring(
        archive.read(
            "xl/workbook.xml"
        )
    )

    relationships_root = ET.fromstring(
        archive.read(
            "xl/_rels/workbook.xml.rels"
        )
    )


    # -----------------------------------------------------------------------
    # Map relationship IDs to workbook XML targets.
    # -----------------------------------------------------------------------

    relationship_targets = {}

    for relationship in relationships_root:
        relationship_id = relationship.attrib.get(
            "Id"
        )

        target = relationship.attrib.get(
            "Target"
        )

        if relationship_id and target:
            relationship_targets[
                relationship_id
            ] = target


    # -----------------------------------------------------------------------
    # Map worksheet names to their internal XML paths.
    # -----------------------------------------------------------------------

    sheet_paths = {}

    sheets_element = workbook_root.find(
        "main:sheets",
        NS,
    )

    for sheet in sheets_element:
        sheet_name = sheet.attrib.get(
            "name"
        )

        relationship_id = sheet.attrib.get(
            f"{{{REL_NS}}}id"
        )

        target = relationship_targets[
            relationship_id
        ]

        if target.startswith(
            "/"
        ):
            path = target.lstrip(
                "/"
            )
        else:
            path = (
                "xl/"
                + target.lstrip(
                    "/"
                )
            )

        sheet_paths[
            sheet_name
        ] = path


    # -----------------------------------------------------------------------
    # Decode shared strings.
    # -----------------------------------------------------------------------

    shared_strings = []

    if (
        "xl/sharedStrings.xml"
        in archive_names
    ):
        shared_root = ET.fromstring(
            archive.read(
                "xl/sharedStrings.xml"
            )
        )

        for string_item in shared_root.findall(
            "main:si",
            NS,
        ):
            text_parts = []

            for text_node in string_item.iter(
                f"{{{MAIN_NS}}}t"
            ):
                if text_node.text is not None:
                    text_parts.append(
                        text_node.text
                    )

            shared_strings.append(
                "".join(
                    text_parts
                )
            )


    # -----------------------------------------------------------------------
    # Generic cell decoder.
    # -----------------------------------------------------------------------

    def decode_cell(cell):
        cell_type = cell.attrib.get(
            "t"
        )

        if cell_type == "inlineStr":
            inline = cell.find(
                "main:is",
                NS,
            )

            if inline is None:
                return None

            text_parts = []

            for text_node in inline.iter(
                f"{{{MAIN_NS}}}t"
            ):
                if text_node.text is not None:
                    text_parts.append(
                        text_node.text
                    )

            return "".join(
                text_parts
            )

        value_element = cell.find(
            "main:v",
            NS,
        )

        if value_element is None:
            return None

        raw_value = value_element.text

        if cell_type == "s":
            try:
                return shared_strings[
                    int(
                        raw_value
                    )
                ]
            except (
                TypeError,
                ValueError,
                IndexError,
            ):
                return None

        if cell_type == "b":
            return (
                raw_value == "1"
            )

        return raw_value


    # -----------------------------------------------------------------------
    # Generic worksheet row reader.
    # -----------------------------------------------------------------------

    def read_sheet_rows(sheet_name):
        worksheet_root = ET.fromstring(
            archive.read(
                sheet_paths[
                    sheet_name
                ]
            )
        )

        sheet_data = worksheet_root.find(
            "main:sheetData",
            NS,
        )

        rows = []

        for row_element in sheet_data.findall(
            "main:row",
            NS,
        ):
            row_number = int(
                row_element.attrib.get(
                    "r",
                    "0",
                )
            )

            values = {}

            for cell in row_element.findall(
                "main:c",
                NS,
            ):
                column_number = (
                    cell_reference_to_column(
                        cell.attrib.get(
                            "r"
                        )
                    )
                )

                if column_number is None:
                    continue

                value = decode_cell(
                    cell
                )

                if value not in (
                    None,
                    "",
                ):
                    values[
                        column_number
                    ] = value

            if values:
                rows.append(
                    {
                        "row_number": row_number,
                        "values": values,
                    }
                )

        return rows


    # -----------------------------------------------------------------------
    # 5. Read the main structured fixture list.
    # -----------------------------------------------------------------------

    full_rows_raw = read_sheet_rows(
        "List - Full Year"
    )

    assert full_rows_raw, (
        "Full-year fixture sheet is empty."
    )

    header_values = full_rows_raw[
        0
    ]["values"]

    header_by_column = {
        column_number: str(
            value
        ).strip()
        for column_number, value in header_values.items()
    }

    expected_headers = {
        1: "Date",
        2: "Weekday",
        3: "Course",
        4: "Time",
        5: "CourseGroup",
        6: "Region",
        7: "Code",
        8: "Surface",
        9: "Type",
    }

    assert header_by_column == expected_headers, (
        "Unexpected full-year fixture header schema."
    )


    # -----------------------------------------------------------------------
    # Convert each worksheet row to a field-named dictionary.
    # -----------------------------------------------------------------------

    full_fixtures = []

    for raw_row in full_rows_raw[
        1:
    ]:
        values = raw_row[
            "values"
        ]

        fixture = {
            header_by_column[column_number]: (
                values.get(
                    column_number
                )
            )
            for column_number in header_by_column
        }

        fixture[
            "DateRaw"
        ] = fixture[
            "Date"
        ]

        fixture[
            "Date"
        ] = decode_excel_date(
            fixture[
                "Date"
            ]
        )

        fixture[
            "source_row"
        ] = raw_row[
            "row_number"
        ]

        full_fixtures.append(
            fixture
        )


    # -----------------------------------------------------------------------
    # 6. Inventory categorical domains.
    # -----------------------------------------------------------------------

    CATEGORY_FIELDS = [
        "Weekday",
        "Time",
        "CourseGroup",
        "Region",
        "Code",
        "Surface",
        "Type",
    ]

    category_counts = {}

    for field in CATEGORY_FIELDS:
        counts = Counter(
            fixture[
                field
            ]
            for fixture in full_fixtures
        )

        category_counts[
            field
        ] = dict(
            sorted(
                counts.items(),
                key=lambda item: (
                    str(
                        item[0]
                    )
                ),
            )
        )


    # -----------------------------------------------------------------------
    # 7. Inspect basic population/date properties.
    # -----------------------------------------------------------------------

    dates = [
        fixture[
            "Date"
        ]
        for fixture in full_fixtures
        if fixture[
            "Date"
        ] is not None
    ]

    unique_courses = sorted(
        {
            fixture[
                "Course"
            ]
            for fixture in full_fixtures
            if fixture[
                "Course"
            ]
        }
    )

    duplicate_date_course = defaultdict(
        list
    )

    for fixture in full_fixtures:
        key = (
            fixture[
                "Date"
            ],
            fixture[
                "Course"
            ],
        )

        duplicate_date_course[
            key
        ].append(
            fixture
        )

    duplicate_date_course_groups = {
        key: rows
        for key, rows in duplicate_date_course.items()
        if len(
            rows
        ) > 1
    }


    # -----------------------------------------------------------------------
    # 8. Read the separate Premier fixture sheet.
    # -----------------------------------------------------------------------

    premier_rows_raw = read_sheet_rows(
        "Premier Fixture List"
    )

    premier_header = {
        column_number: str(
            value
        ).strip()
        for column_number, value in premier_rows_raw[
            0
        ]["values"].items()
    }

    expected_premier_headers = {
        1: "Date",
        2: "Weekday",
        3: "Course",
        4: "Time",
        5: "CourseGroup",
        6: "Region",
        7: "Code",
        8: "Surface",
    }

    assert (
        premier_header
        == expected_premier_headers
    ), (
        "Unexpected Premier fixture header schema."
    )

    premier_fixtures = []

    for raw_row in premier_rows_raw[
        1:
    ]:
        values = raw_row[
            "values"
        ]

        fixture = {
            premier_header[column_number]: (
                values.get(
                    column_number
                )
            )
            for column_number in premier_header
        }

        fixture[
            "DateRaw"
        ] = fixture[
            "Date"
        ]

        fixture[
            "Date"
        ] = decode_excel_date(
            fixture[
                "Date"
            ]
        )

        fixture[
            "source_row"
        ] = raw_row[
            "row_number"
        ]

        premier_fixtures.append(
            fixture
        )


    # -----------------------------------------------------------------------
    # 9. Test Premier fixtures against the full-year population.
    # -----------------------------------------------------------------------
    #
    # Date + Course is the narrowest shared identity available between these
    # two sheets.
    #
    # We report ambiguity rather than pretending it is impossible.

    full_by_date_course = defaultdict(
        list
    )

    for fixture in full_fixtures:
        full_by_date_course[
            (
                fixture[
                    "Date"
                ],
                fixture[
                    "Course"
                ],
            )
        ].append(
            fixture
        )

    premier_match_results = []

    for premier in premier_fixtures:
        key = (
            premier[
                "Date"
            ],
            premier[
                "Course"
            ],
        )

        matches = full_by_date_course.get(
            key,
            [],
        )

        premier_match_results.append(
            {
                "date": premier[
                    "Date"
                ],
                "course": premier[
                    "Course"
                ],
                "match_count": len(
                    matches
                ),
                "full_rows": [
                    match[
                        "source_row"
                    ]
                    for match in matches
                ],
            }
        )

    premier_exact_matches = [
        result
        for result in premier_match_results
        if result[
            "match_count"
        ] == 1
    ]

    premier_unmatched = [
        result
        for result in premier_match_results
        if result[
            "match_count"
        ] == 0
    ]

    premier_ambiguous = [
        result
        for result in premier_match_results
        if result[
            "match_count"
        ] > 1
    ]


    # -----------------------------------------------------------------------
    # 10. Read the workbook's own 2026 total from "Fixture Numbers".
    # -----------------------------------------------------------------------

    fixture_number_rows = read_sheet_rows(
        "Fixture Numbers"
    )

    workbook_total_2026 = None

    for raw_row in fixture_number_rows:
        values = raw_row[
            "values"
        ]

        if values.get(
            1
        ) == "Total":
            # Both summary blocks contain the same total. Capture the first.
            workbook_total_2026 = values.get(
                10
            )

            break

    workbook_total_2026 = (
        int(
            workbook_total_2026
        )
        if workbook_total_2026 is not None
        else None
    )


# ---------------------------------------------------------------------------
# 11. Report the fixture population.
# ---------------------------------------------------------------------------

print(
    "BHA FULL-YEAR FIXTURE LIST — CATEGORY + PREMIER PROBE"
)
print(
    "====================================================="
)

print(
    "\nFULL-YEAR POPULATION"
)
print(
    "===================="
)

print(
    "Fixture rows:",
    len(
        full_fixtures
    ),
)

print(
    "Workbook published 2026 total:",
    workbook_total_2026,
)

print(
    "Row count matches published total:",
    (
        "YES"
        if len(
            full_fixtures
        ) == workbook_total_2026
        else "NO"
    ),
)

print(
    "Earliest fixture date:",
    min(
        dates
    ),
)

print(
    "Latest fixture date:",
    max(
        dates
    ),
)

print(
    "Distinct courses:",
    len(
        unique_courses
    ),
)

print(
    "Duplicate Date+Course groups:",
    len(
        duplicate_date_course_groups
    ),
)


# ---------------------------------------------------------------------------
# 12. Report categorical domains and frequencies.
# ---------------------------------------------------------------------------

print(
    "\nCATEGORY INVENTORY"
)
print(
    "=================="
)

for field in CATEGORY_FIELDS:
    print(
        f"\n{field}"
    )
    print(
        "-" * len(
            field
        )
    )

    for value, count in category_counts[
        field
    ].items():
        print(
            f"{value!r}: {count}"
        )


# ---------------------------------------------------------------------------
# 13. Report Premier subset evidence.
# ---------------------------------------------------------------------------

print(
    "\nPREMIER FIXTURE SUBSET"
)
print(
    "======================"
)

print(
    "Premier fixture rows:",
    len(
        premier_fixtures
    ),
)

print(
    "Exact Date+Course matches:",
    len(
        premier_exact_matches
    ),
)

print(
    "Unmatched Premier rows:",
    len(
        premier_unmatched
    ),
)

print(
    "Ambiguous Premier rows:",
    len(
        premier_ambiguous
    ),
)

print(
    "Premier is exact Date+Course subset:",
    (
        "YES"
        if (
            len(
                premier_unmatched
            ) == 0
            and len(
                premier_ambiguous
            ) == 0
            and len(
                premier_exact_matches
            ) == len(
                premier_fixtures
            )
        )
        else "NO"
    ),
)


if premier_unmatched:
    print(
        "\nUnmatched Premier rows:"
    )

    for result in premier_unmatched:
        print(
            result
        )


if premier_ambiguous:
    print(
        "\nAmbiguous Premier rows:"
    )

    for result in premier_ambiguous:
        print(
            result
        )


# ---------------------------------------------------------------------------
# 14. If the main list itself contains same-course same-day duplicates,
#     print them because that affects fixture identity assumptions later.
# ---------------------------------------------------------------------------

if duplicate_date_course_groups:
    print(
        "\nFULL-LIST DUPLICATE DATE+COURSE GROUPS"
    )
    print(
        "======================================"
    )

    for key, rows in sorted(
        duplicate_date_course_groups.items()
    ):
        print(
            f"\n{key[0]} | {key[1]}"
        )

        for row in rows:
            print(
                {
                    field: row[
                        field
                    ]
                    for field in (
                        "Time",
                        "Code",
                        "Surface",
                        "Type",
                        "source_row",
                    )
                }
            )


# ---------------------------------------------------------------------------
# 15. Persist compact derived evidence.
# ---------------------------------------------------------------------------

derived_inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Full-Year Fixture Lists"
    ),
    "product": (
        "2026 Fixture List — Excel"
    ),
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "full_year_fixture_count": len(
        full_fixtures
    ),
    "workbook_published_2026_total": (
        workbook_total_2026
    ),
    "count_matches_workbook_total": (
        len(
            full_fixtures
        )
        == workbook_total_2026
    ),
    "earliest_date": min(
        dates
    ),
    "latest_date": max(
        dates
    ),
    "distinct_courses": len(
        unique_courses
    ),
    "category_counts": (
        category_counts
    ),
    "duplicate_date_course_groups": {
        f"{date}|{course}": [
            {
                field: row[
                    field
                ]
                for field in (
                    "Time",
                    "Code",
                    "Surface",
                    "Type",
                    "source_row",
                )
            }
            for row in rows
        ]
        for (
            date,
            course
        ), rows in duplicate_date_course_groups.items()
    },
    "premier_fixture_count": len(
        premier_fixtures
    ),
    "premier_exact_date_course_matches": len(
        premier_exact_matches
    ),
    "premier_unmatched": (
        premier_unmatched
    ),
    "premier_ambiguous": (
        premier_ambiguous
    ),
    "network_requests": 0,
    "database_v4_queried": False,
}

OUTPUT_FILE.write_text(
    json.dumps(
        derived_inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# 16. State provenance/acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Workbook:",
    WORKBOOK_FILE,
)

print(
    "Derived inventory:",
    OUTPUT_FILE,
)

print(
    "Network requests: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "PDF files read: 0"
)

print(
    "Historical fixture files requested: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA FULL-YEAR FIXTURE LIST — CATEGORY + PREMIER PROBE

FULL-YEAR POPULATION
Fixture rows: 1458
Workbook published 2026 total: 1458
Row count matches published total: YES
Earliest fixture date: 2026-01-01
Latest fixture date: 2026-12-31
Distinct courses: 59
Duplicate Date+Course groups: 0

CATEGORY INVENTORY

Weekday
-------
'Friday': 233
'Monday': 198
'Saturday': 291
'Sunday': 120
'Thursday': 225
'Tuesday': 190
'Wednesday': 201

Time
----
'Afternoon': 1040
'Evening': 238
'Floodlit': 180

CourseGroup
-----------
'Arena Racing Corporation Limited': 587
'Chester Race Company Limited': 55
'Independent': 494
'Jockey Club Racecourses Limited': 322

Region
------
'Midlands': 478
'North': 453
'South': 527

Code
----
'Both': 1
'Flat': 896
'Jump': 561

Surface
-------
'AWT': 345
'Turf': 1113

Type
----
'National/BHA': 87
'National/BHA Floodlit': 124
'Racecourse/Normal': 1247

PREMIER FIXTURE SUBSET
Premier fixture rows: 52
Exact Date+Course matches: 52
Unmatched Premier rows: 0
Ambiguous Premier

## BHA Full-Year Fixture Lists — source-family conclusion

The BHA publishes an official annual fixture workbook through its public
Full Year fixtures page.

For 2026 the public download surface exposed:

- `2026 Fixture List – Excel`;
- `2026 Fixture List – PDF`;
- `2026 Premier Racedays – PDF`.

The Excel workbook was inspected directly.

It contains four worksheets:

1. `List - Full Year`;
2. `Grid - Full Year`;
3. `Premier Fixture List`;
4. `Fixture Numbers`.

The Excel workbook is therefore substantially more than a presentation-only
calendar.

---

## Main annual fixture list

`List - Full Year` contains one structured row per planned fixture.

Observed columns were:

- `Date`;
- `Weekday`;
- `Course`;
- `Time`;
- `CourseGroup`;
- `Region`;
- `Code`;
- `Surface`;
- `Type`.

For 2026 the sheet contained:

- **1,458 fixture rows**;
- coverage from `2026-01-01` through `2026-12-31`;
- 59 distinct published course labels;
- no duplicate `Date + Course` combinations.

The workbook's separate `Fixture Numbers` summary also reports:

`2026 Total = 1458`

so the detailed-row population reconciles exactly to the workbook's own
published annual total.

---

## Fixture time classification

Observed `Time` values were:

- `Afternoon` — 1,040;
- `Evening` — 238;
- `Floodlit` — 180.

These should currently be treated as BHA fixture-planning categories.

Do not assume that they represent actual race times or that `Floodlit` is merely
another name for evening racing.

---

## Course-group classification

Observed `CourseGroup` values were:

- `Arena Racing Corporation Limited` — 587;
- `Chester Race Company Limited` — 55;
- `Independent` — 494;
- `Jockey Club Racecourses Limited` — 322.

This provides an official BHA grouping attached to each planned fixture.

The exact governance meaning of `CourseGroup` should be established before
treating it as a timeless racecourse attribute.

It may describe the fixture/racecourse operating group rather than the physical
course identity itself.

---

## Regional classification

Observed `Region` values were:

- `Midlands` — 478;
- `North` — 453;
- `South` — 527.

This is an explicit BHA classification and may be useful as official
administrative geography.

It should not automatically be substituted for ordinary geographic regions or
physical-location definitions without checking the BHA meaning.

---

## Racing-code classification

Observed `Code` values were:

- `Flat` — 896;
- `Jump` — 561;
- `Both` — 1.

The existence of one `Both` fixture is important.

The annual fixture source therefore does not require every planned fixture to
belong exclusively to Flat or Jump.

The precise meaning of the observed `Both` fixture should remain unresolved
until examined if it becomes analytically important.

---

## Surface classification

Observed `Surface` values were:

- `Turf` — 1,113;
- `AWT` — 345.

This gives an explicit planned-fixture surface classification.

It should be compared later with the course/track identities already governed
inside Inside Rails rather than assumed equivalent to them.

---

## Fixture `Type`

Observed `Type` values were:

- `Racecourse/Normal` — 1,247;
- `National/BHA` — 87;
- `National/BHA Floodlit` — 124.

This is potentially one of the most useful fields in the workbook because it
appears to encode a fixture-allocation or planning distinction not obvious from
ordinary race-result data.

However:

> the formal meaning of these values has NOT yet been established.

Do not infer from the labels alone that they represent ownership, funding,
licensing, governance or fixture-right categories.

Their semantics require BHA documentation or other direct evidence before
governance.

---

# Premier Racedays

The workbook contains a separate:

`Premier Fixture List`

with the fields:

- `Date`;
- `Weekday`;
- `Course`;
- `Time`;
- `CourseGroup`;
- `Region`;
- `Code`;
- `Surface`.

There were:

- **52 Premier fixture rows**.

Every one of the 52 Premier rows had exactly one matching `Date + Course`
record in the full-year fixture list:

- exact matches = 52;
- unmatched = 0;
- ambiguous = 0.

Therefore, for the observed 2026 workbook:

> the Premier Fixture List is an exact subset of the main annual fixture list
> at `Date + Course` grain.

Premier status is therefore a separate classification of fixtures already
present in the full annual population rather than a separate fixture
population.

---

## Grid worksheet

`Grid - Full Year` presents the same fixture programme in a calendar/grid form.

It also visually encodes distinctions such as:

- Flat versus Jump;
- Evening;
- Floodlit;
- Premier Raceday.

This worksheet is useful for human presentation but is not the preferred
machine-readable interface because the structured `List - Full Year` sheet
already exposes the fixture rows directly.

---

# Fixture Numbers historical summary

The workbook also contains a `Fixture Numbers` worksheet with annual counts from
2018 through 2026.

Observed categories include:

### By racing/surface category

- Flat Turf;
- Flat AW;
- Jump;
- Total.

For 2026:

- Flat Turf = 552;
- Flat AW = 345;
- Jump = 561;
- Total = 1,458.

### By broad time category

- Afternoon;
- Evening;
- Total.

For 2026:

- Afternoon = 1,040;
- Evening = 418;
- Total = 1,458.

The second summary combines the detailed `Evening` and `Floodlit` classifications:

`238 + 180 = 418`.

Therefore:

> the workbook itself operates at more than one aggregation level for fixture
> timing.

The detailed fixture-list categories should be preserved rather than replacing
them with the coarser historical summary.

---

## Important temporal interpretation

This source is an **annual fixture-list/planning product**.

The 2026 workbook includes fixtures through `2026-12-31`, including fixtures
that were still in the future when this investigation was performed.

Therefore it must NOT be interpreted as evidence that all listed fixtures:

- actually took place;
- ran exactly as programmed;
- were not abandoned;
- retained their original course/code/session classification;
- produced races or results.

For historical race-population completeness, actual run/result resources remain
the relevant evidence.

The annual workbook instead provides official evidence of the **planned fixture
programme and its classifications**.

---

## Course-count warning

The workbook contains 59 distinct published 2026 course labels.

This number should NOT be interpreted as:

- the number of British racecourses;
- the number of physical venues;
- the number of governed Inside Rails course identities;
- the number of racing tracks/subcourses.

It is simply the number of distinct `Course` values appearing in this one
annual fixture programme.

Course identity remains governed by the separate racecourse/course research.

---

## Source assessment

**Potential value: high for fixture planning, classification and validation.**

Useful demonstrated information includes:

- official annual fixture population;
- planned fixture date;
- course label;
- weekday;
- fixture time classification;
- racecourse operating/group classification;
- BHA region;
- Flat/Jump/Both classification;
- Turf/AWT classification;
- fixture `Type`;
- Premier Raceday status;
- annual fixture-count summaries back to 2018.

Several of these concepts are not ordinary race-result facts.

---

## Potential later roles inside Inside Rails

The annual fixture source may be useful for:

### Planning/programme provenance

What fixtures were officially scheduled.

### Fixture classification

Including:

- time category;
- region;
- course group;
- code;
- surface;
- fixture type;
- Premier status.

### Validation

Compare planned fixtures with:

- BHA live fixture resources;
- actual result-bearing fixtures;
- Database v4 race populations.

### Research

Potential questions include:

- abandonment/addition rates;
- planned versus actual programme;
- distribution of fixtures by operator/region/code/surface;
- Premier Raceday scheduling;
- changes in fixture composition over time.

---

## Important unresolved semantics

Preserve as unresolved:

- formal definition of `Type`;
- formal definition/governance of `CourseGroup`;
- formal BHA regional classification rules;
- meaning and circumstances of `Code = Both`;
- whether annual workbooks are revised after publication;
- whether past workbook versions can be recovered reliably;
- whether historical annual fixture files use a stable schema.

Those questions only need investigation if the corresponding fields become
valuable enough to govern.

---

## Decision

The Full-Year Fixture Lists source family is sufficiently mapped for the
site-wide inventory.

Do not inspect the equivalent PDF.

Do not search historical workbook years yet.

The Excel workbook should be classified as:

**high-value official fixture-planning/classification data**, with the live
fixture/result resources remaining authoritative for observed race execution.

Move to the next BHA public-source family:

**Racing Statistics / Racing Data Packs.**

In [47]:
# BHA Racing Statistics — public report/download surface discovery
#
# WHAT
# ----
# Inspect the public BHA Racing Statistics page and build a structured inventory
# of the report/data-product families it exposes.
#
# This cell will identify, without downloading any reports:
#
#   1. Full-year Racing Data Packs;
#   2. Monthly Racing Data Packs;
#   3. Horse Population Reports;
#   4. Race Off-Times datasets;
#   5. any other clearly statistics-related downloadable resources that appear
#      in the same public page.
#
# WHY
# ---
# We already know this BHA area contains several distinct data products.
#
# The aim now is to preserve the actual public source surface in the notebook
# before selecting individual PDFs for deeper inspection.
#
# These products are potentially useful for very different purposes:
#
#   - aggregate validation;
#   - official definitions;
#   - administrative horse-population measures;
#   - race punctuality / delay analysis;
#   - macro industry context.
#
# They must therefore not be collapsed into one generic "statistics" source.
#
# READS
# -----
# One public BHA page:
#
#   https://www.britishhorseracing.com/
#       regulation/reports-and-statistics/racing-statistics/
#
# No BHA Authorization credential is read or sent.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       racing_statistics_surface_discovery/
#
# Files:
#
#   racing_statistics_page.json
#   racing_statistics_inventory.json
#
# EXPECTED RESULT
# ---------------
# A structured inventory showing:
#
#   - product family;
#   - visible link text;
#   - resolved URL;
#   - surrounding section heading;
#   - any visible year/date;
#   - apparent file format where recoverable;
#   - counts and date/year ranges by family.
#
# ACQUISITION BOUNDARY
# --------------------
# - page discovery only;
# - no PDF downloads;
# - no report text extraction;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
from html.parser import HTMLParser
import hashlib
import html
import json
from pathlib import Path
import re
import subprocess
from urllib.parse import urljoin, urlsplit


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored research-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "racing_statistics_surface_discovery"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAGE_CACHE_FILE = (
    CACHE_DIR
    / "racing_statistics_page.json"
)

INVENTORY_FILE = (
    CACHE_DIR
    / "racing_statistics_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact public BHA Racing Statistics page.
# ---------------------------------------------------------------------------

RACING_STATISTICS_URL = (
    "https://www.britishhorseracing.com/"
    "regulation/reports-and-statistics/racing-statistics/"
)


# ---------------------------------------------------------------------------
# 3. Public page request helper.
# ---------------------------------------------------------------------------
#
# Preserve HTTP status and redirects as evidence.
#
# Do not use `curl --fail`, because a non-200 response should remain visible
# rather than disappearing into a generic subprocess exception.

STATUS_MARKER = "__BHA_HTTP_STATUS__"
URL_MARKER = "__BHA_FINAL_URL__"


def curl_public_html(url):
    result = subprocess.run(
        [
            "curl",
            "-L",
            "--silent",
            "--show-error",
            "--compressed",
            "--max-time",
            "30",
            "--user-agent",
            (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/140.0 Safari/537.36"
            ),
            "--header",
            (
                "Accept: text/html,application/xhtml+xml,"
                "application/xml;q=0.9,*/*;q=0.8"
            ),
            "--header",
            "Accept-Language: en-GB,en;q=0.9",
            "--write-out",
            (
                f"\n{STATUS_MARKER}%{{http_code}}"
                f"\n{URL_MARKER}%{{url_effective}}"
            ),
            url,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    output = result.stdout

    status_position = output.rfind(
        f"\n{STATUS_MARKER}"
    )

    final_url_position = output.rfind(
        f"\n{URL_MARKER}"
    )

    if (
        status_position == -1
        or final_url_position == -1
        or final_url_position < status_position
    ):
        return {
            "status": None,
            "final_url": None,
            "body": "",
            "curl_return_code": result.returncode,
            "stderr": result.stderr.strip(),
        }

    body = output[
        :status_position
    ]

    status_text = output[
        status_position + len(f"\n{STATUS_MARKER}") :
        final_url_position
    ].strip()

    final_url = output[
        final_url_position + len(f"\n{URL_MARKER}") :
    ].strip()

    try:
        status = int(
            status_text
        )
    except ValueError:
        status = None

    return {
        "status": status,
        "final_url": final_url,
        "body": body,
        "curl_return_code": result.returncode,
        "stderr": result.stderr.strip(),
    }


# ---------------------------------------------------------------------------
# 4. Fetch/cache the Racing Statistics page.
# ---------------------------------------------------------------------------

if PAGE_CACHE_FILE.exists():
    page_envelope = json.loads(
        PAGE_CACHE_FILE.read_text(
            encoding="utf-8"
        )
    )

    assert (
        page_envelope["request_url"]
        == RACING_STATISTICS_URL
    )

    page_source = "cache"

else:
    response = curl_public_html(
        RACING_STATISTICS_URL
    )

    raw_html = response[
        "body"
    ]


    # -----------------------------------------------------------------------
    # Defensively prevent an unexpected operational Bearer value being cached.
    #
    # None is expected on a normal public WordPress page.
    # -----------------------------------------------------------------------

    literal_bearer_present = bool(
        re.search(
            r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
            raw_html,
            flags=re.IGNORECASE,
        )
    )

    safe_html = re.sub(
        r"""Bearer\s+[A-Za-z0-9._~+/=-]{10,}""",
        "Bearer [REDACTED]",
        raw_html,
        flags=re.IGNORECASE,
    )


    page_envelope = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Racing Statistics"
        ),
        "request_url": RACING_STATISTICS_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": response[
            "status"
        ],
        "final_url": response[
            "final_url"
        ],
        "curl_return_code": response[
            "curl_return_code"
        ],
        "stderr": response[
            "stderr"
        ],
        "content_sha256": (
            hashlib.sha256(
                raw_html.encode(
                    "utf-8"
                )
            ).hexdigest()
            if raw_html
            else None
        ),
        "response_html": safe_html,
        "literal_bearer_was_present": (
            literal_bearer_present
        ),
        "authorization_sent": False,
    }

    temp_file = PAGE_CACHE_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_text(
        json.dumps(
            page_envelope,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    temp_file.replace(
        PAGE_CACHE_FILE
    )

    page_source = "network"


# ---------------------------------------------------------------------------
# 5. Report page-access evidence first.
# ---------------------------------------------------------------------------

print(
    "BHA RACING STATISTICS — PUBLIC SOURCE DISCOVERY"
)
print(
    "=============================================="
)

print(
    "Loaded page from:",
    page_source,
)

print(
    "HTTP status:",
    page_envelope[
        "response_status"
    ],
)

print(
    "Final URL:",
    page_envelope[
        "final_url"
    ],
)

print(
    "Page SHA-256:",
    page_envelope[
        "content_sha256"
    ],
)


page_html = page_envelope[
    "response_html"
]

assert (
    page_envelope["response_status"]
    == 200
), (
    "Racing Statistics page did not return HTTP 200."
)

assert page_html.strip(), (
    "Racing Statistics page was empty."
)


# ---------------------------------------------------------------------------
# 6. Parse headings and hyperlinks while retaining page-section context.
# ---------------------------------------------------------------------------
#
# A normal regex-only hyperlink extraction would lose the relationship between
# a resource and the heading under which BHA publishes it.
#
# We therefore use Python's standard-library HTMLParser and retain the latest
# H1/H2/H3/H4 heading for every anchor.

class RacingStatisticsParser(
    HTMLParser
):
    def __init__(self):
        super().__init__(
            convert_charrefs=True
        )

        self.current_heading_tag = None
        self.current_heading_parts = []
        self.latest_heading = None

        self.current_anchor_href = None
        self.current_anchor_parts = []

        self.links = []


    def handle_starttag(
        self,
        tag,
        attrs,
    ):
        tag = tag.lower()

        attributes = dict(
            attrs
        )

        if tag in {
            "h1",
            "h2",
            "h3",
            "h4",
        }:
            self.current_heading_tag = tag
            self.current_heading_parts = []

        elif tag == "a":
            self.current_anchor_href = (
                attributes.get(
                    "href"
                )
            )

            self.current_anchor_parts = []


    def handle_data(
        self,
        data,
    ):
        if self.current_heading_tag:
            self.current_heading_parts.append(
                data
            )

        if self.current_anchor_href is not None:
            self.current_anchor_parts.append(
                data
            )


    def handle_endtag(
        self,
        tag,
    ):
        tag = tag.lower()

        if (
            self.current_heading_tag
            and tag
            == self.current_heading_tag
        ):
            heading_text = " ".join(
                "".join(
                    self.current_heading_parts
                ).split()
            )

            if heading_text:
                self.latest_heading = (
                    heading_text
                )

            self.current_heading_tag = None
            self.current_heading_parts = []


        elif (
            tag == "a"
            and self.current_anchor_href
            is not None
        ):
            link_text = " ".join(
                "".join(
                    self.current_anchor_parts
                ).split()
            )

            self.links.append(
                {
                    "href": (
                        self.current_anchor_href
                    ),
                    "text": link_text,
                    "section_heading": (
                        self.latest_heading
                    ),
                }
            )

            self.current_anchor_href = None
            self.current_anchor_parts = []


parser = RacingStatisticsParser()

parser.feed(
    page_html
)


# ---------------------------------------------------------------------------
# 7. Resolve URLs and classify statistics products conservatively.
# ---------------------------------------------------------------------------

resolved_records = []

for link in parser.links:
    href = html.unescape(
        link["href"]
        or ""
    ).strip()

    if not href:
        continue

    resolved_url = urljoin(
        page_envelope[
            "final_url"
        ]
        or RACING_STATISTICS_URL,
        href,
    )

    parsed_url = urlsplit(
        resolved_url
    )

    text = (
        link["text"]
        or ""
    )

    section = (
        link["section_heading"]
        or ""
    )

    combined = (
        f"{section} "
        f"{text} "
        f"{resolved_url}"
    ).lower()


    # -----------------------------------------------------------------------
    # Assign product family from explicit page terminology.
    #
    # Do not infer report content beyond what the public page labels.
    # -----------------------------------------------------------------------

    if (
        "full year"
        in combined
        and "data pack"
        in combined
    ):
        product_family = (
            "racing_data_pack_full_year"
        )

    elif (
        "data pack"
        in combined
        and any(
            month in combined
            for month in (
                "january",
                "february",
                "march",
                "april",
                "may",
                "june",
                "july",
                "august",
                "september",
                "october",
                "november",
                "december",
            )
        )
    ):
        product_family = (
            "racing_data_pack_monthly"
        )

    elif (
        "horse population"
        in combined
    ):
        product_family = (
            "horse_population_report"
        )

    elif (
        "off-time"
        in combined
        or "off time"
        in combined
        or "off-times"
        in combined
        or "off times"
        in combined
    ):
        product_family = (
            "race_off_times"
        )

    else:
        product_family = None


    if product_family is None:
        continue


    # -----------------------------------------------------------------------
    # Recover visible years and dates where present.
    # -----------------------------------------------------------------------

    visible_years = [
        int(
            value
        )
        for value in re.findall(
            r"\b(20[0-9]{2})\b",
            (
                f"{section} "
                f"{text} "
                f"{resolved_url}"
            ),
        )
    ]

    date_matches = re.findall(
        (
            r"\b("
            r"\d{1,2}\s+"
            r"(?:January|February|March|April|May|June|July|August|"
            r"September|October|November|December)"
            r"\s+20\d{2}"
            r")\b"
        ),
        (
            f"{section} "
            f"{text}"
        ),
        flags=re.IGNORECASE,
    )


    # -----------------------------------------------------------------------
    # Apparent file type is based only on URL suffix.
    #
    # A page/link without a suffix remains "unknown" until actually fetched.
    # -----------------------------------------------------------------------

    suffix = Path(
        parsed_url.path
    ).suffix.lower()

    if suffix == ".pdf":
        apparent_format = "pdf"

    elif suffix in {
        ".xls",
        ".xlsx",
    }:
        apparent_format = "excel"

    elif suffix == ".csv":
        apparent_format = "csv"

    else:
        apparent_format = "unknown"


    resolved_records.append(
        {
            "product_family": (
                product_family
            ),
            "section_heading": (
                section
            ),
            "link_text": (
                text
            ),
            "url": (
                resolved_url
            ),
            "host": (
                parsed_url.hostname
            ),
            "suffix": (
                suffix
            ),
            "apparent_format": (
                apparent_format
            ),
            "visible_years": (
                visible_years
            ),
            "visible_dates": (
                date_matches
            ),
        }
    )


# ---------------------------------------------------------------------------
# 8. Deduplicate exact resources.
# ---------------------------------------------------------------------------

unique_records = []

seen = set()

for record in resolved_records:
    key = (
        record[
            "product_family"
        ],
        record[
            "url"
        ],
        record[
            "link_text"
        ],
    )

    if key in seen:
        continue

    seen.add(
        key
    )

    unique_records.append(
        record
    )


# ---------------------------------------------------------------------------
# 9. Group the public source inventory by product family.
# ---------------------------------------------------------------------------

FAMILY_ORDER = [
    "racing_data_pack_full_year",
    "racing_data_pack_monthly",
    "horse_population_report",
    "race_off_times",
]

family_records = {
    family: [
        record
        for record in unique_records
        if record[
            "product_family"
        ]
        == family
    ]
    for family in FAMILY_ORDER
}


# ---------------------------------------------------------------------------
# 10. Report each source family at a useful but bounded level.
# ---------------------------------------------------------------------------

DISPLAY_NAMES = {
    "racing_data_pack_full_year": (
        "FULL-YEAR RACING DATA PACKS"
    ),
    "racing_data_pack_monthly": (
        "MONTHLY RACING DATA PACKS"
    ),
    "horse_population_report": (
        "HORSE POPULATION REPORTS"
    ),
    "race_off_times": (
        "RACE OFF-TIMES"
    ),
}


for family in FAMILY_ORDER:
    records = family_records[
        family
    ]

    print(
        f"\n{DISPLAY_NAMES[family]}"
    )

    print(
        "=" * len(
            DISPLAY_NAMES[
                family
            ]
        )
    )

    print(
        "Recovered resources:",
        len(
            records
        ),
    )


    years = sorted(
        {
            year
            for record in records
            for year in record[
                "visible_years"
            ]
        }
    )

    if years:
        print(
            "Visible year range:",
            (
                f"{min(years)} "
                f"to {max(years)}"
            ),
        )

        print(
            "Visible years:",
            years,
        )

    else:
        print(
            "Visible years: NONE"
        )


    # -----------------------------------------------------------------------
    # Full-year packs are few enough to show completely.
    # -----------------------------------------------------------------------

    if family == (
        "racing_data_pack_full_year"
    ):
        records_to_print = records


    # -----------------------------------------------------------------------
    # For larger families show the latest bounded sample according to page
    # order rather than dumping dozens of links into the notebook.
    # -----------------------------------------------------------------------

    else:
        records_to_print = records[
            :12
        ]


    for index, record in enumerate(
        records_to_print,
        start=1,
    ):
        print(
            f"\n  Resource {index}"
        )

        print(
            "    Section:",
            (
                record[
                    "section_heading"
                ]
                or "[none]"
            ),
        )

        print(
            "    Text:",
            (
                record[
                    "link_text"
                ]
                or "[no visible text]"
            ),
        )

        print(
            "    URL:",
            record[
                "url"
            ],
        )

        print(
            "    Apparent format:",
            record[
                "apparent_format"
            ],
        )

        if record[
            "visible_dates"
        ]:
            print(
                "    Visible dates:",
                record[
                    "visible_dates"
                ],
            )


    if (
        len(records)
        > len(records_to_print)
    ):
        print(
            "\n  Additional resources not printed:",
            (
                len(records)
                - len(records_to_print)
            ),
        )


# ---------------------------------------------------------------------------
# 11. Produce a compact high-level source-family summary.
# ---------------------------------------------------------------------------

print(
    "\nSOURCE-FAMILY SUMMARY"
)
print(
    "====================="
)

for family in FAMILY_ORDER:
    records = family_records[
        family
    ]

    years = sorted(
        {
            year
            for record in records
            for year in record[
                "visible_years"
            ]
        }
    )

    formats = sorted(
        {
            record[
                "apparent_format"
            ]
            for record in records
        }
    )

    print(
        f"{family}: "
        f"resources={len(records)}, "
        f"years={years or 'none'}, "
        f"formats={formats or 'none'}"
    )


# ---------------------------------------------------------------------------
# 12. Persist the derived source inventory.
# ---------------------------------------------------------------------------

inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Racing Statistics"
    ),
    "page_url": (
        RACING_STATISTICS_URL
    ),
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "resources": (
        unique_records
    ),
    "family_counts": {
        family: len(
            family_records[
                family
            ]
        )
        for family in FAMILY_ORDER
    },
    "authorization_sent": False,
    "report_files_downloaded": 0,
    "database_v4_queried": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 13. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Page cache:",
    PAGE_CACHE_FILE,
)

print(
    "Derived inventory:",
    INVENTORY_FILE,
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "PDF files downloaded: 0"
)

print(
    "Excel files downloaded: 0"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA RACING STATISTICS — PUBLIC SOURCE DISCOVERY
Loaded page from: network
HTTP status: 200
Final URL: https://www.britishhorseracing.com/regulation/reports-and-statistics/racing-statistics/
Page SHA-256: 428cb08742624a00cf0ab48b2173f65714fd0dbe916b3daf8f5512838dfacb25

FULL-YEAR RACING DATA PACKS
Recovered resources: 11
Visible year range: 2014 to 2026
Visible years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

  Resource 1
    Section: Racing Data Packs | Full Year
    Text: Full year racing data pack | 2025 573.19 KB
    URL: https://www.britishhorseracing.com/wp-content/uploads/2026/05/2025_Annual-Data-Pack-1.pdf
    Apparent format: pdf

  Resource 2
    Section: Racing Data Packs | Full Year
    Text: Full year racing data pack | 2024
    URL: https://media.britishhorseracing.com/bha/Racing_Statistics/Racing_Data_Packs_Full_Year/Full_year_2024.pdf
    Apparent format: pdf

  Resource 3
    Section: Racing Data Packs | Full Year
    Text: Full yea

In [48]:
# BHA Racing Statistics — 2025 full-year Racing Data Pack structure probe
#
# WHAT
# ----
# Download and inspect the latest completed annual BHA Racing Data Pack:
#
#   Full year racing data pack | 2025
#
# Public source discovered from the Racing Statistics page:
#
#   https://www.britishhorseracing.com/
#       wp-content/uploads/2026/05/2025_Annual-Data-Pack-1.pdf
#
# This cell will:
#
#   1. download/cache the PDF once;
#   2. preserve HTTP status, byte size and SHA-256;
#   3. extract machine-readable text using the system `pdftotext` utility;
#   4. retain page boundaries;
#   5. print a compact page-by-page heading/sample inventory;
#   6. search the extracted text for major statistical concepts;
#   7. cache both the extracted text and a compact derived inventory.
#
# WHY
# ---
# Surface discovery established several distinct BHA statistical product
# families.
#
# The latest completed full-year Racing Data Pack is the best first
# representative because it should show the broadest official annual measures
# before we decide whether monthly packs contribute genuinely different fields.
#
# We are NOT trying to reproduce every table yet.
#
# The immediate research question is:
#
#   What kinds of official measures and definitions does the annual data pack
#   contain, and which could later validate or supplement Inside Rails?
#
# READS
# -----
# - one public BHA PDF;
# - no BHA Authorization;
# - no Database v4.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       racing_statistics_annual_pack_probe/
#
# Files:
#
#   2025_Annual_Data_Pack.pdf
#   2025_Annual_Data_Pack.txt
#   2025_Annual_Data_Pack_inventory.json
#
# EXPECTED RESULT
# ---------------
# A bounded structural inventory showing:
#
#   - PDF size / SHA-256;
#   - page count if recoverable;
#   - first meaningful lines from each page;
#   - pages containing major concepts such as fixtures, races, entries,
#     declarations, non-runners, prize money, field sizes and competitiveness.
#
# ACQUISITION BOUNDARY
# --------------------
# - one annual PDF only;
# - no monthly pack yet;
# - no Horse Population Report yet;
# - no Race Off-Times PDF yet;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import shutil
import subprocess
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "racing_statistics_annual_pack_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PDF_FILE = (
    CACHE_DIR
    / "2025_Annual_Data_Pack.pdf"
)

TEXT_FILE = (
    CACHE_DIR
    / "2025_Annual_Data_Pack.txt"
)

INVENTORY_FILE = (
    CACHE_DIR
    / "2025_Annual_Data_Pack_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact annual pack discovered from the BHA page.
# ---------------------------------------------------------------------------

PDF_URL = (
    "https://www.britishhorseracing.com/"
    "wp-content/uploads/2026/05/"
    "2025_Annual-Data-Pack-1.pdf"
)


# ---------------------------------------------------------------------------
# 3. Confirm that the machine-readable PDF tools required by this probe exist.
# ---------------------------------------------------------------------------
#
# `pdftotext` is preferred here because previous BHA PDF work established that
# it is available in the project environment and extracts useful text without
# introducing another Python dependency.
#
# `pdfinfo` is optional; if unavailable, text extraction still proceeds.

PDFTOTEXT = shutil.which(
    "pdftotext"
)

PDFINFO = shutil.which(
    "pdfinfo"
)

assert PDFTOTEXT, (
    "System utility `pdftotext` is not available."
)


# ---------------------------------------------------------------------------
# 4. Download/cache the PDF.
# ---------------------------------------------------------------------------
#
# Preserve failed responses as metadata before stopping.
#
# Do not assume Content-Type alone proves the response is genuinely a PDF.

download_source = "cache"
download_evidence = {}

if not PDF_FILE.exists():
    download_source = "network"

    request = Request(
        PDF_URL,
        headers={
            "Accept": (
                "application/pdf,*/*;q=0.8"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/reports-and-statistics/"
                "racing-statistics/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_bytes = response.read()

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_bytes = error.read()

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )

    download_evidence = {
        "response_status": status,
        "content_type": content_type,
        "transport_error": transport_error,
        "bytes_received": len(
            response_bytes
        ),
        "sha256": (
            hashlib.sha256(
                response_bytes
            ).hexdigest()
            if response_bytes
            else None
        ),
    }

    assert status == 200, (
        "Annual Data Pack request did not return HTTP 200."
    )

    assert response_bytes, (
        "Annual Data Pack response was empty."
    )


    # -----------------------------------------------------------------------
    # A genuine PDF should begin with the `%PDF-` signature.
    #
    # This prevents an HTML error page being cached as `.pdf`.
    # -----------------------------------------------------------------------

    assert response_bytes.startswith(
        b"%PDF-"
    ), (
        "Response does not begin with a PDF signature."
    )

    temp_file = PDF_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_bytes(
        response_bytes
    )

    temp_file.replace(
        PDF_FILE
    )


# ---------------------------------------------------------------------------
# 5. Recalculate provenance from the cached local PDF.
# ---------------------------------------------------------------------------

pdf_bytes = PDF_FILE.read_bytes()

pdf_sha256 = hashlib.sha256(
    pdf_bytes
).hexdigest()

pdf_size = len(
    pdf_bytes
)

assert pdf_bytes.startswith(
    b"%PDF-"
), (
    "Cached file does not appear to be a PDF."
)


# ---------------------------------------------------------------------------
# 6. Use pdfinfo when available to recover document-level metadata.
# ---------------------------------------------------------------------------

pdfinfo_fields = {}

if PDFINFO:
    info_result = subprocess.run(
        [
            PDFINFO,
            str(
                PDF_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if info_result.returncode == 0:
        for line in info_result.stdout.splitlines():
            if ":" not in line:
                continue

            key, value = line.split(
                ":",
                1,
            )

            pdfinfo_fields[
                key.strip()
            ] = value.strip()


# ---------------------------------------------------------------------------
# 7. Extract the complete machine-readable text while preserving page breaks.
# ---------------------------------------------------------------------------
#
# `-layout` helps retain table/header relationships better than plain-flow
# extraction.
#
# pdftotext uses form-feed characters between pages, which we preserve in the
# cached text file and use below for page-level inventory.

if not TEXT_FILE.exists():
    extract_result = subprocess.run(
        [
            PDFTOTEXT,
            "-layout",
            str(
                PDF_FILE
            ),
            str(
                TEXT_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    assert extract_result.returncode == 0, (
        "pdftotext failed: "
        f"{extract_result.stderr.strip()}"
    )


document_text = TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)

assert document_text.strip(), (
    "pdftotext produced no readable text."
)


# ---------------------------------------------------------------------------
# 8. Split text by PDF page.
# ---------------------------------------------------------------------------
#
# Preserve the actual extraction boundary rather than trying to infer pages from
# headings or whitespace.

pages = document_text.split(
    "\f"
)

# pdftotext commonly leaves one empty item after the final form-feed.
if (
    pages
    and not pages[-1].strip()
):
    pages = pages[
        :-1
    ]


# ---------------------------------------------------------------------------
# 9. Helper: retain meaningful lines while avoiding an enormous notebook dump.
# ---------------------------------------------------------------------------

def meaningful_lines(
    page_text,
):
    output = []

    for raw_line in page_text.splitlines():
        compact = " ".join(
            raw_line.split()
        )

        if not compact:
            continue

        output.append(
            compact
        )

    return output


# ---------------------------------------------------------------------------
# 10. Build a compact page inventory.
# ---------------------------------------------------------------------------
#
# We show only the first 12 meaningful lines on each page.
#
# That is enough to reveal page titles, table headings and major measures while
# preserving the complete extraction in the cache for later targeted searches.

MAX_LINES_PER_PAGE = 12

page_inventory = []

for page_number, page_text in enumerate(
    pages,
    start=1,
):
    lines = meaningful_lines(
        page_text
    )

    page_inventory.append(
        {
            "page": page_number,
            "meaningful_line_count": len(
                lines
            ),
            "first_lines": lines[
                :MAX_LINES_PER_PAGE
            ],
        }
    )


# ---------------------------------------------------------------------------
# 11. Define major concepts relevant to Inside Rails.
# ---------------------------------------------------------------------------
#
# These are discovery markers, not assumptions about the exact report schema.
#
# Matching a concept means only that the term appears on a page and warrants
# inspection; it does not prove a specific table or formal definition exists.

CONCEPT_PATTERNS = {
    "fixtures": [
        r"\bfixtures?\b",
        r"\babandon(?:ed|ments?)\b",
    ],
    "races": [
        r"\braces?\b",
        r"\brace types?\b",
    ],
    "entries_declarations": [
        r"\bentries\b",
        r"\bdeclarations?\b",
        r"\beliminations?\b",
    ],
    "non_runners": [
        r"\bnon[- ]runners?\b",
        r"\bwithdrawals?\b",
    ],
    "prize_money": [
        r"\bprize money\b",
        r"\bprizemoney\b",
        r"\brace value\b",
        r"\brace values\b",
    ],
    "field_size": [
        r"\bfield size\b",
        r"\bfield sizes\b",
        r"\brunners per race\b",
    ],
    "competitiveness": [
        r"\bcompetitiveness\b",
        r"\bfavourites?\b",
        r"\bwinning favourites?\b",
    ],
    "horses_in_training": [
        r"\bhorses in training\b",
        r"\bhorse population\b",
    ],
    "punctuality": [
        r"\bpunctuality\b",
        r"\boff[- ]times?\b",
        r"\bdelay\b",
        r"\bmedian delay\b",
    ],
    "race_codes": [
        r"\bflat turf\b",
        r"\bawt\b",
        r"\bchase\b",
        r"\bhurdle\b",
        r"\bnhf\b",
        r"\bhunter\b",
    ],
}

concept_pages = {}

for concept, patterns in CONCEPT_PATTERNS.items():
    matched_pages = []

    for page_number, page_text in enumerate(
        pages,
        start=1,
    ):
        if any(
            re.search(
                pattern,
                page_text,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            matched_pages.append(
                page_number
            )

    concept_pages[
        concept
    ] = matched_pages


# ---------------------------------------------------------------------------
# 12. Report document-level evidence.
# ---------------------------------------------------------------------------

print(
    "BHA 2025 ANNUAL RACING DATA PACK — STRUCTURE PROBE"
)
print(
    "=================================================="
)

print(
    "Loaded PDF from:",
    download_source,
)

print(
    "PDF path:",
    PDF_FILE,
)

print(
    "PDF bytes:",
    pdf_size,
)

print(
    "PDF SHA-256:",
    pdf_sha256,
)

print(
    "Extracted pages:",
    len(
        pages
    ),
)

if pdfinfo_fields:
    print(
        "pdfinfo Pages:",
        pdfinfo_fields.get(
            "Pages",
            "[not reported]",
        ),
    )

    print(
        "PDF title:",
        pdfinfo_fields.get(
            "Title",
            "[not reported]",
        ),
    )

    print(
        "PDF creation date:",
        pdfinfo_fields.get(
            "CreationDate",
            "[not reported]",
        ),
    )


# ---------------------------------------------------------------------------
# 13. Print the compact page-by-page inventory.
# ---------------------------------------------------------------------------

print(
    "\nPAGE INVENTORY"
)
print(
    "=============="
)

for item in page_inventory:
    print(
        f"\nPage {item['page']}"
    )

    print(
        "-" * 50
    )

    print(
        "Meaningful extracted lines:",
        item[
            "meaningful_line_count"
        ],
    )

    for line in item[
        "first_lines"
    ]:
        print(
            " ",
            line,
        )


# ---------------------------------------------------------------------------
# 14. Print the major-concept page map.
# ---------------------------------------------------------------------------

print(
    "\nMAJOR-CONCEPT PAGE MAP"
)
print(
    "======================"
)

for concept, matched_pages in concept_pages.items():
    print(
        f"{concept}:",
        (
            matched_pages
            if matched_pages
            else "NONE"
        ),
    )


# ---------------------------------------------------------------------------
# 15. Persist the compact derived inventory.
# ---------------------------------------------------------------------------

inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Racing Statistics / Full-Year Racing Data Pack"
    ),
    "product": (
        "2025 Annual Data Pack"
    ),
    "source_url": PDF_URL,
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "pdf_file": str(
        PDF_FILE
    ),
    "text_file": str(
        TEXT_FILE
    ),
    "pdf_bytes": pdf_size,
    "pdf_sha256": pdf_sha256,
    "pdfinfo": pdfinfo_fields,
    "extracted_page_count": len(
        pages
    ),
    "page_inventory": (
        page_inventory
    ),
    "concept_pages": (
        concept_pages
    ),
    "authorization_sent": False,
    "database_v4_queried": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 16. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "PDF cache:",
    PDF_FILE,
)

print(
    "Extracted text:",
    TEXT_FILE,
)

print(
    "Derived inventory:",
    INVENTORY_FILE,
)

print(
    "Network PDF requests:",
    (
        1
        if download_source == "network"
        else 0
    ),
)

print(
    "PDF extraction tool:",
    PDFTOTEXT,
)

print(
    "Monthly Racing Data Packs downloaded: 0"
)

print(
    "Horse Population Reports downloaded: 0"
)

print(
    "Race Off-Times PDFs downloaded: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA 2025 ANNUAL RACING DATA PACK — STRUCTURE PROBE
Loaded PDF from: network
PDF path: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/racing_statistics_annual_pack_probe/2025_Annual_Data_Pack.pdf
PDF bytes: 586947
PDF SHA-256: 4fe61968b7b8ec042e7ad5c47247f05947bb6e80758b6b8184d13a0d3bdd82cd
Extracted pages: 7
pdfinfo Pages: 7
PDF title: 
PDF creation date: Fri May 15 11:33:18 2026 BST

PAGE INVENTORY

Page 1
--------------------------------------------------
Meaningful extracted lines: 2
  Racing and Industry Statistics
  2021-2025

Page 2
--------------------------------------------------
Meaningful extracted lines: 56
  Fixtures
  `Mixed' Fixtures are recategorised as `Flat'.
  Fixtures Programmed
  Code 2021 2022 2023 2024 2025
  Flat Turf 562 (37.8%) 567 (38.1%) 567 (38.1%) 554 (37.7%) 548 (37.5%)
  Flat AWT 335 (22.5%) 335 (22.5%) 334 (22.4%) 347 (23.6%) 349 (23.9%)
  Jump 589 (39.6%) 586 (39.4%) 587 (39.4%) 567 (38.6%) 563 (38.6%)
  Total 

In [49]:
# BHA Racing Statistics — May 2026 monthly Racing Data Pack probe
#
# WHAT
# ----
# Download and inspect the latest currently-listed BHA monthly Racing Data Pack:
#
#   BHA racing data pack May 2026
#
# Source discovered from the public Racing Statistics page:
#
#   https://media.britishhorseracing.com/
#       bha/Racing_Statistics/Racing_Data_Packs_By_Month_2026/May26.pdf
#
# This cell will:
#
#   1. download/cache the May 2026 PDF once;
#   2. preserve file provenance and SHA-256;
#   3. extract machine-readable text with `pdftotext -layout`;
#   4. print a compact page-by-page inventory;
#   5. search for concepts that may distinguish monthly packs from the
#      full-year annual pack;
#   6. compare those observed concept families with the already-cached
#      2025 Annual Data Pack text.
#
# WHY
# ---
# The annual pack has already demonstrated strong value for aggregate
# validation and official statistical definitions.
#
# It contains:
#
#   - programmed / run fixtures;
#   - race counts;
#   - entries / declarations / eliminations / non-runners;
#   - prize/value statistics;
#   - field sizes;
#   - runner populations;
#   - competitiveness KPIs;
#   - selected innovation/policy measures.
#
# It did NOT expose horse-population snapshots or punctuality/off-time material.
#
# The monthly pack may therefore contribute genuinely different administrative
# and operational measures rather than simply repeating annual tables at a
# shorter interval.
#
# READS
# -----
# - one public BHA monthly PDF;
# - already-cached 2025 annual-pack extracted text for comparison;
# - no BHA Authorization;
# - no Database v4.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       racing_statistics_monthly_pack_probe/
#
# Files:
#
#   May_2026_Racing_Data_Pack.pdf
#   May_2026_Racing_Data_Pack.txt
#   May_2026_Racing_Data_Pack_inventory.json
#
# EXPECTED RESULT
# ---------------
# Evidence showing:
#
#   - monthly pack page structure;
#   - major measures present;
#   - concepts present in May 2026 but absent from the annual-pack extraction;
#   - whether monthly packs warrant treatment as a distinct useful source.
#
# ACQUISITION BOUNDARY
# --------------------
# - one monthly PDF only;
# - no other months;
# - no Horse Population Report yet;
# - no Race Off-Times PDF yet;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import shutil
import subprocess
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "racing_statistics_monthly_pack_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PDF_FILE = (
    CACHE_DIR
    / "May_2026_Racing_Data_Pack.pdf"
)

TEXT_FILE = (
    CACHE_DIR
    / "May_2026_Racing_Data_Pack.txt"
)

INVENTORY_FILE = (
    CACHE_DIR
    / "May_2026_Racing_Data_Pack_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact monthly pack demonstrated by the public page.
# ---------------------------------------------------------------------------

PDF_URL = (
    "https://media.britishhorseracing.com/"
    "bha/Racing_Statistics/"
    "Racing_Data_Packs_By_Month_2026/"
    "May26.pdf"
)


# ---------------------------------------------------------------------------
# 3. Locate the already-cached annual-pack text for direct comparison.
# ---------------------------------------------------------------------------
#
# We compare source products using their extracted text only.
#
# No network request is needed for the annual pack.

ANNUAL_TEXT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "racing_statistics_annual_pack_probe"
    / "2025_Annual_Data_Pack.txt"
)

assert ANNUAL_TEXT_FILE.is_file(), (
    "Expected cached annual-pack text was not found: "
    f"{ANNUAL_TEXT_FILE}"
)


# ---------------------------------------------------------------------------
# 4. Confirm PDF extraction tools are available.
# ---------------------------------------------------------------------------

PDFTOTEXT = shutil.which(
    "pdftotext"
)

PDFINFO = shutil.which(
    "pdfinfo"
)

assert PDFTOTEXT, (
    "System utility `pdftotext` is not available."
)


# ---------------------------------------------------------------------------
# 5. Download/cache the May 2026 monthly pack.
# ---------------------------------------------------------------------------
#
# Failed requests remain inspectable rather than disappearing behind
# `urlopen()` exceptions.

download_source = "cache"

if not PDF_FILE.exists():
    download_source = "network"

    request = Request(
        PDF_URL,
        headers={
            "Accept": (
                "application/pdf,*/*;q=0.8"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/reports-and-statistics/"
                "racing-statistics/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_bytes = response.read()

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_bytes = error.read()

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Preserve request evidence before asserting success.
    # -----------------------------------------------------------------------

    request_evidence = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Racing Statistics / Monthly Racing Data Pack"
        ),
        "product": (
            "May 2026 Racing Data Pack"
        ),
        "request_url": PDF_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "transport_error": transport_error,
        "bytes_received": len(
            response_bytes
        ),
        "sha256": (
            hashlib.sha256(
                response_bytes
            ).hexdigest()
            if response_bytes
            else None
        ),
        "authorization_sent": False,
    }

    INVENTORY_FILE.write_text(
        json.dumps(
            request_evidence,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    assert status == 200, (
        "Monthly Data Pack request did not return HTTP 200."
    )

    assert response_bytes, (
        "Monthly Data Pack response was empty."
    )

    assert response_bytes.startswith(
        b"%PDF-"
    ), (
        "Response does not begin with a PDF signature."
    )

    temp_file = PDF_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_bytes(
        response_bytes
    )

    temp_file.replace(
        PDF_FILE
    )


# ---------------------------------------------------------------------------
# 6. Recalculate local PDF provenance.
# ---------------------------------------------------------------------------

pdf_bytes = PDF_FILE.read_bytes()

assert pdf_bytes.startswith(
    b"%PDF-"
), (
    "Cached monthly file does not appear to be a PDF."
)

pdf_size = len(
    pdf_bytes
)

pdf_sha256 = hashlib.sha256(
    pdf_bytes
).hexdigest()


# ---------------------------------------------------------------------------
# 7. Recover optional document metadata.
# ---------------------------------------------------------------------------

pdfinfo_fields = {}

if PDFINFO:
    info_result = subprocess.run(
        [
            PDFINFO,
            str(
                PDF_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if info_result.returncode == 0:
        for line in info_result.stdout.splitlines():
            if ":" not in line:
                continue

            key, value = line.split(
                ":",
                1,
            )

            pdfinfo_fields[
                key.strip()
            ] = value.strip()


# ---------------------------------------------------------------------------
# 8. Extract monthly-pack text while retaining PDF page breaks.
# ---------------------------------------------------------------------------

if not TEXT_FILE.exists():
    extract_result = subprocess.run(
        [
            PDFTOTEXT,
            "-layout",
            str(
                PDF_FILE
            ),
            str(
                TEXT_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    assert extract_result.returncode == 0, (
        "pdftotext failed: "
        f"{extract_result.stderr.strip()}"
    )


monthly_text = TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)

assert monthly_text.strip(), (
    "Monthly Data Pack produced no readable text."
)

annual_text = ANNUAL_TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)


# ---------------------------------------------------------------------------
# 9. Split monthly extraction into pages.
# ---------------------------------------------------------------------------

pages = monthly_text.split(
    "\f"
)

if (
    pages
    and not pages[-1].strip()
):
    pages = pages[
        :-1
    ]


# ---------------------------------------------------------------------------
# 10. Helper for bounded page summaries.
# ---------------------------------------------------------------------------

def meaningful_lines(
    page_text,
):
    lines = []

    for raw_line in page_text.splitlines():
        compact = " ".join(
            raw_line.split()
        )

        if compact:
            lines.append(
                compact
            )

    return lines


MAX_LINES_PER_PAGE = 14

page_inventory = []

for page_number, page_text in enumerate(
    pages,
    start=1,
):
    lines = meaningful_lines(
        page_text
    )

    page_inventory.append(
        {
            "page": page_number,
            "meaningful_line_count": len(
                lines
            ),
            "first_lines": lines[
                :MAX_LINES_PER_PAGE
            ],
        }
    )


# ---------------------------------------------------------------------------
# 11. Search for concept families that could distinguish monthly packs.
# ---------------------------------------------------------------------------
#
# The categories include both concepts already seen annually and concepts that
# may represent monthly-only administrative/operational measures.
#
# A match proves only text presence, not formal semantics.

CONCEPT_PATTERNS = {
    "fixtures": [
        r"\bfixtures?\b",
    ],
    "races": [
        r"\braces?\b",
    ],
    "entries_declarations": [
        r"\bentries\b",
        r"\bdeclarations?\b",
    ],
    "non_runners": [
        r"\bnon[- ]runners?\b",
    ],
    "prize_money": [
        r"\bprize money\b",
        r"\bprizemoney\b",
        r"\brace value\b",
    ],
    "field_size": [
        r"\bfield size\b",
        r"\bfield sizes\b",
    ],
    "competitiveness": [
        r"\bcompetitiveness\b",
        r"\bfavourites?\b",
    ],
    "horses_in_training": [
        r"\bhorses in training\b",
    ],
    "horse_population": [
        r"\bhorse population\b",
    ],
    "punctuality": [
        r"\bpunctuality\b",
    ],
    "median_delay": [
        r"\bmedian delay\b",
    ],
    "race_clashes": [
        r"\brace clashes?\b",
        r"\bclashes?\b",
    ],
    "off_times": [
        r"\boff[- ]times?\b",
        r"\boff times?\b",
    ],
    "funding": [
        r"\bfunding\b",
        r"\bracecourse contribution\b",
        r"\blevy\b",
    ],
    "abandonments": [
        r"\babandon(?:ed|ment|ments)\b",
    ],
    "races_7_plus": [
        r"\b7\+\s*races?\b",
        r"\b7 or more races?\b",
    ],
    "small_fields": [
        r"\b<\s*6\s*runners\b",
        r"\bless than 6 runners\b",
    ],
}


# ---------------------------------------------------------------------------
# 12. Build page maps for the monthly pack.
# ---------------------------------------------------------------------------

monthly_concept_pages = {}

for concept, patterns in CONCEPT_PATTERNS.items():
    matched_pages = []

    for page_number, page_text in enumerate(
        pages,
        start=1,
    ):
        if any(
            re.search(
                pattern,
                page_text,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            matched_pages.append(
                page_number
            )

    monthly_concept_pages[
        concept
    ] = matched_pages


# ---------------------------------------------------------------------------
# 13. Compare simple concept presence against the annual-pack extraction.
# ---------------------------------------------------------------------------
#
# This comparison answers:
#
#   Is a concept demonstrably present in the monthly pack but absent from the
#   annual pack we inspected?
#
# It does NOT prove the concept never appears in other annual years.

annual_concept_presence = {}

monthly_concept_presence = {}

for concept, patterns in CONCEPT_PATTERNS.items():
    annual_concept_presence[
        concept
    ] = any(
        re.search(
            pattern,
            annual_text,
            flags=re.IGNORECASE,
        )
        for pattern in patterns
    )

    monthly_concept_presence[
        concept
    ] = any(
        re.search(
            pattern,
            monthly_text,
            flags=re.IGNORECASE,
        )
        for pattern in patterns
    )


monthly_only_observed = [
    concept
    for concept in CONCEPT_PATTERNS
    if (
        monthly_concept_presence[
            concept
        ]
        and not annual_concept_presence[
            concept
        ]
    )
]

present_in_both = [
    concept
    for concept in CONCEPT_PATTERNS
    if (
        monthly_concept_presence[
            concept
        ]
        and annual_concept_presence[
            concept
        ]
    )
]

annual_only_observed = [
    concept
    for concept in CONCEPT_PATTERNS
    if (
        annual_concept_presence[
            concept
        ]
        and not monthly_concept_presence[
            concept
        ]
    )
]


# ---------------------------------------------------------------------------
# 14. Report document-level evidence.
# ---------------------------------------------------------------------------

print(
    "BHA MAY 2026 MONTHLY RACING DATA PACK — STRUCTURE PROBE"
)
print(
    "======================================================"
)

print(
    "Loaded PDF from:",
    download_source,
)

print(
    "PDF path:",
    PDF_FILE,
)

print(
    "PDF bytes:",
    pdf_size,
)

print(
    "PDF SHA-256:",
    pdf_sha256,
)

print(
    "Extracted pages:",
    len(
        pages
    ),
)

if pdfinfo_fields:
    print(
        "pdfinfo Pages:",
        pdfinfo_fields.get(
            "Pages",
            "[not reported]",
        ),
    )

    print(
        "PDF creation date:",
        pdfinfo_fields.get(
            "CreationDate",
            "[not reported]",
        ),
    )


# ---------------------------------------------------------------------------
# 15. Print compact page inventory.
# ---------------------------------------------------------------------------

print(
    "\nPAGE INVENTORY"
)
print(
    "=============="
)

for item in page_inventory:
    print(
        f"\nPage {item['page']}"
    )

    print(
        "-" * 50
    )

    print(
        "Meaningful extracted lines:",
        item[
            "meaningful_line_count"
        ],
    )

    for line in item[
        "first_lines"
    ]:
        print(
            " ",
            line,
        )


# ---------------------------------------------------------------------------
# 16. Report monthly concept pages.
# ---------------------------------------------------------------------------

print(
    "\nMONTHLY CONCEPT PAGE MAP"
)
print(
    "========================"
)

for concept, matched_pages in monthly_concept_pages.items():
    print(
        f"{concept}:",
        (
            matched_pages
            if matched_pages
            else "NONE"
        ),
    )


# ---------------------------------------------------------------------------
# 17. Report observed annual/monthly differences.
# ---------------------------------------------------------------------------

print(
    "\nANNUAL / MONTHLY CONCEPT COMPARISON"
)
print(
    "==================================="
)

print(
    "Observed in BOTH inspected products:"
)

for concept in present_in_both:
    print(
        " -",
        concept,
    )


print(
    "\nObserved in MONTHLY but not 2025 ANNUAL:"
)

if monthly_only_observed:
    for concept in monthly_only_observed:
        print(
            " -",
            concept,
        )
else:
    print(
        " NONE"
    )


print(
    "\nObserved in 2025 ANNUAL but not MONTHLY:"
)

if annual_only_observed:
    for concept in annual_only_observed:
        print(
            " -",
            concept,
        )
else:
    print(
        " NONE"
    )


# ---------------------------------------------------------------------------
# 18. Persist compact derived evidence.
# ---------------------------------------------------------------------------

inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Racing Statistics / Monthly Racing Data Pack"
    ),
    "product": (
        "May 2026 Racing Data Pack"
    ),
    "source_url": PDF_URL,
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "pdf_file": str(
        PDF_FILE
    ),
    "text_file": str(
        TEXT_FILE
    ),
    "pdf_bytes": pdf_size,
    "pdf_sha256": pdf_sha256,
    "pdfinfo": pdfinfo_fields,
    "extracted_page_count": len(
        pages
    ),
    "page_inventory": (
        page_inventory
    ),
    "monthly_concept_pages": (
        monthly_concept_pages
    ),
    "monthly_concept_presence": (
        monthly_concept_presence
    ),
    "annual_concept_presence": (
        annual_concept_presence
    ),
    "monthly_only_observed": (
        monthly_only_observed
    ),
    "present_in_both": (
        present_in_both
    ),
    "annual_only_observed": (
        annual_only_observed
    ),
    "authorization_sent": False,
    "database_v4_queried": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 19. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Monthly PDF cache:",
    PDF_FILE,
)

print(
    "Monthly extracted text:",
    TEXT_FILE,
)

print(
    "Derived inventory:",
    INVENTORY_FILE,
)

print(
    "Annual comparison text:",
    ANNUAL_TEXT_FILE,
)

print(
    "Network PDF requests:",
    (
        1
        if download_source == "network"
        else 0
    ),
)

print(
    "Horse Population Reports downloaded: 0"
)

print(
    "Race Off-Times PDFs downloaded: 0"
)

print(
    "Other monthly packs downloaded: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA MAY 2026 MONTHLY RACING DATA PACK — STRUCTURE PROBE
Loaded PDF from: network
PDF path: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/racing_statistics_monthly_pack_probe/May_2026_Racing_Data_Pack.pdf
PDF bytes: 513450
PDF SHA-256: f43c6710f161ff3e50a22c3eb6e9253c613d0467732b88f8f44ba6c537d6ceb6
Extracted pages: 4
pdfinfo Pages: 4
PDF creation date: Mon Jun  1 10:00:28 2026 BST

PAGE INVENTORY

Page 1
--------------------------------------------------
Meaningful extracted lines: 39
  YTD Fixture Volume YTD Race Volume
  Fixt Type Flat Jump Total Race Type Flat Jump
  Year Prog Aband Add Run Prog Aband Add Run Prog Aband Add Run
  2022 309 7 6 308 292 4 1 289 602 11 6 597
  2023 298 5 9 302 293 32 7 268 592 37 15 570 2,189 2,020 2,155 2,185 2,223 2,239
  1,876 1,734 1,791 1,742
  2024 297 7 9 299 298 47 8 259 596 54 16 558
  2025 299 2 2 299 290 30 7 267 590 32 8 566
  Competitiveness & Race Volume
  2026 304 21 20 303 287 38 6 255 592 59 25

In [50]:
# BHA Racing Statistics — 31 May 2026 Horse Population Report probe
#
# WHAT
# ----
# Download and inspect the latest currently-listed BHA Horse Population Report:
#
#   Horse Population Report | 31 May 2026
#
# Public source discovered from the Racing Statistics page:
#
#   https://media.britishhorseracing.com/
#       bha/Racing_Statistics/Horse_Population_Reports/
#       Horse_Population_Report_20260531.pdf
#
# This cell will:
#
#   1. download/cache the report once;
#   2. preserve file size and SHA-256;
#   3. extract machine-readable text with `pdftotext -layout`;
#   4. print a bounded page-by-page structural inventory;
#   5. locate text concerning definitions, snapshot dates and population rules;
#   6. search for horse/trainer/category dimensions exposed by the report;
#   7. compare the report with the already-cached May 2026 monthly pack;
#   8. test whether the monthly-pack 2026 headline horse-population figures
#      also appear in this dedicated report.
#
# WHY
# ---
# The May 2026 monthly Racing Data Pack contains a "Horses in Training"
# snapshot for 31 May with:
#
#   Flat = 9,467
#   Jump = 2,838
#   Dual = 460
#   Total = 12,763
#
# The BHA also publishes a separate monthly Horse Population Report.
#
# We need to establish whether that dedicated report:
#
#   - simply republishes those headline counts; or
#   - contributes additional administrative definitions, dimensions or
#     populations worth treating as a distinct source.
#
# READS
# -----
# - one public BHA Horse Population Report PDF;
# - already-cached May 2026 monthly Racing Data Pack text;
# - no BHA Authorization;
# - no Database v4.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       horse_population_report_probe/
#
# Files:
#
#   Horse_Population_Report_20260531.pdf
#   Horse_Population_Report_20260531.txt
#   Horse_Population_Report_20260531_inventory.json
#
# EXPECTED RESULT
# ---------------
# Evidence showing:
#
#   - report page structure;
#   - observed population definitions / notes;
#   - dimensions and categories in the dedicated report;
#   - whether the monthly-pack headline totals reconcile to it;
#   - whether Horse Population Reports deserve a distinct source role.
#
# ACQUISITION BOUNDARY
# --------------------
# - one Horse Population Report only;
# - no historical population reports;
# - no Race Off-Times report yet;
# - no Database v4 query;
# - no database writes.

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import shutil
import subprocess
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "horse_population_report_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PDF_FILE = (
    CACHE_DIR
    / "Horse_Population_Report_20260531.pdf"
)

TEXT_FILE = (
    CACHE_DIR
    / "Horse_Population_Report_20260531.txt"
)

INVENTORY_FILE = (
    CACHE_DIR
    / "Horse_Population_Report_20260531_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact dedicated Horse Population Report.
# ---------------------------------------------------------------------------

PDF_URL = (
    "https://media.britishhorseracing.com/"
    "bha/Racing_Statistics/"
    "Horse_Population_Reports/"
    "Horse_Population_Report_20260531.pdf"
)


# ---------------------------------------------------------------------------
# 3. Locate the May 2026 monthly Racing Data Pack text.
# ---------------------------------------------------------------------------
#
# This provides the comparison source containing the headline 31 May horse
# population snapshot already observed.

MONTHLY_PACK_TEXT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "racing_statistics_monthly_pack_probe"
    / "May_2026_Racing_Data_Pack.txt"
)

assert MONTHLY_PACK_TEXT_FILE.is_file(), (
    "Expected cached May 2026 monthly-pack text was not found: "
    f"{MONTHLY_PACK_TEXT_FILE}"
)


# ---------------------------------------------------------------------------
# 4. Confirm system PDF tools.
# ---------------------------------------------------------------------------

PDFTOTEXT = shutil.which(
    "pdftotext"
)

PDFINFO = shutil.which(
    "pdfinfo"
)

assert PDFTOTEXT, (
    "System utility `pdftotext` is not available."
)


# ---------------------------------------------------------------------------
# 5. Download/cache the dedicated Horse Population Report.
# ---------------------------------------------------------------------------
#
# Cache the original binary exactly as served.
#
# Preserve request evidence before raising on a failed response.

download_source = "cache"

if not PDF_FILE.exists():
    download_source = "network"

    request = Request(
        PDF_URL,
        headers={
            "Accept": (
                "application/pdf,*/*;q=0.8"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/reports-and-statistics/"
                "racing-statistics/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_bytes = response.read()

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_bytes = error.read()

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Persist failed/empty-response evidence before asserting success.
    # -----------------------------------------------------------------------

    request_evidence = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Racing Statistics / Horse Population Reports"
        ),
        "product": (
            "Horse Population Report | 31 May 2026"
        ),
        "request_url": PDF_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "transport_error": transport_error,
        "bytes_received": len(
            response_bytes
        ),
        "sha256": (
            hashlib.sha256(
                response_bytes
            ).hexdigest()
            if response_bytes
            else None
        ),
        "authorization_sent": False,
    }

    INVENTORY_FILE.write_text(
        json.dumps(
            request_evidence,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    assert status == 200, (
        "Horse Population Report request did not return HTTP 200."
    )

    assert response_bytes, (
        "Horse Population Report response was empty."
    )

    assert response_bytes.startswith(
        b"%PDF-"
    ), (
        "Response does not begin with a PDF signature."
    )

    temp_file = PDF_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_bytes(
        response_bytes
    )

    temp_file.replace(
        PDF_FILE
    )


# ---------------------------------------------------------------------------
# 6. Recalculate provenance from the cached local file.
# ---------------------------------------------------------------------------

pdf_bytes = PDF_FILE.read_bytes()

assert pdf_bytes.startswith(
    b"%PDF-"
), (
    "Cached Horse Population Report does not appear to be a PDF."
)

pdf_size = len(
    pdf_bytes
)

pdf_sha256 = hashlib.sha256(
    pdf_bytes
).hexdigest()


# ---------------------------------------------------------------------------
# 7. Recover optional PDF metadata.
# ---------------------------------------------------------------------------

pdfinfo_fields = {}

if PDFINFO:
    info_result = subprocess.run(
        [
            PDFINFO,
            str(
                PDF_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if info_result.returncode == 0:
        for line in info_result.stdout.splitlines():
            if ":" not in line:
                continue

            key, value = line.split(
                ":",
                1,
            )

            pdfinfo_fields[
                key.strip()
            ] = value.strip()


# ---------------------------------------------------------------------------
# 8. Extract the full report text while preserving page boundaries.
# ---------------------------------------------------------------------------

if not TEXT_FILE.exists():
    extract_result = subprocess.run(
        [
            PDFTOTEXT,
            "-layout",
            str(
                PDF_FILE
            ),
            str(
                TEXT_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    assert extract_result.returncode == 0, (
        "pdftotext failed: "
        f"{extract_result.stderr.strip()}"
    )


population_text = TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)

assert population_text.strip(), (
    "Horse Population Report produced no readable text."
)

monthly_pack_text = MONTHLY_PACK_TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)


# ---------------------------------------------------------------------------
# 9. Split the dedicated report by page.
# ---------------------------------------------------------------------------

pages = population_text.split(
    "\f"
)

if (
    pages
    and not pages[-1].strip()
):
    pages = pages[
        :-1
    ]


# ---------------------------------------------------------------------------
# 10. Produce bounded page summaries.
# ---------------------------------------------------------------------------

def meaningful_lines(page_text):
    lines = []

    for raw_line in page_text.splitlines():
        compact = " ".join(
            raw_line.split()
        )

        if compact:
            lines.append(
                compact
            )

    return lines


MAX_LINES_PER_PAGE = 18

page_inventory = []

for page_number, page_text in enumerate(
    pages,
    start=1,
):
    lines = meaningful_lines(
        page_text
    )

    page_inventory.append(
        {
            "page": page_number,
            "meaningful_line_count": len(
                lines
            ),
            "first_lines": lines[
                :MAX_LINES_PER_PAGE
            ],
        }
    )


# ---------------------------------------------------------------------------
# 11. Search for concepts relevant to the source's administrative meaning.
# ---------------------------------------------------------------------------
#
# These markers are deliberately broad.
#
# Their purpose is to identify pages requiring semantic attention, not to
# impose definitions that the document does not state.

CONCEPT_PATTERNS = {
    "snapshot": [
        r"\bsnapshot\b",
        r"\bas at\b",
    ],
    "definition_or_method": [
        r"\bdefinition\b",
        r"\bmethod(?:ology)?\b",
        r"\bcriteria\b",
        r"\bincludes?\b",
        r"\bexcludes?\b",
    ],
    "weatherbys": [
        r"\bweatherbys\b",
    ],
    "horses_in_training": [
        r"\bhorses in training\b",
    ],
    "flat": [
        r"\bflat\b",
    ],
    "jump": [
        r"\bjump\b",
    ],
    "dual": [
        r"\bdual\b",
    ],
    "trainers": [
        r"\btrainers?\b",
    ],
    "yards": [
        r"\byards?\b",
    ],
    "age": [
        r"\bage\b",
        r"\byear[- ]olds?\b",
        r"\b2yo\b",
        r"\b3yo\b",
    ],
    "sex": [
        r"\bsex\b",
        r"\bfill(?:y|ies)\b",
        r"\bmares?\b",
        r"\bgeldings?\b",
        r"\bcolts?\b",
    ],
    "gb": [
        r"\bGB\b",
        r"\bGreat Britain\b",
    ],
    "ireland": [
        r"\bIRE\b",
        r"\bIreland\b",
        r"\bIrish\b",
    ],
    "registered": [
        r"\bregistered\b",
        r"\bregistration\b",
    ],
    "licensed": [
        r"\blicen[cs]ed\b",
        r"\blicen[cs]e\b",
    ],
}

concept_pages = {}

for concept, patterns in CONCEPT_PATTERNS.items():
    matched_pages = []

    for page_number, page_text in enumerate(
        pages,
        start=1,
    ):
        if any(
            re.search(
                pattern,
                page_text,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            matched_pages.append(
                page_number
            )

    concept_pages[
        concept
    ] = matched_pages


# ---------------------------------------------------------------------------
# 12. Print bounded contexts around especially important semantic markers.
# ---------------------------------------------------------------------------
#
# The first occurrence of each marker is enough for source-family discovery.
#
# We do not dump every repeated instance.

CONTEXT_MARKERS = {
    "snapshot": r"\bsnapshot\b",
    "weatherbys": r"\bweatherbys\b",
    "horses_in_training": (
        r"\bhorses in training\b"
    ),
    "definition": r"\bdefinition\b",
    "includes": r"\bincludes?\b",
    "excludes": r"\bexcludes?\b",
}

semantic_contexts = {}

for marker_name, pattern in CONTEXT_MARKERS.items():
    match = re.search(
        pattern,
        population_text,
        flags=re.IGNORECASE,
    )

    if not match:
        semantic_contexts[
            marker_name
        ] = None

        continue

    start = max(
        0,
        match.start() - 500,
    )

    end = min(
        len(
            population_text
        ),
        match.end() + 1200,
    )

    context = population_text[
        start:end
    ]

    compact_context = "\n".join(
        line.rstrip()
        for line in context.splitlines()
    ).strip()

    semantic_contexts[
        marker_name
    ] = compact_context


# ---------------------------------------------------------------------------
# 13. Test the headline May-2026 counts observed in the monthly pack.
# ---------------------------------------------------------------------------
#
# Normalise commas/whitespace so:
#
#   9,467
#
# and:
#
#   9467
#
# can be compared safely as textual numeric evidence.
#
# This does not yet prove that identical numbers have identical population
# definitions; semantic equivalence still depends on the report wording.

def normalise_numeric_text(text):
    return re.sub(
        r"[,\s]+",
        "",
        text,
    )


normalised_population_text = (
    normalise_numeric_text(
        population_text
    )
)

normalised_monthly_text = (
    normalise_numeric_text(
        monthly_pack_text
    )
)

HEADLINE_COUNTS = {
    "Flat": "9467",
    "Jump": "2838",
    "Dual": "460",
    "Total": "12763",
}

headline_presence = {}

for category, expected_value in HEADLINE_COUNTS.items():
    headline_presence[
        category
    ] = {
        "expected_value": (
            expected_value
        ),
        "present_in_population_report": (
            expected_value
            in normalised_population_text
        ),
        "present_in_monthly_pack": (
            expected_value
            in normalised_monthly_text
        ),
    }


# ---------------------------------------------------------------------------
# 14. Report document-level evidence.
# ---------------------------------------------------------------------------

print(
    "BHA HORSE POPULATION REPORT — 31 MAY 2026"
)
print(
    "========================================="
)

print(
    "Loaded PDF from:",
    download_source,
)

print(
    "PDF path:",
    PDF_FILE,
)

print(
    "PDF bytes:",
    pdf_size,
)

print(
    "PDF SHA-256:",
    pdf_sha256,
)

print(
    "Extracted pages:",
    len(
        pages
    ),
)

if pdfinfo_fields:
    print(
        "pdfinfo Pages:",
        pdfinfo_fields.get(
            "Pages",
            "[not reported]",
        ),
    )

    print(
        "PDF creation date:",
        pdfinfo_fields.get(
            "CreationDate",
            "[not reported]",
        ),
    )


# ---------------------------------------------------------------------------
# 15. Print the bounded page inventory.
# ---------------------------------------------------------------------------

print(
    "\nPAGE INVENTORY"
)
print(
    "=============="
)

for item in page_inventory:
    print(
        f"\nPage {item['page']}"
    )

    print(
        "-" * 50
    )

    print(
        "Meaningful extracted lines:",
        item[
            "meaningful_line_count"
        ],
    )

    for line in item[
        "first_lines"
    ]:
        print(
            " ",
            line,
        )


# ---------------------------------------------------------------------------
# 16. Print the concept/page map.
# ---------------------------------------------------------------------------

print(
    "\nCONCEPT PAGE MAP"
)
print(
    "================"
)

for concept, matched_pages in concept_pages.items():
    print(
        f"{concept}:",
        (
            matched_pages
            if matched_pages
            else "NONE"
        ),
    )


# ---------------------------------------------------------------------------
# 17. Print important semantic contexts.
# ---------------------------------------------------------------------------

print(
    "\nSEMANTIC CONTEXTS"
)
print(
    "================="
)

for marker_name, context in semantic_contexts.items():
    print(
        f"\n{marker_name}"
    )

    print(
        "-" * len(
            marker_name
        )
    )

    if context:
        print(
            context[:5000]
        )
    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 18. Report headline-count reconciliation evidence.
# ---------------------------------------------------------------------------

print(
    "\nMONTHLY-PACK HEADLINE COUNT CHECK"
)
print(
    "================================="
)

for category, evidence in headline_presence.items():
    print(
        f"{category}: "
        f"value={evidence['expected_value']}, "
        f"population_report="
        f"{evidence['present_in_population_report']}, "
        f"monthly_pack="
        f"{evidence['present_in_monthly_pack']}"
    )


# ---------------------------------------------------------------------------
# 19. Persist compact derived evidence.
# ---------------------------------------------------------------------------

inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Racing Statistics / Horse Population Reports"
    ),
    "product": (
        "Horse Population Report | 31 May 2026"
    ),
    "source_url": PDF_URL,
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "pdf_file": str(
        PDF_FILE
    ),
    "text_file": str(
        TEXT_FILE
    ),
    "pdf_bytes": pdf_size,
    "pdf_sha256": pdf_sha256,
    "pdfinfo": (
        pdfinfo_fields
    ),
    "extracted_page_count": len(
        pages
    ),
    "page_inventory": (
        page_inventory
    ),
    "concept_pages": (
        concept_pages
    ),
    "semantic_contexts": (
        semantic_contexts
    ),
    "monthly_pack_headline_count_check": (
        headline_presence
    ),
    "authorization_sent": False,
    "database_v4_queried": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 20. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Horse Population PDF:",
    PDF_FILE,
)

print(
    "Extracted text:",
    TEXT_FILE,
)

print(
    "Derived inventory:",
    INVENTORY_FILE,
)

print(
    "Monthly comparison text:",
    MONTHLY_PACK_TEXT_FILE,
)

print(
    "Network PDF requests:",
    (
        1
        if download_source == "network"
        else 0
    ),
)

print(
    "Other Horse Population Reports downloaded: 0"
)

print(
    "Race Off-Times PDFs downloaded: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA HORSE POPULATION REPORT — 31 MAY 2026
Loaded PDF from: network
PDF path: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/horse_population_report_probe/Horse_Population_Report_20260531.pdf
PDF bytes: 4315337
PDF SHA-256: 553651011f69ad3a3ae1f10b77132b097ec11400381a0c4ff9b6b56a811e1096
Extracted pages: 27
pdfinfo Pages: 27
PDF creation date: Wed Jun  3 10:05:48 2026 BST

PAGE INVENTORY

Page 1
--------------------------------------------------
Meaningful extracted lines: 2
  Horse Population Report
  Updated on 31 May 2026

Page 2
--------------------------------------------------
Meaningful extracted lines: 46
  Introduction
  Welcome to the latest update on the Racehorse Population from the British Horseracing Authority.
  The aim of this document is to give stakeholders and the wider Public an insight into the current state of the Horse Population in
  Great Britain. It is hoped that this transparency will ensure that all decision makers ha

## BHA Horse Population Reports — source-family conclusion

The dedicated BHA Horse Population Report is a materially richer source than
the horse-population summary contained in the monthly Racing Data Pack.

The inspected source was:

**Horse Population Report — updated 31 May 2026**

It contained 27 pages of administrative definitions, population snapshots,
historical comparisons and runner-population analysis.

---

## Source basis

The report states that its Horses in Training data comes from:

> the Weatherbys Racing Administration System

and that the information is supplied by trainers and should be updated when a
horse enters or leaves their care as part of the conditions of their licence.

This is important because the source is therefore not simply reconstructed from
race appearances.

It represents an administrative **Horses in Training population**.

That gives it a different grain and meaning from race-result data.

---

## Snapshot semantics

The report states:

> Unless otherwise stated, snapshots of Horses in Training will be from the
> 15th day of the month in question.

Year-to-date figures instead run up to and including the date stated on the
front of the report.

Therefore:

**snapshot populations and YTD populations are different measures and must not
be combined.**

This also explains an important difference from the May 2026 monthly Racing
Data Pack.

The monthly pack explicitly described its horse-population comparison as:

**Taken as a Snapshot on 31st May of Each Year**

whereas the dedicated Horse Population Report uses the 15th of the month unless
otherwise stated.

The different figures seen in the two products must therefore NOT be treated as
a source discrepancy until their exact observation rules are aligned.

---

## Observed May 2026 Horses in Training snapshot

The dedicated report's May 2026 snapshot reported:

- Flat — 9,512;
- Jump — 3,435;
- Dual — 426;
- Hunter — 1,055;
- Total — 13,373.

The report's `Total` is:

`Flat + Jump + Dual`

and therefore excludes Hunter Chasers from that headline total.

This is consistent with the report's stated population rules.

---

## Important population rules

The report explicitly documents several inclusion/exclusion and recoding rules.

Observed rules include:

- horses temporarily racing abroad for GB trainers are excluded at the time of
  reporting;
- yearlings are excluded from total Flat and/or Dual counts unless shown
  separately;
- yearlings and two-year-olds are excluded from Jump totals;
- yearlings, two-year-olds and three-year-olds are excluded from Hunter Chaser
  totals;
- yearling and two-year-old horses recorded as Dual are recoded as Flat;
- Hunter Chasers are not included in Jump or overall headline population
  figures unless separately stated.

These rules mean that apparently simple questions such as:

> How many horses are in training?

cannot safely be answered by counting horse records without reproducing the
BHA's population definition.

---

## Rating semantics

For rating-band analysis, the report states that horse ratings come from the
most recently published BHA Handicapping Team ratings.

The report also gives an example showing that:

- the Horses in Training population comes from the relevant snapshot date;
- the rating assigned to those horses comes from the most recently published
  ratings available for that snapshot.

Therefore rating-band tables combine:

**administrative population state + separately published rating state.**

This temporal relationship must be preserved if these statistics are ever
reconstructed.

---

## Gender semantics

The report states that horse gender is taken from the reporting date in
question, regardless of whether the horse's recorded gender subsequently
changes.

For YTD reporting, where gender or training type changes during the YTD period,
the most recent state is used.

Therefore historical population tables are not necessarily based on a horse's
current attributes.

---

## Training location and regional definitions

The report provides unusually useful formal geography.

A horse's training location is defined from:

**the trainer's primary training location.**

Training regions use the same regional system as the BHA Fixture List.

The report explicitly defines:

- **North** — north of latitude `53.42911`;
- **South** — south of latitude `51.88002`;
- **Midlands** — between those two latitude boundaries.

This gives direct BHA semantic evidence for the `Region` field previously
observed in the annual Fixture List.

Therefore the Fixture List regional classification is no longer merely an
unexplained label.

This definition should be retained for later governance work.

---

# Population dimensions demonstrated

The dedicated report contains substantially more information than the monthly
Racing Data Pack summary.

Observed dimensions include:

### Training type

- Flat;
- Jump;
- Dual;
- Hunter.

### Age

Examples include:

Flat:

- 2YO;
- 3YO;
- 4YO;
- 5YO+.

Jump:

- 3YO;
- 4–5YO;
- 6–7YO;
- 8YO+.

Hunter Chasers use their own age groups.

### Rating bands

The report provides population breakdowns by official rating bands.

Examples observed include Flat bands such as:

- Unrated;
- 0–50;
- 51–60;
- 61–70;
- 71–80;
- 81–90;
- 91–100;
- 101+.

Jump uses its own rating-band structure.

### Gender

Observed categories include:

- Colt/Entire;
- Gelding;
- Filly/Mare;
- Rig.

### Breeding

The report breaks horse populations down by country of breeding, including
examples such as:

- GB;
- IRE;
- FR;
- USA;
- GNY.

These are further split by sex.

### Location

Horse populations are analysed by:

- North;
- Midlands;
- South;

and by home nation:

- England;
- Scotland;
- Wales.

### Time

The report contains:

- monthly Horses in Training series;
- historical comparisons;
- YTD populations;
- previous-year comparisons;
- pre-Covid comparisons.

---

# Young horses entering training

The report includes information about young horses entering training, including:

- Flat yearlings in training;
- foals entering training for the first time;
- cumulative entry-to-training patterns.

This information cannot be recovered reliably merely from race appearances,
because it concerns administrative entry into training before or independently
of racing.

This is therefore genuinely new information relative to ordinary race-result
data.

---

# Runner populations

The report also contains substantial race-participation analysis.

Observed measures include:

- YTD total runners;
- YTD individual runners;
- Flat and Jump runner populations;
- runner age categories;
- country trained;
- GB-trained runners racing abroad;
- rating-band distributions;
- gender distributions.

The report explicitly defines:

**Total Runners**

as a count of all runners in relevant races,

while:

**Individual Runners**

counts each individual horse once.

Withdrawn horses are excluded.

The report states that relevant runner metrics count runners that actually ran
in GB races, including voided races.

Hunter Chases are generally excluded from the main runner section but may be
shown separately.

These definitions are valuable later when comparing BHA aggregates with Inside
Rails.

---

# Frequency of runs

The report contains measures including:

- average runs per horse;
- proportion of horses with 1 run;
- 2 runs;
- 3 runs;
- 4 runs;
- 5+ runs.

It distinguishes Flat, Jump and Dual participation.

These aggregates should be reproducible from sufficiently complete
race/participant history, but the BHA values provide a strong official
validation target because the population definitions are explicitly stated.

---

# Relationship to the monthly Racing Data Pack

The monthly Racing Data Pack contains a compact horse-population summary.

The dedicated Horse Population Report provides the semantic and analytical
depth behind that subject.

Therefore the two products should not be treated as interchangeable.

A useful distinction is:

### Monthly Racing Data Pack

Broad operational racing dashboard containing:

- fixture/race volume;
- competitiveness;
- prize/value measures;
- horse-population headline measures;
- punctuality;
- clashes;
- other KPIs.

### Horse Population Report

Dedicated administrative horse-population source containing:

- population definitions;
- snapshot methodology;
- detailed demographics;
- ratings distributions;
- breeding;
- geography;
- age;
- training type;
- young horses entering training;
- runner populations;
- frequency of racing.

---

# Source assessment

**Potential value: very high for population research and official definitions.**

Particularly valuable information that cannot simply be inferred from Database
v4 race results includes:

- Horses in Training population;
- entry/exit-from-training administrative state;
- young horses entering training;
- horses that are in training but have not raced;
- official population inclusion/exclusion rules;
- historical population snapshots;
- training-location populations;
- formal BHA regional boundaries.

Other measures, such as runner counts and frequency of runs, are potentially
reconstructible from race-level data but provide valuable official validation
targets.

---

## Important analytical warning

Do NOT treat:

- Horses in Training;
- Total Runners;
- Individual Runners;
- horses appearing in Database v4

as equivalent populations.

They answer different questions.

A horse can be in training without racing during the study period.

Likewise, YTD runner populations are participation measures rather than
administrative training-population measures.

---

## Historical depth

The public Racing Statistics page exposed 66 Horse Population Reports covering
visible dates from 2020 through 2026.

The inspected 2026 report itself contains historical comparison series extending
back to at least 2017.

Therefore useful historical population data may extend substantially further
back than the individual downloadable-report archive.

Do not investigate the historical boundary further unless this source becomes
a selected dataset for Inside Rails.

---

## Decision

The Horse Population Reports source family is sufficiently mapped.

Classify it as:

**high-value official administrative horse-population and population-definition
data.**

Do not download additional monthly Horse Population Reports now.

Preserve the distinction between:

- monthly snapshot measures;
- YTD measures;
- runner populations;
- Horses in Training populations.

The next distinct Racing Statistics source to inspect is:

**Race Off-Times data.**

In [51]:
# BHA Racing Statistics — Race Off-Times data structure probe
#
# WHAT
# ----
# Download and inspect the latest rolling 12-month BHA Race Off-Times report:
#
#   Race Off-Times Data — 1 April 2025 to 31 March 2026
#
# Public source discovered from the BHA Racing Statistics page:
#
#   https://media.britishhorseracing.com/
#       bha/Racing_Statistics/Race_off_times/2026_Q1.pdf
#
# This cell will:
#
#   1. download/cache the PDF once;
#   2. preserve size and SHA-256 provenance;
#   3. extract machine-readable text using `pdftotext -layout`;
#   4. inventory the PDF page structure;
#   5. search for race-level timing fields and punctuality definitions;
#   6. search for delay-reason / clash / required-delay concepts;
#   7. print bounded contexts around the most important semantic markers;
#   8. determine whether the report appears:
#
#        - aggregate only;
#        - race-level;
#        - or mixed.
#
# WHY
# ---
# The May 2026 monthly Racing Data Pack already demonstrated aggregate metrics
# such as:
#
#   - percentage of GB races off within 120 seconds of scheduled time;
#   - race-clash counts;
#   - clash percentage.
#
# The dedicated Race Off-Times source may be substantially more useful if it
# exposes individual races, scheduled times, actual off-times or delay reasons.
#
# That distinction determines whether this source is:
#
#   - merely a validation/context product; or
#   - potentially new race-level data for Inside Rails.
#
# READS
# -----
# - one public BHA Race Off-Times PDF;
# - no BHA Authorization;
# - no Database v4.
#
# WRITES
# ------
# Ignored research evidence under:
#
#   data/cache/bha_official_source_feasibility/
#       race_off_times_probe/
#
# Files:
#
#   Race_Off_Times_2025-04-01_to_2026-03-31.pdf
#   Race_Off_Times_2025-04-01_to_2026-03-31.txt
#   Race_Off_Times_2025-04-01_to_2026-03-31_inventory.json
#
# EXPECTED RESULT
# ---------------
# Evidence showing:
#
#   - page count;
#   - table / section structure;
#   - whether individual races are present;
#   - whether scheduled and actual off-times are present;
#   - whether delays / required delays / clashes are described;
#   - whether this is useful as race-level enrichment.
#
# ACQUISITION BOUNDARY
# --------------------
# - one Off-Times PDF only;
# - no other rolling periods;
# - no historical expansion;
# - no Database v4 comparison;
# - no database writes.

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import shutil
import subprocess
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


# ---------------------------------------------------------------------------
# 1. Establish explicit repository and ignored-cache locations.
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/rob/Documents/inside-rails-horse-racing"
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha_official_source_feasibility"
    / "race_off_times_probe"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PDF_FILE = (
    CACHE_DIR
    / "Race_Off_Times_2025-04-01_to_2026-03-31.pdf"
)

TEXT_FILE = (
    CACHE_DIR
    / "Race_Off_Times_2025-04-01_to_2026-03-31.txt"
)

INVENTORY_FILE = (
    CACHE_DIR
    / "Race_Off_Times_2025-04-01_to_2026-03-31_inventory.json"
)


# ---------------------------------------------------------------------------
# 2. Define the exact demonstrated BHA resource.
# ---------------------------------------------------------------------------

PDF_URL = (
    "https://media.britishhorseracing.com/"
    "bha/Racing_Statistics/"
    "Race_off_times/"
    "2026_Q1.pdf"
)


# ---------------------------------------------------------------------------
# 3. Confirm system PDF tools.
# ---------------------------------------------------------------------------

PDFTOTEXT = shutil.which(
    "pdftotext"
)

PDFINFO = shutil.which(
    "pdfinfo"
)

assert PDFTOTEXT, (
    "System utility `pdftotext` is not available."
)


# ---------------------------------------------------------------------------
# 4. Download/cache the PDF.
# ---------------------------------------------------------------------------
#
# Preserve request evidence before asserting success.
#
# The raw PDF is retained exactly as served so later work can be reproduced.

download_source = "cache"

if not PDF_FILE.exists():
    download_source = "network"

    request = Request(
        PDF_URL,
        headers={
            "Accept": (
                "application/pdf,*/*;q=0.8"
            ),
            "Referer": (
                "https://www.britishhorseracing.com/"
                "regulation/reports-and-statistics/"
                "racing-statistics/"
            ),
            "User-Agent": "Mozilla/5.0",
        },
    )

    status = None
    content_type = None
    response_bytes = b""
    transport_error = None

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            status = response.status

            content_type = response.headers.get(
                "Content-Type"
            )

            response_bytes = response.read()

    except HTTPError as error:
        status = error.code

        content_type = error.headers.get(
            "Content-Type"
        )

        response_bytes = error.read()

        transport_error = (
            f"HTTPError status={error.code}"
        )

    except URLError as error:
        transport_error = (
            f"URLError reason={error.reason!r}"
        )


    # -----------------------------------------------------------------------
    # Persist request evidence even if the resource is unavailable.
    # -----------------------------------------------------------------------

    request_evidence = {
        "provider": (
            "British Horseracing Authority"
        ),
        "source_family": (
            "Racing Statistics / Race Off-Times"
        ),
        "product": (
            "Race Off-Times Data — "
            "1 April 2025 to 31 March 2026"
        ),
        "request_url": PDF_URL,
        "retrieved_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "response_status": status,
        "content_type": content_type,
        "transport_error": transport_error,
        "bytes_received": len(
            response_bytes
        ),
        "sha256": (
            hashlib.sha256(
                response_bytes
            ).hexdigest()
            if response_bytes
            else None
        ),
        "authorization_sent": False,
    }

    INVENTORY_FILE.write_text(
        json.dumps(
            request_evidence,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    assert status == 200, (
        "Race Off-Times request did not return HTTP 200."
    )

    assert response_bytes, (
        "Race Off-Times response was empty."
    )

    assert response_bytes.startswith(
        b"%PDF-"
    ), (
        "Response does not begin with a PDF signature."
    )

    temp_file = PDF_FILE.with_suffix(
        ".tmp"
    )

    temp_file.write_bytes(
        response_bytes
    )

    temp_file.replace(
        PDF_FILE
    )


# ---------------------------------------------------------------------------
# 5. Recalculate provenance from the cached local file.
# ---------------------------------------------------------------------------

pdf_bytes = PDF_FILE.read_bytes()

assert pdf_bytes.startswith(
    b"%PDF-"
), (
    "Cached Race Off-Times file does not appear to be a PDF."
)

pdf_size = len(
    pdf_bytes
)

pdf_sha256 = hashlib.sha256(
    pdf_bytes
).hexdigest()


# ---------------------------------------------------------------------------
# 6. Recover optional PDF metadata.
# ---------------------------------------------------------------------------

pdfinfo_fields = {}

if PDFINFO:
    info_result = subprocess.run(
        [
            PDFINFO,
            str(
                PDF_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if info_result.returncode == 0:
        for line in info_result.stdout.splitlines():
            if ":" not in line:
                continue

            key, value = line.split(
                ":",
                1,
            )

            pdfinfo_fields[
                key.strip()
            ] = value.strip()


# ---------------------------------------------------------------------------
# 7. Extract the complete text while preserving PDF layout/page boundaries.
# ---------------------------------------------------------------------------

if not TEXT_FILE.exists():
    extract_result = subprocess.run(
        [
            PDFTOTEXT,
            "-layout",
            str(
                PDF_FILE
            ),
            str(
                TEXT_FILE
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    assert extract_result.returncode == 0, (
        "pdftotext failed: "
        f"{extract_result.stderr.strip()}"
    )


document_text = TEXT_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)

assert document_text.strip(), (
    "Race Off-Times PDF produced no readable text."
)


# ---------------------------------------------------------------------------
# 8. Split extraction by PDF page.
# ---------------------------------------------------------------------------

pages = document_text.split(
    "\f"
)

if (
    pages
    and not pages[-1].strip()
):
    pages = pages[
        :-1
    ]


# ---------------------------------------------------------------------------
# 9. Helper for compact notebook display.
# ---------------------------------------------------------------------------

def meaningful_lines(page_text):
    lines = []

    for raw_line in page_text.splitlines():
        compact = " ".join(
            raw_line.split()
        )

        if compact:
            lines.append(
                compact
            )

    return lines


MAX_LINES_PER_PAGE = 18

page_inventory = []

for page_number, page_text in enumerate(
    pages,
    start=1,
):
    lines = meaningful_lines(
        page_text
    )

    page_inventory.append(
        {
            "page": page_number,
            "meaningful_line_count": len(
                lines
            ),
            "first_lines": lines[
                :MAX_LINES_PER_PAGE
            ],
        }
    )


# ---------------------------------------------------------------------------
# 10. Search for concepts indicating source grain and timing semantics.
# ---------------------------------------------------------------------------
#
# These patterns deliberately cover several possible phrasings.
#
# Presence of a word does NOT by itself establish semantics; it simply directs
# us to the relevant page/context.

CONCEPT_PATTERNS = {
    "racecourse": [
        r"\bracecourse\b",
        r"\bcourse\b",
    ],
    "race_date": [
        r"\brace date\b",
        r"\bdate\b",
    ],
    "scheduled_time": [
        r"\bscheduled time\b",
        r"\bscheduled off\b",
        r"\bscheduled\b",
    ],
    "actual_off_time": [
        r"\bactual off\b",
        r"\boff time\b",
        r"\boff-time\b",
        r"\boff times\b",
    ],
    "delay_seconds": [
        r"\bseconds?\b",
        r"\bsecs?\b",
        r"\bdelay\b",
    ],
    "required_delay": [
        r"\brequired delay\b",
        r"\breq(?:uired)?\.?\s*delay\b",
        r"\breq delays?\b",
    ],
    "clash": [
        r"\bclash(?:ed|es|ing)?\b",
    ],
    "ireland": [
        r"\bIRE\b",
        r"\bIreland\b",
        r"\bIrish\b",
    ],
    "punctuality": [
        r"\bpunctuality\b",
    ],
    "within_120_seconds": [
        r"\b120\s*(?:seconds?|secs?)\b",
        r"\bwithin\s+120\b",
    ],
    "median": [
        r"\bmedian\b",
    ],
    "average": [
        r"\baverage\b",
        r"\bmean\b",
    ],
    "reason": [
        r"\breason\b",
        r"\breasons\b",
    ],
    "race_id": [
        r"\brace\s*id\b",
        r"\braceid\b",
    ],
}

concept_pages = {}

for concept, patterns in CONCEPT_PATTERNS.items():
    matched_pages = []

    for page_number, page_text in enumerate(
        pages,
        start=1,
    ):
        if any(
            re.search(
                pattern,
                page_text,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            matched_pages.append(
                page_number
            )

    concept_pages[
        concept
    ] = matched_pages


# ---------------------------------------------------------------------------
# 11. Search for recurring line shapes that might indicate race-level rows.
# ---------------------------------------------------------------------------
#
# A race-level report would usually contain many repeated:
#
#   - dates;
#   - clock times;
#   - course names / race rows.
#
# We count date/time-like tokens only as structural evidence.
#
# We do NOT yet parse those tokens into governed race records.

DATE_PATTERN = re.compile(
    r"\b(?:"
    r"\d{1,2}[/-]\d{1,2}[/-](?:20)?\d{2}"
    r"|"
    r"\d{1,2}\s+"
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
    r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
    r"\s+20\d{2}"
    r")\b",
    flags=re.IGNORECASE,
)

CLOCK_PATTERN = re.compile(
    r"\b(?:[01]?\d|2[0-3]):[0-5]\d(?::[0-5]\d)?\b"
)

all_lines = meaningful_lines(
    document_text
)

date_token_count = sum(
    len(
        DATE_PATTERN.findall(
            line
        )
    )
    for line in all_lines
)

clock_token_count = sum(
    len(
        CLOCK_PATTERN.findall(
            line
        )
    )
    for line in all_lines
)


# ---------------------------------------------------------------------------
# 12. Capture bounded contexts around important semantic markers.
# ---------------------------------------------------------------------------

CONTEXT_MARKERS = {
    "scheduled_time": (
        r"\bscheduled time\b"
    ),
    "off_time": (
        r"\boff[- ]time\b"
    ),
    "required_delay": (
        r"\brequired delay\b"
    ),
    "punctuality": (
        r"\bpunctuality\b"
    ),
    "clash": (
        r"\bclash(?:ed|es|ing)?\b"
    ),
    "120_seconds": (
        r"\b120\s*(?:seconds?|secs?)\b"
    ),
    "reason": (
        r"\breasons?\b"
    ),
}

semantic_contexts = {}

for marker_name, pattern in CONTEXT_MARKERS.items():
    match = re.search(
        pattern,
        document_text,
        flags=re.IGNORECASE,
    )

    if not match:
        semantic_contexts[
            marker_name
        ] = None

        continue

    start = max(
        0,
        match.start() - 700,
    )

    end = min(
        len(
            document_text
        ),
        match.end() + 1600,
    )

    context = document_text[
        start:end
    ]

    semantic_contexts[
        marker_name
    ] = "\n".join(
        line.rstrip()
        for line in context.splitlines()
    ).strip()


# ---------------------------------------------------------------------------
# 13. Produce a conservative grain classification.
# ---------------------------------------------------------------------------
#
# This is deliberately based on structural evidence rather than intuition.
#
# A high number of clock/date tokens suggests repeated race-level rows.
# If those are absent and the report consists mainly of percentages/summary
# metrics, classify it as aggregate.
#
# "mixed" remains available when both detailed rows and summary material appear.

has_many_clock_tokens = (
    clock_token_count >= 50
)

has_many_date_tokens = (
    date_token_count >= 20
)

has_summary_semantics = any(
    concept_pages[
        concept
    ]
    for concept in (
        "punctuality",
        "within_120_seconds",
        "median",
        "average",
        "clash",
    )
)

if (
    has_many_clock_tokens
    and has_many_date_tokens
    and has_summary_semantics
):
    apparent_grain = (
        "mixed_race_level_and_aggregate"
    )

elif (
    has_many_clock_tokens
    and has_many_date_tokens
):
    apparent_grain = (
        "race_level_or_near_race_level"
    )

else:
    apparent_grain = (
        "aggregate_or_summary"
    )


# ---------------------------------------------------------------------------
# 14. Report document-level evidence.
# ---------------------------------------------------------------------------

print(
    "BHA RACE OFF-TIMES — STRUCTURE PROBE"
)
print(
    "===================================="
)

print(
    "Source period:",
    "2025-04-01 to 2026-03-31",
)

print(
    "Loaded PDF from:",
    download_source,
)

print(
    "PDF path:",
    PDF_FILE,
)

print(
    "PDF bytes:",
    pdf_size,
)

print(
    "PDF SHA-256:",
    pdf_sha256,
)

print(
    "Extracted pages:",
    len(
        pages
    ),
)

if pdfinfo_fields:
    print(
        "pdfinfo Pages:",
        pdfinfo_fields.get(
            "Pages",
            "[not reported]",
        ),
    )

    print(
        "PDF creation date:",
        pdfinfo_fields.get(
            "CreationDate",
            "[not reported]",
        ),
    )


# ---------------------------------------------------------------------------
# 15. Print bounded page inventory.
# ---------------------------------------------------------------------------

print(
    "\nPAGE INVENTORY"
)
print(
    "=============="
)

for item in page_inventory:
    print(
        f"\nPage {item['page']}"
    )

    print(
        "-" * 50
    )

    print(
        "Meaningful extracted lines:",
        item[
            "meaningful_line_count"
        ],
    )

    for line in item[
        "first_lines"
    ]:
        print(
            " ",
            line,
        )


# ---------------------------------------------------------------------------
# 16. Print concept/page map.
# ---------------------------------------------------------------------------

print(
    "\nCONCEPT PAGE MAP"
)
print(
    "================"
)

for concept, matched_pages in concept_pages.items():
    print(
        f"{concept}:",
        (
            matched_pages
            if matched_pages
            else "NONE"
        ),
    )


# ---------------------------------------------------------------------------
# 17. Print structural evidence for likely grain.
# ---------------------------------------------------------------------------

print(
    "\nGRAIN EVIDENCE"
)
print(
    "=============="
)

print(
    "Date-like tokens:",
    date_token_count,
)

print(
    "Clock-time tokens:",
    clock_token_count,
)

print(
    "Apparent source grain:",
    apparent_grain,
)


# ---------------------------------------------------------------------------
# 18. Print bounded semantic contexts.
# ---------------------------------------------------------------------------

print(
    "\nSEMANTIC CONTEXTS"
)
print(
    "================="
)

for marker_name, context in semantic_contexts.items():
    print(
        f"\n{marker_name}"
    )

    print(
        "-" * len(
            marker_name
        )
    )

    if context:
        print(
            context[:5000]
        )

    else:
        print(
            "NONE RECOVERED"
        )


# ---------------------------------------------------------------------------
# 19. Persist compact derived evidence.
# ---------------------------------------------------------------------------

inventory = {
    "provider": (
        "British Horseracing Authority"
    ),
    "source_family": (
        "Racing Statistics / Race Off-Times"
    ),
    "product": (
        "Race Off-Times Data — "
        "1 April 2025 to 31 March 2026"
    ),
    "source_url": PDF_URL,
    "analysed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "pdf_file": str(
        PDF_FILE
    ),
    "text_file": str(
        TEXT_FILE
    ),
    "pdf_bytes": pdf_size,
    "pdf_sha256": pdf_sha256,
    "pdfinfo": (
        pdfinfo_fields
    ),
    "extracted_page_count": len(
        pages
    ),
    "page_inventory": (
        page_inventory
    ),
    "concept_pages": (
        concept_pages
    ),
    "date_like_token_count": (
        date_token_count
    ),
    "clock_time_token_count": (
        clock_token_count
    ),
    "apparent_source_grain": (
        apparent_grain
    ),
    "semantic_contexts": (
        semantic_contexts
    ),
    "authorization_sent": False,
    "database_v4_queried": False,
}

temp_file = INVENTORY_FILE.with_suffix(
    ".tmp"
)

temp_file.write_text(
    json.dumps(
        inventory,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temp_file.replace(
    INVENTORY_FILE
)


# ---------------------------------------------------------------------------
# 20. State provenance and acquisition boundary explicitly.
# ---------------------------------------------------------------------------

print(
    "\nPROVENANCE"
)
print(
    "=========="
)

print(
    "Race Off-Times PDF:",
    PDF_FILE,
)

print(
    "Extracted text:",
    TEXT_FILE,
)

print(
    "Derived inventory:",
    INVENTORY_FILE,
)

print(
    "Network PDF requests:",
    (
        1
        if download_source == "network"
        else 0
    ),
)

print(
    "Other Off-Times PDFs downloaded: 0"
)

print(
    "BHA Authorization read: NO"
)

print(
    "BHA Authorization sent: NO"
)

print(
    "Database v4 queried: NO"
)

print(
    "Database writes: NONE"
)

BHA RACE OFF-TIMES — STRUCTURE PROBE
Source period: 2025-04-01 to 2026-03-31
Loaded PDF from: network
PDF path: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha_official_source_feasibility/race_off_times_probe/Race_Off_Times_2025-04-01_to_2026-03-31.pdf
PDF bytes: 282043
PDF SHA-256: 262665b19297438d1f5dc399041210434bfeb963723a71bbb81d6179682b6103
Extracted pages: 1
pdfinfo Pages: 1
PDF creation date: Sun Apr  5 10:00:31 2026 BST

PAGE INVENTORY

Page 1
--------------------------------------------------
Meaningful extracted lines: 88
  Race Times Update
  Report Date Range: 'Punctuality' and 'Late Races' refers to races starting within 2 minutes (120 seconds) of the
  Tuesday, April 01, 2025 Tuesday, March 31, 2026 Scheduled Time, which takes account of any BHA Racing Department Requested Delays.
  Start Date End Date
  0:45 82.4% Punctuality
  1779
  Median Race Delay Late Races
  Racecourse League Table Count of Late Flat Races by Primary Late Reason
  Shown with Most Com

## BHA Race Off-Times Data — source-family conclusion

The BHA publishes a dedicated public **Race Off-Times** reporting product within
its Racing Statistics area.

The inspected report covered:

**1 April 2025 to 31 March 2026**

and was published as a one-page PDF.

This source is not a race-by-race off-time dataset.

It is an **aggregate operational punctuality report** containing:

- overall punctuality measures;
- median delay;
- number of late races;
- racecourse-level punctuality comparisons;
- aggregate reasons for late races.

---

## BHA definition of punctuality

The report explicitly defines `Punctuality` and `Late Races` using whether a
race starts:

**within two minutes — 120 seconds — of the Scheduled Time.**

Crucially, the report states that this Scheduled Time:

**takes account of any BHA Racing Department Requested Delays.**

Therefore the BHA punctuality calculation is NOT necessarily:

`actual off-time - originally advertised race time`

A race whose original advertised time was changed through an official requested
delay may still count as punctual against the adjusted BHA Scheduled Time.

This distinction is essential for any future attempt to reproduce BHA
punctuality statistics from race-level data.

---

## Overall reported measures

For the period:

**1 April 2025 to 31 March 2026**

the report displayed:

- **Punctuality — 82.4%**
- **Median Race Delay — 0:45**
- **Late Races — 1,779**

The median delay is therefore 45 seconds for the population represented by the
report.

These are aggregate report-level statistics rather than individual-race
observations.

---

# Racecourse league table

The report contains a racecourse-level league table.

Observed fields include:

- `Course`;
- `Races`;
- `Late Races`;
- `Punctuality`;
- `Median Race Delay`;
- `Most Common Reason`.

Examples from the inspected report include:

### Fakenham

- races — 65;
- late races — 2;
- punctuality — 96.9%;
- median race delay — 0:20;
- most common reason — `Other*`.

### Warwick

- races — 120;
- late races — 5;
- punctuality — 95.8%;
- median race delay — 0:25;
- most common reason — `Other*`.

### Taunton

- races — 81;
- late races — 4;
- punctuality — 95.1%;
- median race delay — 0:30;
- most common reason — `Unruly/Loose horse`.

### Plumpton

- races — 109;
- late races — 6;
- punctuality — 94.5%;
- median race delay — 0:36;
- most common reason — `Late to post`.

This establishes that the source provides useful **course-level operational
performance** rather than only one national headline statistic.

---

# Late-race reasons

The report also contains aggregate counts of late Flat races by primary late
reason.

Observed reason labels include examples such as:

- `Late to post`;
- `Loading/Starting Issues`;
- `Unruly/Loose horse`;
- `Avoiding Clash`;
- `Track Issue`;
- `Ambulance Issue`;
- `Equipment/Shoeing`;
- `Television/Broadcast...`;
- `Knock on from pre...`;
- `Other`.

The report notes that `Other` may include matters such as:

- false starts;
- operational issues;
- other matters identified through Stewards' observation.

The exact complete taxonomy was not formally extracted from this single report,
so these labels should currently be treated as observed examples rather than a
governed closed enumeration.

---

## Relationship to Stewards information

The report describes the late-reason analysis as based on:

**Stewards' Observation.**

This is important because late-reason statistics may therefore derive from the
same broader officiating/stewards information environment investigated earlier.

However, the inspected Off-Times PDF does not expose the underlying
race-by-race Stewards observations.

It only exposes aggregate counts and racecourse-level dominant reasons.

---

# What the source does NOT provide

The inspected Race Off-Times report did not expose demonstrated fields for:

- BHA race ID;
- race date per individual race;
- racecourse + race number per record;
- original scheduled off-time per race;
- adjusted scheduled off-time per race;
- actual off-time per race;
- delay in seconds per race;
- requested-delay duration per race;
- primary late reason per individual race.

No individual race rows were demonstrated.

Therefore the source cannot currently be treated as a structured race-level
off-time feed.

---

# Source grain

The demonstrated grains are:

### Report level

One reporting period containing:

- overall punctuality;
- overall median delay;
- total late races.

### Racecourse level

One row per racecourse containing:

- races;
- late races;
- punctuality percentage;
- median delay;
- most common late reason.

### Late-reason aggregate

Counts of races assigned to broad late-reason categories.

There is no demonstrated individual-race grain in the public PDF.

---

# Relationship to the monthly Racing Data Pack

The May 2026 monthly Racing Data Pack had already demonstrated YTD metrics for:

- punctuality;
- race clashes;
- race-time operational performance.

The dedicated Race Off-Times report provides more detail about the same subject
area by adding:

- an explicit 120-second punctuality definition;
- the treatment of BHA-requested delays;
- racecourse league tables;
- aggregate late-reason information.

Therefore the dedicated report is useful even though it is not race-level.

---

# Potential Inside Rails value

The source has **moderate value**, primarily for validation, definitions and
operational research.

Useful applications include:

### Official punctuality benchmark

Compare Inside Rails race-level off-time calculations with the BHA's published
aggregate punctuality.

### Racecourse comparison

Analyse which racecourses are more or less punctual.

### Delay context

Study broad causes of racing delays.

### Definition validation

Ensure any Inside Rails punctuality measure distinguishes between:

- original advertised time;
- officially adjusted scheduled time;
- actual off-time.

### Publication/research questions

Potential future analyses include:

- Which British racecourses are most punctual?
- How much does race punctuality vary by course?
- What are the most common causes of late races?
- Are delays becoming more or less common over time?
- How different is naive advertised-time punctuality from the BHA's
  requested-delay-adjusted definition?

---

# Limitation for database enrichment

This source does NOT currently justify adding race-level off-time fields to the
database by itself.

Its useful information is mostly:

- aggregate;
- course-level;
- definitional.

If individual BHA race resources already expose actual off-times, those remain
the better source for race-level enrichment.

The Race Off-Times report can then be used as an official validation target for
those detailed records.

---

# Historical availability

Public-source discovery identified genuine Race Off-Times PDFs covering periods
across 2024–2026, including:

- rolling twelve-month periods;
- full-year reports.

The observed public page included seven genuine PDF resources.

No historical expansion is necessary yet.

If race punctuality becomes a selected Inside Rails research subject, the
available reports can later be acquired systematically and compared over time.

---

# Decision

The BHA Race Off-Times source family is sufficiently mapped.

Classify it as:

**official aggregate race-punctuality, racecourse-performance and late-reason
data.**

Do not treat it as a race-level off-time dataset.

Preserve the key BHA semantic rule:

> punctuality means starting within 120 seconds of the Scheduled Time after
> accounting for BHA Racing Department Requested Delays.

No further Race Off-Times PDFs need to be downloaded during the current
site-wide inventory.

## BHA Racing Statistics — source-family conclusion

The BHA Racing Statistics area contains several distinct public statistical
products rather than one homogeneous dataset.

The investigated families were:

- full-year Racing Data Packs;
- monthly Racing Data Packs;
- Horse Population Reports;
- Race Off-Times reports.

They serve different analytical purposes and should not be collapsed into one
generic statistics source.

---

# 1. Full-year Racing Data Packs

The inspected completed annual source was:

**2025 Annual Racing Data Pack**

It contained five-year comparisons covering 2021–2025.

Observed subjects included:

- fixtures programmed;
- fixtures run;
- abandonments/additions;
- races run;
- entries;
- declarations;
- eliminations;
- non-runners;
- race/prize values;
- average field size;
- total runners;
- individual runners;
- average runs per horse;
- race-card size KPIs;
- small-field measures;
- competitiveness measures;
- selected policy/innovation measures.

The annual pack therefore has substantial value as an official aggregate
validation and industry-context source.

It is NOT a replacement for race-level results.

Most of its measures should eventually be compared against Inside Rails
aggregations rather than imported as race records.

---

# 2. Monthly Racing Data Packs

The inspected monthly source was:

**May 2026 Racing Data Pack**

It contained YTD/current operational measures including:

- fixture volume;
- programmed fixtures;
- abandoned fixtures;
- added fixtures;
- fixtures run;
- race volume;
- average field size;
- races with 8+ runners;
- favourite/competitiveness measures;
- total prize money;
- handicap race values;
- Horses in Training summaries;
- punctuality;
- race clashes.

The monthly pack therefore contains genuinely useful material not present in
the inspected annual pack.

Particularly notable are operational measures such as:

- punctuality;
- clashes;
- current/YTD horse-population summaries.

Some measures overlap with the annual pack at a different reporting frequency.

Other measures are operational dashboard metrics that appear specifically in
the monthly reporting product.

---

# 3. Horse Population Reports

The dedicated Horse Population Reports are a distinct high-value source.

The inspected report was:

**Horse Population Report — updated 31 May 2026**

It states that Horses in Training information comes from the Weatherbys Racing
Administration System and is updated by trainers as horses enter or leave their
care.

This gives the source an administrative population basis rather than a
race-appearance basis.

Important demonstrated semantics include:

- monthly Horses in Training snapshots normally use the 15th of the month
  unless otherwise stated;
- YTD measures run through the report date;
- horse ratings are taken from the most recently published BHA ratings
  available for the relevant snapshot;
- gender is interpreted according to reporting-date/YTD rules;
- horses temporarily racing abroad for GB trainers are excluded from the
  relevant snapshot;
- age/training-type rules determine inclusion in Flat, Jump, Dual and Hunter
  populations;
- Hunter Chasers are excluded from headline Jump/overall populations unless
  shown separately.

The report provides extensive analysis by:

- training type;
- age;
- rating band;
- gender;
- breeding country;
- training region;
- home nation;
- runner population;
- country trained;
- frequency of runs.

It also contains information about young horses entering training.

That administrative information cannot reliably be reconstructed simply by
looking at horses that appeared in races.

---

# 4. BHA regional definition

The Horse Population Report provides direct semantics for the same regional
classification observed in the annual Fixture List.

Training regions are defined as:

- North — north of latitude `53.42911`;
- South — south of latitude `51.88002`;
- Midlands — between those boundaries.

This materially improves our understanding of the BHA `Region` field.

The classification is an explicit BHA administrative geography rather than an
ordinary informal regional label.

---

# 5. Race Off-Times reports

The inspected source covered:

**1 April 2025 to 31 March 2026**

The report is a one-page operational summary rather than a race-level dataset.

It defines:

**Punctuality / Late Races**

as whether races start within two minutes — 120 seconds — of the scheduled
time.

Crucially, the scheduled time used for this calculation takes account of:

**BHA Racing Department Requested Delays.**

Therefore a naive calculation of:

`actual off-time - originally advertised time`

would not necessarily reproduce the BHA punctuality metric.

---

## Off-Times measures observed

The report exposed headline measures including:

- punctuality percentage;
- median race delay;
- number of late races.

For the inspected reporting period the displayed headline values included:

- punctuality — 82.4%;
- median race delay — 0:45;
- late races — 1,779.

It also includes a racecourse league table with fields such as:

- course;
- races;
- late races;
- punctuality;
- median race delay;
- most common late reason.

---

## Late-reason information

The report contains aggregate counts and course-level dominant reasons for late
races.

Observed reason labels include examples such as:

- Late to post;
- Loading/Starting Issues;
- Unruly/Loose horse;
- Avoiding Clash;
- Track Issue;
- Ambulance Issue;
- Equipment/Shoeing;
- Television/Broadcast-related reasons;
- Other.

The report notes that `Other` may include operational matters such as false
starts and other issues identified through Stewards' observation.

These categories could provide useful operational context but are not exposed
here at individual-race grain.

---

## Race-level limitation

The inspected Off-Times report did NOT expose demonstrated fields for:

- race ID;
- individual race date;
- individual scheduled time;
- individual actual off-time;
- individual delay duration;
- individual requested-delay value;
- individual late-reason record.

Therefore this PDF should NOT currently be treated as a source for adding
race-level off-time observations to Inside Rails.

The live BHA race resources and existing Inside Rails off-time work remain the
relevant sources for individual races.

The Off-Times PDFs are instead useful for:

- official punctuality definitions;
- aggregate validation;
- racecourse punctuality comparisons;
- operational late-reason context.

---

# Historical availability

Public source discovery found:

### Full-year Racing Data Packs

11 visible annual products covering labelled years:

**2015–2025**

The apparent `2014` recovered during automated discovery came from old
WordPress URL paths and is NOT evidence of a 2014 annual pack.

### Monthly Racing Data Packs

119 public resources were recovered.

### Horse Population Reports

66 public resources were recovered, with visible report dates spanning
2020–2026.

The reports themselves contain historical comparison series extending earlier
than the downloadable archive.

### Race Off-Times

Seven genuine PDF resources were identified on the page, covering reporting
periods across 2024–2026.

An eighth automatically recovered item was merely the Racing Statistics page
linking to itself and is not an Off-Times dataset.

Historical-boundary work is unnecessary until one of these datasets is
selected for systematic acquisition.

---

# Source-role classification

The Racing Statistics products should be treated separately.

**Annual Racing Data Packs**

Official aggregate racing-volume, participation, value and competitiveness
validation/context.

**Monthly Racing Data Packs**

Current/YTD operational dashboard and KPI source.

**Horse Population Reports**

High-value administrative horse-population and population-definition source.

**Race Off-Times reports**

Official aggregate punctuality, delay-definition and late-reason context.

---

# Important methodological lesson

Several apparently simple BHA statistics depend on rules not visible in
ordinary result data.

Examples now demonstrated include:

- programmed fixtures versus fixtures actually run;
- administrative Horses in Training versus horses appearing in races;
- snapshot population versus YTD population;
- Total Runners versus Individual Runners;
- Hunter exclusions;
- age/training-type recoding;
- rating publication timing;
- requested-delay-adjusted punctuality.

Therefore:

> matching a BHA headline number is not merely a counting exercise.

The underlying BHA population and timing definitions must be understood before
a Database v4 comparison is interpreted as agreement or disagreement.

---

# Decision

The BHA Racing Statistics source family is sufficiently mapped for the
site-wide inventory.

Do not download further annual/monthly/population/off-time reports now.

Do not yet reconstruct these aggregates from Database v4.

Potential value is:

**high**, particularly for official definitions, validation, administrative
horse populations and operational context.

Move to the next BHA public-source family:

**Claiming-race records.**